<a href="https://colab.research.google.com/github/yy3462-create/textual_analysis_project/blob/main/final_project(Youtube%2C_Factiva%2C_%22AI%22).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YouTube + VADER Sentiment — Student Guide & Live Coding Notebook

> Run this end-to-end in Google Colab. Cells are heavily commented so you can follow what each line does.

In [1]:
# --- Install dependencies ---
!pip -q install --upgrade plotly kaleido nltk requests tqdm

# --- Imports ---
import pandas as pd
import nltk
import requests
from tqdm import tqdm
from urllib.parse import urlencode
from nltk.sentiment import SentimentIntensityAnalyzer
import plotly.express as px
import plotly.io as pio
import os
from getpass import getpass

# --- Setup ---
pio.renderers.default = "colab"
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

# --- Set your YouTube API Key ---
os.environ["YOUTUBE_API_KEY"] = getpass("Paste your API Key: ")
API_KEY = os.environ.get("YOUTUBE_API_KEY")
BASE_URL = "https://www.googleapis.com/youtube/v3"
assert API_KEY, "API key not set."

# --- Helper function to call YouTube API ---
def yt_get(resource: str, params: dict) -> dict:
    q = {**params, "key": API_KEY}
    url = f"{BASE_URL}/{resource}?{urlencode(q)}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()


# --- 1 Search videos ---
QUERY = "AI"
TARGET_VIDEOS = 60
MAX_RESULTS = 50

video_hits = []
page_token = None

with tqdm(total=TARGET_VIDEOS, desc="Searching videos") as pbar:
    while len(video_hits) < TARGET_VIDEOS:
        params = {
            "part": "snippet",
            "q": QUERY,
            "type": "video",
            "maxResults": MAX_RESULTS,
            "order": "relevance",
        }
        if page_token:
            params["pageToken"] = page_token

        data = yt_get("search", params)
        items = data.get("items", [])
        for it in items:
            vid = it.get("id", {}).get("videoId")
            if not vid:
                continue
            snip = it.get("snippet", {})
            video_hits.append({
                "video_id": vid,
                "publishedAt": snip.get("publishedAt"),
                "title": snip.get("title"),
                "channelId": snip.get("channelId"),
                "channelTitle": snip.get("channelTitle"),
            })
            pbar.update(1)
            if len(video_hits) >= TARGET_VIDEOS:
                break

        page_token = data.get("nextPageToken")
        if not page_token:
            break

videos_df = pd.DataFrame(video_hits)
print("✅ Fetched video search results:", len(videos_df))


# --- 2 Fetch video details ---
def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i+size]

video_ids = videos_df["video_id"].dropna().unique().tolist()
video_details = []

for batch in tqdm(list(chunked(video_ids, 50)), desc="Fetching video details"):
    params = {"part": "snippet,statistics", "id": ",".join(batch)}
    data = yt_get("videos", params)
    for it in data.get("items", []):
        snip = it.get("snippet", {})
        stats = it.get("statistics", {})
        video_details.append({
            "video_id": it.get("id"),
            "title": snip.get("title"),
            "description": snip.get("description"),
            "publishedAt": snip.get("publishedAt"),
            "channelTitle": snip.get("channelTitle"),
            "viewCount": int(stats.get("viewCount", 0) or 0),
            "likeCount": int(stats.get("likeCount", 0) or 0),
            "commentCount": int(stats.get("commentCount", 0) or 0),
        })

video_details_df = pd.DataFrame(video_details)
print("✅ Video details fetched:", len(video_details_df))


# --- 3 Fetch comments ---
all_comments = []

for vid in tqdm(video_details_df["video_id"].tolist(), desc="Fetching comments"):
    page_token = None
    fetched = 0
    try:
        while True:
            params = {
                "part": "snippet",
                "videoId": vid,
                "maxResults": 100,
                "order": "relevance",
            }
            if page_token:
                params["pageToken"] = page_token

            data = yt_get("commentThreads", params)
            items = data.get("items", [])
            for it in items:
                top = it.get("snippet", {}).get("topLevelComment", {})
                s = top.get("snippet", {})
                all_comments.append({
                    "video_id": vid,
                    "comment_id": top.get("id"),
                    "author": s.get("authorDisplayName"),
                    "publishedAt": s.get("publishedAt"),
                    "likeCount": s.get("likeCount", 0),
                    "text": s.get("textOriginal", ""),
                })
                fetched += 1

            page_token = data.get("nextPageToken")
            if not page_token or fetched >= 300:
                break

    except requests.HTTPError as e:
        print(f"⚠️ Skipping {vid} due to HTTP error: {e}")
        continue

comments_df = pd.DataFrame(all_comments)
print("✅ Comments fetched:", len(comments_df))


# --- 4 Sentiment analysis ---
def compound_score(text):
    return sia.polarity_scores(text or "")["compound"]

video_details_df["title_compound"] = video_details_df["title"].fillna("").apply(compound_score)
video_details_df["description_compound"] = video_details_df["description"].fillna("").apply(compound_score)

if not comments_df.empty:
    comments_df["compound"] = comments_df["text"].fillna("").apply(compound_score)
    POS, NEG = 0.05, -0.05
    comments_df["sentiment_label"] = comments_df["compound"].apply(
        lambda c: "pos" if c > POS else ("neg" if c < NEG else "neu")
    )
    agg = (comments_df.groupby("video_id").agg(
        n_comments=("comment_id", "count"),
        mean_compound=("compound", "mean"),
        pct_pos=("sentiment_label", lambda s: (s == "pos").mean()),
        pct_neg=("sentiment_label", lambda s: (s == "neg").mean()),
        pct_neu=("sentiment_label", lambda s: (s == "neu").mean()),
    ).reset_index())
else:
    agg = pd.DataFrame(columns=["video_id", "n_comments", "mean_compound", "pct_pos", "pct_neg", "pct_neu"])

summary = (
    video_details_df.merge(agg, on="video_id", how="left")
    .assign(
        title_compound=lambda d: d["title_compound"].round(3),
        description_compound=lambda d: d["description_compound"].round(3),
        mean_compound=lambda d: d["mean_compound"].round(3),
        pct_pos=lambda d: (d["pct_pos"]*100).round(1),
        pct_neg=lambda d: (d["pct_neg"]*100).round(1),
        pct_neu=lambda d: (d["pct_neu"]*100).round(1),
    )
)

print("✅ Summary ready. Shape:", summary.shape)


# --- 5 Plot 1: Top 10 videos by mean comment sentiment ---
if not summary.empty and summary['mean_compound'].notna().any():
    top10 = summary.sort_values("mean_compound", ascending=False).head(10).copy()
    top10["title_short"] = top10["title"].str.slice(0, 60) + top10["title"].apply(lambda t: "…" if len(str(t)) > 60 else "")
    fig_bar = px.bar(
        top10, x="title_short", y="mean_compound",
        hover_data=["title", "channelTitle", "viewCount", "likeCount", "n_comments"],
        title="Top 10 videos by mean comment sentiment (compound)",
        labels={"title_short": "Video title (truncated)", "mean_compound": "Mean compound sentiment"},
    )
    fig_bar.update_layout(xaxis_tickangle=-30)
    fig_bar.show()
    fig_bar.write_html("plot_top10_sentiment.html", include_plotlyjs="cdn", full_html=True)
else:
    print("⚠️ No sentiment data to plot.")


# --- 6 Plot 3: View count vs mean sentiment ---
if not summary.empty and summary['mean_compound'].notna().any():
    scatter_df = summary.dropna(subset=["mean_compound"]).copy()
    fig_scatter = px.scatter(
        scatter_df,
        x="viewCount",
        y="mean_compound",
        hover_name="title",
        hover_data=["channelTitle", "likeCount", "n_comments"],
        title="View count vs mean comment sentiment",
        labels={"viewCount": "Views of AI", "mean_compound": "Mean compound sentiment"},
        trendline="ols"
    )
    fig_scatter.update_xaxes(type="log")
    fig_scatter.show()

# --- Top 10 most positive & most negative videos by mean comment sentiment ---

import plotly.express as px

if 'summary' in globals() and not summary.empty and summary['mean_compound'].notna().any():

    #  Top 10 most positive & most negative videos
    top10_pos = summary.sort_values("mean_compound", ascending=False).head(10).copy()
    top10_neg = summary.sort_values("mean_compound", ascending=True).head(10).copy()

    # painted
    for df, label, color in [
        (top10_pos, "Most Positive", "green"),
        (top10_neg, "Most Negative", "red")
    ]:
        # shorten the title
        df["title_short"] = df["title"].fillna("").str.slice(0, 60) + df["title"].apply(lambda t: "…" if len(str(t)) > 60 else "")

        fig = px.bar(
            df,
            x="title_short",
            y="mean_compound",
            hover_data=["title", "channelTitle", "viewCount", "likeCount", "n_comments"],
            title=f"Top 10 Videos — {label} Mean Comment Sentiment",
            labels={"title_short": "Video Title (truncated)", "mean_compound": "Mean Compound Sentiment"},
        )

        # show more directly
        if label == "Most Negative":
            fig.update_yaxes(autorange="reversed")

        # visualize
        fig.update_layout(
            xaxis_tickangle=-30,
            bargap=0.3,
            template="plotly_white",
            title_font=dict(size=18),
            yaxis=dict(range=[-1, 1])  # emotion score[-1,1]
        )

        fig.show()

        # save as HTML
        safe_label = label.replace(" ", "_").lower()
        fig.write_html(f"plot_top10_{safe_label}.html", include_plotlyjs="cdn", full_html=True)

    print("✅ Saved: plot_top10_most_positive.html & plot_top10_most_negative.html")

else:
    print("⚠️ No sentiment summary available. Please ensure 'summary' DataFrame was created successfully.")

# --- 8️⃣ Save CSVs ---
videos_df.to_csv("videos_search_hits.csv", index=False)
video_details_df.to_csv("video_details.csv", index=False)
comments_df.to_csv("video_comments.csv", index=False)
summary.to_csv("video_sentiment_summary.csv", index=False)
print("✅ Saved: videos_search_hits.csv, video_details.csv, video_comments.csv, video_sentiment_summary.csv")


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Paste your API Key: ··········


Searching videos:   0%|          | 0/60 [00:01<?, ?it/s]


HTTPError: 413 Client Error: Request Entity Too Large for url: https://www.googleapis.com/youtube/v3/search?part=snippet&q=AI&type=video&maxResults=50&order=relevance&key=+%3C%21DOCTYPE+html+PUBLIC+%22-%2F%2FW3C%2F%2FDTD+XHTML+1.0+Transitional%2F%2FEN%22+%22http%3A%2F%2Fwww.w3.org%2FTR%2Fxhtml1%2FDTD%2Fxhtml1-transitional.dtd%22%3E+%3Chtml%3E+%3Chead%3E+%3Cmeta+name%3D%22robots%22+content%3D%22noindex%2C+nofollow%22+%2F%3E+%09%3Cmeta+http-equiv%3D%22Content-Type%22+content%3D%22text%2Fhtml%3B+charset%3DUTF-8%22%2F%3E+%09%3Ctitle%3EFactiva%3C%2Ftitle%3E+%3Cscript%3Ewindow.ddjskey%3D%27D428D51E28968797BC27FB9153435D%27%3Bwindow.ddoptions%3D%7BenableCookieDomainFallback%3Atrue%7D%3B%3C%2Fscript%3E%3Cscript+type%3D%27text%2Fjavascript%27+src%3D%27%2Fdatadome%2Ftags.js%27+async%3E%3C%2Fscript%3E+%3Clink+rel%3D%22stylesheet%22+type%3D%22text%2Fcss%22+media%3D%22all%22+href%3D%22%2Fcss%2Fui.dotcom1392700ui4sr.ashx%22+%2F%3E+%3Clink+rel%3D%22stylesheet%22+type%3D%22text%2Fcss%22+media%3D%22all

In [2]:
# Factiva search AI

In [3]:
html_code = """
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html>
<head>
<meta name="robots" content="noindex, nofollow" />
	<meta http-equiv="Content-Type" content="text/html; charset=UTF-8"/>
	<title>Factiva</title>
<script>window.ddjskey='D428D51E28968797BC27FB9153435D';window.ddoptions={enableCookieDomainFallback:true};</script><script type='text/javascript' src='/datadome/tags.js' async></script>
<link rel="stylesheet" type="text/css" media="all" href="/css/ui.dotcom1392700ui4sr.ashx" />
<link rel="stylesheet" type="text/css" media="all" href="/css/DotComHeadlines1392700ui4sr.ashx" />
<link rel="stylesheet" type="text/css" media="all" href="/css/fcp/print1392700ui4sr.ashx" />

<script type="text/javascript" src="/gen/modernizr.underscore0392700ui4sr.ashx"></script>
<script type="text/javascript" src="/jquery/jquery.bundle0392700ui4sr.ashx"></script>
<script type="text/javascript" src="/xlib/x_combined0392700ui4sr.ashx"></script>
<script type="text/javascript" src="/gen/RecordGenericOD0392700ui4sr.ashx"></script>
<script type="text/javascript" src="/gen/ui.dotcom0392700ui4sr.ashx"></script>
<script type="text/javascript" src="/controls/search/jquery.overlay.1.30392700ui4sr.ashx"></script>

</head>
<body class=''><a id="skip-main" class="skip-main"  href="#PageBaseForm">Skip to main content</a>
<form name="LinkForm" id="LinkForm" method="post"><input type="hidden" id="_XFORMSTATE" name="_XFORMSTATE" value="" /><input type="hidden" id="_XFORMSESSSTATE" name="_XFORMSESSSTATE" value="" /><div id="LinkFormExElem"></div></form>
<script type="text/javascript" src="../gen/funcTwo.js"></script>
<div id="navcontainer" class="fcpNavContainer">
<table cellpadding="0" cellspacing="0" border="0" width="100%">
<tr>
<td class="factivalogo"><h1>Dow Jones Factiva</h1></td>
<td class="djrlogo" align="right"><span>Dow Jones</span></td>
</tr>
</table>
</div>
<form name="PageBaseForm" method="post" action="/hp/printsavews.aspx?ppstype=Article&amp;pp=Print&amp;hc=All" id="PageBaseForm">
<div>
<input type="hidden" name="PageScriptManager_HiddenField" id="PageScriptManager_HiddenField" value="" />
<input type="hidden" name="_XFORMSESSSTATE" id="_XFORMSESSSTATE" value="H4sIAAAAAAAEAHWSbVeiQBTHv0pnXmPdOwwI9GbFUNMCiXS1h7MHBBNDMcWshO++d7A9pzfLAGfu8/x/cATrOLLYxWJzsdmm62IXvieH3Xm423wwBY2mdUSLmWZiRDGKBswhaQiYq40IMWqEusZnySwytHjOFN1iyZopdQlKC/XaogmaBQrIcHHYLTcKp9YWi4tVvlbg0nau+xOFEmQ2AzB0Vsl4E/QrEJrhqB0bgOvNlmaAYdot0eYOB0cFVUAHTdA6QBG7BYBQcnq0En4srdSg/N+SqRyEUSLQVGGx2T5rrJPtr1me7VdRGp4n8Z4pmsUc3fN4b9z2e+Nhp+v3AzF9aHrTq5HPFKNWiSS2UrgpjccjJ25Mqo7zYpavpGRePVeKSSwQayCUh+RGkissNBWuSRecXMKiFqCodab0AfXW6xraNGUUle8GnDyGxQjnWQMINpSonOX0Mbj6XYAUdvMiKMJtkcR0GNVirSyjDPFvJlQ1dtUBoQOAQ7CJOkE1DKME4gocfl6I8vW9lzeemDKpkdX89lka0Qg6JI3DUdCSQKDmHEiojwyFYM/ED2TNKlrRT0As2LHruNYxuLee2HTYhqG5/41T2PvhIm3FmTp78zfQ7o3y21Tr+t37lH+60y+B7uz9hQ9escjeRv270G+9x/nyc/CH23ATX6X+pN+2h803TfWE1hs4Aufjl+i6M4gnB+9mudht/QPlTu9SfzcfuEP3YT22BY8iD+4HZhBHxgF6L9sgnsQaBna6XHyG+Z069Tojr4+vy3je9263mH50XDF3YRSHX81s1eei5E+sqkganEQ24ASiqv4C3z1upIADAAA=" />
<input type="hidden" name="_XFORMSTATE" id="_XFORMSTATE" value="H4sIAAAAAAAEAL1d/3LjNpJ+FZxvL9mtNSSR+q3U1ZZsy7ZmbMuxNOOabG1tQSQkcUwSDEFa1mRSlXuG2z92H+Ke4u5J8iTX3aBk0TY59ljJVOKRRBnorwF0f91oYH6yej/Bf/VGz9q3Gr3afqe3F84Xnk5U7DnC11LEzkIsdbS33+3tHb87u9rbt63e3uSz8NjkswzZxWdfsNFnB/7/M3wi4sRzfAmvZh78Nfq8Dy9jGak4yd4s5TQS8/WjyHOSNF6/m/pqTi8vPs+CZN0sdNkE2erwv9VGSeFFq2fBJ018V+91961m768/mc/3GBuGTiyF9sI5O/MSGQtnxVTIkoVkY0cEmk1EPJcJPRfwl9Q9diJD+GLi3UrWBwwzz/GEDy0l0ve9uQwduc+OvLmXwKcT6SxCBbJ6Uu8zEbrUtGmKHaogSEMvWe3to7DN3p4bhz3UI7RdCeVSZzqqHL85mNTgj12zmxb8kIFl4XuLlD37OE329i0LIdqAzO7tyXAPtPBTrbfXN03s/bwPb4rb+Xn/XinHsZgHEvC4IGnoakdEUjOh2ZWcpTAeMxWzYxVLnbADL3Y180J2KW5gLoiQMA5D1xM91mfjlU5kAGgd+N1bTy4RqV2GtD/+YTQ62UhYl75ziwK2CanQn5SaPw9rcUvbWAezmXQSpmbsUMRTGPqxSmMHRtZ1vcTD9wmMtcTxw4lxDa9j9n0qfBi2fXYSq2WyYJcyBp0EAoae8J/isoBBX5FmZChkqtmtCEMRSI++cTI8ZhPPF5EnKpUKaqX+FVqpfdqVVqClba0MQ/7eS2KVjSa8uVWsHybeVDigAJzvRnEaNTcWd94M5oxgaTCFRZD6IhGAN64AeAemOr4F5Q20s4BfBniCOcr32FAr+CpMs1msAvZB3GjURONrNNHYmSYaOU38MDgbXIzffvj1l3+N2cXgmo1HZ+8mw9EFStosXbOjq/7wfq21oH0P51/HrFkVC++Zi7awoW1BD/qTydlgzMRU3UoUrvUVwtXsHQkHDW0LNwGjl0aappOrliFNG7SER+kU3EPfi/VCkXFof41K3V2p1M1J3ffZcRprMGpn3GqypQdLPQYTiHYeHEIUqylpuvM1ml7sStOLnMzjhSd9l/WH7F14Kz1fAwTfX7F+mqhQBQoM0fvJ6Iwde/MFmrI3MkEI3a+BoHcFQT+tdlhzmsG8IG3jEJB5AG/+FbK2diVrK+8+3LkkMd/I0BMLmuChWrJRgM2CsOVuvaCPu10Je/doFQr3ltwUrL/+kCQsd8fvh2D0HjccGnN7CzRm+UxzW9jSQxGvhV7AiMN8ZUdiNVXqhvGMdPnsAoRjg1tgJxo+PU6REpppUepAj/ofDraITxN6d3AwU8LhQjfPQ1HczjaKU5gBgp34agoS//rLPw99D0iQBA6kZQLTBZhfCov0cCFCoFNoB2k5sv5ceCGwqmsRB6ABglXqDS/G129zfM6IUzfeMNTLm+fBKm5nG9ZbQANsBlkQqj9R7C3O9evFyjCeeMXeShkhT4ykINIM/AdGjF0q1xGaLI1V6jQLBLE/7QaQnac5hyr8MUUae+kDOQN5v9XsXMQ3MMEE+KhxIqMIURyBy0L+ezgYEYRHrjWLVCrXh82aBZR6a5JPUYwmib906CkhqD2B4FpOL6GVPfx0kSSR7lWry+USWhd+sgB6CQ+Rl1ccFVQdEUsZ8wC8va46GRAeZUC45gEA4RKBcJ0B4eh7udDckcroqVjgXEAAZN+bh8AEobMERcCBFexCJXysuJno3icgctcq9t0euwDLMDczwPB+dhkrR2otKVigGS+MFYIHCZJEFXo6gPWfsiPpMNQSKPoRG7hXdKO1JXdzLbeVKZqevljRzlREbkXF86orEwHO8+8yrAgd3f3Fc//zov/3i77dPByd2+3+WntFUuzIppUyiydsEfbeunm9Tcva2RGKUnJR1LvcEQqZQ3FNS2mcCOeGXak0dNOox5pNL8oiHZzfh75KXTa4i/CjdfjnQIQbJ4AUNBBRFPuIhhQYgc2saOzeCIA397SKuU4jzJvg8k8wEtVV83WNQHlsgHLEyQEU9wingzi5JJxgIjKcPNrC+ZSN2MKzrdlv/v3OmtXr1negqMRZsP55jzzDeBULJ9WGJcVgIm5kkviYT4glA9YfJxgaYtjIYphb0DGYB0+zlRQx6fkRg7rXc7tebzQLbQA9fZGeI5ScFAszlK9UfFPVmfjmGRcBXy5WfP0pmFmAxDNIHCBxgsQRD8/w8JXkoXLv+FrQtVqLxM9F4sMJO1DBVACFD0MYSLCiZD9tC/zWIEtSgB0d8HEaBF6yz/raE+DIzjBzhSmaVGPeSqO3c3xxK1HflsUtGy2thIjd6PkRD7zX8+lFf3zxQFBrUc/0vAiFDl88n0WI3FFXvBCVravTTMyq54H6CDHM1QwxB8QcEXNpEHM146BrQgwOzQOf5hu8fN0QdzK8HL5OeLmb4V0jgUndsmvr4ShCmfOEmPU7AItHWT8gPEdSg2cUKJNGP8mA68mMHWWu78JEtvYjglpgMBoPJ/IODQYoCJb5hiZUZyAHTNYNHlDRPR4OpoQ7Bg9PFEx1xANzefmkYWg8PYP7MWZ9mA4FMMM0YqdimcAaCSWG/Wi1SDnlSZ/h1fVobdstu7vuyMTQwouX6plBSGFDu3F1dim5fcJFCeRa7de7uqydHaEoTR0V9b7cEYplfvakiQooR3hFlIx9w956St+Arw7VrTAp2oVaYvp9HhvK2R8CyYT1p6U/41rGt55DeRq7jE52O+32di5QbPN2evriFQiqdVH7XiC1cdcGP9i4e2m58DhKy3PSrullgVQ7GueX0ssuev3g9eOctbMjFC+ll9R7Z0coOjtCUS/NahX1vgOSnLWzIxSl6a6i3q0docj7nXH/4HDcY32T8KK4E/xgPEcLMfWVcvlUaLArOo0xUepTWixRytf7DKRmIopiJcB3kS+PgOd5DpmXJJbCBMGAOgI/aTZ90DqpmL4A/k5EuGMknFhpvd7hKWFYjWa7lXOk3Zz5wacvNj+o+qWnZd70aDF1gKlmWgHTw9da4Vta4dta4aQVjpz4XilICO6VwjdK4WulUKSxUQrfKIUbpRClc0E8gTQ6kk4SpwExDmDahnJgwp1PcX824Q4KEm/sYoG6tsf/QsUz5d9g+glkBmsbsAiogga3EHoqhlFeAbmcM+GmvuFrC+lHzEHqmTBfhdI3vBkH16OdKnA2h8rHtHk/BCIFrJ/GtYzYtWutDcXH7EoHBe2siR0+fTmxS27idZCCroQ06az3kquhwU0hTAacE3CeAeeEmxvcOAEWIohgoUOcKFwaV1QEN4rg94owoeNaERsKWIAwv53IBHvLBg5uQaz24c0xqPgQOShmSCEch4mEOTaTFz0Aii1csjJf/5s0MqWskmxIPmneuafd08JxyWzRNczOE5jSC8+BzqzeX3+Cb+Ge0MdNb0dvBqOr03X79VrL6lY6nUqj2Wp3P3sBTpGPkZx/hmUQeXfY/h4920Oxj+BDX6yw6T36bhW/i3p92E/1iX6quETaVbvTblZLgEKP+BXqcBbCYn1Gd0/CanRaDbvSaTXr28AS0yT0gk+ol8kiDaYXT/X0N5pRxcLmM+yRqUPAUPhUBTDu6coEsvVSDl7Q/LT5mw16o2PjoNvtdtsqGHR69tpBh36qdrvT7FZtq1EvGnQEij3CV1436Airju1Xmo2m/fSg45PXDPo0b9GHybeaTYBN08h/UCn8kCLmAzDQlxBIgtnzNlE355z12Rw8GqWS6Fd0opwb/esv/0CTPvXmjPZbKCMtnQV8HogVm0oWiBv04yvsIdo0DF6QxZ6+WVVogydRa79eGi49gNbMoNnL327Ctes04Trtbr1owuGzV0+4dh3sS6djVe1Wrftwwm0DxR7hK6+ccACrbjW6tUq72e48PeHwyRcm3LN6aqACLQi+WgUKpGevV2CjanW7rUbVtuv2lxQIX3mtAhsV27a6rUqzbbULVmwbK61evmK3hM1F71Hkg30WuJcmVkC2R2nCRqb0A6L0K+HIilnW1x4lpIBYrZZiZdZVaS1IgQAZt3pmJFHcyDaKkj+jWGAN40DEKL1m/ZjK+uA14kKi8m3CTrB2aoV7YxAqTBZYTwKU9t6SUbUhmiYDuzQqL7CU7t0rYG81ktsdBUFhkM7FjaSQ8F1lXGHnChAOZpguBI7+F9wCBEqGqehQpfNFhezuydqswoR0UwgQbqla7kaukGZTO5IYneewOZXQ7TM0vVpBuAnEOk7we3I6rZCj3wRQpZF+EXGwd6CZJ8qZzhWAHgboHgSEgsPQTXUSr9hQ45D3h9m8PqUsLG7WEoRGaZhftKiS385XNLqZqbOLGCk9e7Wpa3TR1FntqtUEe1cCFHuEr7zS1AEsoDgdq1J/QLXvTV19TbS/1tQluTlxCZqB8ORaeJQkOBGwuHFA2JWCuPrOLHCaAqU5koIJ2HF3MIs7+fI2rEZBEbBcA2axpqWLzKYfCn+l4YOx9wmWfYQWLRKhZ3I8jdKCoYKum/4O5IdGcs6FHUpfTmOwLWCkcJsjZkdK0vo7SUUMy1JCXOBhDUKSxqGRvrRMqMgf2L/dAjQhod1qWIVkDZ+9egFCSGhDoA5kzeq2ixYgAqXooNt+5QIEWHan1e1AdNCuFUUH7dprFuADo3yMNcRXSgv/f/9HoP/AEm4NDuVKyH30sZing3HCiR0I19RdYWgAVnqKG6XCn6aBZjRLShMH15eHo/NH8tgfzdGHZeSo4HnzvLihr03HkvAv3YRCZdZ3kErO2nmV7C/deqI+P+5I9o+vk72Urxb1Ge5I9q+voyTZX7oVRH1GO5L9lfP9pRtA1Gd3R7J380ln6PlN6s4lU7GLtSxHkwHaohspIxapJbgogCSA38YqnfoQER1JeOmB9cG6G9pVAPMBUdPdG+V7EBcsJB4sodMGJRVHjWar2yxwW7h/AE9fnme+W326zzN/RFTcoOJugpsCHFFxQoU1FiLha1TcNaj4BhXPUN1n8J8WOHekQSaYaz/2QtqbgYhhJmM8X9Uzxv44FqlLlt5xVBomYurhsRz26y//zSbCuaG6cfPbeGLlMMaAq0+7D2yQxhBZkFpL+dhw8G7w/gnfY5LEnkzl7fPmT3FDeU6D+xWDWAD4OQRRGVJzjopOTVxB5BQT6q0P+1RASWjKNpk6na69vWuSnyT49MWT5EdzFMr1sPaGZgtI7bkS/lrvz3OaP7gdAauXTwkX10Z4HhOczVtBQDjI17GblUUS+JsJUyD8tvbE/XE8b+s4HjsbD07YQUwROtb3ygSsSiLwzJ7CKuzk5HJCyivZyWnVunZuKx76t6JWpjx6+mLlRasAbNhmf86IzrdF53YVu6z6Ws75lBBQYa9MuAsITOWAAwjmUbLWVJGk25o6Ex6LJZ14TKRGkxN4xvqARlw5k7irOT7tXw2O2HA8fjfoHYI9FpqJFIt3PIjRPREyKh7UWIPhQYSAx8GQVCGA2NQQwUhsdqaIhGHCg2qJAy8EKibjLLZvljKuSX94+SAgaON7U4KWCCzJfM4iLG4nt3OsACTuOlIpXch8KciWowGCqW2qn/uXg8MseUEGhzZM6SRns5R/XZ6OJ/2rxyKYGD/Cks34eWCKW9oGcy5AhlCabUzwJF6UZht4KGop3SrqwN+ZqPmA7kiiwjVbYvaofz24YGIpwNZhjszBTUosEieT7nuRFyo2jfF4KgEp5V5F3X/cGZA8cxzc8UQKKnvzQtwhBz+vbuGdCw5zhvk0kNF3cW8hxA3aaGEmTikJ+zA6H58OH3ftEIiVCoAxPQ9EcUsPqAym/2jvn4bg8pTNvBgcTkony6RLMpeSr/P+xfCJ9WaZ3Tacjc+TuLidXAomlkA3cIZcwuxY6U1ydUYVKugolyqmJEyrNA9X0F123PbVYj84bDsWvtQgKjiqqUzomPHlKQSloHwjOAlcylKK9FPbkZ5reZICs8KTPI2Y8AKqnHDSeMrc7Mz7jGgZ7pSR4KXpoqIOGzsSPF+jDk9bm0oeFfbYEK05UkYHxAXL8+sv/6DwA97RdPl2mLCloIwSUEd8GqwQvuu5zEv+7VsCWJpRKhLM2hHAPH389Zd/fhi9+/WXf52dscPT/sXJgE1OB+x6dHV2BJ+SuOUHykZXk9PJ4xmbHShTcbJ45lUDxS3l5j5SDJhNEVnxVvnZsIIWk53JlvecBynYaLQmgv3BmobMjaUIzFF17cR4HIBkLnWhh6N35/3HprYek8wQtQTPPeJZ3FJuzwao4FQlGg8ZeuHMT4n/gkW5VcC1jJJLXWX/6P1k/NjB2VmptXub6Ge6yuKWHtiS4cV6kr69POsfDnpsdDkZng/H5+x4dEXPYF5fDPoXRwMYt/7ZGNH1z0cXJ+zdmCCVOs7xu4vx5PFoWybw0Wmok2fOm+KWconI4cVkcHg6ZpdnowmbjNYr8eIHdg4T76QPr88GF0dDkB8ADt4PrghEqSct6LoW7ApELV+P2wepB8ek6ZN3w6OBwXE1HE/O+2O8vmIy3ozO8Wh0NBzgOILxuWJnw+MBHeUv9bFFYix2Big/0YbnYPq37ukwhxl6zE/v0hhik1l2PrVdvj3zbnx98cPjvrINglQvw0/P3OMobCl3Xm2Bm0rjJbgsvWAOeCtMgE1FeANsMhCYH9EswD3ZBPeVhzdS0EGrQIIdEL5PK75d6oIvfjg9O3rCsxhI4aeF7z7TqhY2lItHhmd8NB7Cyu332BsFsSLWBiR0RGV9KwSdbTqX5t6d7DO6U4bDb2W/dA1MjjbaYgVERFEFIMadbCLjGCJUHbA/vrk+4YeTP1Ho04Y4bfuwThZ3tku992X/qn929hCS3TS77UA2QcXPDBwKW3pKOf0ZXqrUY4P5Kkp6SGy1CTwHPh8DUyHlaPgcM1rnWSRNMDdv8EiYmzrmhAQ+oauI8Kqm+89JAaV8oEjs1c4UsMopYDSbIZ8UTCsfOBbVAnwHxsjHDDCmTRemguJGz71bHH69UOaM9lLKGwkwEeoSvCAQ6FDeJeyPjopWf/qOoJbSi/HZ+egxw1rv6mg/eOYZo+J2HnJoCAQ1lqHhxmWsfEycsAXixlKIDNd3tCMbbsdieHDSwasDEPbD72Mb2LMBXMpNCgSdursBPM3vNp9JY6xgdHEQ8RyfivfZOA1R/CPpVFj7u+d9i6CVspgCkTr+bqB18nmLdzex8ELZo8ggVqE5MQCWOsSwLlg/h0iCClTb5WzlFG3oYyuaOfrFM41xcTP5RIXjpxpE6LETodncw1IWDSuJELgKNzLw0qYAT0Yw7QtwQhGYJlMJ0C5nLMBzngp+zcaLpnzos5AUtvOU5XwPdlL1TBnDkEFIB+EeHaoiHn/+f/8FroH5sHhSvFnuG/aZXQ+O4edERViiEdM9X8bjXIMe6HqfUh7ztF2z7rwdWUhsaRvoyaTPWmDl8EYRNO2+FDeYfJwJZ32R1uCOX2GhKxjP7IsqJlbgykAB8ZE+wSo5WN1stzub3Rl7rfF0c2wBnr442Y2ZMJHc0qFfzHgn66vxVtW5wIq56jwRvMU30DhC42toHKFxecfjDBpfQ9MccXHERUes8aRvzeZWzep2G+37cwpPQ8qxLazXEWRtHRFEKdWwmSuq0K3grWUp1vApLIzbZ8PheT+bYlneCafNOluczaL7FE6nvHBmcvRha5pvJDQp7mnirp7JKwvbyWeF18eCwcauWUKQAu3CC0WIDINTwjWfgN2NpY8XiQATQcjgWQ8WMK8T/EvfmKr7TnldTYFQ1o7A5bMgJX9A/kNzDAzWuU7AWM9hVejISyQO7wIWqzSOdVOg6K+vQCSY5SdKCsSzdwTzwfbohJ8fXfUNM8ZBodP+fOwsiCZcAUsCID1KaNEzrMGlOXlI05jwlB+WKJCjsSM8+VzQibpN2ObIPxsvtFWrTdk8Xt82NFVYc+mYy5S4DrBOVMzBHaX+Zo19qX7kfFTLWdgWyvHjuhbgubyguKEHHC+U0tXZxj/OM9pv8/EEgYzlt3S7GSIzF3KgtXHWpXZYOzsztYRk1HHcIqARFRpPumUCPRiYHwExrJNdSKroHhDabqN05TQlzpwFO51S0nR+9fb68B4U7hzW4V09S1XGN0tn9wVwVrNjYQFcvd4uPBNFz15ZAIf9VOv1Tqtdtdp2o1oCFitQ4SuvKoAjWPVms2tVGt1WQQUqPnlWAVyxsA83bGAuTaiaHLwhGAM80YcXb2ngNyFdMQZm/gCCKbwFY44BVatXr7HLczYYT9ixFLCO6NIm+L1PMtxnB55670lzc+3kA3uDp27gublGh6ZUKZntH14PrwYP14llm/hZOEsvfiYLLG7p4cVrJ3jkB48cXJ6YolusjkgFXoJ4IGX4F5L6EXPd4j64Xf6gn8bUX3OfDnmtF3CfmSlfqawE2OX7eho8mcSjOaczSVj3QCJyC0hKrdaqtXPFD0Uy5dKOPgznmXcjIVw+g9CSrnV8RF3x3mRdOYDZZLc6j+xXRjYcelqIEibQ3KxuaPzd1VkOrt1oLzHxtC4GQcOG5I7KGCy7WmtVZz6eKwVJueA+Srqu8yyUKpfI0+z9aMT+zL7//nuyiu/8ZH29XgLziUjNMcRdqS9osLuPiG65Ehq/ixI8zW+V4j/++COd300zEEBj1yDwMDSC+JJ68l70WzqvEQAjxlgGaH+PlATRKF5iFUsq10FySlvQC1wsR0NYKuu7qZH8xuvK8CzJMgN+7qNPWqhI2nXgyHPFavUeHukE/ZaUHDVhibYfSFxPV/cX23xF1YyeUgBREalZSZG5XJDLyNPKldUAFMDXCiDlGvA8A493B23Ac9eDRUhH1eGLWKS0Bs8NeL4GzxF81b6r33XDRevjZmkWIMxfgSPRcIJpQm1O8IyiI9FdT+I0iCg0+4MNIwn2yoMRh7jjcCHBcB2s4EXs6SQQ2kzlRyy7fCqL32Uq47VXK5XyuaKD/hiKETyOR/ojzREa2DkDDVQP0Ph0BS8yaF+a4OJRSpxWOupyIGIfz9yAwY9dqgmF5/ADtU3nl2YzrJYbmktuH5H3cu1Zu9Ye0D+NNVE88wnbSsRrxeIMFpcIi4OJgDjL5RJg8SXAIi07IuSCYHEv+ZLq8gzhUMUSXDdMvFOP0sfIMNdVk9cLvIYfT+x5WJaabeYf4MEJUKVe3O+HUu3cIV4Qb3YUu4+CiHK9dn6XWekA2CWC5QsAy6NtsKBNAIu3WSBYnlUu8CmCBQXrBbw0YE2ZnZOB/YK2H1wGAzEzO4xTjVeIbE4IIjGZeZ8gHgCSls7nvszuCgNi60BoNno/PMIzzDJMjQt/FNGUK7f7uyg3AWzcMdhwthI2vFk0Imx4naDBZswBYeOOuvVcvEWMsH1Blw/rqpOZ792xo6xqmg5/n6tQ0hX7lo2vk4WG9Y55CzwVTgmYMzGfr/MypvCTVPooDCpXafN3UWloIHI3K6HG0+c8QIh4R4dl42uACGsfIHI8uKLRk/kGIrkvUxn6Jc3mi6beRazb+o/1tsE+hA+eznjztef7eKVG33Wlu07Jj7+5ZEBRMZCgdLxtk0ofhQHlKm39LipNI95tofYoLYjXSmZ0ewnI+BQMACLDKYra0xEHYFjJ7kqHA64v6DFfUYth14QSL+CFYjyRFmPO/xxzaucmnXYgsfgV1Bb5aoUMjzT3KBQp15y9a80hu0lEvMot78X60h+JYHBiARiOCUJ4jWBAewgGVLUB8wV15ZNX2S0N3zAtEx9oQM84b7pWGP8tALptX6UJeysxKXsGowa+MNmHEM9seF2qSDNTEc3wEiGqNdYR+E9zzXSt5LTEwXhycfIwlJyv09tTnYTzF7PSqYKpFZqrmU0+Cm9ldmScZPcm6m0Fm2/zbPeORwBmfb1iphCuV0G0UOGKY1JyzTaLJP/5b/s2/hsy+C/zWF34UW/hK/xh42d2DT+ze/a+1cEHNXhl01MECIoaH0AXltVAjPbPP//8/9RRXTPwZwAA" />
<input type="hidden" name="__VIEWSTATE" id="__VIEWSTATE" value="" />
</div>

<script type="text/javascript">
<!--
if((typeof RootNewWindowDirectory == 'undefined') && (!RootNewWindowDirectory)){RootNewWindowDirectory = 'https://global-factiva-com.ezproxy.cul.columbia.edu/en';}
isPostProcessing = true;
languageCode = 'en';
logOmniture = true;
// -->
</script><script type="text/javascript">
<!--
function translate(t) {
var r={"yes":"Yes","no":"No","ok":"OK","djheaderaWNPLikeLink":"${djheaderaWNPLikeLink}","djheaderDontShowMe":"Don\'t show me this again","djheaderFirstView":"${djheaderFirstView}","djheaderLoremipsum":"Factiva is powered by the most comprehensive collection of business information in the world.","djheaderGetStarted":"Get Started","djheaderTryitlater":"Try it later","snapShotPrd":"Homepage","snapShotNavBeta":"${snapShotNavBeta}","nameIsRequired":"Please enter a 1-25 character name.","snapShotBeta":"Homepage","tryItNow":"Try It Now!","whatsNewPopUpTitle":"What\'s New","invalidEmail":"You have entered an invalid e-mail address.","profValidEmail":"Please enter a valid e-mail address.","sSun":"Sun","sMon":"Mon","sTue":"Tue","sWed":"Wed","sThu":"Thu","sFri":"Fri","sSat":"Sat","sunday":"Sunday","monday":"Monday","tuesday":"Tuesday","wednesday":"Wednesday","thursday":"Thursday","friday":"Friday","saturday":"Saturday","sJan":"Jan","sFeb":"Feb","sMar":"Mar","sApr":"Apr","sMay":"May","sJun":"Jun","sJul":"Jul","sAug":"Aug","sSep":"Sep","sOct":"Oct","sNov":"Nov","sDec":"Dec","january":"January","february":"February","march":"March","april":"April","may":"May","june":"June","july":"July","august":"August","september":"September","october":"October","november":"November","december":"December","smallAm":"${smallAm}","smallPm":"${smallPm}","capitalAm":"${capitalAm}","capitalPm":"${capitalPm}"};
return (r && t && r[t] || r[t] === '')? r[t] : t || null;
}
// -->
</script>
<script src="/ScriptResource.axd?d=na9ThF35vypYMd7Jk_rxYO38HjXffKotc0wvG_70TpcLVKfZBZPIM8kH05SebEymwP8VMdB5A4PWnvSTWMpmgkzkPMRWw3LUebC7IY1-RzB_CjMRRSx8x83UUIIagDyIsgi9fBsYewhw-mD8wu8dSyqNoNY1&amp;t=5c0e0825" type="text/javascript"></script><link href="/WebResource.axd?d=NhJjFR_XCMs18rNVWoDInluTIB_uBACnN1nBI-8xV6GiPce6R6t2x0Iz5ENz8Px7hAME9tFr3xqyYeeLelCj5XdG2swm5QVvr04cmRtyfAQYQSvyeyzPxiKMkscYlRJ3KlT37g2&amp;t=638484482820000000" type="text/css" rel="stylesheet" /><div id="contentWrapper"><div id="contentLeft" class="carryOverOpen"><span></span><div id="article-FJBT000020251202em1100001" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/fjbtLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Increasing Literacy on the Scams Targeting Latines: Generative Artificial Intelligence, Digital Technologies, and the Latine Community</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>Aguilar Gabriel Lorenzo </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>114 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>1 January 2026</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Journal of Business & Technical Communication</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>FJBT</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>1-23</td></tr><tr><td align="right" valign="top" class="index"><b>VOL</b>&nbsp;</td><td>Volume 40; Issue 1; ISSN:1050-6519</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2026 Journal of Business & Technical Communication. Provided by ProQuest Information and Learning. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >This article builds a heuristic that raises the artificial intelligence (AI) literacy of Latine students. Nefarious people are exploiting marginalized Latine communities by using AI in creative partnerships, similar to those described in technical communication research, to build social profiles of Latines. These people are rhetorically using AI in passive-income and voice-over scams that target Latines who are insecure about their financial and citizenship situations. The heuristic offered here guides instructors on how to increase Latine students’ AI literacy by making these students aware of the rhetorical relationships between nefarious individuals and AI.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td></td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302022 : Artificial Intelligence Technologies | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>ccat : Corporate/Industrial News | gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcom : Society/Community | gcrim : Crime/Legal Action | gcsci : Computer Science | gedu : Education | gfraud : Fraud | ggenai : Generative AI | glit : Literacy/Illiteracy | gsci : Sciences/Humanities | gsoc : Social Issues | nabst : Abstracts | ncat : Content Types | nfact : Factiva Filters | nfcpex : C&E Executive News Filter</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>Article | artificial intelligence | Feature | Latino studies | SAGE PUBLICATIONS, INC. | scams | Scholarly Journals | technical communication</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Sage Publications, Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document FJBT000020251202em1100001</td></tr></table><br/></div></div><br/><span></span><div id="article-ASZOOG0020251203elcv00017" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/aszoogLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Fragmented Landscapes as Refuge for Forest Birds in Pakistan and India: A Systematic Review</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>Muhammad Azeem Akhter, Mark E Hostetler, Ihsan Qadir and Muhammad Talha Imtiaz </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>5728 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>31 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Pakistan Journal of Zoology</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>ASZOOG</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>2959</td></tr><tr><td align="right" valign="top" class="index"><b>VOL</b>&nbsp;</td><td>57</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025. Zoological Society of Pakistan </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Key words</p>
<p class="articleParagraph enarticleParagraph" >Biodiversity, Forest fragments, Birds, Avian ecology, Urbanization</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >INTRODUCTION</p>
<p class="articleParagraph enarticleParagraph" >The average human population in developing countries is increasing rapidly and half of the world population is now living in the cities. The report projections state that the current 7.88 billion population on Earth will increase to 8.5 billion by 2030, 9.7 billion in 2050, and 11.2 billion in 2100 (World Population Prospects, Population Division, United Nations, 2022). The process of human development changes natural landscapes into agricultural and urbanized areas, impermeable ground layers, hydrological process disruptions, exotic vegetation growth followed by a rise in human population, and altered energy and nutrient flows (Menon and Rangaswamy, 2016). Declining biodiversity in cities is mostly associated with habitat alteration, especially vegetation cover and structural changes due to urbanization (Sadam et al., 2021).</p>
<p class="articleParagraph enarticleParagraph" >The rapid expansion of urban and rural areas has had profound effects on natural ecosystems and biodiversity worldwide, and it has affected the distribution of birds (Archer et al., 2019). Human-dominated areas are highly modified and fragmented habitats that affect the types of birds that occur in cities (Grimm et al., 2008; Aronson et al., 2014; Venter et al., 2016). In many developing countries, some wildlife and birds survive outside protected areas on farmlands, pasture lands, and urban areas (Bolwig et al., 2006). Land transformation poses significant challenges for wildlife, including forest birds (Lampila et al., 2005), as their habitat are increasingly fragmented and transformed, and depending on how these areas are designed and managed, only certain species utilize these areas (Hostetler and Holling, 2000; Zaman et al., 2023).</p>
<p class="articleParagraph enarticleParagraph" >Forest birds, especially those living in and around the human-dominated areas are among the taxa that are most at risk from land transformation because they depend on good forest habitats for breeding, food, and shelter (Blair, 1996; Kang et al., 2015). Because forest birds have specific ecological niches and environmental needs, they face several challenges as their natural habitats become urbanized (Blair, 1996; Lepczyk et al., 2017). However, human-dominated environments and urban green spaces have the potential to function as essential resources and refuges for some bird species, assisting in their survival (Chace and Walsh, 2006).</p>
<p class="articleParagraph enarticleParagraph" >Forest birds are important for pollination, seed dissemination, and pest management (Whelan et al., 2008). Their presence in urban and rural areas can promote avian diversity and improve human well-being by adding aesthetic and recreational value (Shochat et al., 2006). Forest birds also act as environmental quality indicators and can promote an understanding of the ecological integrity of cities (Clergeau et al., 2001). We focused on forest birds because most cities have the ability to conserve and plant trees in urban areas. Further, most urban bird surveys are done in parks and residential areas that contain trees (Hostetler and Holling, 2000; Lepczyk et al., 2017). To plan for the conservation of urban forest birds in human-dominated landscapes, conservationists must first understand which forest bird species can be found in cities and agricultural spaces.</p>
<p class="articleParagraph enarticleParagraph" >With such knowledge, it makes it possible to recognize priority species and their unique habitat needs, enabling targeted conservation efforts (Clergeau et al., 2001).</p>
<p class="articleParagraph enarticleParagraph" >The prime objective of the current study is to determine the forest bird species of India and Pakistan that are potentially adapted to urban environments while maintaining their wilderness ranges because they are somehow vulnerable to rapid changes in the urban environment. The result of this review provides a list of forest birds that could utilize urban environments, helping in the decision-making process for avian conservation.</p>
<p class="articleParagraph enarticleParagraph" >MATERIALS AND METHODS</p>
<p class="articleParagraph enarticleParagraph" >Systematic review</p>
<p class="articleParagraph enarticleParagraph" >We carried out a systematic review of peer-reviewed studies of urban forest birds surveyed in urban and rural areas that contained trees and/or small forest fragments. The objective was to create a list of birds that are known to consistently use urban or rural areas that are highly fragmented. The geographic focus was India and Pakistan. These countries share about 3323 km of border length starting from Gujrat/Sindh to Kashmir (Pakistan, 2016). Both countries have been facing issues of uncontrolled urbanization since the start of the 21st century and both of them share the same zoogeographic region i.e. Indomalayan realm and have similar climatic conditions along with many cultural similarities along the borders (Bibi and Metais, 2016). Most of the mountain ranges, plane areas, deserts, and even coastal areas are shared in both of these countries. That is why they share over 70% of flora and fauna (Pakistan-Himalayas, Karakoram, Indus, Britannica, n.d.).</p>
<p class="articleParagraph enarticleParagraph" >We looked for bird studies that were conducted in urban and suburban areas with urban tree canopies and in rural areas that had forest fragments embedded in an agricultural matrix. We considered birds that were either year-round residents or migrants that were using the habitat as a stopover or winter habitat. We focused on these taxa: Accipitridiformes, Anseriformes, Apodiformes, Bucerotiformes, Charadriiformes, Ciconiformes, Columbiformes, Coraciiformes, Cuculiformes, Falconiformes, Gruiformes, Passeriformes, Pelecaniformes, Piciformes, Psittaciformes, Strigiformes, and Suliformes.</p>
<p class="articleParagraph enarticleParagraph" >Literature search strategy</p>
<p class="articleParagraph enarticleParagraph" >An online literature search was performed on articles published from 2000 to 2023 in English. Only original research articles were included in the review and we excluded editorials, comments, and opinion essays. To be included in this review, the research articles had to have reported bird species abundance/presence and were analyzed within urban/rural forest fragments either in the form of urban parks or small forest patches and/ or surveys in residential/commercial areas. To identify appropriate articles, we identified a series of keywords and used these as a search string in the Web of Science© online database and Science Direct. We developed our systematic review following the protocol suggested by the Collaboration for Environmental Evidence (CEE, 2013), using the population-intervention-comparator-outcomes (PICO) framework. This framework utilizes a combination of words that maximizes the discovery of relevant articles.</p>
<p class="articleParagraph enarticleParagraph" >This search strategy corresponded to the following: (a) Population - descriptors of those objects of study such as urban forest birds, (b) Interventions - aspects influencing these objects defined such as forest fragment, pocket park, urban trees, (c) Comparator-descriptors of the interventions identified such as fragment size, small fragment, large fragment, and (d) Outcomes-descriptors of outcomes related to objects with identified interventions such as occurrence, abundance, density. Additionally, we also included the words India and Pakistan to define the geographical scope of the review. Likewise, we used the words identified in the previous step to build a search string using Boolean operators like “AND” to link the groups of words between and “OR” to link those words inside of each search. We also used the string the symbol * to find words that have some suffix o prefix commonly used but with a similar meaning (e.g., urban* = urbanization or *urban=suburban).</p>
<p class="articleParagraph enarticleParagraph" >Additionally, we also included in the string the Boolean operator “NOT” linked to broader topics such as climate change and other study topics not considered for this review.</p>
<p class="articleParagraph enarticleParagraph" >Article selection</p>
<p class="articleParagraph enarticleParagraph" >Based on the literature identified in the first step, we developed a list of articles considered suitable for inclusion in the review. We reviewed titles, abstracts, and the full text to determine relevant studies. We screened titles and abstracts of articles discovered through the scientific search engines and rated each as either “0” (not useful) or “1” (potentially useful). For those that received a “1,” we read the full text. We considered articles with forest bird records in urban/rural forest fragments or residential commercial areas with trees in the Indo-Pak subcontinent.</p>
<p class="articleParagraph enarticleParagraph" >Information extraction, analysis, and synthesis</p>
<p class="articleParagraph enarticleParagraph" >We reported the total number of articles selected for this review and information on the year of publication and country of study. Additionally, from these articles, we constructed a list of all birds that are known to consistently use urban or rural areas that are highly fragmented. Forest birds made the list when they were reported in at least two articles and had a minimum of five individuals seen in a study. For each species, we provided the following: (a) the English common name and scientific name of each selected bird following the taxonomic classification, (b) existing sites associated with the reporting time and dates, (c) the Method used to count the bird population, and (d) the urban/rural green spaces where the forest birds occurred in at least two decades.</p>
<p class="articleParagraph enarticleParagraph" >RESULTS</p>
<p class="articleParagraph enarticleParagraph" >We found a total of 127 possible bird studies in India and Pakistan in human-dominated landscapes, but only 10 studies met our criteria where we were able to generate a list of forest birds reported in and around urban habitats like urban green spaces, agricultural lands, and residential/commercial areas (Supplementary Table I). Of these 10 studies, 3 studies included all three habitats, 3 studies included residential areas and agricultural lands, 2 of the studies just reported birds' density in urban green spaces and residential areas while 1 study just reported urban green spaces, and 1 reported residential areas. Green spaces are areas that have scattered trees in high human areas and some portions covered by small forest fragments. Overall, most studies were published in the last 10 years (80%) with 5 studies in Pakistan and 5 studies in India (Supplementary Table I).</p>
<p class="articleParagraph enarticleParagraph" >We documented a total of 101 bird species considered to be utilizing trees in human-dominated landscapes. We found 95 bird species (94%) reported in urban green spaces. Out of these 95 species, 65% were primary residents, 23% were winter migrants, 5% were summer breeders, and 7% of the species were considered to be partially resident and winter migrants in the Indo-Pak subcontinent (Table I).</p>
<p class="articleParagraph enarticleParagraph" >Table I. List of 101 forest bird species in this review identified as users of fragmented landscapes in human dominated areas of India and Pakistan. </p>
<p class="articleParagraph enarticleParagraph" ><pre class="articlePre" >S. Order/ Common name                                            Migratory/         Urban    Agri-       Residential/   References
No. (Scientific name)                                            resident status    green    cultural    commercial
                                                                                    spaces   lands       areas
Order: Accipitridiformes
1    Shikra (Accipiter badius)                                   Resident           ✓        ✓           ✓              2,6,8
2    Northern goshawk (Accipiter gentilis)                       Winter migrant     ✓        ✓           -              2,9
3    Common buzzard (Buteo buteo)                                Winter migrant     ✓        ✓           ✓              1, 2
4    Long-legged buzzard (Buteo rufinus)                         Winter migrant     ✓        ✓           -              1, 2
5    Brahminy kite (Haliastur indus)                             Resident           ✓        ✓           - 	        2, 8
6    Black kite (Milvus migrans)                                 Resident           ✓        ✓           ✓              1, 2, 3, 4, 5, 6, 7,
                                                                                                                        8, 9, 10
7    Tawny eagle (Aquila rapax nipalensis)                       Winter migrant     ✓        ✓           ✓              1, 2
8    Crested honey buzzard (Pernis ptilorhynchus)                Winter migrant     ✓        ✓           ✓              1, 2, 8
Order: Anatoidea
9    Ruddy shelduck (Tadorna ferruginea)                         Winter migrant     ✓        ✓           -              1, 2
10   Common shelduck (Tadorna tadorna)                           Winter migrant     ✓        ✓           -              1, 2, 8
Order: Apodiformes
11   House swift (Apus affinis)                                  Resident           ✓        ✓           ✓              2, 4, 6, 8, 9
12   Asian palm swift (Cypsiurus balasiensis)                    Resident           ✓        ✓           ✓              6, 8
Order: Bucerotiformes
13   Common hoopoe (Upupa epops)                                 Resident           ✓        ✓           ✓              1, 2, 3, 4, 8, 9
Order: Charadriiformes
14   Wood sandpiper (Tringa glareola)                            Winter migrant     ✓        -           -              1, 2, 7
15   Common greenshank (Tringa nebularia)                        Winter migrant     ✓        -           -              2, 1
16   Green sandpiper (Tringa ochropus)                           Winter migrant              -           -              1, 2, 7
17   Marsh sandpiper (Tringa stagnatilis)                        Winter migrant              ✓           -              1, 2, 7
Order: Ciconiiformes
18   Painted stork (Mycteria leucocephala)                       Resident           ✓        -           -              1, 2
19   Open-billed stork (Anastomus oscitanas)                     Resident           -        ✓           ✓              6, 8
Order: Columbiformes
20   Indian ring dove (Streptopelia decaocto)                    Resident           ✓        ✓           ✓              1, 2, 8, 9, 10
21   Oriental turtle dove (Streptopelia orientalis)              Winter migrant     ✓        ✓           ✓              1, 5
22   Little brown dove (Streptopelia senegalensis)               Resident           ✓        ✓           ✓              1, 2, 3, 4, 9, 10
23   Red turtle dove (Streptopelia tranquebarica)                Summer breeder     ✓        ✓           ✓              1, 4
24   Yellow-footed green pigeon (Treron phoenicoptera)           Resident           ✓        ✓           ✓              2, 4, 6, 9, 10
Order: Coraciiformes
25   Indian roller/blue jay (Coracias benghalensis)              Resident           ✓        ✓           ✓              1, 2, 3, 8,
26   European roller (Coracias garrulous)                        Passage migrant    ✓        ✓           ✓              1, 2
27   Green bee-eater (Merops orientalis)                         Resident/ winter   ✓        ✓           ✓              2, 3, 5, 6, 8, 9,
                                                                 migrant                                                10
28   Indian grey hornbill (Ocyceros birostris)                   Resident           ✓        ✓           ✓              4, 9
Order: Cuculiformes
29   Common crow pheasant (Centropus sinensis)                   Resident           ✓        ✓           ✓              1, 2, 6, 8, 9, 10
30   Asian koel (Eudynamus scolopacea)                           Summer breeder     ✓        ✓           ✓              1, 3, 4, 6, 8, 9
Order: Falconiformes
31   Red necked falcon (Falco chicquera)                         Resident           ✓        ✓           -              1, 2
Order: Gruiformes
32   White-breasted waterhen (Amaurornis phoenicurus)            Resident           ✓        ✓           ✓              1, 2, 3, 4, 6, 7, 8
Order: Passeriformes
33   Bank myna (Acridotheres ginginianus)                        Resident           ✓        ✓           ✓              1, 2, 3, 6, 8, 9, 10
34   Common myna (Acridotheres tristis)                          Resident           ✓        ✓           ✓              1, 2, 3, 6, 8, 9, 10
35   Blyth's reed warbler (Acrocephalus dumetorum)               Summer breeder     ✓        ✓           ✓              1, 2, 8
36   Moustached sedge warbler (Acrocephalus melanopogon)         Winter migrant     -        ✓           -              1, 2
37   Small skylark (Alauda gulgula)                              Winter migrant     ✓        ✓           ✓              1, 2
38   House crow (Corvus splendens)                               Resident           ✓        ✓           ✓              1, 2, 3, 4, 6, 8,
                                                                                                                        9, 10
39   Rufous tree pie (Dendrocitta vagabunda)                     Resident           ✓        ✓           ✓              1, 2, 3, 4, 5, 6, 8,
                                                                                                                        9, 10
40   Forest wagtail (Dendronanthus indicus)                      Winter migrant     ✓        ✓           -              2, 8
41   Black drongo/king crow (Dicrurus macrocercus)               Resident           ✓        ✓           ✓              1, 2, 3, 4, 5, 6, 8,
                                                                                                                        9, 10
42   Red-headed bunting (Emberiza bruniceps)                     Winter migrant     ✓        ✓           -              1, 2
43   Red-breasted flycatcher (Ficedula parva)                    Winter migrant     ✓        ✓           ✓              1, 2
44   Long-tailed shrike (Lanius schach)                          Resident           ✓        ✓           ✓              2, 3, 4, 8
45   Bay-backed shrike (Lanius vittatus)                         Resident           ✓        ✓           ✓              2, 4
46   Indian silverbill (Lonchura malabarica)                     Resident           ✓        ✓           -              1, 2
47   Blue throat (Luscinia svecica)                              Winter migrant     ✓        ✓           -              2, 4, 8
48   Purple-rumped sunbird (Nectarinia asiatica)                 Resident           ✓        ✓           ✓              1, 2, 3, 5, 6, 9
49   Golden oriole (Oriolus oriolus)                             Summer breeder     ✓        ✓           ✓              1, 2, 4, 6, 8
50   Common tailorbird (Orthotomus sutorius)                     Resident           ✓        ✓           ✓              1, 2, 6, 8
51   House sparrow (Passer domesticus)                           Resident           ✓        ✓           ✓              1, 2, 3, 6, 8, 10
52   Small minivet (Pericrocotus cinnamomeus)                    Resident           ✓        ✓           -              2, 9
53   Long-tailed minivet (Pericrocotus ethologus)                Resident           ✓        ✓           -              1, 2
54   Greenish warbler (Phylloscopus trochiloides)                Winter migrant     ✓        ✓           ✓              2, 8
55   Baya weaver (Ploceus philippinus)                           Resident           ✓        ✓           ✓              1, 2, 3, 8, 10
56   Yellow-bellied prinia (Prinia flaviventris)                 Resident           -        ✓           ✓              2, 8
57   Red-vented bulbul (Pycnonotus cafer)                        Resident           ✓        ✓           ✓              1, 2, 3, 5, 6, 8,
                                                                                                                        9, 10
58   White-eared bulbul (Pycnonotus leucotis)                    Resident           ✓        ✓           ✓              2, 9
59   White-browned fantail flycatcher (Rhipidura aureola)        Resident/ winter   ✓        ✓           ✓              1, 2, 4
                                                                 migrant
60   Lesser whitethroat (Sylvia curruca)                         Winter migrant     ✓        ✓           -              1, 2
61   Lesser/common woodshrike (Tephrodornis pondicerianus)       Resident           ✓        ✓           ✓              1, 2
62   Asian paradise-flycatcher (Terpsiphone paradise)            Summer breeder     -        ✓           -              2, 8
63   Common babbler (Turdoides caudatus)                         Resident           ✓        ✓           ✓              1, 2
64   Striated babbler (Turdoides earlei)                         Resident           ✓        ✓           ✓              1, 2, 3, 10
65   Oriental white-eye (Zosterops palpebrosus)                  Resident           ✓        ✓           ✓              2, 8, 5
66   Common iora (Aegithina tiphia)                              Resident           ✓        ✓           ✓              5, 8
67   Greater short-toed lark (Calandrella brachydactyla)         Winter migrant     ✓        ✓           -              1, 2
68   Purple sunbird (Cinnyris asiaticus)                         Summer breeder     ✓        ✓           ✓              8, 10
69   Oriental magpie robin (Copsychus saularis)                  Resident           ✓        ✓           ✓              4, 6, 8, 9, 10
70   Yellow bellied flower pecker (Dicaeum melanoxanthum)        Resident           ✓        -           ✓              5, 9
71   Ashy drongo (Dicrurus leucophaeus)                          Resident           -        ✓           ✓              6, 8
72   Asian pied starling (Gracupica contra)                      Resident           ✓        ✓           ✓              8, 10
73   Black-naped monarch (Hypothymis azurea)                     Resident                    -           -              8, 9
74   Black-headed munia (Lonchura atricapilla)                   Resident           ✓        ✓           ✓              8, 9
75   Black hooded oriole (Oriolus xanthornus)                    Resident           ✓        ✓           ✓              6, 8, 9
76   Great tit (Parus major)                                     Resident                    ✓           ✓              1, 5, 8, 9
77   Whiskered bulbul (Pycnonotus jocosus)                       Resident           ✓        ✓           ✓              6, 8, 9, 10
78   White-cheeked bulbul (Pycnonotus leucogenys)                Resident           ✓        ✓           ✓              1, 5
79   Asian pied starling (Sturnus contra)                        Winter migrant              ✓           ✓              3, 6, 8, 9
80   Jungle babbler (Turdoides striata)                          Resident           ✓        ✓           ✓              1, 2, 3, 5, 6, 8,
                                                                                                                        9, 10
81   Orange headed thrush (Zoothera citrina)                     Winter migrant     ✓        ✓           -              6, 8
     Order: Pelecaniformes
82   Grey heron (Ardea cinerea)                                  Ardea cinerea      ✓        ✓           -              1, 2
83   Purple heron (Ardea purpurea)                               Ardea purpurea     ✓        ✓           -              1, 8, 2
84   Indian pond heron (Ardeola grayii)                          Ardeola grayii     ✓        ✓           ✓              1, 2, 3, 4, 6, 7, 8,
85   Cattle egret (Bubulcus ibis)                                Bubulcus ibis      ✓        ✓           ✓              1, 2, 3, 4, 6, 7, 8
86   Little egret (Egretta garzetta)                             Egretta garzetta   ✓        ✓           ✓              1, 2, 3, 6, 7, 8
87   Intermediate egret (Egretta intermedia)                     Egretta intermedia ✓        ✓           ✓              1, 2, 7
88   Little cormorant (Microcarbo niger)                         Resident           ✓        -           ✓              2, 8
89   Great Indian cormorant (Phalacrocorax carbo)                Resident           ✓        -           -              2, 7, 8
90   Night heron (Nycticorax nycticorax)                         Resident           ✓        ✓           ✓              1, 4, 6, 8
Order: Piciformes
91    Yellow crowned woodpecker (Dendrocopos mahrattensis)       Resident           ✓        ✓           -              2, 9
92    Black-rumped flameback (Dinopium benghalense)              Resident           ✓        ✓           ✓              2, 3, 4, 6, 8, 9,
93    Coppersmith barbet (Megalaima haemacephala)                Resident           ✓        ✓           ✓              2, 4, 6, 8, 9
94    Blue throated barbet (Megalaima asiatica)                  Resident           ✓        ✓           ✓              6, 8
95    Brown-headed barbet (Megalaima zeylanica)                  Resident           ✓        -           -              9, 10
Order: Psittaciformes
96    Large Indian parakeet (Psittacula eupatria)                Resident           ✓        ✓           ✓              1, 2, 5, 8, 9
97    Rose-ringed parakeet (Psittacula krameri)                  Resident           ✓        ✓           ✓              2, 4, 6, 8, 9, 10
Order: Strigiformes
98    Spotted little owlet (Athene brama)                        Resident           ✓        ✓           ✓              1, 2, 3, 4, 6, 8, 9
Order: Suliformes
99    Darter (Anhinga melanogaster)                              Winter migrant     ✓        ✓           ✓              1, 2, 6, 8
Order: Suliformes
100 Indian cormorant (Phalacrocorax fuscicollis)                 Resident           ✓        -           -              7, 8
101 Little cormorant (Phalacrocorax niger)                       Resident           ✓        ✓           ✓              1, 6
</pre>
</p>
<p class="articleParagraph enarticleParagraph" >We found 91 (90%) species in agricultural lands. Out of which 72% of species were primarily year-round residents to the reported areas, 22% were winter migrants and seventy-two species (71%) were reported in residential/commercial areas. Out of these, 9 species were winter migrants, 5 of them were only summer breeders, 5 species were partial resident and winter migrants and the remaining 53 bird species were year-round residents in India and Pakistan (Table I). With a total of 101 bird species identified from 17 orders, Passeriformes exhibited the highest diversity of bird species than the other orders in all three habitats with 47 species in cultivated lands, 45 species in green spaces, and 37 species in residential and commercial areas (Table I). The second largest number of species belonged to Pelecaniformes, and then Accipitridiformes.</p>
<p class="articleParagraph enarticleParagraph" >This diversity emphasizes that human-dominated areas contain possible habitats for a wide array of bird species, including both residents and migratory visitors.</p>
<p class="articleParagraph enarticleParagraph" >DISCUSSION</p>
<p class="articleParagraph enarticleParagraph" >The list of 101 forest bird species spans across several genera. Notably, large species such as shikra (Accipiter badius) and Northern goshawk (Accipiter gentilis) occurred in fragmented areas with trees. Also, migratory species were sighted, such as wood sandpiper (Tringa glareola) and common greenshank (Tringa nebularia); human-dominated areas can provide stopover and wintering habitats along bird migratory flyways (Archer et al., 2019; Xu et al., 2021). However, not all residential landscapes are created equal. Managing the quality and quantity of urban vegetation is key to creating good breeding and stopover habitat within the built environment and native vegetation can increase native bird diversity (Schneider and Miller, 2014). As observed in one of the review studies, greater bird densities and diversities were discovered in residential areas with higher vegetation cover (Sengupta et al., 2014).</p>
<p class="articleParagraph enarticleParagraph" >Our study results are similar to several review studies of North and South America. In a study of Neotropical migrant birds in South America (Amaya-Espinel and Hostetler, 2019), researchers found that small forest patches and urban tree cover provided some migrants with stopover and wintering habitats. Another study found that certain interior-forest specialists, which are defined as being dependent on extensive forest expanses for successful breeding, utilized small forest fragments in both urban and rural settings and tree canopies within suburban residential areas as crucial stopover sites during migration seasons (Archer et al., 2019). However, some migrating species may primarily forage or take shelter near the centers of forest patches; Dawson and Hostetler (2010) found several migrant species that avoided the edges of forest patches, indicating that the interior of these small urban forest fragments may hold significance for these species.</p>
<p class="articleParagraph enarticleParagraph" >Further, Buron et al. (2022) demonstrated that birds that primarily foraged under the tree canopy typically utilized urban forest patches more than residential treed areas. Additionally, remnant forest patches, located next to developments, are still utilized by migrating and resident bird species (Hostetler et al., 2005). Overall, studies have suggested that both city trees in residential areas and forest fragments in and around cities can provide habitat for different types of forest birds (Archer et al., 2019).</p>
<p class="articleParagraph enarticleParagraph" >Synanthropic species, or urban dwellers, are birds that can effectively take advantage of human-caused changes and disruptions in an urban environment (Fischer et al., 2015). The presence of forest generalist species in urban environments of India and Pakistan, such as the house crow (Corvus splendens) and common myna (Acridotheres tristis), highlights their adaptability to cities, exploiting food waste generated by humans (Marzluff et al., 2012; Tariq et al., 2024). Other resident species like the house swift (Apus affinis) and Indian ring dove (Streptopelia decaocto) also demonstrate successful adaptation to human-dominated landscapes (Pal et al., 2019). Despite crows being seen in cities, a significant decline in the urban population of house crows has been observed (Radadia, 2013).</p>
<p class="articleParagraph enarticleParagraph" >Factors contributing to this decline could be due to the reduction of trees (Marzluff et al., 2001), crowded and heavily built-up areas (Bernat-Ponce et al., 2018), toxicity in urban environments (Benmazouz et al., 2021; Seress and Liker, 2015), and fluctuations in food availability (Mustafa et al., 2015). This decline in crows since 2011 suggests the vulnerability of synanthropic species in the urban areas of India and Pakistan, underscoring the complex interactions between urbanization and bird ecology.</p>
<p class="articleParagraph enarticleParagraph" >For forest birds in cities, it is important to conserve forest fragments and trees as cities expand. Developers and city planners must take into account design, construction, and post-construction factors to promote the long-term health of trees and forest patches (Hostetler, 2012). For example, construction practices such as parking heavy machinery in forested areas, failure to protect root zones of trees with proper fencing, and failure to recognize and remove invasive vegetation transported from other areas can greatly reduce the ability of trees to survive and forested areas to retain plant and animal diversity (Hostetler, 2012). Further, nearby residents may impact conserved areas through pollution, exotic animals, and the spreading of invasive plants. Conserving forest patches within an urban area can have limited effects on biodiversity when steps are not taken to ensure its biological integrity.</p>
<p class="articleParagraph enarticleParagraph" >Without long-term management, urban forest patches can become ecological traps-habitats that an organism might favor despite increased livelihood of species mortality and decline (Battin, 2004).</p>
<p class="articleParagraph enarticleParagraph" >It's important to note that while this review lists forest birds seen in urban and fragmented rural areas of Pakistan and India, the mere presence of a species does not guarantee thriving populations or long-term persistence. Factors such as disease, competition, pollution, and predation can adversely affect certain species (Wilson et al., 2019). However, these lists serve as a starting point for identifying local species of concern and potential conservation. City planners can then collaborate with ecologists to assess population vitality and conduct further research on breeding success and foraging availability in fragmented and residential areas. While areas with trees could become bird habitats, potential threats to city birds must be carefully addressed through conservation measures and sustainable urban planning (Hostetler, 2012; Wilson et al., 2019).</p>
<p class="articleParagraph enarticleParagraph" >DECLARATIONS</p>
<p class="articleParagraph enarticleParagraph" >Acknowledgement</p>
<p class="articleParagraph enarticleParagraph" >We extend our sincere gratitude to the <span class="companylink">Higher Education Commission of Pakistan</span> (HEC) for their generous financial support, which was instrumental in the successful completion of our review article. Our heartfelt thanks go to the Department of Wildlife Ecology and Conservation, IFAS, UF, for their invaluable support. We are also deeply thankful to the dedicated faculty members at Bhauddin Zakariya University Multan's Department of Forestry and Range Management for their kind cooperation and assistance during the crucial phase of data collection.</p>
<p class="articleParagraph enarticleParagraph" >Funding</p>
<p class="articleParagraph enarticleParagraph" >The study was partially funded by <span class="companylink">Higher Education Commission (HEC)</span>, Pakistan.</p>
<p class="articleParagraph enarticleParagraph" >Declaration of generative AI</p>
<p class="articleParagraph enarticleParagraph" >During the preparation of this work, the authors used Chat GPT to enhance the Grammar and readability of the article. After using this service, the authors reviewed and edited the content as needed and took full responsibility for the content of the publication.</p>
<p class="articleParagraph enarticleParagraph" >Supplementary material</p>
<p class="articleParagraph enarticleParagraph" >There is supplementary material associated with this article. Access the material online at: <span class="colorLinks">http://dx.doi [http://dx.doi]</span>. org/10.17582/journal.pjz/..........</p>
<p class="articleParagraph enarticleParagraph" >Statement of conflict of interest</p>
<p class="articleParagraph enarticleParagraph" >The authors have declared no conflict of interest.</p>
<p class="articleParagraph enarticleParagraph" >REFERENCES</p>
<p class="articleParagraph enarticleParagraph" >Amaya-Espinel, J.D. and Hostetler, M.E., 2019. The value of small forest fragments and urban tree canopy for Neotropical migrant birds during winter and migration seasons in Latin American countries: A systematic review. Landsc. Urban Plann., 190: 103592. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j]</span>. landurbplan.2019.103592</p>
<p class="articleParagraph enarticleParagraph" >Archer, J.M.J., Hostetler, M.E., Acomb, G. and Blair, R., 2019. A systematic review of forest bird occurrence in North American forest fragments and the built environment. Landsc. Urban Plann., 185: 1-23. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j.landurbplan.2019.01.005 [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j.landurbplan.2019.01.005]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Aronson, M.F.J., La Sorte, F.A., Nilon, C.H., Katti, M., Goddard, M.A., Lepczyk, C.A., Warren, P.S., Williams, N.S.G., Cilliers, S., Clarkson, B., Dobbs, C., Dolan, R., Hedblom, M., Klotz, S., Kooijmans, J.L., Kuhn, I., Macgregor-Fors, I., Mcdonnell, M., Mortberg, U. and Winter, M., 2014. A global analysis of the impacts of urbanization on bird and plant diversity reveals key anthropogenic drivers. Proc. R. Soc. B Biol. Sci., 281: 1780. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1098/rspb.2013.3330</p>
<p class="articleParagraph enarticleParagraph" >Battin, J., 2004. When good animals love bad habitats: Ecological traps and the conservation of animal populations. Conserv. Biol., 18: 1482-1491. https:// doi.org/10.1111/j.1523-1739.2004.00417.x</p>
<p class="articleParagraph enarticleParagraph" >Benmazouz, I., Jokimäki, J., Lengyel, S., Juhász, L., Kaisanlahti-Jokimäki, M.L., Kardos, G., Paládi, P. and Kover, L., 2021. Corvids in urban environments: A systematic global literature review. Animals, 11: 3226. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.3390/ani11113226 [https://doi-org.ezproxy.cul.columbia.edu/10.3390/ani11113226]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Bernat-Ponce, E., Gil-Delgado, J.A. and Guijarro, D., 2018. Factors affecting the abundance of House Sparrows Passer domesticus in urban areas of southeast of Spain. Bird Study, 65: 404-416. https:// doi.org/10.1080/00063657.2018.1518403</p>
<p class="articleParagraph enarticleParagraph" >Bibi, F. and Metais, G., 2016. Evolutionary history of the large herbivores of south and Southeast Asia (Indomalayan Realm). Ecol. Large Herb. South Southeast Asia, pp. 15-88. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1007/978-94-017-7570-0_2</p>
<p class="articleParagraph enarticleParagraph" >Blair, R.B., 1996. Land use and avian species diversity along an urban gradient. Ecol. Appl., 6: 506-519. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.2307/2269387 [https://doi-org.ezproxy.cul.columbia.edu/10.2307/2269387]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Bolwig, S., Pomeroy, D., Tushabe, H. and Mushabe, D., 2006. Crops, trees, and birds: Biodiversity change under agricultural intensification in Uganda's farmed landscapes. Geogr. Tids. Danish J. Geogr., 106: 115-130. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1080/00167223.2 [https://doi-org.ezproxy.cul.columbia.edu/10.1080/00167223.2]</span> 006.10649561</p>
<p class="articleParagraph enarticleParagraph" >Buron, R., Hostetler, M.E. and Andreu, M., 2022. Urban forest fragments vs residential neighborhoods: Urban habitat preference of migratory birds. Landsc. Urban Plann., 227: 104538. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1016/j.landurbplan.2022.104538</p>
<p class="articleParagraph enarticleParagraph" >CEE, 2013. Guidelines for systematic reviews in environmental management. Version 4.2, March, 80. <span class="colorLinks">http://www.environmentalevidence.org/wp-content/uploads/2014/06/Review-guidelines-version-4.2-final.pdf [http://www.environmentalevidence.org/wp-content/uploads/2014/06/Review-guidelines-version-4.2-final.pdf]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Chace, J.F. and Walsh, J.J., 2006. Urban effects on native avifauna: A review. Landsc. Urban Plann., 74: 46-69. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j]</span>. landurbplan.2004.08.007</p>
<p class="articleParagraph enarticleParagraph" >Clergeau, P., Jokimäki, J. and Savard, J.L., 2001. Are urban bird communities influenced by the bird diversity of adjacent landscapes? J. appl. Ecol., 38: 1122-1134. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1046/j.1365-2664.2001.00666.x [https://doi-org.ezproxy.cul.columbia.edu/10.1046/j.1365-2664.2001.00666.x]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Dawson, D. and Hostetler, M.E., 2010. Forest remnants: Conserving and observing bird diversity in urban settings. <span class="companylink">University of Florida</span>, IFAS Extension. Vol. 298. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.32473/edis-uw343-2010 [https://doi-org.ezproxy.cul.columbia.edu/10.32473/edis-uw343-2010]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Fischer, J.D., Schneider, S.C., Ahlers, A.A. and Miller, J.R., 2015. Categorizing wildlife responses to urbanization and conservation implications of terminology. Conserv. Biol., 29: 1246-1248. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1111/cobi.12451 [https://doi-org.ezproxy.cul.columbia.edu/10.1111/cobi.12451]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Grimm, N.B., Faeth, S.H., Golubiewski, N.E., Redman, C.L., Wu, J., Bai, X. and Briggs, J.M., 2008. Global change and the ecology of cities. Science, 319: 756-760. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1126/science.1150195 [https://doi-org.ezproxy.cul.columbia.edu/10.1126/science.1150195]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Hostetler, M., 2012. The green leap: A primer for conserving biodiversity in subdivision development. University of California Press. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1525/california/9780520271104.001.0001</p>
<p class="articleParagraph enarticleParagraph" >Hostetler, M. and Holling, C.S., 2000. Detecting the scales at which birds respond to structure in urban landscapes. Urban Ecosyst., 4: 25-54. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1023/A:1009587719462</p>
<p class="articleParagraph enarticleParagraph" >Hostetler, M., Duncan, S. and Paul, J., 2005. Post-construction effects of an urban development on migrating, resident, and wintering birds. Southe. Natl., 4: 421-434. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1656/1528-7092(2005)004[0421:PEOAUD]2.0.CO;2 [https://doi-org.ezproxy.cul.columbia.edu/10.1656/1528-7092(2005)004[0421:PEOAUD]2.0.CO;2]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Kang, W., Minor, E.S., Park, C.R. and Lee, D., 2015. Effects of habitat structure, human disturbance, and habitat connectivity on urban forest bird communities. Urban Ecosyst., 18: 857-870. https:// doi.org/10.1007/s11252-014-0433-5</p>
<p class="articleParagraph enarticleParagraph" >Khera, N., Mehta, V. and Sabata, B.C., 2009. Interrelationship of birds and habitat features in urban greenspaces in Delhi, India. Urban Forest. Urban Green., 8:187-196.</p>
<p class="articleParagraph enarticleParagraph" >Lampila, P., Monkkonen, M. and Desrochers, A., 2005. Demographic responses by birds to forest fragmentation. Conserv. Biol., 19: 1537-1546. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1111/j.1523-1739.2005.00201.x [https://doi-org.ezproxy.cul.columbia.edu/10.1111/j.1523-1739.2005.00201.x]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Lepczyk, C.A., La Sorte, F.A., Aronson, M.F.J., Goddard, M.A., MacGregor-Fors, I., Nilon, C.H. and Warren, P.S., 2017. Global patterns and drivers of urban bird diversity. Ecol. Conserv. Birds Urban Environ., pp. 13-33. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1007/978-3-319-43314-1_2 [https://doi-org.ezproxy.cul.columbia.edu/10.1007/978-3-319-43314-1_2]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Marzluff, J.M., Bowman, R. and Donnelly, R., 2012. Avian ecology and conservation in an urbanizing world. Springer Science and Business Media.</p>
<p class="articleParagraph enarticleParagraph" >Marzluff, J.M., McGowan, K.J., Donnelly, R. and Knight, R.L., 2001. Causes and consequences of expanding American crow populations. Avian Ecol. Conserv. Urban. World, pp. 331-363. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1007/978-1-4615-1531-9_16</p>
<p class="articleParagraph enarticleParagraph" >Marzluff, J. and Rodewald, A., 2008. Conserving biodiversity in urbanizing areas: nontraditional views from a bird's perspective. Cities Environ. (CATE), 1: 6. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.15365/ [https://doi-org.ezproxy.cul.columbia.edu/10.15365/]</span> cate.1262008</p>
<p class="articleParagraph enarticleParagraph" >Menon, M. and Rangaswamy, M., 2016. Avifaunal richness and abundance along an urban rural gradient with emphasis on vegetative and anthropogenic attributes in Tiruchirappalli, India. Landsc. Res., 41: 131-148. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1080 [https://doi-org.ezproxy.cul.columbia.edu/10.1080]</span> /01426397.2014.910294</p>
<p class="articleParagraph enarticleParagraph" >Mustafa, I., Arif, N., Hussain, S.M., Malik, I.U., Javid, A., Ullah, M.I., Asif, S., Khan, M.R., Waqas, A. and Eqani, S.A.M., 2015. Population dynamics of house sparrow (Passer domesticus) and house crow (Corvus splendens) in Punjab (District Sargodha), Pakistan. Pakistan J. Zool., 47: 1147-1155.</p>
<p class="articleParagraph enarticleParagraph" >Naithani, A. and Bhatt, D., 2012. Bird community structure in natural and urbanized habitats along an altitudinal gradient in Pauri district (Garhwal Himalaya) of Uttarakhand state, India. Biologia., 67: 800-808.</p>
<p class="articleParagraph enarticleParagraph" >Pakistan Himalayas, Karakoram, Indus, Britannica. (n.d.). Retrieved April 17, 2024, from <span class="colorLinks">https://www [https://www]</span>. britannica.com/place/Pakistan/The-Himalayan-and-Karakoram-ranges Pakistan, C.G., 2016. No title about Pakistan. https:// pakconsulatela.org/about-pakistan/</p>
<p class="articleParagraph enarticleParagraph" >Pal, M., Pop, P., Mahapatra, A., Bhagat, R. and Hore, U., 2019. Diversity and structure of bird assemblages along urban-rural gradient in Kolkata, India. Urban For. Urban Green., 38: 84-96. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1016/j.ufug.2018.11.005</p>
<p class="articleParagraph enarticleParagraph" >Radadia, B., 2013. Population estimation of Indian house crow (Corvus splendens) in Junagadh, Gujarat. Int. J. Res. Edu., 2 (Sp. Issue 1), January 2013.</p>
<p class="articleParagraph enarticleParagraph" >Rajpar, M.N., Hassan-Aboushiba, A.B., Ullah, S., Ozdemir, I., Ullah, A. and Zakaria, M., 2019. Determining the temporal changes in avian population inhabiting urban seasonally waterlogged areas in Hyderabad, Sindh, Pakistan. Appl. Ecol. environ. Res., 17: 10831-10843. <span class="colorLinks">http://dx.doi [http://dx.doi]</span>. org/10.15666/aeer/1705_1083110843</p>
<p class="articleParagraph enarticleParagraph" >Sadam, A., Khan, R.U. and Mahmood, S., 2021. Identifying bird traits that enable them to become urban exploiters in an urban area of Mardan, Pakistan. Pakistan J. Zool., 53: 1813-1822. https:// doi.org/10.17582/journal.pjz/20190805080803</p>
<p class="articleParagraph enarticleParagraph" >Schneider, S.C. and Miller, J.R., 2014. Response of avian communities to invasive vegetation in urban forest fragments. Condor: Ornithol. Appl., 116: 459-471. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1650/CONDOR-13-009R1.1 [https://doi-org.ezproxy.cul.columbia.edu/10.1650/CONDOR-13-009R1.1]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Sengupta, S., Mondal, M. and Basu, P., 2014. Bird species assemblages across a rural urban gradient around Kolkata, India. Urban Ecosyst., 17: 585- 596. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1007/s11252-013-0335-y [https://doi-org.ezproxy.cul.columbia.edu/10.1007/s11252-013-0335-y]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Seress, G. and Liker, A., 2015. Habitat urbanization and its effects on birds. Acta Zool. Acad. Sci. Hung., 61: 373-408. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.17109/ [https://doi-org.ezproxy.cul.columbia.edu/10.17109/]</span> AZH.61.4.373.2015</p>
<p class="articleParagraph enarticleParagraph" >Sidra, S., Ali, Z. and Chaudhry, M.N., 2013. Avian diversity at New Campus of Punjab University in relation to land use change. Pakistan J. Zool., 45:1069-1082.</p>
<p class="articleParagraph enarticleParagraph" >Shochat, E., Warren, P.S., Faeth, S.H., McIntyre, N.E. and Hope, D., 2006. From patterns to emerging processes in mechanistic urban ecology. Trends Ecol. Evol., 21: 186-191. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j]</span>. tree.2005.11.019</p>
<p class="articleParagraph enarticleParagraph" >Tariq, A., Ahmad, S.R. and Qadir, A., 2024. Nesting material adaptation of native bird species with anthropogenic litter along an urbanization gradient in Pakistan. Environ. Res., pp. 118435. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1016/j.envres.2024.118435</p>
<p class="articleParagraph enarticleParagraph" >Tiwary, N.K. and Urfi, A.J., 2016. Spatial variations of bird occupancy in Delhi: The significance of woodland habitat patches in urban centres. Urban Forest. Urban Green., 20:338-347.</p>
<p class="articleParagraph enarticleParagraph" >Venter, O., Sanderson, E.W., Magrach, A., Allan, J.R., Beher, J., Jones, K.R., Possingham, H.P., Laurance, W.F., Wood, P. and Fekete, B.M., 2016. Sixteen years of change in the global terrestrial human footprint and implications for biodiversity conservation. Nat. Commun., 7: 12558. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1038/ncomms12558</p>
<p class="articleParagraph enarticleParagraph" >Whelan, C.J., Wenny, D.G. and Marquis, R.J., 2008. Ecosystem services provided by birds. Annls N.Y. Acad. Sci., 1134: 25-60. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1196/ [https://doi-org.ezproxy.cul.columbia.edu/10.1196/]</span> annals.1439.003</p>
<p class="articleParagraph enarticleParagraph" >Wilson, S., Schuster, R., Rodewald, A.D., Bennett, J.R., Smith, A.C., La Sorte, F.A., Verburg, P.H. and Arcese, P., 2019. Prioritize diversity or declining species? Trade-offs and synergies in spatial planning for the conservation of migratory birds in the face of land cover change. Biol. Conserv., 239: 108285. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j.biocon.2019.108285 [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j.biocon.2019.108285]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >World Population Prospects, Population Division, United Nations, 2022. <span class="colorLinks">https://population.un.org/ [https://population.un.org/]</span> wpp/</p>
<p class="articleParagraph enarticleParagraph" >Xu, Y., Kieboom, M., Van Lammeren, R.J.A., Si, Y. and De Boer, W.F., 2021. Indicators of site loss from a migration network: Anthropogenic factors influence waterfowl movement patterns at stopover sites. Glob. Ecol. Conserv., 25: e01435. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1016/j.gecco.2020.e01435</p>
<p class="articleParagraph enarticleParagraph" >Zaman, A., Rafique, A., Jabeen, F. and Sultana, T., 2023. Diversity, Abundance and Seasonal Assessment of Wild Birds in Urban Habitat of District Chiniot, Pakistan. Pakistan J. Zool., 55: 525. <span class="colorLinks">https://doi [https://doi]</span>. org/10.17582/journal.pjz/20211215151255</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gbiol : Biology | gcat : Political/General News | genv : Natural Environment | gnatcn : Environmental Protection | gsci : Sciences/Humanities</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>asiaz : Asia | devgcoz : Emerging Market Countries | dvpcoz : Developing Economies | india : India | pakis : Pakistan | sasiaz : South Asia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>The Zoological Society of Pakistan</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document ASZOOG0020251203elcv00017</td></tr></table><br/></div></div><br/><span></span><div id="article-ASZOOG0020251203elcv0000z" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/aszoogLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Effect of Carbon Source Addition Strategies on Water Quality, Growth Performance and Histology in Penaeus vannamei and GIF Tilapia (Oreochromis niloticus) in Polyculture Model in Lined Pond - BFT Aquaculture Systems</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>Malreddy Joshna, Baboonsundaram Ahilan, Cheryl Antony, Krishnan Ravaneswaran, Pushparaj Chidambaram, Arumugam Uma and Ruby Ponnusamy </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>4378 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>31 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Pakistan Journal of Zoology</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>ASZOOG</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>2877</td></tr><tr><td align="right" valign="top" class="index"><b>VOL</b>&nbsp;</td><td>57</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025. Zoological Society of Pakistan </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Key words</p>
<p class="articleParagraph enarticleParagraph" >Biofloc, GIF tilapia, Penaeus vannamei, Polyculture, Lined pond</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >INTRODUCTION</p>
<p class="articleParagraph enarticleParagraph" >Nowadays aquaculture industry is growing rapidly and cutting edge technologies are being practiced to improve quantity and quality production. Generally, aquaculture species retains at least 20-30% of feed nutrients (Avnimelech and Ritvo, 2003); dense aquaculture results in rapid accumulation of organic and inorganic compounds, which disturbs the environment by the discharge of waste water due to water exchange. To overcome the environmental damage and increase sustainable aquaculture production (Avnimelech, 2009), one of the promising technologies that developed was an ecofriendly culture technology known as biofloc technology (BFT) (Martinez-Cordova et al., 2017).</p>
<p class="articleParagraph enarticleParagraph" >BFT is a protein rich live food formed by aggregates of algae, protozoa, bacteria and particulate organic matter and held together in a loose matrix of mucus secreted by bacteria and bound by filamentous microorganisms. Biofloc has two major advantages viz., treating wastes from feeding and providing nutrients from floc consumption. However, certain drawbacks can encounter in the biofloc technology such as high concentration of solids generated, excessive accumulation of suspended solids in the water and the solids removed are an effluent rich in nitrogen and phosphorous compounds.</p>
<p class="articleParagraph enarticleParagraph" >To overcome the above-mentioned obstacle, the utilization of polyculture model could result in a better development in a BFT system. Polyculture is an aquaculture model, where simultaneous cultivation of different trophic levels in the same environment system, resulting in the conversion of culture residues in to food, for the other species (Chopin et al., 2001). The use of polyculture model could contribute to increased productivity and would allow for maximum utilization of nutrients present in BFT system, based on different trophic level species. The interaction between farmed aquatic organisms in polyculture depends mainly on the stocking density and biological characteristics of the species.</p>
<p class="articleParagraph enarticleParagraph" >Shrimp is the main exporting species in the world, since it is widely cultured in tropical and sub-tropical regions. Some characteristics that made it widely known based on expanding production chain due to its adaptability, rapid growth, disease resistance (Hossain and Islam, 2006) and adaptability to polyculture with fish (Haque et al., 2018). Moreover, shrimp with fish culture improves the production efficiency, greater profits (De-Shang and Shuang-Lin, 2000) and improved ecological balance of pond (Uddin et al., 2006) and less environmental impact (Santos and Valenti, 2002). In polyculture model, synergistic interactions should be improvements in feed availability and environmental conditions (Milstein, 1992) and antagonistic interactions are competition for food, space, oxygen and other resources.</p>
<p class="articleParagraph enarticleParagraph" >Above mentioned fundamentals can be matched by the Penaeus vannamei and GIF tilapia because of different spatial distribution and feeding habits. P. vannamei are benthic in production ponds, omnivorous, eating detritus, waste and feces (D'Abramo and New, 2010). Whereas, GIF tilapia are pelagic, filtering phytoplankton, omnivorous and eating periphyton (Tadesse, 1999). However, the growth and production performance of shrimp and GIF tilapia in polyculture based BFT system in lined pond have been poorly documented to date (Reinoso et al., 2019). Therefore, the documentation of this hybrid technique using lined pond based biofloc technology in polyculture model of P. vannamei and GIF tilapia is highly essential for further adoption.</p>
<p class="articleParagraph enarticleParagraph" >MATERIALS AND METHODS</p>
<p class="articleParagraph enarticleParagraph" >Experimental design and experimental units</p>
<p class="articleParagraph enarticleParagraph" >The experiment was conducted at Tamil Nadu Dr. J. Jayalalithaa Fisheries University; Pulicat Research Farm Facility (PRFF), Pazhaverkadu, Chennai, Tamil Nadu, India for a period of 90 days during September to December, 2023. It was conducted in four uniform HDPE (High Density Polyethylene Ponds) lined ponds with an area of 0.12 ha (12 X 10 X 1.5 m), by following completely randomized design (CRD) in duplicate, with and without addition of carbon source for development of biofloc and considered as biofloc treatment and clear water treatment, respectively. The biofloc was developed and maintained as per Avnimelech (1999) with minor modifications in two lined ponds. One horse power aerator was placed at the center of the pond in order to maintain a circulation of water.</p>
<p class="articleParagraph enarticleParagraph" >The shrimp species used in the experiment was white leg shrimp (P. vannamei) with an average initial weight of 1.06±0.08 g and stocking density of 60 shrimp/ m3. The shrimp were fed with commercial sinking pellet feed of 36% crude protein level and fed with only 50% of feeding rate compared to actual feeding rate (Table I) in both treatments. GIF tilapia used as a fish species, had an average initial weight of 0.42±0.01 g and was maintained in hapa at the shrimp lined ponds at a density of 5 fish/ m3. GIF tilapia was fed with commercial floating pellet feed of 24% crude protein level. Both the species were fed four times a day (06.00; 10.00; 14.00 and 18.00 Hrs.). Weights of 10% of total number of animals were measured individually at fortnight interval with a view to estimate the animal biomass and to adjust the feeding rate accordingly.</p>
<p class="articleParagraph enarticleParagraph" >Table I. Feeding rate for Penaeus vannamei stocked in a biofloc based lined pond. </p>
<p class="articleParagraph enarticleParagraph" ><pre class="articlePre" >Days of     Weight of           Feeding rate
culture     P. vannamei      Lin, 1991    Followed
0-15        1.06 g           10%          5%
15-30       3.50 g           6.5%         3.25%
30-45       8.09 g           5%           2.5%
45-60       11.80 g          4.2%         2.1%
60-75       14.80 g          4.0%         2.0%
75-90       17.70 g          3.5%         1.75%
</pre>
</p>
<p class="articleParagraph enarticleParagraph" >Water quality monitoring</p>
<p class="articleParagraph enarticleParagraph" >During the experimental period, water temperature (o C), dissolved oxygen, pH and salinity were measured daily. Nitrogen compounds such as ammonia, nitrite and nitrate were analyzed weekly with the double beam UV visible spectrophotometer KLUV-2100 model. Total hardness, calcium hardness, magnesium hardness and alkalinity were determined as per the method described in APHA (2005).</p>
<p class="articleParagraph enarticleParagraph" >Growth performance analysis of shrimp and GIF tilapia</p>
<p class="articleParagraph enarticleParagraph" >Fortnightly, growth performance of fish and shrimp was recorded by measuring the 10% of total number of animals, randomly with minimal stress, and the individual final body weight was recorded. The collected animal weight were used to calculate weight gain (g), feed conversion ratio (FCR), feed efficiency ratio (FER), protein efficiency ratio (PER), specific growth rate (SGR-%/day), average daily gain (ADG-g/animal/day) and survival rate (%) by adopting standard formulae (Dong et al., 2018; Tan et al., 2018).</p>
<p class="articleParagraph enarticleParagraph" >Histology</p>
<p class="articleParagraph enarticleParagraph" >Ten animals in similar size from biofloc and clear water treatment were euthanized for histology. The hepatopancreas, gut of shrimp and gill, gut of GIF tilapia were collected and immediately fixed in 10% neutralized buffered formalin (NBF), dehydrated in graded ethanol levels, embedded in paraffin wax and blocked at 58OC and sectioned using Rotary microtome (Leica RM2255, India) (Bell and Lightner, 1988). The sections of 6µm were stained with hematoxylin and eosin (H and E) with Microm HMS7. Finally, sections were evaluated under light microscope at 100X (Olympus CX21, India) and photographed for further examination (Krogdahl et al.,2003; Wang et al., 2017).</p>
<p class="articleParagraph enarticleParagraph" >Statistical analysis</p>
<p class="articleParagraph enarticleParagraph" >The results of water quality and growth were analyzed in <span class="companylink">SPSS</span> Version 24 software using student's t-test. Significance level for the test was set as p&gt;0.05. Histology analyses were analyzed descriptively.</p>
<p class="articleParagraph enarticleParagraph" >RESULTS AND DISCUSSION</p>
<p class="articleParagraph enarticleParagraph" >Water quality parameters</p>
<p class="articleParagraph enarticleParagraph" >Water quality parameters were recorded during the experimental trial are given in Table II. pH, temperature, salinity, alkalinity, total hardness, calcium hardness, magnesium hardness, nitrite and dissolved oxygen did not shown any significant difference between the treatment groups. Mean ammonia and nitrate ranged from 0.33±0.02 to 0.91±0.02 mg/L and 1.21±0.07 to 1.88±0.13 mg/L with the lowest value observed in biofloc based lined pond system.</p>
<p class="articleParagraph enarticleParagraph" >In case of ammonia and nitrate, significant difference was observed in between the treatments. The decrease level of ammonia and nitrate in biofloc treatment lined pond is due to the occurrence of nitrification process by chemoautotrophic bacteria and the removal of ammonia by heterotrophic bacteria, present in the biofloc system (Ebeling et al., 2006). As the addition of carbon source in the biofloc lined pond, having the ability to consume ammonia and nitrate for the growth and multiplication of bacterial population. Polyculture of shrimp and fish in biofloc at lined pond has been known for minimizing the environmental impact of effluents, particularly related to nitrogenous wastes (Martinez-Porchas et al., 2010).</p>
<p class="articleParagraph enarticleParagraph" >Table II. Mean water quality parameters recorded in lined pond with biofloc and clear water in polyculture model. </p>
<p class="articleParagraph enarticleParagraph" ><pre class="articlePre" >Parameters                               Biofloc                 Clear water          p value
pH                                       8.57±0.09               8.53±0.09            0.017
Temperature (°C)                         27.21±0.44              27.28±0.42           0.465
Salinity (ppt)                           5.64±0.14               5.40±0.06            0.158
Alkalinity (mg/L)                        210.92±1.54             217.00±2.56          0.072
Total hardness (mg/L)                    2984.20±17.59           2974.00±27.32        0.793
Calcium hardness (mg/L)                  143.75±5.80             135.33±2.57          0.162
Magnesium hardness (mg/L)                902.08±26.50            937.50±5.59          0.226
Ammonia (mg/L)                           0.33±0.02 b             0.91±0.02a           0.000
Nitrite (mg/L)                           0.21±0.10               0.40±0.01            0.078
Nitrate (mg/L)                           1.21±0.07b              1.88±0.13a           0.000
Dissolved oxygen (mg/L)                  6.04±0.22               5.78±0.41            0.611
</pre>
</p>
<p class="articleParagraph enarticleParagraph" >Table III. Growth performance of P. vannamei and GIF tilapia reared using lined pond maintained with biofloc and clear water in polyculture model. </p>
<p class="articleParagraph enarticleParagraph" ><pre class="articlePre" >                                                       Penaeus vannamei                                               GIF tilapia
                                        Biofloc           Clear water          p-value           Biofloc              Clear water      p-value
Initial weight (g)                      1.06±0.08         1.06±0.08            -                 0.42±0.01            0.42±0.01        -
Final weight (g)                        23.34±0.90a       15.68±0.66b          0.000             16.70±0.35a          9.39±0.50b       0.000
Average body weight gain (g)            22.28±0.94a       14.62±0.71b          0.000             16.28±0.35a          8.97±0.50b       0.000
FCR                                     0.84±0.03b        1.29±0.06 a          0.000             1.04±0.02b           1.93±0.11a       0.000
FER                                     1.21±0.05a        0.79±0.04b           0.000             0.97±0.02a           0.53±0.03b       0.000
PER                                     0.62±0.03a        0.41±0.02b           0.000             0.68±0.01a           0.37±0.02b       0.000
SGR (%/day)                             0.04±0.00 a       0.03±0.00 b          0.000             0.04±0.001a          0.03±0.00b       0.000
Average daily gain(g/animal/day)        0.25±0.01a        0.16±0.01b           0.000             0.18±0.00a           0.10±0.01b       0.000
Survival rate (%)                       63.31±0.47a       41.16±0.23b          0.000             85.90±1.44a          68.83±0.84b      0.000
</pre>
</p>
<p class="articleParagraph enarticleParagraph" >Growth performance of shrimp and GIF tilapia</p>
<p class="articleParagraph enarticleParagraph" >Growth parameters recorded in between the treatment groups are given in Table III, the study found significant difference in growth performance of shrimp and GIF tilapia reared in biofloc lined pond treatment compared to clear water. The present study found 35% and 45% of significantly higher weight gain in biofloc treatment lined pond in P. vannamei and GIF tilapia, respectively, compared to clear water system. Similarly, in polyculture of giant freshwater prawn and Nile tilapia reared in biofloc has reported 40% and 34%, respectively of higher weight gain compared to clear water system (Hisano et al., 2019). Contradictory to our results, Barbosa et al. (2022) have reported no significant difference in weight gain of Nile tilapia and freshwater shrimp in polyculture based biofloc technology.</p>
<p class="articleParagraph enarticleParagraph" >Similar to the present study, FCR values of L. vannamei and GIF tilapia reared in biofloc system has displayed a 1.45 (Xu and Pan, 2012) and 0.83 (Long et al., 2015), respectively, which was closer to the present study FCR. On the other hand, tilapia and prawn reared in biofloc system in polyculture model with complete feeding rate has displayed a FCR of 1.22-1.44 and 2.88-4.31, respectively, which was higher FCR than present study and it may be due to difference in feeding rate followed in the present study (50% feeding rate). Further it confirms that, P. vannamei and GIF tilapia reared in lined pond biofloc treatment does not have any effect on the culture, as the species utilize different niche in the culture system (Khan et al., 2016).</p>
<p class="articleParagraph enarticleParagraph" >Histology</p>
<p class="articleParagraph enarticleParagraph" >The histological observations of gill (Fig. 1A) of GIF tilapia exposed to biofloc has shown deformities such as tips of few primary lamellae were congested, whereas gill of GIF tilapia exposed to clear water has shown deformities such as tips of few primary and secondary lamellae were congested. On the other side, no abnormalities were observed in the gut of GIF tilapia in both the treatments (Fig. 1B). Interestingly, no deformities in hepatopancreas (Fig. 2A) and gut (Fig. 2B) of P. vannamei in biofloc treatment lined pond. Abnormalities were observed in clear water lined pond reared P. vannamei such as few scattered hepatopancreatic tubules showed hyperplastic changes in hepatopancreas and autolytic changes in gut.</p>
<p class="articleParagraph enarticleParagraph" >In fish, gills are the main respiratory organs for the simple diffusion of gases (oxygen and carbon dioxide). Moderate alterations in gill were observed for the fish reared under biofloc and clear water lined pond systems, tips of few primary lamellae were congested in biofloc treatment lined pond system and tips of few primary and secondary lamellae were congested in clear water lined pond system. Similar to the present study, zero water exchange system of Nile tilapia (Suloma, 2013) and tilapia raised in BFT and RAS system (Vincent, 2006) show alterations like congestion in the gill. On the other side, gill of O. niloticus (Azim and Little, 2008), C. carpio (Haghparast et al., 2020) and African cat fish (Romano et al., 2018) did not produce any potential gill damage when reared in biofloc system. The intestinal histology can be used to ascertain the gut condition (Khojasteh, 2012). The gut of GIF tilapia did not shown any damage in the presence of biofloc in culture water.</p>
<p class="articleParagraph enarticleParagraph" >Consistent with present study, Nile tilapia fed with biofloc meal did not cause any damage to the gut (Hersi et al., 2023). In present biofloc treatment study showed normal and healthy hepatopancreas, whereas the clear water shrimp has shown some deformities such as few scattered hepatopancreatic tubules showed hyperplasia changes. Similar to the present study, biofloc system did not show any histological changes in hepatopancreas of L. vannamei (Moss et al., 2001) and M. monoceros (Kaya et al., 2019). Presence of biofloc in the culture water did not cause any damage to the gut of shrimp. Similar to our results, Zheng et al. (2018) and Won et al. (2020) reported that shrimp grown in biofloc did not cause any damage to the gut. The study demonstrate that raising of GIF tilapia and shrimp in biofloc had fewer and less severe histopathological lesions in gill of GIF tilapia and hepatopancreas of P. vannamei and may consider as biofloc did not affect the normal physiological activity of GIF tilapia and shrimp.</p>
<p class="articleParagraph enarticleParagraph" >CONCLUSION</p>
<p class="articleParagraph enarticleParagraph" >The present polyculture study with Penaeus vannamei and GIF tilapia in a biofloc system has a potential to improve water quality, promoted the growth of shrimp and GIF tilapia and is beneficial for the growth of microbial biomass. Further, the present study has demonstrated that the P. vannamei and GIF tilapia polyculture is technically feasible, environmentally friendly and economically attractive with the appropriate feeding strategy.</p>
<p class="articleParagraph enarticleParagraph" >DECLARATIONS</p>
<p class="articleParagraph enarticleParagraph" >Acknowledgements</p>
<p class="articleParagraph enarticleParagraph" >The authors are grateful for the financial support provided by the Prime Minister's Fellowship for Doctoral Research, a joint initiative of <span class="companylink">Confederation of Indian Industry (CII)</span> and Science and Engineering Research Board (SERB) and Murugappa Fish Feeds. Special thanks are due to Tamil Nadu Dr. J. Jayalalithaa Fisheries University, Nagapattinam for providing the necessary facilities and guidance for the study.</p>
<p class="articleParagraph enarticleParagraph" >Funding</p>
<p class="articleParagraph enarticleParagraph" >Prime Minister's Fellowship for Doctoral Research, a joint initiative of <span class="companylink">Confederation of Indian Industry (CII)</span> and Science and Engineering Research Board (SERB) and Murugappa Fish Feeds for sustainable and maximum profit from unit area (17th Batch).</p>
<p class="articleParagraph enarticleParagraph" >IRB approval</p>
<p class="articleParagraph enarticleParagraph" >The study was approved by Institutional Review Board of Tamil Nadu Dr. J. Jayalalithaa Fisheries University, Tamil Nadu, India.</p>
<p class="articleParagraph enarticleParagraph" >Ethical approval</p>
<p class="articleParagraph enarticleParagraph" >After the approval of the statutory authorities of the Tamil Nadu Dr. J. Jayalalithaa Fisheries University, Nagapattinam, Tamil Nadu, India, the research work was carried out in adherence with the current animal welfare laws in India. The care and treatment of the experimental animal was carried out by guidelines of the CPCSEA (Committee for the Purpose of Control and Supervision of Experiments on Animals), Ministry of Environment and Forests (Animal Welfare Division, Govt. of India).</p>
<p class="articleParagraph enarticleParagraph" >Statement of conflict of interest</p>
<p class="articleParagraph enarticleParagraph" >The authors have declared no conflict of interest.</p>
<p class="articleParagraph enarticleParagraph" >REFERENCES</p>
<p class="articleParagraph enarticleParagraph" >APHA (<span class="companylink">American Public Health Association</span>), 2005. Standard methods for examination of water and waste water, 20th edition, Port City Press, Baltimore, Maryland, USA.</p>
<p class="articleParagraph enarticleParagraph" >Avnimelech, Y., 1999. Carbon/ nitrogen ratio as a control element in aquaculture systems. Aquaculture, 176: 227-235. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/S0044-8486(99)00085-X [https://doi-org.ezproxy.cul.columbia.edu/10.1016/S0044-8486(99)00085-X]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Avnimelech, Y., 2009. Biofloc technology: A practical guide book. World Aquaculture Society. Baton Rouge, LA, pp. 182.</p>
<p class="articleParagraph enarticleParagraph" >Avnimelech, Y. and Ritvo, G., 2003. Shrimp and fish pond soils: Processes and management. Aquaculture, 220: 549-567. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/S0044-8486(02)00641-5 [https://doi-org.ezproxy.cul.columbia.edu/10.1016/S0044-8486(02)00641-5]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Azim, M.E. and Little, D.C., 2008. The biofloc technology (BFT) in indoor tanks: Water quality, biofloc composition, and growth and welfare of Nile tilapia (Oreochromis niloticus). Aquaculture, 283: 29-35. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j]</span>. aquaculture.2008.06.036</p>
<p class="articleParagraph enarticleParagraph" >Barbosa, P.T.L., Povh, J.A., Farias, K.N.N., da Silva, T.V., Teodoro, G.C., Ribeiro, J.S., Stringhetta, G.R., dos Santos Fernandes, C.E. and CorrêaFilho, R.A.C., 2022. Nile tilapia production in polyculture with freshwater shrimp using an aquaponic system and biofloc technology. Aquaculture, 551: 737916. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j.aquaculture.2022.737916 [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j.aquaculture.2022.737916]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Bell, T.A. and Lightner, D.V., 1988. A handbook of normal penaeid shrimp histology.</p>
<p class="articleParagraph enarticleParagraph" >Chopin, T., Buschmann, A.H., Halling, C., Troell, M., Kautsky, N., Neori, A., Kraemer, G.P., Zertuche-González, J.A., Yarish, C. and Neefus, C., 2001. Integrating seaweeds into marine aquaculture systems: A key toward sustainability. J. Phycol., 37: 975-986. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1046/j.1529-8817.2001.01137.x [https://doi-org.ezproxy.cul.columbia.edu/10.1046/j.1529-8817.2001.01137.x]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >D'Abramo, L.R. and New, M.B., 2000. Nutrition, feeds and feeding. Freshwater prawn culture: The farming of Macrobrachium rosenbergii, pp. 203-220. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1002/9780470999554.ch13 [https://doi-org.ezproxy.cul.columbia.edu/10.1002/9780470999554.ch13]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >De-shang, L. and Shuang-lin, D., 2000. Summary of studies on closed-polyculture of penaeid shrimp with fishes and moluscans. Chinese J. Oceanol. Limnol., 18: 61-66. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1007/ [https://doi-org.ezproxy.cul.columbia.edu/10.1007/]</span> BF02842543</p>
<p class="articleParagraph enarticleParagraph" >Dong, J., Zhao, Y.Y., Yu, Y.H., Sun, N., Li, Y.D., Wei, H., Yang, Z.Q., Li, X.D. and Li, L., 2018. Effect of stocking density on growth performance, digestive enzyme activities, and nonspecific immune parameters of Palaemonetes sinensis. Fish Shellfish Immunol., 73: 37-41. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j]</span>. fsi.2017.12.006</p>
<p class="articleParagraph enarticleParagraph" >Ebeling, J.M., Timmons, M.B. and Bisogni, J.J., 2006. Engineering analysis of the stoichiometry of photoautotrophic, autotrophic, and heterotrophic removal of ammonia-nitrogen in aquaculture systems. Aquaculture, 257: 346-358. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1016/j.aquaculture.2006.03.019</p>
<p class="articleParagraph enarticleParagraph" >Haghparast, M.M., Alishahi, M., Ghorbanpour, M. and Shahriari, A., 2020. Evaluation of hemato-immunological parameters and stress indicators of common carp (Cyprinus carpio) in different C/N ratio of biofloc system. Aquacult. Int., 28: 2191-2206. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1007/s10499-020-00578-1 [https://doi-org.ezproxy.cul.columbia.edu/10.1007/s10499-020-00578-1]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Haque, M.R., Islam, M.A., Khatun, Z., Hossain, M.A. and Wahab, M.A., 2018. Effects of stocking densities of tilapia Oreochromis niloticus (Linnaeus, 1758) with the inclusion of silver carp Hypophthalmichthys molitrix (Valenciennes, 1844) in C/N-CP prawn Macrobrachium rosenbergii (De Man, 1879) culture pond. Aquacult. Int., 26: 523-541. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1007/s10499-017-0229-8 [https://doi-org.ezproxy.cul.columbia.edu/10.1007/s10499-017-0229-8]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Hersi, M.A., Genc, E., Pipilos, A. and Keskin, E., 2023. Effects of dietary synbiotics and biofloc meal on the growth, tissue histomorphology, whole-body composition and intestinal microbiota profile of Nile tilapia (Oreochromis niloticus) cultured at different salinities. Aquaculture, 570: 739391. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j.aquaculture.2023.739391 [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j.aquaculture.2023.739391]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Hisano, H., Barbosa, P.T., Hayd, L.A. and Mattioli, C.C., 2019. Evaluation of Nile tilapia in monoculture and polyculture with giant freshwater prawn in biofloc technology system and in recirculation aquaculture system. Int. Aquat. Res., 11: 335-346. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1007/s40071-019-00242-2</p>
<p class="articleParagraph enarticleParagraph" >Hossain, M.A. and Islam, M.S., 2006. Optimization of stocking density of freshwater prawn Macrobrachium rosenbergii (de Man) in carp polyculture in Bangladesh. Aquacult. Res., 37: 994-1000. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1111/j.1365-2109.2006.01518.x [https://doi-org.ezproxy.cul.columbia.edu/10.1111/j.1365-2109.2006.01518.x]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Kaya, D., Genc, M.A., Aktas, M., Yavuzcan, H., Ozmen, O. and Genc, E., 2019. Effect of biofloc technology on growth of speckled shrimp, Metapenaeus monoceros (Fabricus) in different feeding regimes. Aquacult. Res., 50: 2760-2768. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1111/are.14228</p>
<p class="articleParagraph enarticleParagraph" >Khan, M.S.R., Khan, M.M., Akter, N. and Wahab, M.A., 2016. Strain performance of tilapia in 445 freshwater prawn polyculture. J. Bangladesh Agric. Univ., 14: 127-134. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.3329/jbau [https://doi-org.ezproxy.cul.columbia.edu/10.3329/jbau]</span>. v14i1.30607</p>
<p class="articleParagraph enarticleParagraph" >Khojasteh, S.B., 2012. The morphology of the post-gastric alimentary canal in teleost fishes: A brief review. Int. J. aquat. Sci., 3: 71-88.</p>
<p class="articleParagraph enarticleParagraph" >Krogdahl, Å., Bakke-McKellep, A.M. and Baeverfjord, G., 2003. Effects of graded levels of standard soybean meal on intestinal structure, mucosal enzyme activities, and pancreatic response in Atlantic salmon (Salmo salar L.). Aquacult. Nutr., 9: 361-371. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1046/j.1365-2095.2003.00264.x [https://doi-org.ezproxy.cul.columbia.edu/10.1046/j.1365-2095.2003.00264.x]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Long, L., Yang, J., Li, Y., Guan, C. and Wu, F., 2015. Effect of biofloc technology on growth, digestive enzyme activity, hematology, and immune response of genetically improved farmed tilapia (Oreochromis niloticus). Aquaculture, 448: 135-141. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1016/j.aquaculture.2015.05.017</p>
<p class="articleParagraph enarticleParagraph" >Martínez, P.M., Martínez, C.L.R., Porchas-Cornejo, M.A. and López-Elías, J.A., 2010. Shrimp polyculture: A potentially profitable, sustainable, but uncommon aquacultural practice. Rev. Aquacult., 2: 73-85. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1111/j.1753-5131.2010.01023.x [https://doi-org.ezproxy.cul.columbia.edu/10.1111/j.1753-5131.2010.01023.x]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Martínez-Córdova, L.R., Martínez-Porchas, M., Emerenciano, M.G.C., Miranda-Baeza, A. and Gollas-Galván, T., 2017. From microbes to fish the next revolution in food production. Crit. Rev. Biotechnol., 37: 287-295. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.3109/0 [https://doi-org.ezproxy.cul.columbia.edu/10.3109/0]</span> 7388551.2016.1144043</p>
<p class="articleParagraph enarticleParagraph" >Milstein, A., 1992. Ecological aspects of fish species interactions in polyculture ponds. Hydrobiologia, 231: 177-186. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1007/BF00018201 [https://doi-org.ezproxy.cul.columbia.edu/10.1007/BF00018201]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Moss, S.M., Divakaran, S. and Kim, B.G., 2001. Stimulating effects of pond water on digestive enzyme activity in the Pacific white shrimp, Litopenaeus vannamei (Boone). Aquacult. Res., 32: 125-131. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1046/j.1365-2109.2001.00540.x [https://doi-org.ezproxy.cul.columbia.edu/10.1046/j.1365-2109.2001.00540.x]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Reinoso, S., Munoz, D., Cedeno, R., Tirado, J.O., Bangeppagari, M. and Mulla, S.I., 2019. Adaptation of” Biofloc” aquatic system for polyculture with tilapia (Oreochromis sp.) and river prawn (Macrobrachium sp.). J. Microbiol. Biotechnol. Fd. Sci., 8: 1130. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.15414/ [https://doi-org.ezproxy.cul.columbia.edu/10.15414/]</span> jmbfs.2019.8.5.1130-1134</p>
<p class="articleParagraph enarticleParagraph" >Romano, N., Dauda, A.B., Ikhsan, N., Karim, M. and Kamarudin, M.S., 2018. Fermenting rice bran as a carbon source for biofloc technology improved the water quality, growth, feeding efficiencies, and biochemical composition of African catfish Clarias gariepinus juveniles. Aquacult. Res., 49: 3691-3701. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1111/are.13837 [https://doi-org.ezproxy.cul.columbia.edu/10.1111/are.13837]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Santos, M.J.D. and Valenti, W.C., 2002. Production of Nile tilapia Oreochromis niloticus and freshwater prawn Macrobrachium rosenbergii stocked at different densities in polyculture systems in Brazil. J. World Aquacult. Soc., 33: 369-376. https:// doi.org/10.1111/j.1749-7345.2002.tb00513.x</p>
<p class="articleParagraph enarticleParagraph" >Suloma, A., 2013. Application of new strategies to reduce suspended solids in zero-exchange system: I. Histological alterations in the gills of Nile tilapia. J. appl. Sci. Res., 9: 1186-1192.</p>
<p class="articleParagraph enarticleParagraph" >Tadesse, Z., 1999. The nutritional status and digestibility of Oreochromis niloticus L. diet in Lake Langeno, Ethiopia. Hydrobiologia, 416: 97-106. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1023/A:1003807318933</p>
<p class="articleParagraph enarticleParagraph" >Tan, C., Sun, D., Tan, H., Liu, W., Luo, G. and Wei, X., 2018. Effects of stocking density on growth, body composition, digestive enzyme levels and blood biochemical parameters of Anguilla marmorata in a recirculating aquaculture system. Turk. J. Fish. aquat. Sci., 18: 9-16.</p>
<p class="articleParagraph enarticleParagraph" >Uddin, S., Ekram, U., Azim, M., Wahab, A. and Verdegem, M.C., 2006. The potential of mixed culture of genetically improved farmed tilapia (Oreochromis niloticus) and freshwater giant prawn (Macrobrachium rosenbergii) in periphyton based systems. Aquacult. Res., 37: 241-247. https:// doi.org/10.1111/j.1365-2109.2005.01424.x</p>
<p class="articleParagraph enarticleParagraph" >Vincent, Y.R., 2006. Use of gill condition to assess welfare of tilapia raised in two intensive production systems. M.Sc. thesis, <span class="companylink">University of Stirling</span>, UK.</p>
<p class="articleParagraph enarticleParagraph" >Wang, J., Tao, Q., Wang, Z., Mai, K., Xu, W., Zhang, Y. and Ai, Q., 2017. Effects of fish meal replacement by soybean meal with supplementation of functional compound additives on intestinal morphology and microbiome of Japanese seabass (Lateolabrax japonicus). Aquacult. Res., 48: 2186-2197. https:// doi.org/10.1111/are.13055</p>
<p class="articleParagraph enarticleParagraph" >Won, S., Hamidoghli, A., Choi, W., Bae, J., Jang, W.J., Lee, S. and Bai, S.C., 2020. Evaluation of potential probiotics Bacillus subtilis WB60, Pediococcus pentosaceus, and Lactococcus lactis on growth performance, immune response, gut histology and immune-related genes in whiteleg shrimp, Litopenaeus vannamei. Microorganisms, 8: 281.</p>
<p class="articleParagraph enarticleParagraph" >Xu, W.J. and Pan, L.Q., 2012. Effects of bioflocs on growth performance, digestive enzyme activity and body composition of juvenile Litopenaeus vannamei in zero-water exchange tanks manipulating C/N ratio in feed. Aquaculture, 356: 147-152. https:// doi.org/10.1016/j.aquaculture.2012.05.022</p>
<p class="articleParagraph enarticleParagraph" >Zheng, X., Duan, Y., Dong, H. and Zhang, J., 2018. Effects of dietary Lactobacillus plantarum on growth performance, digestive enzymes and gut morphology of Litopenaeus vannamei. Probiot. Antimicrobe. Proteins, 10: 504-510. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1007/s12602-017-9300-z</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i0 : Agriculture | i01001 : Farming | i03001 : Aquaculture</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gbiol : Biology | gcat : Political/General News | gsci : Sciences/Humanities</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>asiaz : Asia | dvpcoz : Developing Economies | pakis : Pakistan | sasiaz : South Asia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>The Zoological Society of Pakistan</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document ASZOOG0020251203elcv0000z</td></tr></table><br/></div></div><br/><span></span><div id="article-ASZOOG0020251203elcv00004" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/aszoogLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>In-Vitro and In-Vivo Antibacterial Effects of Saxifraga umbellulata var. Pectinata on Escherichia coli Isolated from Yaks</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>Kuanhui Liu, Shihui Xie, Ting Li, Reem M. Aljowaie, Mohamed S. Elshikh and Kun Li </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>4095 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>31 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Pakistan Journal of Zoology</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>ASZOOG</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>2535</td></tr><tr><td align="right" valign="top" class="index"><b>VOL</b>&nbsp;</td><td>57</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025. Zoological Society of Pakistan </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Key words</p>
<p class="articleParagraph enarticleParagraph" >Anti-microbial, Diarrhea, Escherichia coli, MIC, MBC, Saxifraga umbellulata var. Pectinata, Yak</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >INTRODUCTION</p>
<p class="articleParagraph enarticleParagraph" >Yaks are economically important and essential bovine ruminant on the Qinghai-Xizang plateau (Li et al., 2023; Chen et al., 2022). These animals provide milk, meat, fur or related products and as well as serving as means of transport for native people (Lu et al., 2023; Wang et al., 2023). However, diarrhea poses a frequent problem in these animals, leading to severe losses and affecting ruminant health (Li et al., 2022). Diarrhea in cattle, especially calf diarrhea is a world-wide issue in the cattle industry (Choi et al., 2021; Rasheed et al., 2022; Anwar et al., 2022). Previous studies have confirmed that pathogens such as viruses (bovine viral diarrhea virus, torovirus), bacteria (Escherichia coli, Salmonella spp.), and parasites (Cryptosporidium parvum, Giardia duodenalis) are factors that lead to diarrhea in animals (Chang et al., 2021; Shi et al., 2020; Gelalcha et al., 2022; Arsenault et al., 2022; Ali et al., 2024; Taghipour et al., 2022).</p>
<p class="articleParagraph enarticleParagraph" >Among them, E. coli is a commonly detected bacterial pathogen that bring severe challenges to public and livestock health (Frankel and Ron, 2018; Li et al., 2023). Moreover, an increasing number of multi-resistant bacteria have been isolated from food animals, further complicating the issue due to limited available antibacterial agents (Roth et al., 2019; Ullah et al., 2023).</p>
<p class="articleParagraph enarticleParagraph" >Traditional Chinese medicines are valuable and effective means for curing diseases and enhancing health (Chi et al., 2021). Among them, many herbs have antibacterial effects, such as Andrographis paniculata, Sanguisorba officinalis L. and garlic (Dai et al., 2019; Zhou et al., 2021; Tesfaye, 2021). The long history of Xizang medicine has integrated medical systems developed from traditional Chinese medicine and other medicines like Arabian medicine, which have greatly contributed to the health of plateau herdsmen (Huang et al., 2023).</p>
<p class="articleParagraph enarticleParagraph" >The Xizang medicine of Saxifraga umbellulata var. Pectinata (SUP) is a representative plateau perennial herb, which belongs to Saxifragaceae family. In regions with altitude over 3 km, this herb is a recognized traditional Xizang medicine named Songdi used for treating liver and gallbladder diseases as well as digestive diseases (Huang et al., 2023). Previous reports have indicated that SUP extraction has antibacterial activity, hepatoprotective effect (Huang et al., 2023). However, not much information is available about the antibacterial effect of SUP on E. coli isolated from yaks. Therefore, we conducted this study to investigate the in vitro and in vivo antibacterial effect of SUP on E. coli isolated from yaks.</p>
<p class="articleParagraph enarticleParagraph" >MATERIALS AND METHODS</p>
<p class="articleParagraph enarticleParagraph" >Bacterial isolate</p>
<p class="articleParagraph enarticleParagraph" >Multi-drug resistant E. coli was previously isolated from diarrhea yaks and stored in the clinical veterinary laboratory of Nanjing Agricultural University. This bacterium was resistant to penicillin, ampicillin, erythromycin, tetracycline, streptomycin and gentamicin.</p>
<p class="articleParagraph enarticleParagraph" >Preparation of the SUP extracts solution</p>
<p class="articleParagraph enarticleParagraph" >SUP (200 g) was obtained from Tibetan medicine factory (Lhasa, China). Ethyl acetate extraction of SUP was performed according to previously described protocols (Hou et al., 2022; Liu et al., 2022). Initially the herbs were crushed and soaked with 2 L of ethanol (75%) overnight. The following day, herbs were heated to 85 °C and subjected to reflux extraction for 2h, followed by filtration. The SUP herbs were extracted and filtered three times and all the products were mixed together. At last, the mixed SUP products were dried and reconstituted in sterile water at 1g/mL for further use.</p>
<p class="articleParagraph enarticleParagraph" >In vivo antibacterial effect of SUP on E. coli</p>
<p class="articleParagraph enarticleParagraph" >The anti-E. coli activity of SUP was examined via commonly utilized methods of minimum inhibitory concentration (MIC) and minimum bactericidal concentration (MBC) detection (Han and Guo, 2012; Parvekar et al., 2020). In a 96-well plate, 100 µL of SUP at various concentrations (0, 512 mg/mL, 256 mg/mL, 128 mg/mL, 64 mg/mL, 32 mg/mL, 16 mg/mL, 8 mg/mL, 4 mg/mL, 2 mg/mL, 1 mg/mL, 0.5 mg/mL and 0.25 mg/mL) was added to an equal volume of bacteria solution (106 CFU/mL). Subsequently, 50 µL of LB medium (Hangzhou Binhe Microorganism Reagent Co., Ltd, China) was added to each well and the plate was incubated at 37 °C for 18 h. The MIC endpoint was determined when no visible growth of E. coli was observed in the well. To determine the MBC, 100 µL of medium from wells before and after MIC well, and MIC well plated onto LB agar plates and incubated at 37 °C overnight. The MBC endpoint was reached when the plates had fewer than 0.25 x 105 CFU (0.1%) colonies.</p>
<p class="articleParagraph enarticleParagraph" >Next, we evaluated the growth inhibition curve of E. coli in response to different doses of SUP. In sterile tubes containing 4.90 mL LB medium, 50 µL of bacteria solution (106 CFU/mL) was added along with 50 µL of SUP at concentration of 0, MIC, 2MIC and 4MIC mg/mL. Each concentration had 18 independent repeat tubes. The tubes were then incubated at 37 °C in shaker and at time point of 0, 2h, 4h, 6h, 8h, 10h and 12h, three tubes of each concentration were sampled to check OD600 value.</p>
<p class="articleParagraph enarticleParagraph" >The effect of SUP on the biological film and membrane permeability of E. coli</p>
<p class="articleParagraph enarticleParagraph" >The effect of SUP on the biofilm was assessed using crystal violet staining (Bai et al., 2022). Ina 96 well plate, 100 µL of E. coli (OD600 =0.05) was mixed with 100 µL of SUP (4MIC) and incubated at 37 °C for 6 h and 12 h. Then the OD620 value was measured, and the plate was washed by <span class="companylink">PBS</span> three times and fixed with 200 µL methanol for 15 min. Finally, the plate was washed with <span class="companylink">PBS</span> for three times and stained with 0.1% crystal violet for 30 min. Finally, the plate was washed with <span class="companylink">PBS</span> and 200 ul of 30% glacial acetic acid added for measuring absorbance at 540 nm. Three independent repeats were performed for all wells, and equal volume of sterile water was added to control wells. The ratio value of OD540 /OD620 was calculated to determine the effect of SUP on the biofilm.</p>
<p class="articleParagraph enarticleParagraph" >To evaluate the effect of SUP on the membrane permeability of E. coli, 100 uL bacteria solution (106 CFU/mL) was mixed with SUP (4MIC) in a 96-well plate, and then incubated at 37 °C. The OD260 and OD280 values of supernatant were examined at 0, 2, 4 and 8 h. Three independent repeats were set for all wells, and an equal volume of sterile water was added in control wells.</p>
<p class="articleParagraph enarticleParagraph" >Animal experiments</p>
<p class="articleParagraph enarticleParagraph" >A total of 39 male Kunming mice (five weeks, average weight of 23.5±1.3 g) were obtained from Qinglongshan animal breeding (Nanjing, China). The mice were reared and housed in the animal facility having free access to feed and water. All of the mice were given three days for acclimatization and then grouped into control (C), infection (I) and treatment (T) groups. Mice in group I and T were intra-peritoneally infected with a bacteria solution (107 CFU/mL), while group T received treatment with SUP (200 mg/Kg) for three days. Group C and I were treated with an equal volume of sterile water. On the second day, three mice from each group were euthanized to collect organ and intestines samples. Daily weights and mortality of mice were recorded. Those collected tissues were ground with sterile <span class="companylink">PBS</span>, and then used for bacteria culture on LB agar plate. Colony forming units (CFUs) were counted to analyze tissue bacterial loads. Additionally, the jejunum was employed for pathologic analysis.</p>
<p class="articleParagraph enarticleParagraph" >Pathologic analysis</p>
<p class="articleParagraph enarticleParagraph" >The jejunum from Kunming animals was fixed in paraformaldehyde (4%) and then subjected to H and E staining in Pinuofei Biological Technology (Wuhan, China). An Olympus CX23 microscope (Olympus Co., Japan) was used for pathologic analysis. Statistical analysis of villus height and crypt depth of mice in C, I and T were performed.</p>
<p class="articleParagraph enarticleParagraph" >Statistical analysis</p>
<p class="articleParagraph enarticleParagraph" >Non-parametric tests were conducted using <span class="companylink">IBM</span> SPSS (27.0) software. Data are presented as means ± SD and statistical significance was determined at P &lt; 0.05.</p>
<p class="articleParagraph enarticleParagraph" >RESULTS</p>
<p class="articleParagraph enarticleParagraph" >In vitro antibacterial effect of SUP on E. coli</p>
<p class="articleParagraph enarticleParagraph" >The MIC and MBC of SUP anti-against E. coli were 8 mg/mL and 16 mg/mL, respectively. The growth inhibition curve clearly demonstrated that SUP significantly inhibited the growth of E. coli, indicating a dose-dependent bactericidal effect (Fig. 1).</p>
<p class="articleParagraph enarticleParagraph" >Crystal violet staining showed that SUP significantly inhibited the formation of E. coli biofilm at 6 h (P&lt;0.0001) and 12 h (P&lt;0.0001) (Fig. 2A). Membrane permeability analysis of E. coli revealed that SUP markedly increased bacterial leakage at 4 h (P&lt;0.0001) and 8 h (P&lt;0.0001) (Fig. 2B).</p>
<p class="articleParagraph enarticleParagraph" >In vivo antibacterial effect of SUP on E. coli</p>
<p class="articleParagraph enarticleParagraph" >Animal study revealed that E. coli infection led to mice mortality within 2-24h, while treatment with SUP saved the animals lives (Fig. 3A). Weight analysis showed that E. coli caused weight loss in mice, while animals treated with SUP showed slightly higher body weights (Fig. 3B). Pathologic analysis indicated that E. coli obviously disrupted the integrity of intestinal villi in mice, whereas SUP alleviated intestinal damage in animals (Fig. 3B). Villus height (P&lt;0.0001) and the ratio of villus height/crypt depth (P&lt;0.001) in group I were markedly lower than in group C, while crypt depth in group I was significantly higher (P&lt;0.0001). However, animals treated with SUP presented noticeably increased villus height (P&lt;0.0001), ratio of villus height/crypt depth (P&lt;0.0001), and decreased crypt depth (P&lt;0.001) (Fig. 3B).</p>
<p class="articleParagraph enarticleParagraph" >Organ bacterial loads showed that E. coli loads in the heart (P&lt;0.5), liver (P&lt;0.001), spleen (P&lt;0.001), lung (P&lt;0.001), kidney (P&lt;0.001), duodenum (P&lt;0.001), jejunum (P&lt;0.001), ileum (P&lt;0.001), cecum (P&lt;0.01), colon (P&lt;0.01) and rectum (P&lt;0.001) in group I were significantly increased. Interestingly, animals fed with SUP exhibited markedly lower bacteria loads in heart (P&lt;0.5), liver (P&lt;0.01), spleen (P&lt;0.5), lung (P&lt;0.001), duodenum (P&lt;0.05), jejunum (P&lt;0.01), ileum (P&lt;0.01), cecum (P&lt;0.01), colon (P&lt;0.01) and rectum (P&lt;0.05) (Fig. 4).</p>
<p class="articleParagraph enarticleParagraph" >DISCUSSION</p>
<p class="articleParagraph enarticleParagraph" >Cattle are important food-producing ruminants, providing nutritious products for citizens. Therefore, cattle disease not only harm animal health, but potentially threaten the protein food supply. Especially on the cold plateaus, yaks are crucial food resources (Li and Liu, 2022). E. coli is a common opportunistic pathogen causing diarrhea, resulting in significant economic losses to farming industry due to medical costs and animal deaths (Zhang et al., 2022). With antibiotic abuse in veterinary and medical practices, drug-resistant E. coli has been detected in the environment, animals and people (Hu and Cheng, 2016). There is an urgent need to screen novel antibacterial drugs with fewer side effects.</p>
<p class="articleParagraph enarticleParagraph" >Medicinal herbs have been popularly used for thousands of years due to their antimicrobial properties, offering promising alternatives to conventional antibacterial drugs (Alanazi et al., 2023). Previous research has found herbs such as Chrysanthemum, Lagotis brachystachya and Oak bark for their anti-E. coli effects (Kim et al., 2013; Hou et al., 2022; Šukele et al., 2022). Consistent with these findings, our study confirmed that SUP could inhibit the growth of multi-drug resistant E. coli from yaks both in vitro and in vivo (Figs. 1, 3). In vitro studies showed that SUP could inhibit E. coli at 8 mg/mL, with MBC was 16 mg/mL. The growth inhibition curve indicated that SUP (32 mg/mL) could nearly completely inhibited bacteria growth (Fig. 1).</p>
<p class="articleParagraph enarticleParagraph" >Similar to a previous a study reported gut injuries caused by E. coli (Ismael et al., 2023), the strain of E. coli isolated from yaks proved lethal to mice causing severe intestine damage. However, our in vivo results showed that SUP decreased ice mortality by mitigating intestine damages and reducing bacteria loads (Figs. 3, 4).</p>
<p class="articleParagraph enarticleParagraph" >Furthermore, we investigated the anti-E. coli mechanism of SUP by examining the biofilm and membrane permeability of E. coli. Biofilms consist of numerous bacterial cells aggregated together with extracellular matrix, which can shield bacteria from antibacterial agents and confer resistance to drugs (Lu et al., 2021), as well as protect from the host immune system (Roy et al., 2018). Following treatment for 6 h and 12 h, SUP significantly inhibited the biofilm formation (Fig. 2A), suggesting that SUP may hinder E. coli survival by impeding the biofilm formation. Our results are consistent with previous studies that have shown various herbs can inhibit the bacterial biofilm formation (Hou et al., 2022; Lu et al., 2019). Membrane integrity and permeability are crucial fo r bacteria growth (Yang et al., 2021), as a compromised membrane can result in the leakage of cell contents (Xu et al., 2017).</p>
<p class="articleParagraph enarticleParagraph" >In this study, proteins and nucleic acids were detected in E. coli treated with SUP (Fig. 2B), indicating that SUP could increase the permeability of E. coli membrane.</p>
<p class="articleParagraph enarticleParagraph" >CONCLUSION</p>
<p class="articleParagraph enarticleParagraph" >In this study, we demonstrated that Saxifraga umbellulata var. Pectinata could inhibit E. coli in vitro and in vivo by affecting the biological film and membrane permeability of bacteria. These findings provide insights that could potentially lead to develop novel anti-E. coli drugs or prevent measures for diarrhea in plateau yaks.</p>
<p class="articleParagraph enarticleParagraph" >DECLARATIONS</p>
<p class="articleParagraph enarticleParagraph" >Acknowledgement</p>
<p class="articleParagraph enarticleParagraph" >The authors extend their appreciation to the Researchers Supporting Project number (RSP2024R418), <span class="companylink">King Saud University</span>, Riyadh, Saudi Arabia.</p>
<p class="articleParagraph enarticleParagraph" >Funding</p>
<p class="articleParagraph enarticleParagraph" >The current study was funded by the Natural Science Research Project of Education Department of Anhui Province “Study on the preparation and application in animal production of immune adjuvant nanoparticles of CP-PLGA” (Grant No. KJ2021A1334).</p>
<p class="articleParagraph enarticleParagraph" >Ethical statement</p>
<p class="articleParagraph enarticleParagraph" >All the experiment procedures were conducted in accordance with the guidelines and approval of the Ethics Committee of Nanjing Agricultural University (NJAU. No20240226021).</p>
<p class="articleParagraph enarticleParagraph" >Statement of conflict of interest</p>
<p class="articleParagraph enarticleParagraph" >The authors have declared no conflict of interest.</p>
<p class="articleParagraph enarticleParagraph" >REFERENCES</p>
<p class="articleParagraph enarticleParagraph" >Alanazi, H.H., Elasbali, A.M., Alanazi, M.K. and El-Azab, E.F., 2023. Medicinal herbs: Promising immunomodulators for the treatment of infectious diseases. Molecules, 28: 8045. <span class="colorLinks">https://doi [https://doi]</span>. org/10.3390/molecules28248045</p>
<p class="articleParagraph enarticleParagraph" >Ali, M., Xu, C., Nawaz, S., Ahmed, A.E., Hina, Q. and Li, K., 2024. Anti-cryptosporidial drug-discovery challenges and existing therapeutic avenues: A one-health concern. Life, 14: 80. <span class="colorLinks">https://doi [https://doi]</span>. org/10.3390/life14010080</p>
<p class="articleParagraph enarticleParagraph" >Anwar, M.A., Aziz, S., Ashfaq, K., Aqib, A.I., Shoaib, M., Naseer, M.A., Alvi, M.A., Muzammil, I., Bhutta, Z.A., Sattar, H., Saleem, A., Zaheer, T., Khanum, F. and Mahmood, A., 2022. Trends in frequency, potential risks, and antibiogram of E. coli isolated from semi-intensive dairy systems. Pak. Vet. J., 42: 167-172.</p>
<p class="articleParagraph enarticleParagraph" >Arsenault, R.J., Brown, T.R., Edrington, T.S. and Nisbet, D.J., 2022. Kinome analysis of cattle peripheral lymph nodes to elucidate differential response to Salmonella spp. Microorganisms, 10: 120. https:// doi.org/10.3390/microorganisms10010120</p>
<p class="articleParagraph enarticleParagraph" >Bai, Y., Wang, W., Shi, M., Wei, X., Zhou, X., Li, B. and Zhang, J., 2022. Novel antibiofilm inhibitor ginkgetin as an antibacterial synergist against Escherichia coli. Int. J. Mol. Sci., 23: 8809. https:// doi.org/10.3390/ijms23158809</p>
<p class="articleParagraph enarticleParagraph" >Chang, L., Qi, Y., Liu, D., Du, Q., Zhao, X. and Tong, D., 2021. Molecular detection and genotyping of bovine viral diarrhea virus in Western China. BMC Vet. Res., 17. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1186/s12917-021-02747-7 [https://doi-org.ezproxy.cul.columbia.edu/10.1186/s12917-021-02747-7]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Chen, X., Saeed, N.M., Ding, J., Dong, H., Kulyar, M.F.E.A., Bhutta, Z.A., Mehmood, K., Ali, M.M., Irshad, I., Zeng, J., Liu, J., Wu, Q. and Li, K., 2022. Molecular epidemiological investigation of Cryptosporidium sp., Giardia duodenalis, Enterocytozoon bieneusi and Blastocystis sp. infection in free-ranged yaks and tibetan pigs on the plateau. Pak. Vet. J., 42: 533-539. <span class="colorLinks">https://doi [https://doi]</span>. org/10.29261/pakvetj/2022.060</p>
<p class="articleParagraph enarticleParagraph" >Chi, J., Sun, L., Cai, L., Fan, L., Shao, C., Shang, L. and Zhao, Y., 2021. Chinese herb microneedle patch for wound healing. Bioactive Mater., 6: 3507-3514. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j.bioactmat.2021.03.023 [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j.bioactmat.2021.03.023]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Choi, K., Kang, J., Cho, H., Yu, D. and Park, J., 2021. Changes in serum protein electrophoresis profiles and acute phase proteins in calves with diarrhea. Can. J. Vet. Res., 85: 45-50.</p>
<p class="articleParagraph enarticleParagraph" >Dai, Y., Chen, S.R., Chai, L., Zhao, J., Wang, Y. and Wang, Y., 2019. Overview of pharmacological activities of Andrographis paniculata and its major compound andrographolide. Crit. Rev. Fd. Sci. Nutr., 59(Supp. 1): S17-S29. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.10 [https://doi-org.ezproxy.cul.columbia.edu/10.10]</span> 80/10408398.2018.1501657</p>
<p class="articleParagraph enarticleParagraph" >Frankel, G. and Ron, E.Z., 2018. Escherichia coli, a versatile pathogen (eds. G. Frankel and E.Z. Ron). Vol. 416;416. Springer, Cham, Switzerland. https:// doi.org/10.1007/978-3-319-99664-6</p>
<p class="articleParagraph enarticleParagraph" >Gelalcha, B.D., Ensermu, D.B., Agga, G.E., Vancuren, M., Gillespie, B.E., D'Souza, D.H., Okafor, C.C. and Dego, O.K., 2022. Prevalence of antimicrobial resistant and extended-spectrum beta-lactamase-producing Escherichia coli in dairy cattle farms in East Tennessee. Foodb. Pathog. Dis., 19: 408-416. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1089/fpd.2021.0101 [https://doi-org.ezproxy.cul.columbia.edu/10.1089/fpd.2021.0101]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Han, C. and Guo, J., 2012. Antibacterial and anti-inflammatory activity of traditional Chinese Herb Pairs, Angelica sinensis and Sophora flavescens. Inflammation, 35: 913-919. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1007/ [https://doi-org.ezproxy.cul.columbia.edu/10.1007/]</span> s10753-011-9393-6</p>
<p class="articleParagraph enarticleParagraph" >Hou, S., Guo, J., Liu, L., Qiu, F. and Liu, X., 2022. Antibacterial and antibiofilm activity of Lagotis brachystachya extract against extended-spectrum b-lactamases-producing Escherichia coli from broiler chickens. Poult. Sci., 101: 101555. https:// doi.org/10.1016/j.psj.2021.101555</p>
<p class="articleParagraph enarticleParagraph" >Hu, Y. and Cheng, H., 2016. Health risk from veterinary antimicrobial use in China's food animal production and its reduction. Environ. Pollut., 219: 993-997. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j.envpol.2016.04.099 [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j.envpol.2016.04.099]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Huang, J., Chen, D., Liu, M., Yu, Y., Zhang, Y. and Huang, J., 2023. Seven new phenylhexanoids with antioxidant activity from Saxifraga umbellulata var. Pectinata. Molecules, 28: 3928. <span class="colorLinks">https://doi [https://doi]</span>. org/10.3390/molecules28093928</p>
<p class="articleParagraph enarticleParagraph" >Ismael, M., Qayyum, N., Gu, Y., Zhezhe, Y., Cui, Y., Zhang, Y. and Lu, X., 2023. Protective effect of plantaricin bio-LP1 bacteriocin on multidrug-resistance Escherichia coli infection by alleviate the inflammation and modulate of gut-microbiota in BALB/c mice model. Int. J. Biol. Macromol., 246: 125700. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j]</span>. ijbiomac.2023.125700</p>
<p class="articleParagraph enarticleParagraph" >Kim, K.S., Lim, D.J., Yang, H.J., Choi, E.K., Shin, M.H., Ahn, K.S., Jung, S.H., Um, J.Y., Jung, H.J., Lee, J.H., Lee, S.G., Jung, S.K. and Jang, H.J., 2013. The multi-targeted effects of chrysanthemum herb extract against Escherichia coli O157:H7. Phytother. Res., 27: 1398-1406. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1002/ptr.4859</p>
<p class="articleParagraph enarticleParagraph" >Li, X., Zhu, X. and Xue, Y., 2023. Drug resistance and genetic relatedness of Escherichia coli from mink in Northeast China. Pak. Vet. J., 43: 824-827.</p>
<p class="articleParagraph enarticleParagraph" >Li, B., Zhang, L., Wang, L., Wei, Y., Guan, J., Mei, Q. and Hao, N., 2023. Antimicrobial activity of yak beta-defensin 116 against Staphylococcus aureus and its role in gut homeostasis. Int. J. Biol. Macromol., 253: 126761. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j]</span>. ijbiomac.2023.126761</p>
<p class="articleParagraph enarticleParagraph" >Li, K., Zeng, Z., Liu, J., Pei, L., Wang, Y., Li, A., Kulyar, M.F., Shahzad, M., Mehmood, K., Li, J. and Qi, D., 2022. Effects of short-chain fatty acid modulation on potentially diarrhea-causing pathogens in yaks through metagenomic sequencing. Front. Cell. Infect. Microbiol., 12. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.3389/ [https://doi-org.ezproxy.cul.columbia.edu/10.3389/]</span> fcimb.2022.805481</p>
<p class="articleParagraph enarticleParagraph" >Li, S. and Liu, S., 2022. Estimation of the proteome affecting changes in tenderness of yak meat during storage by label-free mass spectrometry. Vet. Med. Sci., 8: 1640-1649. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1002/ [https://doi-org.ezproxy.cul.columbia.edu/10.1002/]</span> vms3.801</p>
<p class="articleParagraph enarticleParagraph" >Liu, Z., Wang, H., Li, C., Yang, J., Suo, Q., Zhou, Y. and Qie, R., 2022. Ethyl acetate extract of Caesalpinia sappan L. for the treatment of atherosclerosis in ApoE-/- mice and its mechanism. Mol. Omics, 18: 977-990. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1039/D2MO00254J [https://doi-org.ezproxy.cul.columbia.edu/10.1039/D2MO00254J]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Lu, C., Liu, H., Shangguan, W., Chen, S. and Zhong, Q., 2021. Antibiofilm activities of the cinnamon extract against Vibrio parahaemolyticus and Escherichia coli. Arch. Microbiol., 203: 125-135. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1007/s00203-020-02008-5</p>
<p class="articleParagraph enarticleParagraph" >Lu, L., Hu, W., Tian, Z., Yuan, D., Yi, G., Zhou, Y., Cheng, Q., Zhu, J. and Li, M., 2019. Developing natural products as potential anti-biofilm agents. Chinese Med., 14. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1186/s13020-019-0232-2 [https://doi-org.ezproxy.cul.columbia.edu/10.1186/s13020-019-0232-2]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Lu, S., Zou, W., Chen, X., Sun, G., Cidan, Y., Almutairi, M.H., Dunzhu, L., Nazar, M., Mehmood, K., Zhu, Y., Basang, W. and Li, K., 2023. Effects of Cryptosporidium parvum infection on intestinal fungal microbiota in yaks (Bos grunniens). Microb. Pathogen., 183: 106322. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j]</span>. micpath.2023.106322</p>
<p class="articleParagraph enarticleParagraph" >Parvekar, P., Palaskar, J., Metgud, S., Maria, R. and Dutta, S., 2020. The minimum inhibitory concentration (MIC) and minimum bactericidal concentration (MBC) of silver nanoparticles against Staphylococcus aureus. Biomater. Invest. Dent., 7: 105-109. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1080/264152 [https://doi-org.ezproxy.cul.columbia.edu/10.1080/264152]</span> 75.2020.1796674</p>
<p class="articleParagraph enarticleParagraph" >Rasheed, M.B., Ahsan, A., Irshad, H., Shahzad, M.A., Usman, M., Riaz, A., Chaudhry, T.H., Amir, A., Zubair, M, Khan, A. and Yousaf, A., 2023. Occurrence of Shiga toxin-producing E. coli in zoo animals of Rawalpindi and Islamabad zoos. Asian J. Agric. Biol., 2023: 2022080.</p>
<p class="articleParagraph enarticleParagraph" >Roth, N., Käsbohrer, A., Mayrhofer, S., Zitz, U., Hofacre, C. and Domig, K.J., 2019. The application of antibiotics in broiler production and the resulting antibiotic resistance in Escherichia coli: A global overview. Poult. Sci., 98: 1791-1804. <span class="colorLinks">https://doi [https://doi]</span>. org/10.3382/ps/pey539</p>
<p class="articleParagraph enarticleParagraph" >Roy, R., Tiwari, M., Donelli, G. and Tiwari, V., 2018. Strategies for combating bacterial biofilms: A focus on anti-biofilm agents and their mechanisms of action. Virulence, 9: 522-554. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.10 [https://doi-org.ezproxy.cul.columbia.edu/10.10]</span> 80/21505594.2017.1313372</p>
<p class="articleParagraph enarticleParagraph" >Shi, Z., Wang, W., Chen, C., Zhang, X., Wang, J., Xu, Z. and Lan, Y., 2020. First report and genetic characterization of bovine torovirus in diarrhoeic calves in China. BMC Vet. Res., 16. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1186/s12917-020-02494-1</p>
<p class="articleParagraph enarticleParagraph" >Šukele, R., Skadinš, I., Koka, R. and Bandere, D., 2022. Antibacterial effects of oak bark (Quercus robur) and heather herb (Calluna vulgaris L.) extracts against the causative bacteria of bovine mastitis. Vet. World, 2022: 2315-2322. <span class="colorLinks">https://doi [https://doi]</span>. org/10.14202/vetworld.2022.2315-2322</p>
<p class="articleParagraph enarticleParagraph" >Taghipour, A., Sharbatkhori, M., Tohidi, F., Ghanbari, M.R., Karanis, P., Olfatifar, M., Majidiani, H., Khazaei, S., Bahadory, S. and Javanmard, E., 2022. Global prevalence of Giardia duodenalis in cattle: A systematic review and meta-analysis. Prevent. Vet. Med., 203:105632. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1016/j [https://doi-org.ezproxy.cul.columbia.edu/10.1016/j]</span>. prevetmed.2022.105632</p>
<p class="articleParagraph enarticleParagraph" >Tesfaye, A., 2021. Revealing the therapeutic uses of garlic (Allium sativum) and its potential for drug discovery. Sci. World J., 2021: 1-7. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1155/2021/8817288</p>
<p class="articleParagraph enarticleParagraph" >Ullah, M., Rasool, F., Khan, N., Ali, S. and Sheikh, A.A., 2023. Antibiotic resistance and its gene profile in Escherichia coli isolated from diseased farm-raised carps in Punjab, Pakistan. Pak. Vet. J., 43: 470-476.</p>
<p class="articleParagraph enarticleParagraph" >Wang, S., Cao, Z., Wu, Q., Ai, M.H.A. and Dong, H., 2023. A comparative analysis and verification of differentially expressed miRNAs could provide new insights for the treatment of endometritis in yaks. Pak. Vet. J., 43: 486-492.</p>
<p class="articleParagraph enarticleParagraph" >Xu, C., Li, J., Yang, L., Shi, F., Yang, L. and Ye, M., 2017. Antibacterial activity and a membrane damage mechanism of Lachnum YM30 melanin against Vibrio parahaemolyticus and Staphylococcus aureus. Fd. Contr., 73: 1445-1451. <span class="colorLinks">https://doi [https://doi]</span>. org/10.1016/j.foodcont.2016.10.048</p>
<p class="articleParagraph enarticleParagraph" >Yang, H., Gao, Y., Long, L., Cai, Y., Liao, J., Peng, J. and Wang, L., 2021. Antibacterial effect of Blumea balsamifera (L.) DC. essential oil against Staphylococcus aureus. Arch. Microbiol., 203: 3981-3988. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.1007/s00203-021-02384-6 [https://doi-org.ezproxy.cul.columbia.edu/10.1007/s00203-021-02384-6]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Zhang, Q., Wang, M., Ma, X., Li, Z., Jiang, C., Pan, Y. and Zeng, Q., 2022. In vitro investigation on lactic acid bacteria isolated from Yak faeces for potential probiotics. Front. Cell. Infect. Microbiol., 12. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.3389/fcimb.2022.984537 [https://doi-org.ezproxy.cul.columbia.edu/10.3389/fcimb.2022.984537]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Zhou, P., Li, J., Chen, Q., Wang, L., Yang, J., Wu, A., Jiang, N., Liu, Y., Chen, J., Zou, W., Zeng, J. and Wu, J., 2021. A comprehensive review of genus Sanguisorba: Traditional uses, chemical constituents and medical applications. Front. Pharmacol., 12. <span class="colorLinks">https://doi-org.ezproxy.cul.columbia.edu/10.3389/ [https://doi-org.ezproxy.cul.columbia.edu/10.3389/]</span>
                  </p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gbiol : Biology | gcat : Political/General News | gchlra : Infectious Foodborne/Waterborne Diseases | gecol : E. Coli Infections | ghea : Health | gmed : Medical Conditions | gsci : Sciences/Humanities | gspox : Infectious Diseases</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | asiaz : Asia | china : China | chinaz : Greater China | devgcoz : Emerging Market Countries | dvpcoz : Developing Economies | easiaz : East Asia | jiangs : Jiangsu | yunna : Yunnan</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>The Zoological Society of Pakistan</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document ASZOOG0020251203elcv00004</td></tr></table><br/></div></div><br/><span></span><div id="article-FORAI00020251206elci00018" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/foraiLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>ZELENSKY’S NEW SOLUTION</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>2216 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>18 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>AirForces Monthly</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>FORAI</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2025. Key Publishing Ltd. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Mina Adel looks at the rise of agile warfare in the skies over Eastern Europe</p>
<p class="articleParagraph enarticleParagraph" >Gripens for Ukr+aine</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >“For me personally, the Gripen is the only fighter jet in the world for which I would willingly sell my soul even trade my one true love, the MiG-29.” So said renowned Ukrainian fighter pilot Vadym Voroshylov, known by his callsign ‘Karaya’, on his <span class="companylink">Instagram</span> account.</p>
<p class="articleParagraph enarticleParagraph" >On October 22, Ukrainian President Volodymyr Zelensky signed a letter of intent (LOI) with Swedish Prime Minister Ulf Kristersson and stated: “Ukraine will significantly increase its combat aviation numbers. This is an ambitious task and it must be fulfilled. A historic step has been taken now – an agreement with Sweden on Gripen fighter aircraft, and that’s a good choice. We are counting on 150 such aircraft for Ukraine, and the first are expected to arrive next year.</p>
<p class="articleParagraph enarticleParagraph" >“Gripens for Ukraine are part of our security guarantees – with an air force capable of fully protecting our skies. There has never been a combat-aviation deal of this scale for Ukraine before.” He concluded: “This is a historic achievement.”</p>
<p class="articleParagraph enarticleParagraph" >This raises a critical question: why is Ukraine so determined to acquire the Gripen? To explore possible answers, Air Forces Monthly reached out to former Gripen C pilot Mikel Grev and Pierre Chuet, an ex-Rafale M pilot.</p>
<p class="articleParagraph enarticleParagraph" >New threats need new fighters</p>
<p class="articleParagraph enarticleParagraph" >Ukraine received its first batch of US-built F-16 fighter jets on August 1, 2024, and later received its first Mirage 2000-5s from France on February 6, 2025</p>
<p class="articleParagraph enarticleParagraph" >These two fighter jets, specifically, rank among the most successful fourth-generation aircraft. For decades, <span class="companylink">NATO</span> relied on them to secure the skies and carry out a wide range of missions in harsh environments. They have undergone numerous critical upgrades that have kept them operational to this day.</p>
<p class="articleParagraph enarticleParagraph" >The Ukrainian Air Force had sought them since the beginning of the conflict for two main reasons: first, to counter Russian glide bombs such as the UMPK; second, due to their reliability and large numbers, to allow Ukraine to sustain air operations effectively. In practice, their entry into the battlefield has significantly enhanced the efficiency of air defence operations, helping to secure Ukrainian airspace and push back Russian fighter jets as much as possible. However, the Russian side responded with tactical adaptation, introducing technological modifications to its glide bombs to increase their combat effectiveness. The new variant, known as the UMPB-5R, is equipped with a Swiwin SW800Pro-Y engine (see Russia’s new menace: long range KABs , Dec 2025, p21), doubling the traditional targeting range from 62 to 124 miles. It also features a 12-antenna CRPA satellite navigation module (upgraded from the previous 8-antenna version) enhancing guidance and resistance to Ukrainian GPS/GLONASS jamming.</p>
<p class="articleParagraph enarticleParagraph" >With recent upgrades, Russian bombers now enjoy the luxury of releasing their payloads well beyond the reach of Ukrainian air defences, safely and with complete operational safety, under the protective umbrella of air superiority fighters such as the Su-35 and air defence interceptors such as the MiG-31. Both aircraft can launch the Vympel R-37 (AA-13 Axehead ) missile, which boasts a range of 250 miles, compared to the 50-mile range of the MICA missiles fired by Mirage 2000-5 jets and the AIM-120C-5 missiles launched from F-16AMs.</p>
<p class="articleParagraph enarticleParagraph" >This preserves Russia’s advantage in beyond-visual-range (BVR) engagements, allowing it to shoot down Ukrainian fighters from afar and limit their ability to operate at high altitudes.</p>
<p class="articleParagraph enarticleParagraph" >On the other hand, Ukraine’s air defence forces have been manoeuvring their batteries in the field to conduct advanced aerial ambushes (known as ‘SAMBUSH’) using systems like the American Patriot and the Soviet-era S-200 (SA-5 Gammon ). These flexible units have successfully downed several aircraft, including Su-35 Flanker-E , Su-34 Fullback , Mi-8 Hip helicopters, and early warning aircraft. However, such missions are hazardous, as they expose the batteries to detection and counterattacks by enemy artillery or even FPV drones, which operate heavily near the front lines.</p>
<p class="articleParagraph enarticleParagraph" >All these factors underscore Ukraine’s urgent need for a fighter jet capable of understanding and adapting to the fast-paced dynamics of the battlefield – one that can match Russian airpower and push it back more effectively. Such an aircraft would enable other Ukrainian jets to strike more aggressively near and behind Russian lines, providing vital air support to Ukrainian ground units. This requirement has become increasingly critical.</p>
<p class="articleParagraph enarticleParagraph" >However, before such operations can be fully realised, it is essential first to attack, neutralise, and dismantle Russia’s formidable, layered air defence network, which includes systems like the S-400 (SA-21 Growler ), S-300V (SA-10 Grumble ), Buk (SA-11 Gadfly ), and Pantsir (SA-22 Greyhound ).</p>
<p class="articleParagraph enarticleParagraph" >Fight smart, not hard</p>
<p class="articleParagraph enarticleParagraph" >The arrival of Gripen fighter jets to the battlefield in 2026 would be perfectly timed to maximize their operational impact – not only because they could offer solutions to many of the challenges currently facing the Ukrainian Air Force, thereby enhancing its combat effectiveness compared to previous phases, but also because more than a year will have passed since the introduction of the two Western fighter platforms into Ukrainian service.</p>
<p class="articleParagraph enarticleParagraph" >By then, pilots and operational commanders will have absorbed Western technologies, and a new generation of pilots trained in Europe under Western doctrine will have entered service. This will significantly improve their ability to integrate and operate advanced systems compared to the current generation of Ukrainian pilots. All these factors will create optimal conditions for effectively deploying the Swedish fighter. The Gripen NG, as the E/F versions were known, has a compact design and numerous airframe improvements. These include a larger wing area compared to the older C/D versions, enhancing its heavy-load-carrying capabilities with ten hard points and 30% more fuel for extended range. It possesses a wide array of lethal tools, such as the Leonardo ES-05 Raven active electronically scanned array (AESA) radar, providing a whole +100° field of regard, allowing maximum situational awareness and platform survivability. This Wide Field of Regard (WFoR) allows the aircraft to turn away after missile launch, whilst still maintaining datalinks to the missile, according to <span class="companylink">Leonardo</span>. In addition, the Leonardo Skyward G infrared search-and-track (IRST) sensor has been designed and developed as an embedded solution for fifth-generation fighter aircraft. <span class="companylink">Saab</span>’s Arexis Electronic Warfare (EW) integrates a cutting-edge system that combines a variety of offensive and defensive measures to disrupt enemy efforts while protecting itself, ensuring high survivability.</p>
<p class="articleParagraph enarticleParagraph" >Grev stated: “The Gripen E’s split software security system is designed to receive updates, keeping the fighter modern for longer and reducing lag against ever-changing threats. While many still focus on comparing thrust, maximum speed and the number of pylons, what truly matters is the ability to avoid being shot down while maximising the effectiveness of the weapons being fired. This can increase its combat effectiveness by several hundred per cent in beyond-visual-range scenarios.”</p>
<p class="articleParagraph enarticleParagraph" >This is why the Gripen is expected to offer Ukrainian pilots a new, more effective and safer way to engage Russian fighters and air defences – precisely what Ukraine needs in the future to deter and contain Russian airpower. The proposed deal includes 150 Gripen fighter jets to replace a wide range of ageing Soviet-era fighters and bombers, marking a transformative leap for the Ukrainian Air Force.</p>
<p class="articleParagraph enarticleParagraph" >In fact, Saab is prepared to open a final assembly plant in Ukraine as part of this deal, according to the UK’s Financial Times , signalling a long-term strategic partnership and a significant boost to Ukraine’s defence-industrial capacity.</p>
<p class="articleParagraph enarticleParagraph" >Ambush fighter</p>
<p class="articleParagraph enarticleParagraph" >Many experts have long argued that the Ukrainian Air Force cannot shift from a defensive posture to offensive air operations, primarily due to Russia’s qualitative and quantitative superiority. However, this dynamic is gradually changing, and the Gripen is expected to mark the beginning of a new Ukrainian aerial offensive – not only because of its advanced technological capabilities but also because of its exceptional operational flexibility.</p>
<p class="articleParagraph enarticleParagraph" >Gripen’s ability to operate from highways and improvised runways, without relying on fixed airbases, allows it to evade Russian missile strikes and loitering munitions. While this may initially appear to be a defensive feature, with proper tactical planning, it could evolve into a lethal offensive advantage enabling Ukrainian forces to launch surprise attacks and disrupt Russian air operations with greater agility and survivability.</p>
<p class="articleParagraph enarticleParagraph" >Pierre explained: “By design, Gripen jets are built to operate in a dispersed, survivable manner –hiding in multiple locations to endure long-term conflict and execute the kind of aerial guerrilla warfare that Ukrainian pilots have practised for the past three years. One of Gripen’s most significant advantages is its rapid turnaround time. It can be refuelled and rearmed – whether with fuel or munitions – in just ten minutes on the ground. For a modern fighter, that’s exceptional. From the outset, the aircraft was designed for this kind of agile operation: the engine remains running, the jet is serviced quickly, and it takes off again with updated radar plots and support points.”</p>
<p class="articleParagraph enarticleParagraph" >He added: “This enables Gripen to launch, reposition, and re-engage in what are known as pop-up formations—tactical manoeuvres where the aircraft briefly appears to fire a long-range missile and then vanishes before the enemy can retaliate. However, the success of such tactics depends heavily on supporting the Gripen with enhanced firepower and networked targeting capabilities.”</p>
<p class="articleParagraph enarticleParagraph" >In such scenarios, adequate radar coverage and secure data links are essential for relaying enemy positions and providing a well-planned, protected flight path to minimise the risk of interception. Additionally, long-range air-to-air missiles and launch points as close as possible to the target area are critical. Gripen’s ability to engage multiple targets simultaneously offers a tactical edge – especially when executed by numerous jets approaching from different directions to achieve surprise, disorient the adversary, and neutralise them.</p>
<p class="articleParagraph enarticleParagraph" >This approach means that only a limited number of fighters are needed to carry out “free hunt” missions, allowing other Gripen units to perform secondary tasks such as suppressing or distracting enemy air defences in parallel. This grants the hunters more time and freedom to operate, enabling sustained execution of such missions with high efficiency.</p>
<p class="articleParagraph enarticleParagraph" >Felon is coming</p>
<p class="articleParagraph enarticleParagraph" >In November 2022, Lt Col Ilya Sizov, ‘Hero of Russia’ and the commander of 23rd Fighter Aviation Regiment, told the military newspaper Suvorovsky Natisk that the unit’s pilots started theoretical training on the Su-57 Felon at the crew conversion centre in Lipetsk.</p>
<p class="articleParagraph enarticleParagraph" >This elite air unit is expected to be the first to receive Russia’s fifth-generation fighter jets for full-scale combat deployment – not merely for limited operational trials as seen during the first year of the war, when their use was restricted to long-range engagements to avoid confrontation with Ukrainian fighters and air defences. However, this cautious approach is unlikely to continue if Ukraine succeeds in shifting the airpower balance in its favour. In response, Russia is expected to deploy its stealth fighters more aggressively to reinforce existing formations. These aircraft will likely be operated by seasoned pilots with extensive experience in beyond-visual-range (BVR) engagements, capitalising on upgraded weaponry such as the R-77M (AA-12 Adder) missile, which boasts a range of 100 miles and features conventional control fins – allowing it to be carried internally within the Su-57 Felon’ s weapons bay. Additionally, continued use of the R-37 (AA-13 Axehead ) missile – launched from a platform somewhat more complicated to detect than the Su-35 Flanker -E – will further enhance Russia’s long-range capabilities.</p>
<p class="articleParagraph enarticleParagraph" >Thus, the matter does not rest solely on the Gripen itself, but rather on the Ukrainian Air Force’s ability to acquire and integrate the full spectrum of capabilities required to achieve air superiority. Chief among these are advanced detection and tracking systems – both ground-based radar networks and airborne platforms such as AEW&C aircraft.</p>
<p class="articleParagraph enarticleParagraph" >This is precisely where the Saab 340 AEW&C, also known as the ASC 890, comes into play beside the new AI system if it’s tested in Ukraine through their Gripen. Announced on May 29, 2024, this platform is expected to bridge the gap between ground-based radar and airborne fighter operations, enabling a level of networked situational awareness that could surpass the Russian strategy employed since the outset of the war. However, the most significant challenge remains protecting these high-value assets, ensuring they are not destroyed on the ground or intercepted in the air. Their survival will be critical to sustaining a coherent and resilient air defence and command-and-control architecture.</p>
<p class="articleParagraph enarticleParagraph" >To conclude, the arrival of the Gripen on the battlefield is poised to fundamentally reshape the aerial landscape, especially if the aircraft is deployed with its full technological, armament, and data-link capabilities, and if Ukrainian fighter pilots reach a level of operational maturity that ensures maximum combat effectiveness.</p>
<p class="articleParagraph enarticleParagraph" >This would represent a real test of <span class="companylink">NATO</span> technologies against Eastern systems, offering invaluable lessons that will undoubtedly be leveraged in the development of sixth-generation fighter platforms. The goal: to innovate more effective methods of air combat than those currently known and practised.</p>
<p class="articleParagraph enarticleParagraph" >afm</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gairf : Air Force | gcat : Political/General News | gcns : National/Public Security | gdef : Armed Forces</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>asiaz : Asia | dvpcoz : Developing Economies | eeurz : Central/Eastern Europe | eurz : Europe | russ : Russia | ukrn : Ukraine</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Key Publishing Ltd</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document FORAI00020251206elci00018</td></tr></table><br/></div></div><br/><span></span><div id="article-FORAI00020251206elci00002" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/foraiLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>BATTLES above</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>by Henri-Pierre Grolleau. </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>3019 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>18 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>AirForces Monthly</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>FORAI</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2025. Key Publishing Ltd. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Général Jérôme Bellanger, commander of the French Air and Space Force (FASF), recently answered questions about FASF modernisation posed</p>
<p class="articleParagraph enarticleParagraph" >Talking to… French Air and Space Force commander</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Q: What lessons for the French Air and Space Force have you drawn from contemporary conflicts?</p>
<p class="articleParagraph enarticleParagraph" >A: In Ukraine, Iran or Pakistan, at the tactical, operational and strategic levels, contemporary conflicts are of primary interest in testing the relevance of our philosophies and the order of our priorities. My general assessment is that the current international situation, marked by a level of conflict unprecedented in recent history, reinforces the analyses contained in my strategic vision for the FASF, The Sky As A Battlefield. These conflicts are all different, but beyond the lessons specific to each, they all point to a form of aerospace power in modern warfare.</p>
<p class="articleParagraph enarticleParagraph" >In Ukraine, the lack of air superiority explains the persistence of a war of attrition that is extremely costly in human lives and equipment. It also raises the question of mass. The Russians have lost more than 30 Su-34 fighter-bombers since the beginning of the conflict, while building just as many at the same time. The rise of Russian and Ukrainian drone production is particularly important, with thousands of drones being deployed daily into the battlefield and deep into territories of both sides. In terms of industrial performance, operational resilience and tactical innovation – I’m thinking of the Ukrainian Operation Spider Web ( More on Ukraine’s Spider’s Web , AFM , August 2025, p7) against Russian long-range aircraft – there are many lessons to be learned.</p>
<p class="articleParagraph enarticleParagraph" >Regarding the Israeli-American strikes on Iran (see Iran nuclear sites bombed , AFM , August 2025, p6, 32-39), the ability to bypass enemy air defence systems is a decisive condition for freedom of action and operational superiority. These raids, carried out in several stages since October 2024, show that access denial systems are never impenetrable and their collapse exposes the adversary. In terms of duration, effectiveness, political impact, and action, this war is a counter-example to the one waged by Russia in Ukraine.</p>
<p class="articleParagraph enarticleParagraph" >Finally, regarding the fighting between India and Pakistan ( Understanding the Rafale kills , AFM , October 2025, p43-58) when dozens of fighters were engaged by each side in a high-intensity engagement, losses were inevitable, regardless of the level of expertise and determination of the combatants, regardless of the quality of their equipment.</p>
<p class="articleParagraph enarticleParagraph" >In summary, we reaffirm the importance of air superiority, to continue our efforts in the fields of suppression enemy air defences (SEAD) and electronic warfare (EW), and to review the size of our fleets upwards, particularly the number of fighter aircraft. This is very much in line with the recent announcements by the government for the acquisition of 30 additional Rafales.</p>
<p class="articleParagraph enarticleParagraph" >Q: The Rafale is increasingly perceived by our allies, observers and by the foreign press as an ageing machine. How will you ensure its long-term effectiveness?</p>
<p class="articleParagraph enarticleParagraph" >A: Everyone promotes their own interests: our adversaries, of course, our international competitors and even some foreign industrial competitors – who may even be our allies! We listen to everything said, but we’re cautious when it comes to the analysis of the performance of our weapon systems by foreign sources. I am referring here to</p>
<p class="articleParagraph enarticleParagraph" >India and Pakistan. Barely had the fighting ended that we witnessed a disinformation manoeuvre in favour of the Chinese defence industry, aiming to incriminate the quality of the Rafale or the skills of the Indian pilots, to justify what should not be justified. As I mentioned earlier, when you engage in a high-intensity combat, losses and attrition are part of the equation. It’s as simple as that and it clearly underscores the need to increase the size of the French fighter force.</p>
<p class="articleParagraph enarticleParagraph" >To answer your question, the Rafale’s growth potential is truly remarkable and it is clearly one of this aircraft’s key assets. The aircraft that joined our ranks in 2004 does not have much in common with what it has become today, thanks to successive modernisations, and even less with what the future F5 standard promises. Because it will ensure the operational credibility of the airborne nuclear component, notably with the adoption of the future ASN4G airborne nuclear weapon, all the operational capabilities of the F5 standard will be reviewed upwards: sensors, connectivity, weapons, combat performance. Nothing will be left out, including its optronics systems, electronic warfare suite and the capability to suppress enemy air defences. The use of artificial intelligence for data processing and decision support and the reliance on expanded connectivity will prefigure the collaborative combat capabilities brought by the future SCAF. The Rafale F5 is an ambitious and extremely promising programme. It’s important to objectively analyse what it brings to the table, always keeping in mind the fierce industrial competition that’s driving the war of narratives. The impressive and renewed export success of the Rafale is the best proof of its attractiveness and global combat effectiveness.</p>
<p class="articleParagraph enarticleParagraph" >Q: Are you satisfied with the Mirage 2000D modernisation programme?</p>
<p class="articleParagraph enarticleParagraph" >A: We have just declared the modernised Mirage 2000D fully operational. This is an important milestone,enabling these fighter-bombers to maintain their rank for at least ten more years. The 3ème Escadre de Chasse at Nancy continues to provide much-needed power projection capabilities to our combat aviation force, with its modernised assets and its battle-hardened crews seasoned by decades of operations.</p>
<p class="articleParagraph enarticleParagraph" >We are also studying various options for the 2000D, having taken on board lessons learned from current conflicts, such as the possibility of adapting low-cost weapons to deal with new threats such as Shahed drones. These types of drones are proliferating in all theatres, and in the context of a major engagement, it is unthinkable to use our most sophisticated and expensive weapons to deal with them. In Ukraine we will soon surpass the threshold of 1,000 Geran-2 drones launched by Russia in a single night.</p>
<p class="articleParagraph enarticleParagraph" >In terms of modernisation, other so-called ‘agile’ options are being considered after operational feedback, are under consideration. For example, we plan to integrate the Talios targeting pod on the Mirage 2000D RMV, as well as the sovereign A ASM/Hammer precision munitions that have been in service on the Rafale for many years – an extremely effective family of weapons that provides complete satisfaction to all its users.</p>
<p class="articleParagraph enarticleParagraph" >Finally, our ability to work directly on the Mirage 2000D weapon system’s software allows us to develop particularly interesting combat capabilities relying on artificial intelligence (AI). Thanks to the remarkable efforts of our software developers and of the programme development team, 3ème Escadre aircrews have an onboard information system that assists them in all phases of flight, navigation, weapons delivery, close air support and intelligence gathering and processing. The upcoming integration of Talios will also allow us to go further in terms of real-time exploitation of collected imagery.</p>
<p class="articleParagraph enarticleParagraph" >Q: How do you assess the entry into service of the A400M?</p>
<p class="articleParagraph enarticleParagraph" >A: At the 2025 Paris Air Show, we celebrated the approval of the full operational capability of the A400M, the flagship of our transport aviation. This aircraft is one of the pillars that give the FASF its global stature, alongside our fighters and our A330 MRTT tankers. It’s a kind of Swiss Army knife, capable of undertaking both logistical and tactical missions.</p>
<p class="articleParagraph enarticleParagraph" >The A400M is a high-intensity warfare and joint combat tool. We can congratulate ourselves on the spectacular progress made in this regard, with a training exercise of an unprecedented level recently conducted with the 11th Parachute Brigade, during which new-generation Scorpion-series armoured vehicles embarked on A400Ms. The aircraft has also become a key special operations asset, as demonstrated during the multinational Athena exercise last spring. This aircraft also prepares us for the ‘next move’, that is to say for strategic developments over the long term, such as the ability to operate in the extreme Arctic environment. That’s why, with our comrades from the French Army’s High Mountain Military Group, we validated this year our ability to operate beyond the Arctic Circle in Greenland during the UPPICK mission.</p>
<p class="articleParagraph enarticleParagraph" >However, this full operational capability does not mark the end of discussions regarding the use of this aircraft, whose potential is enormous. The A400M could notably become an ‘effector carrier’, opening up major opportunities in the fields of electronic warfare, intelligence, weapons carriage and long-range strikes.</p>
<p class="articleParagraph enarticleParagraph" >For all these reasons, I welcome the upward revision of the size of this fleet, initially planned for 35 aircraft by the military planning law and since increased to 37, with deliveries scheduled to continue until 2028. Each year, our reliance on this fleet intensifies: relief efforts for populations in Mayotte and La Réunion islands in the Indian Ocean, support for internal security forces in New Caledonia, evacuations of nationals, etc. The A400M is remarkable in terms of versatility, but the issue of fleet size remains one of our major areas of focus.</p>
<p class="articleParagraph enarticleParagraph" >Q: Can you provide an update on the progress of the MRTT program? Are 15 A330 MRTTs sufficient to fulfil all refuelling and strategic transport missions?</p>
<p class="articleParagraph enarticleParagraph" >A: The French military plan provides for 15 MRTTs by 2028, primarily to meet the requirements of the permanent airborne nuclear component. Here again, given the international context, with a nuclear dialectic at the heart of ongoing conflicts and a hardening nuclear situation, this mission is extremely relevant. We can only be impressed and grateful for the truly visionary decisions taken by General Charles de Gaulle in the 1950s. We can clearly see how envious our European allies are of this fully sovereign model and now is clearly not the time to weaken our deterrence posture.</p>
<p class="articleParagraph enarticleParagraph" >As you have seen over the past few months, we have been deploying our MRTTs all around the world. What this fleet allows us in terms of power projection is just spectacular. There are few air forces in the world that possess such capabilities. In the recent past, there have been numerous examples of what we accomplished to demonstrate our global reach. Among the most emblematic achievements, I would cite the annual Pégase missions carried out by the A400M/MRTT/ Rafale triptych in the Indo-Pacific region or, for the first time this year, in the far north, or the evacuations of nationals conducted in Afghanistan, Sudan and Niger, each time in extreme conditions. It is important to understand that, in each of these operations, the MRTTs remained under the command of the Strategic Air Forces and are therefore likely to return to the French mainland at any time as part of the permanent nuclear deterrence posture.</p>
<p class="articleParagraph enarticleParagraph" >Q: Has integration into <span class="companylink">NATO</span> promoted the sharing of data, knowledge and tactics with our allies?</p>
<p class="articleParagraph enarticleParagraph" >A: Yes, for airmen it’s very clear that <span class="companylink">NATO</span> makes us stronger together. <span class="companylink">NATO</span> standards have enabled us to develop, over many years, a true culture of interoperability, particularly in air combat. In-flight refuelling is a simple and telling example: in this central area of air operations, <span class="companylink">NATO</span> provides us with a remarkable level of logistical and operational standardisation. We could multiply the examples, including much more advanced tactical knowhow. Being a <span class="companylink">NATO</span> partner is an obvious strength and provides a clear comparative advantage over our potential adversaries in the context of a major engagement.</p>
<p class="articleParagraph enarticleParagraph" >The ‘return on investment’ is also very positive for the French Air and Space Force. Here again, I will give you a striking example. There are approximately 30 NATO Centres of Excellence, which are incubators of doctrinal innovation, dedicated centres for sharing inter-allied expertise and training, each dedicated to a specific operational domain. France has the privilege of hosting two of these facilities, focusing on two areas that are at the very heart of military aerospace power: air operations, at the Lyon Mont-Verdun base, and space operations, at the Toulouse base. This is a very strong signal of confidence from our allies and a real privilege for us.</p>
<p class="articleParagraph enarticleParagraph" >Finally, in the current American political context, our approach is pragmatic: we must secure our commitment within <span class="companylink">NATO</span> for what it brings us, as I have just described, but also know how to operate outside of this framework if necessary. The best example is the security guarantees in the air domain, in the context of a possible ceasefire in Ukraine. France and the UK demonstrated historic co-leadership in the preparatory work for this ‘coalition of the willing’, with more than 20 air force leaders meeting in Lyon outside of any <span class="companylink">NATO</span> or European framework, to move forward in the right direction. We experienced something unprecedented and quite extraordinary that day, and we stand ready to activate these plans should circumstances require it.</p>
<p class="articleParagraph enarticleParagraph" >Q: Will space continue to gain importance within the FASF?</p>
<p class="articleParagraph enarticleParagraph" >A: I often refer to 2025 as The Year of Space. We have seen major advances: the ramp-up of the Ariane 6 space launcher, the opening of our first space-dedicated air base, the achievement of the first milestones by NATO’s Space Centre of Excellence and the announcement of the Very High Altitude (VHA) strategy, for example. All of this is just the beginning. What we are seeing is very clear and the President of the Republic reiterated it at the last Paris Air Show: space is increasingly contested, unstable and offensive. And space, particularly military space, has become the true gauge of international power.</p>
<p class="articleParagraph enarticleParagraph" >In 1962, 24 satellites orbited the Earth. By 2010, there were 1,000. There are 10,000 of them today, and we will probably have more than 50,000 in orbit by 2030. These orders of magnitude are alarming. We must be highly attentive to the risk that this environment could become a field of uncontrolled confrontation. We are already seeing a rise in tensions: ground-to-space threats but also space-to-space threats, with a sharp increase in non-cooperative closing-in manoeuvres, intentional jamming and the development of dazzling lasers and anti-satellite missiles. I am also particularly concerned by the rise of offensive anti-satellite capabilities in the cyber field, with some international players choosing to develop capabilities to neutralise or take control of satellites through this means, which has the advantage of being more difficult to attribute. And then there is always the risk of kinetic attacks from irresponsible actors. Moreover, should the ‘Big Day’ finally happen, hostile threats will not limit themselves to targeting military assets. We should all remember that, on February 24, 2022, Russian hackers succeeded in interrupting the communications services provided by the KA-SAT satellite to Ukraine by attacking its ground segment.</p>
<p class="articleParagraph enarticleParagraph" >In view of these facts, we must accelerate our ongoing efforts. A national space strategy is currently being finalised. In the meantime, our 2019 defence space strategy remains on the agenda, and the Air and Space Force has implemented it in all areas: doctrine, human resources, infrastructure, etc. We did not wait for the invasion of Ukraine or the current strategic upheavals to act.</p>
<p class="articleParagraph enarticleParagraph" >Today, in terms of capabilities, we have a very efficient model, but one that is too limited. We operate cutting-edge satellites, but too few in number, with development and deployment times that are too long. In fact, this model is built for a permissive strategic environment, which space is becoming less and less. It is not sufficiently resilient given the level of threat. This means that its architecture must be adapted, with more distributed capacities, more modularity and more responsiveness – for example by taking advantage of low orbits that notably allow for increased revisit frequencies. This is where the importance of the One Web and <span class="companylink">Eutelsat</span> constellations and the IRIS² programme lie, true strategic treasures receiving determined support at the European level. And this means that we must equip ourselves with the means to act not only from space, in support of other operating domains, but also in and towards space. The opening of our first space-oriented air base, in Toulouse, last July, and the upcoming inauguration of the new Space Command infrastructure at ‘Général Aubinière’ 101 air base are steps in this direction. We are living in a key moment for the rise of the French defence space component and we are fully committed to this effort.</p>
<p class="articleParagraph enarticleParagraph" >Q: What would be your final word? A: What I have just explained about the dramatic increase in the threat level in space also applies to a layer called Very High</p>
<p class="articleParagraph enarticleParagraph" >Altitude or VHA. This region lies between the ‘air’ up to an altitude of 15km, in which we have traditionally operated until now, or flight level 500, which is the flight ceiling for most combat aircraft, and space, which begins approximately at the Karman line, at an altitude of around 100km.</p>
<p class="articleParagraph enarticleParagraph" >The government unveiled at Le Bourget a French strategy for VHA that places airmen on the front line. We no longer limit ourselves to considering our actions up to flight level 500: today, we must think and act up to flight level 3300 and consider that as a continuum with what is happening in space. The President of the Republic gave us this mandate, in precisely these terms. Whether in the fields of organisation and operations, or even in the field of system resilience, we must consider our challenges and our actions in terms of the air to space continuum. It is this ‘holistic’ vision of the third dimension that I defend and support. We are therefore currently experiencing rapid and significant advances in VHA in terms of detection, identification, neutralisation, observation and communication. You may be aware of the success of our recent test firing on target balloons provided by the CNES, the French National Space Centre. I could also mention the Nostradamus over-the-horizon radar programme in the early warning register and other experiments that aim to take advantage of VHA in terms of extension, permanence and survivability, including the flight of the Balman manoeuvring balloon from French Guyana or the Zephyr solar plane. Things are moving quickly and we are acting and taking measures. afm</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gairf : Air Force | gcat : Political/General News | gcns : National/Public Security | gdef : Armed Forces</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>asiaz : Asia | dvpcoz : Developing Economies | eeurz : Central/Eastern Europe | eurz : Europe | iran : Iran | meastz : Middle East | pakis : Pakistan | sasiaz : South Asia | ukrn : Ukraine</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Key Publishing Ltd</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document FORAI00020251206elci00002</td></tr></table><br/></div></div><br/><span></span><div id="article-FORAI00020251206elci0001d" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/foraiLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The ups and downs of the Dubai Airshow</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>419 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>18 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>AirForces Monthly</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>FORAI</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2025. Key Publishing Ltd. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >I ventured to the Dubai Airshow in November for what has arguably become the leading aviation event in the world. You can’t fail to be impressed with the massive static show that boasted more than 150 military and civil aircraft or the huge exhibition hall that hosted some of the biggest aerospace companies in the world, with Israel the notable exception this year because due to the UAE’s ‘security concerns’. The Russians were there in big numbers, though, which many people found quite bemusing given what Vladimir Putin has been doing in Ukraine since February 2022.</p>
<p class="articleParagraph enarticleParagraph" >The Middle East is the only region where Russian aerospace can show off its new technologies – and watching the impressive Su-57E Felon – albeit a prototype – being put through some quite outrageous manoeuvres was probably the star of the display. The USAF sent an F-35A and an F-16C, but their displays were no match for the agility of Russia’s fifth-generation multirole fighter.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Another piece of great entertainment came from the United Arab Emirates Air Force and Air Defence’s (UAEAF&AD’s) Al Fursan, officially known as the Fursan El Amarat, flying its new Chinese L-15 jet trainers. While the flying was what you would expect, it was the aircraft’s howling Ivchenko AI-222 engines that caught all of us by surprise.</p>
<p class="articleParagraph enarticleParagraph" >The opening ceremony included a flypast of UAEAF&AD aircraft in three different waves: helicopters, transports and heavies with fighter escort. The first to arrive were the 11 choppers bearing down on the show (see pic), which brought to mind The Ride of Valkyries by Richard Wagner, made famous by the 9th Cavalry Regiment led by Lieutenant Colonel Bill Kilgore (Robert Duvall) in the 1979 Vietnam movie Apocalypse Now. Maybe next time the organisers could blast that out over the loudspeakers!</p>
<p class="articleParagraph enarticleParagraph" >Sadly, the event was marred by the crash of an Indian Air Force Tejas during the flying display on the last day, which claimed the life of the pilot. This was when the organisers let themselves down by allowing the air display to carry on, with the crash site still burning, which was horribly disrespectful to the deceased aviator and his IAF colleagues. It was a brutal decision and the display should have been cancelled, as it would have been in Europe.</p>
<p class="articleParagraph enarticleParagraph" >Alan Warnes Editor at Large</p>
<p class="articleParagraph enarticleParagraph" >Contact the Editor at Alan.Warnes@keypublishing.com</p>
<p class="articleParagraph enarticleParagraph" >Visit our website at <span class="colorLinks">www.key.aero/airforcesmonthly [https://www.key.aero/airforcesmonthly]</span>
                  </p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaero : Aero/Air Sports | gairf : Air Force | gcat : Political/General News | gcns : National/Public Security | gdef : Armed Forces | gspo : Sports | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>asiaz : Asia | devgcoz : Emerging Market Countries | dubai : Dubai | eeurz : Central/Eastern Europe | eurz : Europe | meastz : Middle East | russ : Russia | uae : United Arab Emirates</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Key Publishing Ltd</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document FORAI00020251206elci0001d</td></tr></table><br/></div></div><br/><span></span><div id="article-FORAI00020251206elci0000h" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/foraiLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Al Fursan L-15 with refuelling probe</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>254 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>18 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>AirForces Monthly</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>FORAI</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2025. Key Publishing Ltd. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >WHILE AI Fursan were flying their new Hongdu L-15 mounts in the air display, this example, the second No 7 – denoting the number of states in UAE – was on show in the static park. Although painted in the Al Fursan marks, it was fitted with an air-to-air refuelling (AAR) probe. In 2023, a model of an L-15 with an AAR, was shown at the CATIC exhibition and is obviously an option for the UAE. The deal to buy 12 L-15s for Al Fursan can be traced back to 2021 when one of them participated in the flying display. In 2023 there were two L-15s present, one in the flying display and the other in the static, surrounded by a number of weapons. It was obvious the UAE was keen on the Chinese jet and that was confirmed on the opening day of the 2023 show, when the UAE MoD announced a 1.62 AED contract with CATIC for ‘the purchase of airshow aircraft and its accessories‘ to replace the team’s Leonardo MB339NATs.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" > There was also an option for 36 trainers but it would appear that it hasn’t been exercised.</p>
<p class="articleParagraph enarticleParagraph" >Until now, only six L-15s have been exported – to the Zambian Air Force in 2015/16. The UAE Al Fursan L-15s were delivered earlier this year via PAF Base Nur Khan just outside Rawalpindi, Pakistan. But it wasn’t until now that they were seen publicly or otherwise.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>asiaz : Asia | devgcoz : Emerging Market Countries | meastz : Middle East | uae : United Arab Emirates</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Key Publishing Ltd</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document FORAI00020251206elci0000h</td></tr></table><br/></div></div><br/><span></span><div id="article-FORAI00020251206elci0000s" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/foraiLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>
                           Shield AI Unveils Fully Autonomous VTOL Fighter Jet</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>324 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>18 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>AirForces Monthly</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>FORAI</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2025. Key Publishing Ltd. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >
                        <span class="companylink">SHIELD AI</span>, a defence technology company based in San Diego, California, has unveiled its autonomous fighter jet, the X-BAT.</p>
<p class="articleParagraph enarticleParagraph" >This artificial intelligence-piloted aircraft with vertical take-off capabilities is the company’s entry into the growing market for military drones.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >
                     Armor Harris, senior vice-president of aircraft at <span class="companylink">Shield AI</span>, said:</p>
<p class="articleParagraph enarticleParagraph" >“X-BAT is a revolution in airpower because it combines four things – VTOL, range, multi-role capability and autonomy. VTOL plus range solves survivability on the ground and dependency on tankers. Multirole provides critical flexibility as the threat evolves, because no plan survives first contact with the enemy.”</p>
<p class="articleParagraph enarticleParagraph" >Also, according to <span class="companylink">Shield AI</span>, the X-BAT can carry weapons internally and externally and perform strike, counter-air, electronic warfare and intelligence missions. Up to three X-BATs can fit in the deck space of one legacy fighter or helicopter. Brandon Tseng, <span class="companylink">Shield AI</span>’s co-founder, president and former Navy SEAL, emphasised the strategic advantage of runway-independent operations: “Airpower without runways is the holy grail of deterrence,” he said. “It gives our forces persistence, reach and survivability, and it buys diplomacy another day.” The X-BAT unveiling reflects the air force’s accelerating push into autonomous warfare through its Collaborative Combat Aircraft programme, which aims to field AI-enabled drones as force multipliers for crewed fighters, with approximately two CCAs for every advanced fighter. In May 2024, former Air Force Secretary Frank Kendall demonstrated the military’s commitment to AI pilots. He flew aboard an autonomous F-16 at Edwards Air Force Base where the X-62A VISTA, piloted by AI during dogfighting manoeuvres, reached speeds exceeding 550mph. After the flight, Kendall said he would trust the AI with decisions on weapon launches. <span class="companylink">Shield AI</span> has completed ground tests validating the airframe, engine and vertical take-off capability, with first flights that were expected in autumn 2026 and full mission capability demonstrations by 2028.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>olwnlt : Shield AI Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302022 : Artificial Intelligence Technologies | iaer : Aerospace/Defense | idef : Defense Equipment/Products | iindstrls : Industrial Goods | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gairf : Air Force | gcat : Political/General News | gcns : National/Public Security | gcsci : Computer Science | gdef : Armed Forces | gsci : Sciences/Humanities</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usca : California | usw : Western U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Key Publishing Ltd</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document FORAI00020251206elci0000s</td></tr></table><br/></div></div><br/><span></span><div id="article-FORAI00020251206elci00006" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/foraiLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Al Fursan’s howling L-15s</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>338 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>18 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>AirForces Monthly</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>FORAI</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2025. Key Publishing Ltd. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Alan Warnes catches the UAEAF&AD’s Al Fursan with their new Hongdu L-15s making their world premiere</p>
<p class="articleParagraph enarticleParagraph" >Most spectators at the recent Dubai Air Show, on November 17, at around 1335hrs, would have been enthralled with what they were hearing, not just seeing. That is when the United Arab Emirates Air Force and Air Defense (UAEAF&AD) aerobatic team Fursan al Emarat made its world premiere, flying their new Chinese jet trainer – the Hongdu L-15. While the aerobatics and manoeuvres were very slick, what we were not expecting was the howling that came from the jet’s Ivchenko-Progress and <span class="companylink">Motor Sich</span> AI-222 powerplant. It was Starfighter-esque… if you are old enough to remember the charismatic fighter. Every time the seven jets flew by in formation, this howl would arrive with them for five or so seconds until they passed. It was truly memorable and will be a real crowd-puller in the coming years. I asked the team-leader why the aircraft was making the noise and he didn’t know!</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >For two days, the team flew 30-minute displays turning and burning in front of the crowd, who on the first day featured the Ruler, Vice President and Prime Minister of the UAE, His Highness Sheikh Mohammed bin Rashid Al Maktoum. On the third day, they made a flypast with a Fly Dubai Boeing 737 MAX, but did not appear on the Thursday then closed the flying display on the Friday after the tragic Indian Air Force Tejas crash.</p>
<p class="articleParagraph enarticleParagraph" >The Al Fursan, as the team is more commonly known, made its first public display at Dubai air show in 2011 flying the Aermacchi MB-339NAT, but in 2021 a decision was made to replace the ageing jets with the L-15 which was making an appearance at the Dubai Air Show that year. Undoubtedly the team, with that howl, is set to become one of the most charismatic in the world. Long may it last!</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>mtrsch : Motor Sich JSC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i364 : Aerospace Products/Parts | i3640002 : Aircraft Engines | iaer : Aerospace/Defense | iindstrls : Industrial Goods</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaero : Aero/Air Sports | gairf : Air Force | gcat : Political/General News | gcns : National/Public Security | gdef : Armed Forces | gspo : Sports | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>asiaz : Asia | devgcoz : Emerging Market Countries | dubai : Dubai | meastz : Middle East | uae : United Arab Emirates</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Key Publishing Ltd</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document FORAI00020251206elci00006</td></tr></table><br/></div></div><br/><span></span><div id="article-FORAI00020251206elci0000x" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/foraiLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Edge’s Jeniah and now Omen</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>382 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>18 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>AirForces Monthly</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>FORAI</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2025. Key Publishing Ltd. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >EDGE’S FLAGSHIP Jeniah UCAV was once again on show and, according to staff, will fly in 2028. It’s going to be a fully autonomous, low-observable system, designed for high-risk missions such as suppression of enemy air defences and precision strike across land and sea. Khaled Al Zaabi, Edge’s President of Platforms & Systems, told the author: “We haven’t positioned Jeniah as a dedicated collaborative combat aircraft [CCA], but its design and mission set allow it to perform many of those same roles.” When asked if the Jeniah could work with the UAEAF&AD’s new Rafales, Al Zaabi responded: “Anything is possible from a technical standpoint. But that’s not something we’re actively pursuing right now.</p>
<p class="articleParagraph enarticleParagraph" >Integration of that nature would always depend on the customer’s specific requirements.”</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Just before the show, on November 13, Edge signed a joint venture agreement with US tech giant, <span class="companylink">Anduril Industries</span> to develop a hover-to-cruise Autonomous Air Vehicle (AAV) known as Omen. Al Zaabi said: “Our new joint venture with Anduril combines its advanced autonomy and AI command-and-control systems with our rapid production ecosystem here in the UAE. The first project is the Omen AAV and the UAE has already agreed to acquire 50 systems.”</p>
<p class="articleParagraph enarticleParagraph" >Anduril is better known for developing the YFQ-44A Fury CCA, but who is to say that the new Edge-Anduril Production Alliance won’t get involved with the Jeniah?</p>
<p class="articleParagraph enarticleParagraph" >On display next to the Jeniah were the Darkwing and WSM-1cruise missiles that are under development by Edge. Saif Ali al Dahbashi, Edge’s President Missiles and Weapons, said they will provide different capabilities. “The Dark Wing is equipped with a wing kit, has been in development for a year and has almost completed testing. We started working on the [longrange] WSM-1 cruise missile this year and pushing aggressively to complete development within one-and-a-half years.”</p>
<p class="articleParagraph enarticleParagraph" >He confirmed both weapons and the Halcon P32 Thunder are earmarked for the Dassault Rafale, (which should start being deliveredin 2026), so speed to develop the weapons is of the essence. This could open up new sales markets for Edge because both Egypt and Indonesia which are Edge customers, have also ordered the Rafale.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>fpvnol : Anduril Industries Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>iaer : Aerospace/Defense | idef : Defense Equipment/Products | iindstrls : Industrial Goods</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Key Publishing Ltd</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document FORAI00020251206elci0000x</td></tr></table><br/></div></div><br/><span></span><div id="article-AVINEW0020251206elci0000n" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/avinewLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The advance of AI</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>2005 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>18 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Aviation News</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>AVINEW</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2025. Key Publishing Ltd. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Mina Adel sat down with former Gripen fighter pilot Mikael Grev, also CEO of Swedish firm Avioniq, to talk about artificial intelligence in aviation</p>
<p class="articleParagraph enarticleParagraph" >AI In Fighters</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Before sixth-generation fighters enter service around 2040, Europe is looking to bridge the gap in technology seen in frontline platforms. This has meant a continued commitment to upgrading older-generation aircraft, particularly fourth-generation and advanced fourth-generation fighters. AI is playing a big part in their futures.</p>
<p class="articleParagraph enarticleParagraph" >This comes alongside significant developments in the competing aircraft from the East which are being produced at an accelerated pace. Although they may not match Western fighters in terms of technological sophistication, they are no longer lagging as far behind as they did in the past. One could say they are dangerously close to matching Western counterparts, leveraging new weapon systems and electronic technologies integrated into tactics that could prove lethal.</p>
<p class="articleParagraph enarticleParagraph" >One of the most critical and innovative methods by which <span class="companylink">NATO</span> air forces are adapting to emerging threats is the introduction of several new operational concepts. Among the most prominent is artificial intelligence (AI), which is currently being developed in countries such as Sweden, the US and, more recently, the UK. This technology is expected to significantly enhance the combat effectiveness of fighter jets, not only in beyond-visual-range (BVR) and within-visual-range (WVR) aerial engagements, but also in command, control and early warning capabilities.</p>
<p class="articleParagraph enarticleParagraph" >Last summer saw two significant agreements, the first being Sweden’s Saab testing AI on Gripen E fighter jets under the name Centaur, a joint programme with Germany’s Helsing conducted as part of Saab’s broader initiative known as Project Beyond. The second involved a collaboration between <span class="companylink">BAE Systems</span> and the Swedish firm Avioniq to trial an AI-driven decision aid, known as Rattlesnaq, on the Eurofighter Typhoon.</p>
<p class="articleParagraph enarticleParagraph" >The author approached Mikael Grev, CEO of Avioniq, to find out why a pilot flying a fourth-generation jet needs AI.</p>
<p class="articleParagraph enarticleParagraph" >In your experience as a fourth-generation fighter pilot, why does a pilot of a current fighter jet need an AI system?</p>
<p class="articleParagraph enarticleParagraph" >A: In aviation, situational awareness (SA) has always been key to success, dating back to when pilots had to locate their opponents visually. Simply put, achieving good SA requires two components: sensors (hardware) to perceive the environment and a user interface (software) to convey sensor data effectively to the pilot. Information gathered by sensors but inadequately presented – due to suboptimal interfaces or small screens – becomes essentially worthless in immediate combat situations. Historically, sensors have received significant attention and considerable resources to refine their capabilities, generating vast amounts of data and typically ‘seeing’ more than what could realistically be communicated to the pilot. Thus, sophisticated decision-support tools and user interfaces are required to translate this extensive sensor data into actionable information for pilots.</p>
<p class="articleParagraph enarticleParagraph" >Rattlesnaq takes information from advanced sensors through sensor fusion to calculate enemy capabilities and potential actions. The system tracks both historical data, such as previous firing opportunities, and predicts future scenarios. Instead of presenting raw data on enemy positions, Rattlesnaq employs advanced algorithms and onboard simulations of all relevant weapons and platforms, illustrating available – and perhaps more importantly – unavailable tactical options to the pilot. This capability isn’t limited to next-generation aircraft. Any aircraft aware of enemy positions can leverage Rattlesnaq to predict outcomes in BVR combat.</p>
<p class="articleParagraph enarticleParagraph" >Additionally, improvements in propulsion and guidance technologies are increasing the range of air-to-air and ground-to-air missile systems, and this trend is expected to continue. As enemy weapons become increasingly long-range, pilots face growing difficulty in estimating threats based solely on experience. Although experienced BVR pilots develop intuition about engagement timing, long-range weapons complicate these assessments. The gap between the enemy’s weapon employment zone (WEZ) and the pilot’s assessed minimum abort range (MAR) widens. This is why decision-support tools like Rattlesnaq are imperative.</p>
<p class="articleParagraph enarticleParagraph" >As we compare similar systems in the US and even Helsing’s AI, which has been tested on the Gripen, what makes Rattlesnaq unique?</p>
<p class="articleParagraph enarticleParagraph" >A: The distinction between verifiable AI systems like Rattlesnaq, which assist in tactical decision-making, and pure AI systems aiming to replace pilots entirely is twofold. First, non-flyers often underestimate the complexity involved in replacing pilots. While it’s feasible to automate specific BVR combat scenarios – something we achieved nearly a decade ago – moving beyond an 80% success rate becomes increasingly challenging. Like autonomous cars, the progression from initial capabilities to full autonomy becomes exponentially more complex. In aviation, this challenge is even greater since fighter pilots undergo rigorous, multi-year training and selection processes. Second, accurately evaluating AI performance across millions of mission variations requires experienced pilots or fighter controllers, professions already in global shortage. Thus, AI solutions aiming to replace pilots comprehensively will take significantly longer than many anticipate, primarily due to underestimated operational complexity.</p>
<p class="articleParagraph enarticleParagraph" >Rattlesnaq doesn’t aim to replace pilots, but to enhance their effectiveness by managing complex tactical computations. It visualises critical tactical opportunities without taking control of the aircraft, allowing pilots to leverage their training and experience for nuanced judgments. Consequently, Rattlesnaq doesn’t compete with pure AI firms targeting fully autonomous fighter jets. While fully autonomous solutions may not become viable until 2040 at the earliest, Rattlesnaq is available now, ready to integrate into current and next-generation combat aircraft immediately.</p>
<p class="articleParagraph enarticleParagraph" >What is your opinion on the new variants and upgrades of the Typhoon?</p>
<p class="articleParagraph enarticleParagraph" >A: The Eurofighter Typhoon excels in speed and manoeuvrability, foundational traits for all aerial combat. Equipped with weapons like Meteor, IRIS-T and various air-to-ground ordnance, it offers an extensive arsenal. The new ECRS Mk2 AESA radar enhances active sensor capability, enabling detection of relevant threats in BVR engagements. The long-term evolution (LTE) version of the Typhoon, featuring a large display, will significantly enhance the user interface, thereby delivering superior SA with Rattlesnaq. While a considerable screen benefits Rattlesnaq, it’s not essential. Both current (Tranche 2-4) and future Typhoon LTE variants can fully leverage the system, transforming the aircraft into a formidable BVR combat platform.</p>
<p class="articleParagraph enarticleParagraph" >What makes Rattlesnaq a gamechanger for the Typhoon, given that sixth-generation fighters will arrive late and the Typhoon will remain in service for many years?</p>
<p class="articleParagraph enarticleParagraph" >A: Sixth-generation aircraft are primarily defined by their ability to understand and interact with their environment, necessitating advanced computational capabilities and agile software development practices. While civilian software development represents an untapped resource for rapid advancement, it lacks the specialised expertise required for combat-oriented decision-support tools, as understanding the operational context is critical. This is what differentiates Avioniq. Our team brings together a dedicated software team guided by an engineer and software developer who, in addition to their expertise, have also spent a decade as combat pilots and have more than ten years of experience developing this fundamental capability.</p>
<p class="articleParagraph enarticleParagraph" >Rattlesnaq incorporates ‘on-the-edge’ computations and verifiable AI integration. While this competitive platform is designed for the software needs of sixth-generation fighters, it can be integrated into any fifth-generation aircraft with sufficient computing power. This presents a crawl, walk, run capability that can be integrated today and scaled as sixth-generation aircraft begin to enter service. In this way, it provides an effective gap filler, enhancing the capabilities of today’s aircraft.</p>
<p class="articleParagraph enarticleParagraph" >With emerging threats from fifth-generation platforms such as the People’s Liberation Air Force and Russian military equipped with deadly long-range missiles, as seen in conflicts such as the recent India-Pakistan air war in May, how can AI-enabled systems effectively counter these challenges?</p>
<p class="articleParagraph enarticleParagraph" >A: New weapons from China and Russia now reach far beyond the ranges for which today’s decision-support systems were initially designed. A long-range missile takes minutes to arrive and a great deal can happen in flight, both inside the missile, which enjoys many degrees of freedom in its guidance, and for the targeted aircraft, which has substantial opportunities to evade it if it begins its manoeuvres early enough.</p>
<p class="articleParagraph enarticleParagraph" >With current decision-support systems, a pilot typically only sees the enemy’s maximum engagement envelope, known as the weapon engagement zone (WEZ). That makes it difficult to determine precisely when to initiate an evasive manoeuvre, as several minutes of flight time before impact mean the possible outcome space grows exponentially. Missile-specific factors, such as the weapon’s lofting algorithm, also have a significant influence.</p>
<p class="articleParagraph enarticleParagraph" >That is why Rattlesnaq, which is capable of simulating every enemy weapon in real-time, is so effective. The system calculates the distance you can fly in any direction without being hit. The further the missiles travel, the less relevant a static WEZ display becomes and the more you benefit from decision-support that includes onboard simulation of enemy weapons. In short, Rattlesnaq is poised to become indispensable for BVR combat, especially against long-range air-to-air or ground-to-air missiles.</p>
<p class="articleParagraph enarticleParagraph" >In terms of operating within <span class="companylink">NATO</span>’s command and control framework through AEW&C aircraft, how can AI be employed to ensure greater operational effectiveness?</p>
<p class="articleParagraph enarticleParagraph" >A: With an AEW&C resource, a comprehensive real-time operational picture of enemy aircraft is available, making it easier for decision-makers to manage rules of engagement (ROE) and make quick decisions as the enemy acts. An AEW&C enhances the ‘Observe’ phase in the crucial Observe, Orient, Decide, Act (OODA) loop. What further enhances the ‘Orient’ phase is the decision support available. By consolidating the information in one place, advanced AI systems such as Rattlesnaq, can process the sensor data and present it to the operator in a way that maximizes understanding of the actual threat. In BVR combat, the quality of the information that reaches the pilot’s brain is what makes the difference. The combination of an AEW&C’s sensor and modern decision support provides a significant advantage. When that advantage is applied to actual missile engagements, missile performance and the fighter aircraft’s speed become key factors in determining the outcome.</p>
<p class="articleParagraph enarticleParagraph" >Sun Tzu, author of The Art of War , wrote: “Strategy without tactics is the slowest route to victory. Tactics without strategy is the noise before defeat.” From this perspective, the advantage of AI systems becomes clear, seeking a balance between executing a successful strategy and employing lethal combat tactics aimed at neutralising the adversary’s offensive edge and disrupting the progress of hostile operations.</p>
<p class="articleParagraph enarticleParagraph" >At the tactical level, systems such as Rattlesnaq and Centaur are employed on fighter aircraft to support decision-making. They operate within a strategic framework that also relies on AI. The concept known as ‘affordable mass’ has already emerged as a result. This entails reducing the number of fighter jets involved in aerial formations by replacing them with advanced offensive drones, known as collaborative combat aircraft (CCA), which will accompany each fighter to form a specialised unit. The fighter pilot will control this ‘mass’ with the aid of AI to achieve optimal results, while minimising logistical support costs, pilot fatigue and unnecessary risks to human operators.</p>
<p class="articleParagraph enarticleParagraph" >This approach counters adversaries who rely primarily on numerical superiority, be it in terms of aircraft quantity or payload. Consequently, this strategy delivers substantial efficiency with minimal risk and high tactical flexibility, thanks to the ease of deploying drones from any nearby location, whether prepared or not, such as in the Pacific. Ultimately, it will significantly enhance the effectiveness of <span class="companylink">NATO</span>’s future Kill Web [a network connecting sensors, decision-makers and weapons systems to rapidly identify, target and neutralise threats].</p>
<p class="articleParagraph enarticleParagraph" >Despite the theoretical effectiveness of this strategy and the supporting systems, there are many sceptics, primarily due to its dependence on electronics and datalinks that can be disrupted through electronic warfare (EW), which has become increasingly dangerous, as evidenced by the Russian-Ukraine war. Therefore, it is likely that fighter pilots will continue to be heavily relied upon for decades to come to ensure mission performance, with AI serving as a supportive tool rather than the primary component in planning and executing combat missions.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302022 : Artificial Intelligence Technologies | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcsci : Computer Science | gsci : Sciences/Humanities</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Key Publishing Ltd</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document AVINEW0020251206elci0000n</td></tr></table><br/></div></div><br/><span></span><div id="article-DAYB000020251205elcc0006u" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daybLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Washington Daybook - General News Events - Futures</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Federal Information & News Dispatch, Inc./Agence France-Presse </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>188 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>12 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Washington Daybook</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYB</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Federal Information & News Dispatch, Inc. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >9 a.m. Technology - Discussion</p>
<p class="articleParagraph enarticleParagraph" >SPONSOR: <span class="companylink">The Brookings Institution</span>
                     </p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >TOPIC/SUBJECT: holds a discussion on "The energy challenges of Taiwan and Asia's AI ambitions."</p>
<p class="articleParagraph enarticleParagraph" >PARTICIPANTS: Gary Dirke, senior director and professor of practice at <span class="companylink">Arizona State University</span>'s LightWorks; Tarcy Sih-Ting-Jhou, senior researcher at the Asia-Pacific Energy Research Center; R. David Edelman, senior fellow at the Brookings Foreign Policy Program and Brookings China Center; Samantha Gross, director of and fellow at the Brookings Energy Security and Climate Initiative and fellow at the Brookings Foreign Policy Program; Syaru Shirley Lin, founder and chair of the Center for Asia-Pacific Resilience and Innovation; and Ryan Hass, director of the Brookings John Thornton China Center and senior fellow in the Brookings Foreign Policy Program and Brookings Center for Asia Policy Studies</p>
<p class="articleParagraph enarticleParagraph" >DATE: December 12, 2025</p>
<p class="articleParagraph enarticleParagraph" >LOCATION: <span class="companylink">Brookings Institution</span>, 1775 Massachusetts Avenue NW, Falk Auditorium, Washington, D.C.</p>
<p class="articleParagraph enarticleParagraph" >CONTACT: 202-797-6105, events@brookings.edu [Note: Register at <span class="colorLinks">https://www.brookings.edu/events/the-energy-challenges-of-taiwan-and-asias-ai-ambitions/ [https://www.brookings.edu/events/the-energy-challenges-of-taiwan-and-asias-ai-ambitions/]</span> ]</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gdip : International Relations | gpir : Politics/International Relations | gpol : Domestic Politics | ncal : Calendar of Events | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | namz : North America | usa : United States | usdc : Washington DC | uss : Southern U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Federal Information & News Dispatch LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYB000020251205elcc0006u</td></tr></table><br/></div></div><br/><span></span><div id="article-NSWK000020251202elcc00034" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/nswkLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>How a Global ‘Climate Reset’ Could Change the Fight Against Warming</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>Jennifer Wignall </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1308 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>12 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Newsweek</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>NSWK</td></tr><tr><td align="right" valign="top" class="index"><b>ED</b>&nbsp;</td><td>Global Edition</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>1</td></tr><tr><td align="right" valign="top" class="index"><b>VOL</b>&nbsp;</td><td>Volume 185, Number 18</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Newsweek LLC All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >It started in April, just after Earth Day, when former British Prime Minister Tony Blair and his think tank, the <span class="companylink">Tony Blair Institute</span>, called for a “radical reset” in the way the world tackles climate change. Blair argued that current net-zero strategies were “doomed to fail.”</p>
<p class="articleParagraph enarticleParagraph" >Next came Bill Gates, among the world’s richest men and a major force in clean tech through his <span class="companylink">Breakthrough Energy</span> investment group. Gates <span class="colorLinks">released a memo [https://www.newsweek.com/bill-gates-delivers-tough-truths-on-climate-before-u-n-talks-10951942]</span> just before the COP30 climate talks calling for “a different view” and urging leaders to “adjust our strategies” for dealing with climate change.  </p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >“Climate change is not the biggest threat to the lives and livelihoods of people in poor countries, and it won’t be in the future,” Gates wrote, infuriating many in the climate movement. <span class="companylink">University of Pennsylvania</span> climate scientist and author Michael Mann told Newsweek the Gates memo was “horrifying.”</p>
<p class="articleParagraph enarticleParagraph" >President Donald Trump, who has called climate change the world’s “greatest con job,” seized on the Gates memo. “I (WE!) just won the War on the Climate Change Hoax,” Trump posted on social media, and claimed that Gates had “finally admitted that he was completely WRONG on the issue.” (Gates denied this admittance, calling Trump’s comments a “gigantic misreading” of his memo.)</p>
<p class="articleParagraph enarticleParagraph" >Along the way, several major banks and businesses have quietly edged away from earlier climate pledges and dropped out of net-zero groups as they reassess their approach amid <span class="colorLinks">Trump’s hostility to climate action [https://www.newsweek.com/trump-undoing-climate-action-can-clean-energy-investments-survive-2098780]</span> and diminished enthusiasm for it in some other parts of the world.   </p>
<p class="articleParagraph enarticleParagraph" >The call for a climate “reset” is upon us. But just what does that mean and what would a climate reset look like? We asked some leading climate thinkers from business, economics, energy and politics to weigh in.</p>
<p class="articleParagraph enarticleParagraph" >Focus on Climate Costs</p>
<p class="articleParagraph enarticleParagraph" >“I agree with Gates, but he goes the wrong way,” Rhode Island Senator Sheldon Whitehouse said in a briefing with reporters at <span class="colorLinks">November’s COP30 in Belém, Brazil. [https://www.newsweek.com/harvard-economist-cites-important-cop30-development-on-climate-and-trade-11102465]</span> “We need a different approach, a more aggressive approach, an approach that points out the corruption and mischief from the fossil fuel industry.”  </p>
<p class="articleParagraph enarticleParagraph" >Whitehouse is the top Democrat on the Senate Committee on Environment and Public Works and one of the most consistent Congressional voices for climate action. He said Gates’ framing of the issue provides cover for those who oppose a clean-energy transition. “I am sick to death of hearing my Republican colleagues come out of meetings with Bill Gates and tell me, ‘See, we don’t need to do anything after all—Bill Gates says innovation is going to solve this,’” Whitehouse said. “He’s just been wrong on that, and he’s been wrong on that for years.”</p>
<p class="articleParagraph enarticleParagraph" >Whitehouse said climate and energy innovation will be stifled so long as there is no price on carbon pollution.</p>
<p class="articleParagraph enarticleParagraph" >“The economy is tilted to discourage innovation by giving the fossil fuel industry massive political subsidies,” he said.</p>
<p class="articleParagraph enarticleParagraph" >The reset Whitehouse wants is in messaging. He wants to more clearly connect climate change to economic impacts such as rising home insurance rates in places at increased risk of storms, flood and fire.</p>
<p class="articleParagraph enarticleParagraph" >“All of those affordability concerns connect back to the fossil fuel industry and its business,” he said.</p>
<p class="articleParagraph enarticleParagraph" >Democratic Senator Peter Welch of Vermont predicts a “consumer revolt” as ratepayers see their electricity bills rise, and he thinks it will be tied to <span class="colorLinks">Trump’s energy policies. [https://www.newsweek.com/trump-climate-energy-executive-orders-market-risk-2022410]</span> Renewable energy and battery energy storage have together accounted for the bulk of new electricity generation brought online over the past year. But as AI data centers push power demand upward, Welch said, the Trump administration’s attempts to stifle clean energy development make it harder to supply more electricity.  </p>
<p class="articleParagraph enarticleParagraph" >“That is immediately going to spike the retail rates,” Welch told Newsweek during September’s Climate Week NYC. While climate change alone has not motivated voters, he said, cost concerns might. “Climate is a winning message when it’s tied to affordability and jobs,” he added.</p>
<p class="articleParagraph enarticleParagraph" >The Move From Moral to Material Concerns   </p>
<p class="articleParagraph enarticleParagraph" >Andrew Prag is managing director for policy at the We Mean Business Coalition, a nonprofit that works with major companies on climate action. Prag said many companies are changing their climate strategies as the effects from warming become less theoretical.</p>
<p class="articleParagraph enarticleParagraph" >“It’s become much more a question of risks and opportunities than it is around moralizing on climate change,” Prag told Newsweek at COP30. “There’s been a change of narrative which has come about in the business world, partly because of the change in economics.”</p>
<p class="articleParagraph enarticleParagraph" >When the Paris Agreement came about 10 years ago, renewable energy was an expensive option. But as costs for clean tech continue to plummet, <span class="colorLinks">solar power is now the cheapest way  [https://www.newsweek.com/bill-mckibbens-latest-book-argues-seizing-solar-powers-big-moment-2118928]</span>to generate electricity in many markets, according to the <span class="companylink">International Energy Agency</span>.  </p>
<p class="articleParagraph enarticleParagraph" >Similarly, in 2015, EVs made up less than 1 percent of passenger car sales. Today, globally, about one in five new passenger cars is electric, and in China, the world’s largest car market, EVs account for half of new car sales, according to the nonprofit Systems Change Lab. “We’re in a very different place now where a lot of the technologies and the implementation of clean tech is economically attractive and there’s a strong business case for it,” Prag said.</p>
<p class="articleParagraph enarticleParagraph" >Other companies, however, are finding it harder to meet earlier climate pledges. Xia Li, assistant professor of strategy and entrepreneurship at <span class="companylink">London Business School</span>, said that is due to political uncertainty in the U.S. and soft commitments by some companies.</p>
<p class="articleParagraph enarticleParagraph" >“Many were never fully committed,” Li said via email. Companies might have announced ambitious goals but “as external uncertainties and operational constraints have become clearer, they are recalibrating.”</p>
<p class="articleParagraph enarticleParagraph" >Li has written extensively about corporate sustainability, and she said the business shift underway on climate is complex. “Rather than a simple retreat, this looks like sorting under a changing environment,” Li said. Some companies are delaying action but others with stronger commitments are thinking more deeply about climate impacts.</p>
<p class="articleParagraph enarticleParagraph" >“The physical impacts of climate change—heatwaves, floods, wildfire, water stress—are now material business risks,” she said. “They shape operating schedules, supply chains, insurance costs and access to finance.”</p>
<p class="articleParagraph enarticleParagraph" >Corporate climate strategy is now less about high-profile announcements and more about ground-level operations. Investors are also changing how they assess a company’s environmental performance.</p>
<p class="articleParagraph enarticleParagraph" >Traditional ratings provide an “investor checkbox” of a company’s environmental, social and governance record, Li said. But that approach won’t necessarily capture a company’s real impacts on the environment. New methods are emerging to better gauge company performance, she said, such as measures of the emission reduction potential of new innovations in climate tech.</p>
<p class="articleParagraph enarticleParagraph" >“I welcome the shift from box-ticking to impact,” Li said, “because it puts societal outcomes on par with firms’ financial performance.”</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">The Rockefeller Foundation</span>’s SVP and Power Program Leader Ashvin Dayal has spent much of his career using clean energy to expand electricity access in developing economies. Dayal said the talk of a climate reset won’t change the underlying facts about climate science or the economics of clean energy, but it does present a chance to change how those facts are communicated.</p>
<p class="articleParagraph enarticleParagraph" >“I wouldn’t call it a reset, I would call it a reminder [to] really focus on what matters here,” Dayal told Newsweek. “The climate crisis will only ever be addressed if we continuously keep people at the center of it.”</p>
<p class="articleParagraph enarticleParagraph" >When developing nations deploy more renewable energy, it isn’t just to cut CO2, he said. It is done to provide clean power to the millions of people who lack access to electricity.</p>
<p class="articleParagraph enarticleParagraph" >“It’s clean energy for people,” Dayal said. “It is about improving the lives and livelihoods of people across the planet.”</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>swiycz : Breakthrough Energy LLC | ttnbff : Tony Blair Institute</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gclimt : Climate Change | genv : Natural Environment | gglobe : Global/World Issues</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Newsweek LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document NSWK000020251202elcc00034</td></tr></table><br/></div></div><br/><span></span><div id="article-NSWK000020251202elcc0002z" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/nswkLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Kal Penn Wants to Know Why History Keeps Repeating in New Podcast</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>H. Alan Scott </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>4858 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>12 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Newsweek</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>NSWK</td></tr><tr><td align="right" valign="top" class="index"><b>ED</b>&nbsp;</td><td>Global Edition</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>1</td></tr><tr><td align="right" valign="top" class="index"><b>VOL</b>&nbsp;</td><td>Volume 185, Number 18</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Newsweek LLC All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >“I find the whole conversation about how it started not just interesting, but necessary.”</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Initially, Kal Penn was hesitant to do a podcast because “every actor has a podcast.” Fortunately for us, he created Here We Go Again, focused on why history keeps repeating itself. “I loved the fact that you could talk about history repeating itself through pop culture, through politics. But it’s not a political podcast by any means.” That said, Penn, an actor who took a break from Hollywood to work in the Obama administration, still very much has a foot in advocacy. “If you wanna go through the death spiral of social media and make yourself anxious,” go for it, he says, but he’s not going to join. Instead, he’s going to invite people to “come knock on doors…it’s gonna move the needle on real conversations.” And one thing fans continue to discuss is their love for his Harold & Kumar franchise. Recounting a time he ran into political adviser Karl Rove and found out he was a fan, Penn realized, “as long as we stay truthful to the characters, the hope is that as polarized as this world is, we can still make a movie for everybody.”</p>
<p class="articleParagraph enarticleParagraph" >SUBSCRIBE TO <span class="colorLinks">THE PARTING SHOT [https://www.newsweek.com/podcasts/the-parting-shot]</span> WITH H. ALAN SCOTT ON <span class="colorLinks">APPLE PODCASTS [https://podcasts.apple.com/us/podcast/the-parting-shot-with-h-alan-scott/id1608211048]</span> OR <span class="colorLinks">SPOTIFY [https://open.spotify.com/show/3baeI8Lb1Uys8M4WUdL9XR]</span> AND <span class="colorLinks">WATCH ON YOUTUBE [https://www.youtube.com/watch?v=X06ZbeKIohQ]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Editor's Note: This conversation has been edited and condensed for publication.</p>
<p class="articleParagraph enarticleParagraph" >So I actually just interviewed Ed Helms about SNAFU. You being on his network of podcasts just feels like a great fit.</p>
<p class="articleParagraph enarticleParagraph" >Yeah, that's how it's been feeling developing it and getting it up and running, too. I've been a huge fan of Ed's for a while, he was in one of the Harold and Kumar with us. I just think the world of him. And I like that he is both funny, inquisitive, uplifting, kind of all at the same time. And I had held off on doing my own podcast because part of it was just like super self-conscious about the fact that every actor has a podcast. Unless I have a real hook of something that feels authentic to me, I was like, I don't wanna do it just to do it. And there's so many great podcasts, obviously, that are out there. So when we started talking about this idea, I loved the fact that you could talk about history repeating itself through pop culture, through politics. But it's not a political podcast by any means. I find institutions really curious. I'm fascinated by them. And when institutional memory goes away, I'm also fascinated by that. I mean, look, I'm an actor. I moved out to L.A. in college. Hollywood 20 years ago was so completely different than it is now in terms of the content it's making and all of that. And I think that was one of the things that initially made me so fascinated by it. So to work with Ed and that whole team on something that is storytelling, that's an arc, like our first episode, our first guest was Bill Nye, and the topic was the space race. And so the arc there was, the space race in the '80s was because of the Soviet Union in the U.S. and competition. The space race today seems to be between billionaires. So what changed? How has that evolved? And then what can we expect for the future? We had Pete Buttigieg on, and this was probably like the least political podcast he's done. He's also a friend, very gracious of him to come on. But the fun thing about having somebody who was the former transportation secretary on is yes, we did talk about infrastructure, which is not, I think, what we called the episode, because it's such a horrific thing to sell somebody on.  </p>
<p class="articleParagraph enarticleParagraph" >It’s not sexy. Although, I would listen to it.</p>
<p class="articleParagraph enarticleParagraph" >Me too. The nerdy part of me, too, but in making it palatable, it's like, okay, you talk about when Eisenhower, when the U.S. invested in bridges and highways and all of this.   </p>
<p class="articleParagraph enarticleParagraph" >He created the highway system. It's fascinating.</p>
<p class="articleParagraph enarticleParagraph" >Exactly, yes, yes. But then, also what I love about doing this podcast is I completely selfishly can ask pet peeve questions of people. So I'm like, “Pete, you were transportation secretary, how come when your plane is actually early, they make you wait for like 30 minutes for a gate, and the captain will say something like, ‘Well, the folks on the ground just didn't know that we were coming in early.’” Yes, they did. They knew exactly where we were the entire time. That's how this works. So how come there's no gate? And there is an answer to that. So he obviously knew the answer to this and walked us through what it was, but that's the other joy of this, is that it's not serious. And I think the hope is that the listener leaves an episode with an understanding of an in-depth conversation with an expert on a particular topic. And also recognizing where we fit in to all of this. It's not preachy, it doesn't end with like, “Here's how you can get involved.” But you just feel a little more educated and a little bit more uplifted.   </p>
<p class="articleParagraph enarticleParagraph" >One of my favorite things about this podcast is that connection to history, and the connection it has to modern times. For example, we’re talking about health care right now in this country. I think back to when President Johnson signed Medicare into law. Getting rid of Medicare would be a very hard thing to do these days, because it’s become an expected thing in our lives.</p>
<p class="articleParagraph enarticleParagraph" >Yeah, 100 percent. Look, anytime something in politics or business is codified in a way that makes it—I'll never say that you can't ever repeal certain big things, because it's a crazy time we live in—but yeah, when you have something that people rely on that's steadfast, I find the whole conversation about how it started not just interesting, but necessary for us to remember why we have this, and then also how can you build on it? When you look at the good things that are still possible in business, in finance, in government, whatever it is, there's precedent for things. And to look at it and say, here's how stuff works, I think is a positive. I'll give you another example that's actually like, it has nothing to do with my podcast, but I'm always mindful of it since we look both back, present, and forward in all our topics, is, I'm a pretty left-leaning guy, not that that comes up in our episodes really, but when people today, a lot of these young kids today, will say things like, “Oh, Obama's such a disappointment. He's such a moderate, blah, blah blah.” To me, that means that that was a progressive administration that worked. If through a 2025 lens, you're looking back and saying that the things he did were moderate by today's standards, that means they were effective. Could he have gotten more of a lot of the things that those of us on the left would have probably wanted? Yeah, of course. But there's no magic time machine, right? If you're 20 years old and you grew up with things like the Affordable Care Act and the repeal of Don't Ask, Don't Tell and marriage equality—not that that was him, but under his presidency it happened—those are all things that are just normal to you. So that means that the goalposts have moved, and they should always be moving if we're having honest conversations about the trajectory of things.   </p>
<p class="articleParagraph enarticleParagraph" >An example of this is, a couple years, I had cancer. I got it right before the Affordable Care Act became law. If the <span class="companylink">ACA</span> hadn’t passed, my pre-existing condition would have prevented me from getting insurance. That’s just one example of how these laws that become historical benchmarks have very real impacts on people's lives.  </p>
<p class="articleParagraph enarticleParagraph" >It's great. And those are the stories I remember when I was at the White House, when we were working on the Affordable Care Act, and it was one of those things where the national media, especially cable news, they would run stories that are a little bit sensational or whatever. It's all talking head stuff, right? For cable news. But it was local news where the story that you just told was happening all across the country and local news affiliates, the nightly news, where they [would] interview people who said, this person has this particular medical condition and here's how this bill would help. And that really moved mountains because it showed people this is not just something that a bunch of dudes in suits are arguing about on CNN. It's something that affects, intimately affects, people in our own communities. These interviews are like outside of the school that you know or the grocery store. And I still feel like those are some of the more pivotal ways of getting things done is [through] those personal stories.   </p>
<p class="articleParagraph enarticleParagraph" >I suspect you’re a little like me in that I’m the friend in my friend group everybody comes to to be like, “Can they do that?” And sometimes it’s annoying. For example, everyone is asking me if Trump can run for a third time, and I’m like, “GUYS! Did you listen in civics class?” It feels a little bit like a conversation to distract from other, more important news. Do you get angry with stupidity from people you love?</p>
<p class="articleParagraph enarticleParagraph" >Oh, you knew the answer to this when you asked. Yes, I do, and I think you probably have the same perspective on this because of what we do for a living, that those types of conversations are obviously a phenomenal distraction, a purposeful distraction. Our friends in the media, and I'm not slamming it because we both work in it happily, but there's a difference between a legitimate article and a clickbaity, scary article so that they can sell ad space and pay our salaries. That's just the reality of it. And I've gotten to the point where it's very hard to not get angry at your friends when they do all this. But I just very simply have stopped responding with anything that takes the bait and I'm just like, “Hey, here’s an event that you can do. Here's a friend who's running for office if you wanna come knock on doors with us. It's not a thing I'm putting on social media. It's an official event.” It's just like a thing that I've decided I'm gonna do in my life because it's gonna move the needle on real conversations. If you wanna go through the death spiral of social media and make yourself anxious and then have two glasses of wine with a Xanax in order to sleep, be my guest, but my way of going about it is a little bit different, and you're welcome to come join.   </p>
<p class="articleParagraph enarticleParagraph" >What are some areas of history or things that history repeating itself that you're either currently exploring or eager to explore?</p>
<p class="articleParagraph enarticleParagraph" >There are two that stand out. We did an episode, we haven't dropped it yet, but with my friend Alok Vaid-Menon, they’re a fantastic stand-up comedian artist. And it's about gender and gender roles, and they know so much on the history of this and also where we are now. It was such a light, fun conversation about fashion, about insecurity, about who we are as a society. And I am not the expert with these guests, right? So, when the guests talk about the past and they're usually the ones living the present and they are leaders in where things are going to the future. So with Alok, for example, they were just telling me about the past, but also talking very eloquently through and through what's happening now and also getting rid of the distraction part of things and into where we're going. We did one that I'm equally excited about with Lilly Singh. And Lilly was the first, I might be getting some of this wrong, but I believe Lilly was the first woman of color with her own late-night show when she took over that slot, right? She's great, she's so funny. The conversation with her was about how she was one of the original <span class="companylink">YouTube</span> content creators. So, when <span class="companylink">YouTube</span> exploded, she really rose this meteoric rise with the characters that she was doing. And she started doing them in her parents' basement in Toronto, and then now has her own company. And the conversation with her was about, okay, things started digitally. She is a woman of color with these characters that no network executive would have given her a show to do, so, she did it on her own. The technology changed. We had <span class="companylink">YouTube</span>. And then she segued from that to doing her own late-night show to having a movie come out and now as network TV is almost fully dead, especially in the late night space, she's going back to a lot of digital content. And so the things evolved. But the conversation with her then is about the past, how she got to this place. What's happened since, why go back to doing digital content, the control that you have over it, the point of view that you can share that's your own. And in her case, too, it was timely. I mean, all of the conversations around [Jimmy] Kimmel and [Stephen] Colbert's cancelation and what that was like for her as a woman who was doing this show during COVID. They didn't properly, the network didn't properly invest in her. Resources or writer's rooms, at least in what I say not properly, at a commensurate level to other men in that space. So none of these conversations are, none of our guests have been like, “Woe is me, I've had it so hard.” They're just sharing the struggles that they've gone through to get the success that they have. And so we see like, oh, these are things that thankfully would never happen again, or in the case of Lilly’s conversation, she just wrote this phenomenal movie called Doin' It, a great comedy that's out right now, and even talking to her about the way that the industry didn't necessarily know what to do with that film and what a labor of love it is for her to get it off the table. It's those types of connections that I've been really fascinated to explore. And it's just funny. Like, Alok is a stand-up comedian, so when they do an interview, we've been friends for years, but it's such a funny and uplifting conversation.   </p>
<p class="articleParagraph enarticleParagraph" >The conversations around technology are fascinating to me. Like, people were afraid of television at one time. Eventually we got bored of that and tuned to the internet. We get bored of whatever new technology comes to move on to the next thing. Yet we still have this outrage at change.  </p>
<p class="articleParagraph enarticleParagraph" >I have a good friend, she's my old neighbor, lives here in New York [City]. She's 104 years old. Her name is Beulah. She is a retired actor. And some of her early stuff was theater, radio, that kind of stuff. And explaining to her what a podcast is, I basically found myself saying, “It's like I have my own radio show, except you can listen to it on your phone.” And it was just like such an underwhelming description. She was like, “All right, well, I don't know why you would want to do that. But all right. I love it.” But you're right. As things change and come around, we have growing pains with them. I mean, look, I think a necessary scary freak-out thing is how AI impacts all these things, but yes, of course, technological changes. Even look at the last just couple years, it used to be that like, “Oh, there's amazing content on streamers.” And streamers really are the reason that there's so much diversity in content, not just racial, ethnic , etc., but like the types of shows that you can watch and the international content. But then they've gone from like, “Cool, I'll subscribe to this. I'll get rid of my cable.” And now it's like, “Oh, we're also gonna have ads on <span class="companylink">Hulu</span> and <span class="companylink">Netflix</span>.” So this is TV! It's just TV now. [laughs] </p>
<p class="articleParagraph enarticleParagraph" >Looking back to history, is there a moment in history that this current political moment that we're heading toward with the midterm elections reminds you of?   </p>
<p class="articleParagraph enarticleParagraph" >I'm thankfully young enough and not enough of a history buff myself to not have a good frame of reference for that. But what I will say is, I think this last couple of weeks, you saw Democrats around the country sweep, and it's a range of Democrats. You have somebody like a Democratic Socialist like Zohran [Mamdani] in New York, but then you also have far more conservative Democrats who won. So in many ways, I feel like it's a clear referendum on the Trump administration and Republicans and the fact that people's needs aren't being addressed. That said, we're talking on a day when Chuck Schumer and the Democrats have for some insane reason agreed to flush all of their political capital down the toilet and say that, “Yeah, we'll agree to get nothing out of a government shutdown deal”’ So I don't know, what I see is a huge generational shift, not just on the left, but also the right. But for our purposes, we were talking to the left. Huge generational shift. You have a Chuck Schumer, a Hakeem Jeffries, Nancy Pelosi is retiring. And I'm not trying to throw them under the bus, but it's just so clear that they're completely out of touch with younger voters. I think it's probably a moment where my hope is that you'll see a shift where there's more capable leadership and people who are in the pipeline who have started paying their dues or just at the beginning of many of their careers. So, I'm curious to see where that all ends, but it is definitely a wild time, even for folks who don't pay attention to politics.   </p>
<p class="articleParagraph enarticleParagraph" >While we might be polarized, one thing that polarization inspires is different types of people to run for office. And so like, this is probably the most chaotic we've been in U.S. history, at least modern U.S. history, but I think you're seeing that in some candidates, on both sides of the aisle, there’s diversity in beliefs.  </p>
<p class="articleParagraph enarticleParagraph" >Yes, and it is very cool to see. I mean, objectively, if you're a fan of politics, you can look at people on the right and see these candidates and be like, “Wow, this is a direct response to somebody not feeling like their current Republican rep is representing them properly.” Take the Hakeem example. I've been reading a bunch about how he might have a primary challenger, this guy, Chi Ossé, who's in the New <span class="companylink">York City Council</span>, who I think is fantastic. I haven't spoken to him about whether he's running or not, but I've just, from the little things that show up on social media, I'm like, “Oh, that would be such a great matchup.” And that's happening all across the country. You're right. It is hopeful. And this is why I'm saying, [when] you were asking about what happens when friends ask dumb en- of-the-world questions. And these are great examples of them. Like just go knock on doors for an afternoon for the candidate of your choice. It doesn't have to be in a sexy election.   </p>
<p class="articleParagraph enarticleParagraph" >In fact, it should be in the unsexy elections. It should be for the local elections. That’s where everything happens. I mean, the other thing that that makes me angry people being like, “Zohran 2032.” And politics aside, he can’t do that. It’s simple civics.  </p>
<p class="articleParagraph enarticleParagraph" >He wasn't born in the country. Also, I don't know if you get this, too. But this happened when I was working for Obama and it's also happened recently with Zohran, who I've known since he was 14 years old, so I'm heavily biased here, and I helped him out with his state assembly stuff and his work with the taxi workers. Very proud of him. But it was only this last week when I posted some stuff from election night that I had friends texting me, “Aw bro, you're so lucky that you got to work on that campaign.” I’m like, “What? Go back through your texts three years ago, I asked you if you wanted to come to a coffee fundraiser I was doing for his state assembly race and you said, ‘no.'” Just cause you read about it and you like the guy now, which is great—and he needs a lot of support as he moves forward—but like go find the 10 other people who you align with and just go help them out for an afternoon, you know?   </p>
<p class="articleParagraph enarticleParagraph" >Speaking of Zohran, one thing I find fascinating is how he won, by the amount he won by, despite some of the backlash he received from Jewish voters, particularly what he’s said about Israel. In the past, that would be an immediate end of campaign event, but it wasn’t in 2025. How do you think he did it?</p>
<p class="articleParagraph enarticleParagraph" >Yeah, I mean, look, I think it's probably a little too early to speculate how that affects everything broadly, but I think one of the things he did a great job at was coalition building from the very beginning. And loud voices always get a lot of attention, and I'm not trying to minimize that, but he also had an incredible amount of support from the Jewish community as well, especially on the progressive wing. And so, there was a lot of what you see in politics as like getting validation from your friends in different communities as well. But I just think he was genuine. He wanted to meet people where they are. I mean, he’s been like this since he was 14. He's never scared to have a conversation with anybody who might disagree with him. He's willing to listen. One of the first things he did in deciding to run for mayor was talking to Trump voters in New York City and saying, “What made you go and vote for Donald Trump?” From these communities,  what convinced you that he had a plan and had your best interest in mind. And that's just kind of the basic work that a lot of people don't do. And it's going to sound cliche now that he's the mayor-elect, but like, this is a guy who clearly loves his city. Also, that gives you the perspective that most people are not single-issue voters. The affordability crisis is very real. I always laugh when I hear people say, “Well, we got to move to Connecticut.” You know who's not talking about the luxury of getting that second house and moving to Connecticut? The people who are going to benefit from the free bus pilot program. Yes, exactly. So it's as if we're not even having the same conversation. But to answer your question, I just think it's a combination of coalition building really addressing the needs of working New Yorkers and giving folks a reason to show up.   </p>
<p class="articleParagraph enarticleParagraph" >Well, and I would add to that too, similar to Pete Buttigieg going on Fox News. There is this level of, I'm going to meet you where you're at, no matter where you’re at, and not going to give you the typical spin that you're used to getting.</p>
<p class="articleParagraph enarticleParagraph" >It's very refreshing, and the bullsh** meter was so obvious, I think also because of the other candidates who were running against him. Do you remember that crowded debate where the question was like, “Oh, where would you visit first if you were mayor?” And everybody said Israel. And Zohran is like, I” wouldn't go anywhere. I'm mayor of New York City.” But then also, because he knew what he was kind of being baited, he said, also, I just want to be clear, my commitment is to Jewish New Yorkers. I will meet Jewish New Yorkers here in New York. I'm not going to take a trip to Israel as the mayor of New York City that ignores our own New Yorkers, right? So,even that simple thing was very, very worth looking at. </p>
<p class="articleParagraph enarticleParagraph" >Well, my last question for you, and it's a fun one: when I think of the legacy of Harold and Kumar, I think to myself, wouldn't it be fun if they went to the White House? Especially now!  </p>
<p class="articleParagraph enarticleParagraph" >I love these characters, and the opportunity to have done those three movies so much that I would consider myself lucky if I was a hundred years old doing Harold and Kumar 58. So, they could go anywhere as far as I'm concerned. Kumar is way cooler than I will ever be. It’s a joy to play him. There's conversations around a fourth movie, the writers have a deal, and I think they're trying to work through the mechanics. So hopefully we end up being able to make a fourth movie. I don't know where they go, but what I love about all three movies, they’re three totally different movies, but they're all a satire and there are jokes within jokes. Jon Hurwitz and Hayden Schlossberg, really talented guys who created the franchise and wrote all three movies, always craft layer on layer on layers. The second movie we had a wonderful comedian named James Adomian who played George Bush. And a couple months after Harold and Kumar Escape from Guantanamo Bay comes out, I was in D.C., Bush was still president. And I ran into Karl Rove, who was one of Bush's senior advisers. And I went up to him, because in my head I was like, “Oh, you'll piss off so many of your friends if you get a selfie with this guy.” So I go, “Mr. Rove, can I take a picture with you? My name is…,” and he goes, “Kal Penn, I know exactly who you are. You're hilarious.” And I was like, “What?” And the look on my face must have given it away. He goes “What, am I not supposed to find you funny?” I'm like, “No, no. What have you seen?” He goes, “Well, Harold and Kumar Escape From Guantanamo Bay just came out.” I'm, like, “You've seen Harold and Kumar Escape from Guatanamo Bay?” He goes, “Yeah, was I not supposed to find that funny?” Let me just tell you. We have our personal politics, obviously. But the whole goal of these movies is to make everybody laugh. And if you knew that that was satire, and if you and people at the Bush White House could just laugh at it because it's a dumb, fun, stoner comedy that, I want these movies to be movies that you can watch with your crazy uncle at Thanksgiving. And so getting that validation from Karl Rove that he was a Harold and Kumar fan was so wonderful. And I know there are gonna be people listening to this whose like blood is boiling because of the politics of all of this. But I'm just saying from the artistic perspective, the way you asked that question, that is my answer, that like, as long as we stay truthful to the characters, the hope is that as polarized as this world is, we can still make a movie for everybody.   </p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>ccat : Corporate/Industrial News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | nyc : New York City | usa : United States | usdc : Washington DC | use : Northeast U.S. | usny : New York State | uss : Southern U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Newsweek LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document NSWK000020251202elcc0002z</td></tr></table><br/></div></div><br/><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WC50111020251206elcb00005"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  Conquest Planning's Mark Evans Stepping Down as CEO</b><div class="leadFields"><a href="javascript:void(0)">Retail Traffic</a>, 12:00 AM, 11 December 2025, 419 words,  Davis Janowski, (English)</div><div class="snippet ensnippet"> Chief Revenue Officer Brad Joudrie will move into the CEO role as Evans takes on executive chairmanship.Conquest Planning, the artificial intelligence-powered, Canada-based, financial planning platform founded by Mark Evans (creator of ...</div>
<div>(Document WC50111020251206elcb00005)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WC46111020251205elcb00001"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  Foreign Investment in a Not-So-Globalized World: Navigating Review Processes in the Face of Protectionism Thu Dec 11</b><div class="leadFields"><a href="javascript:void(0)">Canadian Bar Association</a>, 02:00 AM, 11 December 2025, 268 words,  Philip Lee LLP, (English)</div><div class="snippet ensnippet"> As geopolitical tensions rise and national security concerns take center stage, governments worldwide are tightening foreign investment rules and demonstrating economic protectionism. Canada has embraced this global trend, with the federal ...</div>
<div>(Document WC46111020251205elcb00001)</div><br/></td></tr>
						</table>
					</div>
				</div><span></span><div id="article-DAYB000020251205elcb0006k" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daybLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Washington Daybook - General News Events - Futures</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Federal Information & News Dispatch, Inc./Agence France-Presse </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>128 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>11 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Washington Daybook</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYB</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Federal Information & News Dispatch, Inc. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >2 p.m. Foreign Affairs - Discussion</p>
<p class="articleParagraph enarticleParagraph" >SPONSOR: <span class="companylink">The Hudson Institute</span>
                     </p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >TOPIC/SUBJECT: holds a discussion on "Building U.S.-Taiwan Defense Supply Chain Collaboration: Opportunities for Co-development and Co-production."</p>
<p class="articleParagraph enarticleParagraph" >PARTICIPANTS: former Taiwan General Staff Chief Adm. Lee Hsi-Min; Betsy Shieh, consultant at Barbet Insights; Brandon Tseng, co-founder and president of <span class="companylink">Shield AI</span>; and Rupert Hammond-Chambers, president of the U.S.-Taiwan Business Council</p>
<p class="articleParagraph enarticleParagraph" >DATE: December 11, 2025</p>
<p class="articleParagraph enarticleParagraph" >LOCATION: <span class="companylink">Hudson Institute</span>, 1201 Pennsylvania Avenue NW, Suite 400, Washington, D.C.</p>
<p class="articleParagraph enarticleParagraph" >CONTACT: David Altman, 202-223-7771, press@hudson.org [Note: Registration and livestream at <span class="colorLinks">https://www.hudson.org/events/building-us-taiwan-defense-supply-chain-collaboration-opportunities-codevelopment [https://www.hudson.org/events/building-us-taiwan-defense-supply-chain-collaboration-opportunities-codevelopment]</span> ]</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>hudson : Hudson Institute | olwnlt : Shield AI Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>iaer : Aerospace/Defense | idef : Defense Equipment/Products | iindstrls : Industrial Goods</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gpir : Politics/International Relations | gpol : Domestic Politics | ncal : Calendar of Events | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usdc : Washington DC | uss : Southern U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>DEF</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Federal Information & News Dispatch LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYB000020251205elcb0006k</td></tr></table><br/></div></div><br/><span></span><div id="article-DAYB000020251205elcb0006e" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daybLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Washington Daybook - General News Events - Futures</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Federal Information & News Dispatch, Inc./Agence France-Presse </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>149 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>11 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Washington Daybook</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYB</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Federal Information & News Dispatch, Inc. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >9 a.m. Technology - Summit</p>
<p class="articleParagraph enarticleParagraph" >SPONSOR: The Government Executive Media Group's ATARC</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >TOPIC/SUBJECT: holds its Public Sector CIO Summit, focusing on "modernization at scale, evolving cyber imperatives, responsible AI and automation, and executive strategies for leading digital government."</p>
<p class="articleParagraph enarticleParagraph" >AGENDA: Highlights:</p>
<p class="articleParagraph enarticleParagraph" >-- 9:20 a.m.: Ankur Saini, acting CTO of the Transportation Department's Federal Motor Carrier Safety Administration, participates in a discussion on "Modernizing U.S. Government IT Infrastructure"</p>
<p class="articleParagraph enarticleParagraph" >-- 11:10 a.m.: Nick Marinos, managing director for information technology and cybersecurity at the <span class="companylink">U.S. Government Accountability Office</span>, participates in a discussion on "Exploring Cybersecurity Imperatives"</p>
<p class="articleParagraph enarticleParagraph" >DATE: December 11, 2025</p>
<p class="articleParagraph enarticleParagraph" >LOCATION: Carahsoft Conference & Collaboration Center, 11493 Sunset Hills Road, Suite 100, Reston, Va.</p>
<p class="articleParagraph enarticleParagraph" >CONTACT: 202-266-7332, events@govexec.com [Note: Register at <span class="colorLinks">https://events.atarc.org/cio-summit25/home/ [https://events.atarc.org/cio-summit25/home/]</span> ]</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gpir : Politics/International Relations | gpol : Domestic Politics | ncal : Calendar of Events | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usdc : Washington DC | uss : Southern U.S. | usva : Virginia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>CNG | TRN</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Federal Information & News Dispatch LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYB000020251205elcb0006e</td></tr></table><br/></div></div><br/><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WC50111020251205elcb00004"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  WealthStack Roundup: 55ip and InvestCloud Expand Strategic Partnership</b><div class="leadFields"><a href="javascript:void(0)">Retail Traffic</a>, 12:00 AM, 11 December 2025, 750 words,  Elaine Misonzhnik, (English)</div><div class="snippet ensnippet"> In other news, BetaNXT partners with Tumelo and Prudential advisors have a new mobile application.Fintech 55ip, a fully owned subsidiary of JPMorgan Chase, and InvestCloud expanded their strategic partnership to enable tax optimization for ...</div>
<div>(Document WC50111020251205elcb00004)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WC73345020251205elcb00001"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  &#x1f331; Patch AM: Why Syracuse’s red kettles are short on bell ringers this year</b><div class="leadFields"><a href="javascript:void(0)">Syracuse Patch</a>, 12:00 AM, 11 December 2025, 604 words, (English)</div><div class="snippet ensnippet"> Also on today&#39;s calendar: AUBURN - NYS Notary Law Exam Prep on Dec 11 at Cayuga CC led by Law Scholar Alfred E. Piombino and 9 more events.News we&#39;re reading</div>
<div>(Document WC73345020251205elcb00001)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WCHNASN020251205elcb001h3"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  IIT Bombay Announces the 21st Edition of E-Summit, Asia's Largest Business Conclave on 11-12 December</b><div class="leadFields"><a href="javascript:void(0)">Asian News International</a>, 02:00 PM, 11 December 2025, 492 words, (English)</div><div class="snippet ensnippet"> New Delhi [India], December 5: The Entrepreneurship Cell IIT Bombay, Asia&#39;s largest student-run non-profit dedicated to promoting entrepreneurship, presents the 21st E-Summit, Asia&#39;s largest business conclave, scheduled for 11th to 12th ...</div>
<div>(Document WCHNASN020251205elcb001h3)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WC50111020251204elcb00001"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  Five Beneficiary Designations For Clients to Review Now</b><div class="leadFields"><a href="javascript:void(0)">Retail Traffic</a>, 12:00 AM, 11 December 2025, 1515 words,  Daniel P. Michaelsen, (English)</div><div class="snippet ensnippet"> Don’t let problems spring up at the worst possible time.Your clients’ beneficiary designations are probably wrong.Not because they made bad decisions, but because they made them once and never looked again. Life changed. Their estate plan ...</div>
<div>(Document WC50111020251204elcb00001)</div><br/></td></tr>
						</table>
					</div>
				</div><span></span><div id="article-AIRWO00020251129elcb0000h" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/airwoLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Archer snaps up Hawthorne Airport</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>152 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>11 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Airliner World</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>AIRWO</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2025. Key Publishing Ltd. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >HAWTHORNE MUNICIPAL Airport, located three miles east of <span class="companylink">Los Angeles International Airport</span>, has been acquired by US air taxi firm <span class="companylink">Archer Aviation</span> for $126m. The airport, built in the 1920s and also known as Jack Northrop Field, sits on an 80-acre site that includes around 190,000sq ft of terminal, office and hangar facilities. San Jose-based Archer plans to make the airport its operational hub for its planned Los Angeles air taxi network, which it intends to launch ahead of the city’s hosting of the Summer 2028 Olympic and Paralympic Games. It is also looking to utilise the facility as “an innovation testbed for the next-generation of AI-powered aviation technologies” that it is currently developing and plans to deploy with its airline and technology partners. This, Archer said, includes AI-powered air traffic and ground operations management, along with other key technologies.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td></td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>buudiq : Archer Aviation Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i364 : Aerospace Products/Parts | i3640010 : Civil Aircraft | i764 : Airports | iaer : Aerospace/Defense | iairtr : Air Transport | iindstrls : Industrial Goods | itsp : Transportation/Logistics</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c23 : Research/Development | ccat : Corporate/Industrial News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>lax : Los Angeles | namz : North America | usa : United States | usca : California | usw : Western U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Key Publishing Ltd</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document AIRWO00020251129elcb0000h</td></tr></table><br/></div></div><br/><span></span><div id="article-DAYB000020251205elca00057" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daybLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Washington Daybook - General News Events - Futures</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Federal Information & News Dispatch, Inc./Agence France-Presse </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>123 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>10 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Washington Daybook</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYB</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Federal Information & News Dispatch, Inc. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Advisory Technology - Discussion</p>
<p class="articleParagraph enarticleParagraph" >SPONSOR: The Henry L. Stimson Center</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >TOPIC/SUBJECT: holds a virtual discussion, beginning at 10 a.m., on "Responsible AI in Democratic Society."</p>
<p class="articleParagraph enarticleParagraph" >PARTICIPANTS: Akintunde Ifeanyichukwu Agunbiade, associate at Aluko & Oyebode; Branka Panic, founding director at AI for Peace; Cristina Martinez Pinto, founder and CEO of the PIT Policy Lab; and Giulia Neaher, research analyst at the Stimson Center's Strategic Foresight Hub</p>
<p class="articleParagraph enarticleParagraph" >DATE: December 10, 2025</p>
<p class="articleParagraph enarticleParagraph" >LOCATION: None given</p>
<p class="articleParagraph enarticleParagraph" >CONTACT: 202-223-5956, communications@stimson.org or Caitlin Goodman, 202-478-3437 or 202-361-0254, cgoodman@stimson.org [Note: Register at <span class="colorLinks">https://www.stimson.org/event/responsible-ai-in-democratic-society/ [https://www.stimson.org/event/responsible-ai-in-democratic-society/]</span> ]</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gpir : Politics/International Relations | gpol : Domestic Politics | ncal : Calendar of Events | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usdc : Washington DC | uss : Southern U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Federal Information & News Dispatch LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYB000020251205elca00057</td></tr></table><br/></div></div><br/><span></span><div id="article-DAYB000020251205elca0005w" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daybLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Washington Daybook - General News Events - Futures</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Federal Information & News Dispatch, Inc./Agence France-Presse </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>126 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>10 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Washington Daybook</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYB</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Federal Information & News Dispatch, Inc. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >2 p.m. Foreign Affairs - Briefing</p>
<p class="articleParagraph enarticleParagraph" >SPONSOR: The Commission on Security and Cooperation in Europe (CSCE)</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >TOPIC/SUBJECT: holds a briefing on "From Production to Procurement: How Europe and Ukraine Are Transforming Defense Supply Chains."</p>
<p class="articleParagraph enarticleParagraph" >PARTICIPANTS: Maj. Gen. Karsten Jensen, defense attache at the Royal Danish Embassy in the United States; Kateryna Bondar, fellow at the Center for Strategic and International Studies's Wadhwani AI Center; and Sophia Besch, senior fellow at the <span class="companylink">Carnegie Endowment for International Peace</span>'s Europe Program</p>
<p class="articleParagraph enarticleParagraph" >DATE: December 10, 2025</p>
<p class="articleParagraph enarticleParagraph" >LOCATION: 2358-C Rayburn House Office Building</p>
<p class="articleParagraph enarticleParagraph" >CONTACT: 202-225-1901, csce.press@mail.house.gov [Note: Livestream at <span class="colorLinks">https://www.youtube.com/live/i8W5tE8eQSU [https://www.youtube.com/live/i8W5tE8eQSU]</span> ]</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gdip : International Relations | gpir : Politics/International Relations | gpol : Domestic Politics | ncal : Calendar of Events | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>dvpcoz : Developing Economies | eeurz : Central/Eastern Europe | eurz : Europe | namz : North America | ukrn : Ukraine | usa : United States | usdc : Washington DC | uss : Southern U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>DEF</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Federal Information & News Dispatch LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYB000020251205elca0005w</td></tr></table><br/></div></div><br/><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WC49877020251203elca00005"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  Automated Retail & Kiosk Innovation ShowIntegrating AI into self-service</b><div class="leadFields"><a href="javascript:void(0)">Vending Times</a>, 12:00 AM, 10 December 2025, 440 words, (English)</div><div class="snippet ensnippet"> Learn how AI will change self-service at the upcoming Automated Retail &amp; Kiosk Innovation Show.The world of self-service is transforming and AI is at the forefront.</div>
<div>(Document WC49877020251203elca00005)</div><br/></td></tr>
						</table>
					</div>
				</div><span></span><div id="article-DAYB000020251205elc90004m" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daybLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Washington Daybook - General News Events - Futures</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Federal Information & News Dispatch, Inc./Agence France-Presse </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>298 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>9 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Washington Daybook</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYB</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Federal Information & News Dispatch, Inc. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >[CODEZ] ,WAXC03C0925G093,WAOFFHILL,,,WAAI,,WAHITECH,,,WAAI,,WAHITECH,,, FOD, CMT, TEL, TLS ,</p>
<p class="articleParagraph enarticleParagraph" >8 a.m. Social Issues - Summit</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >SPONSOR: The Wall Street Journal</p>
<p class="articleParagraph enarticleParagraph" >TOPIC/SUBJECT: holds its CEO Council Summit,with the theme "Law, Politics, Technology and the Changing Context of Business," December 8-9.</p>
<p class="articleParagraph enarticleParagraph" >AGENDA: Highlights:</p>
<p class="articleParagraph enarticleParagraph" >-- 8 a.m.: Josh Stinchcomb, executive vice president and global chief revenue officer at Dow Jones, participates in a discussion on "AI for Growth: Going Beyond Efficiency"</p>
<p class="articleParagraph enarticleParagraph" >-- 9:25 a.m.: Kevin Hassett, director of the <span class="companylink">White House's National Economic Council</span>, delivers remarks on "Making Sense of the Economy"</p>
<p class="articleParagraph enarticleParagraph" >-- 10:15 a.m.: Horacio Rozanski, chairman, president and CEO of <span class="companylink">Booz Allen</span>, delivers remarks on "AI, Space and the Technologies Redefining U.S. Defense"</p>
<p class="articleParagraph enarticleParagraph" >-- 11 a.m.: Bob Hormats, former undersecretary of State for economic growth, energy and the environment and former vice chairman of <span class="companylink">Goldman Sachs</span>, delivers remarks on "Leading a Global Company in a Deglobalizing World"</p>
<p class="articleParagraph enarticleParagraph" >-- 12:40 p.m.: Mike Wirth, chairman and CEO of <span class="companylink">Chevron</span>, participates in a discussion on "Powering the AI Revolution"</p>
<p class="articleParagraph enarticleParagraph" >-- 1:05 p.m.: Chris Nicholas, president and CEO of Sam's Club, delivers remarks on "Delighting Customers in the Age of AI"</p>
<p class="articleParagraph enarticleParagraph" >-- 2:15 p.m.: Brian Nichol, chairman and CEO of <span class="companylink">Starbucks</span>, delivers remarks on "The Next Chapter at <span class="companylink">Starbucks</span>"</p>
<p class="articleParagraph enarticleParagraph" >-- 3:30 p.m.: Maggie Timoney, president and CEO of <span class="companylink">Heineken USA</span>, delivers remarks on "<span class="companylink">Heineken</span>'s High Stakes Bet on the Future"</p>
<p class="articleParagraph enarticleParagraph" >-- 4:25 p.m.: John Stankey, chairman and CEO of <span class="companylink">AT&T</span>, delivers remarks on "Leading Through Reinvention"</p>
<p class="articleParagraph enarticleParagraph" >DATE: December 9, 2025</p>
<p class="articleParagraph enarticleParagraph" >LOCATION: Washington, D.C.</p>
<p class="articleParagraph enarticleParagraph" >CONTACT: CEOCouncil@wsj.com [Note: Register at <span class="colorLinks">https://ceocouncil-wsj-com.ezproxy.cul.columbia.edu/event/ceo-council-summit-11/ [https://ceocouncil-wsj-com.ezproxy.cul.columbia.edu/event/ceo-council-summit-11/]</span> ]</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>usnecc : United States National Economic Council</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c41 : Management | ccat : Corporate/Industrial News | cslmc : Senior Level Management | gcat : Political/General News | gpir : Politics/International Relations | gpol : Domestic Politics | ncal : Calendar of Events | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | nfcpin : C&E Industry News Filter | niwe : IWE Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usdc : Washington DC | uss : Southern U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>DEF | EXE</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Federal Information & News Dispatch LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYB000020251205elc90004m</td></tr></table><br/></div></div><br/><span></span><div id="article-DAYB000020251205elc900048" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daybLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Washington Daybook - General News Events - Futures</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Federal Information & News Dispatch, Inc./Agence France-Presse </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>117 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>9 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Washington Daybook</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYB</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Federal Information & News Dispatch, Inc. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Advisory Technology - Discussion</p>
<p class="articleParagraph enarticleParagraph" >SPONSOR: <span class="companylink">Washington Post</span> Live</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >TOPIC/SUBJECT: holds a virtual discussion, beginning at 1 p.m., on "AI at Play: The Future of Sports, Powered by AI."</p>
<p class="articleParagraph enarticleParagraph" >PARTICIPANTS: Brian Moore, co-founder and CEO of Orreco; Lana Wong, founding member of Moderate the Panel; and Francessca Vasquez, vice president of <span class="companylink">Amazon Web Services</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >DATE: December 9, 2025</p>
<p class="articleParagraph enarticleParagraph" >LOCATION: None given</p>
<p class="articleParagraph enarticleParagraph" >CONTACT: Kathleen Floyd, 202-309-0785, Kathleen.Floyd@washpost.com [Note: Register at <span class="colorLinks">https://www.washingtonpost.com/washington-post-live/2025/12/09/future-sports-powered-by-ai/ [https://www.washingtonpost.com/washington-post-live/2025/12/09/future-sports-powered-by-ai/]</span> for virtual attendance. Livestream at <span class="colorLinks">https://www.youtube.com/c/WashingtonPostLive [https://www.youtube.com/c/WashingtonPostLive]</span>.]</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcsci : Computer Science | gpir : Politics/International Relations | gpol : Domestic Politics | gsci : Sciences/Humanities | ncal : Calendar of Events | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usdc : Washington DC | uss : Southern U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Federal Information & News Dispatch LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYB000020251205elc900048</td></tr></table><br/></div></div><br/><span></span><div id="article-DAYB000020251205elc90004e" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daybLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Washington Daybook - General News Events - Futures</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Federal Information & News Dispatch, Inc./Agence France-Presse </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>87 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>9 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Washington Daybook</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYB</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Federal Information & News Dispatch, Inc. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Advisory Technology - Discussion</p>
<p class="articleParagraph enarticleParagraph" >SPONSOR: The Government Executive Media Group</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >TOPIC/SUBJECT: holds a virtual discussion, beginning at 2 p.m., on "The Pitfalls and Opportunities of Leveraging AI in Research."</p>
<p class="articleParagraph enarticleParagraph" >PARTICIPANTS: Heidi Becker, project manager at Dimensions Research Security</p>
<p class="articleParagraph enarticleParagraph" >DATE: December 9, 2025</p>
<p class="articleParagraph enarticleParagraph" >LOCATION: None given</p>
<p class="articleParagraph enarticleParagraph" >CONTACT: 202-266-7332, events@govexec.com [Note: Register at <span class="colorLinks">https://events.govexec.com/digital-science-the-pitfalls-and-opportunities-of-leveraging-ai-in-research/ [https://events.govexec.com/digital-science-the-pitfalls-and-opportunities-of-leveraging-ai-in-research/]</span> ]</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gpir : Politics/International Relations | gpol : Domestic Politics | ncal : Calendar of Events | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usdc : Washington DC | uss : Southern U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Federal Information & News Dispatch LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYB000020251205elc90004e</td></tr></table><br/></div></div><br/><span></span><div id="article-DAYB000020251205elc900041" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daybLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Washington Daybook - General News Events - Futures</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Federal Information & News Dispatch, Inc./Agence France-Presse </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>156 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>9 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Washington Daybook</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYB</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Federal Information & News Dispatch, Inc. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Advisory Technology - Discussion</p>
<p class="articleParagraph enarticleParagraph" >SPONSOR: <span class="companylink">The Brookings Institution</span>
                     </p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >TOPIC/SUBJECT: holds a virtual discussion, beginning at 10:30 a.m., on "How to matter in the AI age: Lessons from home, school, and work."</p>
<p class="articleParagraph enarticleParagraph" >PARTICIPANTS: Jennie Wallace, journalist, author and founder of the Mattering Institute; Keri Rodrigues, co-founder and founding president of National Parents Union; Rebecca Winthrop, director of the Brookings Center for Universal Education and senior fellow at the Brookings Global Economy and Development Program; and E.J. Dionne, Jr., senior fellow in the Brookings Governance Studies Program and Brookings Center for Effective Public Management and Brookings chair in American governance</p>
<p class="articleParagraph enarticleParagraph" >DATE: December 9, 2025</p>
<p class="articleParagraph enarticleParagraph" >LOCATION: None given</p>
<p class="articleParagraph enarticleParagraph" >CONTACT: 202-797-6105 [Note: Submit questions to events@brookings.edu. Register at <span class="colorLinks">https://www.brookings.edu/events/how-to-matter-in-the-ai-age-lessons-from-home-school-and-work/ [https://www.brookings.edu/events/how-to-matter-in-the-ai-age-lessons-from-home-school-and-work/]</span> ]</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gpir : Politics/International Relations | gpol : Domestic Politics | ncal : Calendar of Events | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usdc : Washington DC | uss : Southern U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Federal Information & News Dispatch LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYB000020251205elc900041</td></tr></table><br/></div></div><br/><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WC45761020251204elc900005"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  SABCS: Advances in emerging blood-based surveillance tools, new approaches to predicting treatment response and tailoring therapies across...</b><div class="leadFields"><a href="javascript:void(0)">Newswise</a>, 12:00 AM, 9 December 2025, 1061 words, (English)</div><div class="snippet ensnippet"> Investigators from the UCLA Health Jonsson Comprehensive Cancer Center will present a wide range of new breast cancer research at the 2025 San Antonio Breast Cancer Symposium (SABCS), highlighting advances in early detection, precision ...</div>
<div>(Document WC45761020251204elc900005)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WC57065020251206elc800008"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  NorfolkNew program pairs seniors, young adults to help combat loneliness and isolationColter Anstaett</b><div class="leadFields"><a href="javascript:void(0)">WTKR-TV</a>, 12:00 AM, 8 December 2025, 451 words,  Colter Anstaett, (English)</div><div class="snippet ensnippet"> NORFOLK, Va. — There&#39;s a new effort underway in Hampton Roads to help combat loneliness and isolation in seniors and young adults.Making silly videos is just one way 22-year-old Carter Scott and 75-year-old Pearl Ross spend time together.</div>
<div>(Document WC57065020251206elc800008)</div><br/></td></tr>
						</table>
					</div>
				</div><span></span><div id="article-B000000020251206elc800001" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/bLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>In a K Economy, a Fat Cannibal Staredown Could Be Bad NewsIn a K Economy, a Fat Cannibal Staredown Could Be Bad News</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>By Jack Hough </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1117 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Barron's</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>B</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>9 9</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025 Dow Jones & Company, Inc. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >
                        <img src="../pro/default.aspx?napc=S&_XFORMSTATE=H4sIAAAAAAAEAD2LQQrCMBBF7zLrECZpmyazlAriRvAGsR1qSq0hFRXa3F1F9C8ePHh%2fUbQodG8QCkPQpYl8as%2fhznLix%2bzTLbQjyw1%2bp1FXSqPhsbUfVyBqgpiup%2bH%2fa%2fbbw3H3Kws0yklrZVmZ2q3h4nuWQ%2bR%2b7cIcwxNESVpoggZEQZhzfgEiOgsllAAAAA%3d%3d"/>

                     </p>
<p class="articleParagraph enarticleParagraph" >PHOTO: TIMOTHY A. CLARY / AFP via <span class="companylink">Getty Images</span>
                     </p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >The economy has shifted from a lowercase k to an uppercase one, a Wall Street researcher told me this past week. I nodded and hoped it wasn't a Kevorkian reference. It turns out it's all in the arms. The k's rising one and falling one signify different groups moving in opposite directions; the rich are thriving, while the rest are struggling. With a capital K, the rising arm is more prominent, just as swelling asset prices make now an even better time than usual to be rich.</p>
<p class="articleParagraph enarticleParagraph" >And the lower arm? It doesn't so much represent the poor, who tend to stay poor but at least receive inflation adjustments on government assistance, says Mike Reid, senior U.S. economist at <span class="companylink">RBC Capital Markets</span>. "It's really the middle-income folks...say, the 20th to 80th percentile. Those are the folks who are being pinched the most." Reid sees implications for investors and policymakers. If top earners are driving spending growth, for example, the economy might be more exposed than usual to stock market volatility.</p>
<p class="articleParagraph enarticleParagraph" >Meanwhile, I see a worrisome trend for letter-based financial metaphors. Already, we have S-shaped growth curves, L-shaped recessions, V- and U-shaped recoveries, and W-shaped whatevers. If interpreting the economy now hinges on fine differences between upper and lowercases, sloppy penmanship could lead to financial ruin. It's time to look beyond the alphabet for something more descriptive.</p>
<p class="articleParagraph enarticleParagraph" >Let's look at an example. Reid points out that from 1970 to 2010, there was just under one retiree for each new worker, whereas now the ratio has ballooned to about 2.5 to 1. "You don't need this strong job growth to support a stable unemployment rate," he says. But if stocks tank, boomers eyeing lower portfolio values could hold on to their jobs for longer, exacerbating any uptick in unemployment. This feels important enough to warrant a metaphor, yet too complicated to be summed up by any single letter—even a cursive capital G, if those still exist.</p>
<p class="articleParagraph enarticleParagraph" >What we need is a name that evokes young people waiting for old people to turn over their spots. I'm torn between Pickleball Hours at the Town Basketball Court Effect and Restroom at a Rolling Stones Concert Dilemma.</p>
<p class="articleParagraph enarticleParagraph" >Other strategists have spotted important market wrinkles. At BofA Securities, Jill Carey Hall, who oversees U.S. small- and mid-cap research, points out that earnings growth underlying the S&P SmallCap 600 index is expected to jump from 6% this year to 17% next year. Small companies would go from being growth laggards to leaders. Historically, when that has happened, small-caps have outperformed large-caps 75% of the time, by an average of nine percentage points a year.</p>
<p class="articleParagraph enarticleParagraph" >This is both too tempting to ignore and too long-awaited to trust. Over the past 15 years, the large-cap S&P 500 index has returned 640%, trouncing its small-cap sibling by more than 280 points. That has left small-caps 30% cheaper than large-caps relative to this year's projected earnings. The ideal metaphor here will call to mind something small that is poised to go fast, but isn't nearly assured of success. If you ask me, 2026 is displaying a Chihuahua on an E-Bike Setup. Tiny dogs appear ready to zoom. But their little paws can't reach the handlebars, so really, anything could happen.</p>
<p class="articleParagraph enarticleParagraph" >Among large-cap stocks, has anyone else noticed a developing Fat Cannibal Staredown? That's where market behemoths, who have been feasting on small competitors for decades, begin to hungrily eye one another's total addressable markets. <span class="companylink">Alphabet</span>, No. 3 by market value, has developed its own zippy artificial-intelligence chips with help from No. 6 <span class="companylink">Broadcom</span>, and is now seen as a threat to No. 1 <span class="companylink">Nvidia</span>, who says its chips are set apart by their software, long the domain of <span class="companylink">Microsoft</span>, which is busy battling for cloud AI work with <span class="companylink">Amazon.com</span>, which is now selling AI chips of its own. <span class="companylink">Tesla</span>, which has fallen out of the top seven, now says it's all about robo-taxis, but <span class="companylink">Alphabet</span>'s <span class="companylink">Waymo</span> is the early leader there. <span class="companylink">Alphabet</span>'s <span class="companylink">YouTube</span> is giving streamers like <span class="companylink">Netflix</span> and <span class="companylink">Amazon</span> pause, but <span class="companylink">Amazon</span>'s rapid rise in advertising is a menace to <span class="companylink">Alphabet</span> and <span class="companylink">Meta Platforms</span>.</p>
<p class="articleParagraph enarticleParagraph" >A Fat Cannibal Staredown is always a volatile situation, especially when combined with a Muffin Top Market—technology, media, and telecom now combine for 45% of the S&P 500 index. If we're lucky, all of that emerging AI wizardry will help more than it hurts. <span class="companylink">Deutsche Bank</span> says that AI next year will either weaken a fragile job market or boost productivity by more than half a point. It's a classic Family Reunion Bearhug From a Cousin You Like but Who Talks Too Much About Crypto Conundrum, and I have mixed feelings.</p>
<p class="articleParagraph enarticleParagraph" >Would it be awkward here to change topics to obesity meds and erectile function? Definitely? I wish you had said something sooner.</p>
<p class="articleParagraph enarticleParagraph" >It's just that <span class="companylink">Oppenheimer & Co.</span> has been highlighting the role that obesity meds, along with new treatments for cancer, infectious diseases, movement disorders, and more, can play in extending longevity and quality of life. Next up might be more attention on erectile function, which <span class="companylink">Oppenheimer</span> calls a harbinger of health. Women outlive men by an average of 5.8 years and rising in the U.S., and heart disease, diabetes, and emotional distress are leading reasons. Erectile dysfunction is often the first detectable symptom of these, and it appears to be both a cause and effect. Treat ED, in other words, and you might extend lives.</p>
<p class="articleParagraph enarticleParagraph" >That makes it surprising that leading ED treatments, called PDE5 inhibitors, and better known as Viagra and Cialis, are about a quarter-century old. <span class="companylink">Oppenheimer</span> points to a tiny Swedish company called <span class="companylink">Dicot Pharma</span> and its experimental treatment LIB-01. It has been shown in preliminary trials to work for weeks, not just hours—in a when-needed way rather than an intrusive one, as I understand it, but I don't have the medical grasp of a trained men's downstairs-ologist. Dicot will need more capital and lengthy trials, and it might succeed or flop, but it's nice to see new movement in a neglected field.</p>
<p class="articleParagraph enarticleParagraph" >Write to Jack Hough at <span class="colorLinks">jack.hough@barrons.com [mailto:jack.hough@barrons.com]</span>. <span class="colorLinks">Follow him on X [https://twitter.com/jackhough]</span> and subscribe to his <span class="colorLinks">Barron's Streetwise podcast [https://www-barrons-com.ezproxy.cul.columbia.edu/podcasts/streetwise?page=1&mod=podcasts_tile]</span>.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302022 : Artificial Intelligence Technologies | i34531 : Semiconductors | iindele : Industrial Electronics | iindstrls : Industrial Goods | iintcir : Integrated Circuits | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c1521 : Analysts' Comments/Recommendations | ccat : Corporate/Industrial News | gcat : Political/General News | gpersf : Personal Finance | gpersi : Personal Investments | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | nimage : Images</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>AMZN | AVGO | DB | DBK.XE | GOOGL | I/ELQ | I/ETK | I/SEM | LLM | M/IDU | M/TEC | META | MSFT | N/DJN | N/GEN | N/PFN | N/WER | NFLX | NVDA | TSLA</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>Barrons.com | Streetwise Barrons</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Dow Jones & Company, Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document B000000020251206elc800001</td></tr></table><br/></div></div><br/><span></span><div id="article-B000000020251206elc8000b5" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/bLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>CLM</b>&nbsp;</td><td>Review</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Hope for the Home Buyer</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>By Shaina Mishkin </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>812 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Barron's</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>B</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>10</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025 Dow Jones & Company, Inc. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >
                        <img src="../pro/default.aspx?napc=S&_XFORMSTATE=H4sIAAAAAAAEAD2LywrCMBBF%2f2XWIUymj8RZSgVxI%2fgHaRpqSq0hFRXa%2fLsF0bs4cODcRfGicLeBUdQMXZrYJncNTy8n%2f5ptegQ3ernH7wipUoS1H53ZtK1AaIaY7u3w%2fzWnw%2fly%2fJUFloakMZK01moNN9t7OUTfr12YY3iDKJkEMTQgCsac8wcL8H%2bZlAAAAA%3d%3d"/>

                     </p>
<p class="articleParagraph enarticleParagraph" >PHOTO: Illustration by Elias Stein</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >It has been a tough few years for home buyers, but there's a glimmer of hope for 2026. Two new forecasts say home prices will rise modestly next year: <span class="companylink">Redfin</span> expects a 1% price increase nationally, while Realtor.com sees a 2.2% gain. Both anticipate prices growing slower than wages, and mortgage rates averaging 6.3%. <span class="companylink">Redfin</span> predicts a 3% lift in sales, while Realtor.com estimates 1.7%. (Barron's parent <span class="companylink">News Corp</span> runs Realtor.com.)</p>
<p class="articleParagraph enarticleParagraph" >But location, of course, is everything. Prices in September continued to rise in Northeastern and Midwestern metros tracked by S&P Cotality Case-Shiller Home Price Indices, and slid in Sunbelt metros like Phoenix, Dallas, and Miami. It's what <span class="companylink">S&P Dow Jones Indices</span>' Nicholas Godec calls a "tale of two markets."</p>
<p class="articleParagraph enarticleParagraph" >Realtor.com forecasts that 2026 prices will fall below year-ago levels in 22 of 100 metro areas. Two Florida locales, Cape Coral and North Port, lead in anticipated declines, with 10.2% and 8.9% price drops, respectively. In October, active listings in both metros were 13.3% higher than 2024, which probably fuels expectation of softer prices.</p>
<p class="articleParagraph enarticleParagraph" >Elsewhere, prices will climb. Metros with the largest anticipated gains include some of the most affordable housing markets: Toledo, Ohio (13.1%); Syracuse, N.Y. (12.4%); and Scranton, Pa. (10.9%). October listing prices in all three areas were below $300,000, according to Realtor.com—well under the national $424,200 median.</p>
<p class="articleParagraph enarticleParagraph" >Write to Shaina Mishkin at <span class="colorLinks">shaina.mishkin@dowjones.com [mailto:shaina.mishkin@dowjones.com]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Last Week</p>
<p class="articleParagraph enarticleParagraph" >Markets</p>
<p class="articleParagraph enarticleParagraph" >Bitcoin continued to swoon, falling 6% on Monday, leading global stocks and bonds down. U.S. stocks fell, with the Dow industrials down nearly a point, then rallied amid volatility on hopes for a Federal Reserve rate cut. ADP said the economy lost 32,000 jobs in November, and late-November jobless claims fell to a three-year low. On the week, the Dow rose 0.5%, the S&P 500 0.3%, and the Nasdaq Composite 0.9%.</p>
<p class="articleParagraph enarticleParagraph" >Companies</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Walt Disney</span>'s Zootopia 2 took in $156 million in the U.S. and Canada, $400 million internationally. Regulators ordered fixes to a software glitch on some 6,000 <span class="companylink">Airbus</span> A320s, and the company cut its 2025 delivery target. With its shares down 60% in the Bitcoin selloff, Strategy established a $1.44 billion U.S.-dollar reserve to pay preferred stock dividends and interest payments. Blackstone, <span class="companylink">Apollo Global</span>, and <span class="companylink">KKR</span> will participate in a <span class="companylink">Bank of England</span> private-market stress test. <span class="companylink">Apple</span> replaced its AI chief, John Giannandrea, with <span class="companylink">Microsoft</span>'s Amar Subramanya. The <span class="companylink">European Union</span> opened an antitrust probe into <span class="companylink">Meta Platforms</span> embedding AI tools in WhatsApp. The White House moved to scrap Biden-era car fuel-efficiency standards.</p>
<p class="articleParagraph enarticleParagraph" >Deals</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Netflix</span> said it would acquire <span class="companylink">Warner Bros. Discovery</span>'s movie and TV studio and <span class="companylink">HBO</span> MAX streaming businesses for $83 billion, including debt, beating out <span class="companylink">Paramount Skydance</span> and <span class="companylink">Comcast</span>...Chip maker <span class="companylink">Marvell Technology</span> agreed to buy <span class="companylink">Celestial AI</span> for $3.25 billion.</p>
<p class="articleParagraph enarticleParagraph" >Next Week</p>
<p class="articleParagraph enarticleParagraph" >Tuesday 12/9</p>
<p class="articleParagraph enarticleParagraph" >The <span class="companylink">Bureau of Labor Statistics</span> releases the Job Openings and Labor Turnover Survey for both September and October. At the end of August, there were 7.22 million job openings, and 1.02 unemployed people for every open position, the highest ratio since April 2021.</p>
<p class="articleParagraph enarticleParagraph" >Wednesday 12/10</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Adobe</span> and <span class="companylink">Oracle</span> report quarterly results on Wednesday, followed by <span class="companylink">Broadcom</span> and <span class="companylink">Costco Wholesale</span> on Thursday.</p>
<p class="articleParagraph enarticleParagraph" >The Federal Open Market Committee announces its monetary-policy decision. The FOMC is widely expected to cut the federal-funds rate by a quarter of a percentage point to 3.5%-3.75%. The central bank also releases its quarterly Summary of Economic Projections. In the September SEP, the median projection for the federal-funds rate by year-end 2026 was 3.4%, which would imply only one more quarter-point cut, assuming that the FOMC cuts as expected at this meeting. Traders are pricing in a roughly 3% federal-funds rate by December 2026, a much more aggressive easing cycle than currently forecast by the central bank. This may be due to the dovish Kevin Hassett, currently director of the National Economic Council and the favorite to replace Jerome Powell, whose term as Fed chair ends in May.</p>
<p class="articleParagraph enarticleParagraph" >The Numbers</p>
<p class="articleParagraph enarticleParagraph" >4.1%</p>
<p class="articleParagraph enarticleParagraph" >Increase in Black Friday retail sales from 2024, excluding autos, up from 2024's 3.4%, <span class="companylink">Mastercard</span> estimated.</p>
<p class="articleParagraph enarticleParagraph" >64%</p>
<p class="articleParagraph enarticleParagraph" >Percentage of venture funding, roughly $161 billion, going into AI over the first nine months of 2025.</p>
<p class="articleParagraph enarticleParagraph" >4 M</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">International Energy Agency</span>'s estimate of excess per-day supply of barrels of oil in 2026, a record.</p>
<p class="articleParagraph enarticleParagraph" >$16 T</p>
<p class="articleParagraph enarticleParagraph" >The combined wealth of the world's billionaires, up 13% since last year, from a <span class="companylink">UBS</span> report.</p>
<p class="articleParagraph enarticleParagraph" >Write to Robert Teitelman at <span class="colorLinks">bob.teitelman@dowjones.com [mailto:bob.teitelman@dowjones.com]</span>
                  </p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c15 : Financial Performance | c151 : Earnings | ccat : Corporate/Industrial News | e11 : Economic Performance/Indicators | e1121 : Home Sales/Housing Affordability Figures | ecat : Economic News | ereal : Real Estate Markets | m11 : Equity Markets | mcat : Commodity/Financial Market News | ncat : Content Types | ncolu : Columns | nfact : Factiva Filters | nfce : C&E Exclusion Filter | nfcpin : C&E Industry News Filter | nimage : Images</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>AAPL | AIR.FR | CMCSA | DIS | EADSY | G/UKBK | KKR | LLM | MA | META | MRVL | MSFT | N/CNW | N/DJN | N/ERN | N/GENI | N/IEN | N/MKT | N/PFM | N/SLS | N/STK | N/WER | NFLX | NWS | NWS.AU | NWSA | R/NME | R/US | WBD</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>Barrons.com | Review</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Dow Jones & Company, Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document B000000020251206elc8000b5</td></tr></table><br/></div></div><br/><span></span><div id="article-B000000020251205elc80002w" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/bLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>It's Time for Your Year-End Portfolio Review --- A good year for stocks—and big gains in tech—may be making your portfolio too risky. How to get it in shape for 2026.</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>By Elizabeth O'Brien </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>2071 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Barron's</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>B</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>16</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025 Dow Jones & Company, Inc. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >It's always smart to take stock of your portfolio in December, but this year it's especially key. The investment landscape may be shifting, and retirees can't afford to coast into 2026 on autopilot.</p>
<p class="articleParagraph enarticleParagraph" >The S&P 500 and the Nasdaq Composite are posting their third consecutive year of double-digit returns. Artificial intelligence has been the market's lifeblood, but it's debatable how much longer the AI trade will last. Also unknown is the impact of AI on corporate profits and the economy—and how that will impact markets.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >"We have yet to see how the AI-driven productivity boom plays out," says Kevin Khang, senior international economist at <span class="companylink">Vanguard</span>.</p>
<p class="articleParagraph enarticleParagraph" >If the tech bulls are right, AI could become as transformative as electricity, he says. That would keep propelling equities and economic growth. It's also possible that AI's impact will be more subdued, while the U.S. grapples with rising debt and deficits.</p>
<p class="articleParagraph enarticleParagraph" >For the next few years, "it will be a horse race between AI and deficits," Khang says.</p>
<p class="articleParagraph enarticleParagraph" >Many investment pros expect the stock market to keep rising in 2026. CFRA Research sees the S&P 500 at 7400 by the end of 2026. That would be an 8% gain from recent levels around 6860. Historically, midterm election years are extra volatile for markets, according to CFRA Chief Investment Strategist Sam Stovall. But he sees a more muted market chugging ahead on the back of the Fed's rate-easing cycle and double-digit earnings growth expectations for 2026 and 2027.</p>
<p class="articleParagraph enarticleParagraph" >Another good year is no guarantee, though. And investors may have forgotten how to play defense after a long stretch of gains. If the economy tips into a recession, the market will almost certainly fall. If your portfolio isn't ready, you could face steep losses.</p>
<p class="articleParagraph enarticleParagraph" >Here are some ways to prepare, and year-end moves to consider.</p>
<p class="articleParagraph enarticleParagraph" >Consolidate Your Accounts</p>
<p class="articleParagraph enarticleParagraph" >It's hard to take a holistic view of your portfolio when your investments are scattered. Nearly a third of near-retirees reported having two or more 401(k)s, according to Allspring's 2025 Retirement Study. If you suspect you have lost sight of an old account, you can search the government's <span class="colorLinks">retirement savings lost-and-found database [https://lostandfound.dol.gov/]</span>.</p>
<p class="articleParagraph enarticleParagraph" >Investors may think they are diversifying by spreading their money around at different firms. What matters is your asset mix, regardless of where it's held.</p>
<p class="articleParagraph enarticleParagraph" >Having multiple accounts at different firms may also be more trouble than it's worth, says Bethany Dever, a certified financial planner at <span class="companylink">Rockland Trust</span>. Investors can lose sight of proper rebalancing, performance tracking, and tax reporting, she says, and they may miss out on volume discounts for bigger balances.</p>
<p class="articleParagraph enarticleParagraph" >More investors are getting the message. From 2024 to 2025, the share of households with investible assets of $1 million to $4.9 million at only one financial institution jumped 11 percentage points to 22%, according to research firm Hearts & Wallets.</p>
<p class="articleParagraph enarticleParagraph" >Even if you're comfortable juggling accounts across multiple firms, chances are your loved ones won't be. Once the primary money manager in a couple becomes infirm or dies, "it's a burden on the spouse to have all these accounts," Dever says. What's more, <span class="colorLinks">many financial institutions have their own power-of-attorney protocols [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/how-ai-could-kill-the-great-wealth-transfer-327bc6ab]</span>. The fewer of them your loved ones have to deal with, the better. </p>
<p class="articleParagraph enarticleParagraph" >
                     <img src="../pro/default.aspx?napc=S&_XFORMSTATE=H4sIAAAAAAAEAD2LwQrCMBAF%2fyXnEDabtkn3KBXEi%2bAfxDTUlLaGVKzQ5t9VFN9hYGDeKmmVUL9BwCtibZrIJncNDy8mv8w23YMbvNjBdwhYSoTSD858dGFcE4vpdun%2fv%2ba4P50Pv7JSUGgljBFodK22MNrOiz76bmvDHMOT8YKQI7GGcUWQc34B%2flFnN5QAAAA%3d"/>

                  </p>
<p class="articleParagraph enarticleParagraph" >PHOTO: Illustration by Jori Bolton</p>
<p class="articleParagraph enarticleParagraph" >Keeping your accounts at one brokerage should provide enough protection against fraud or a firm going bankrupt. Major brokerages carry excess coverage well above the SIPC insurance of $500,000 per account type per member firm. This coverage doesn't protect against market losses, just brokerage failure and certain kinds of fraud. By contrast, <span class="companylink">FDIC</span> insurance protects cash up to $250,000 per depositor per bank. Savers with higher cash balances have an incentive to spread their money across different institutions.</p>
<p class="articleParagraph enarticleParagraph" >Diversify, Truly</p>
<p class="articleParagraph enarticleParagraph" >With your investments in one place, it's easier to tell if your asset mix is aligned with your goals. Your portfolio will drift according to the underlying performance of stocks and bonds, so your balance can get out of whack over time. Investors who began retirement 10 years ago with a portfolio of 60% stocks and 40% bonds, and haven't touched it since, would now have more than 80% in stocks, according to an illustration by <span class="companylink">Morningstar</span>.</p>
<p class="articleParagraph enarticleParagraph" >Holding 80% in equities would be too risky for most retirees, especially given stocks' lofty valuations. The S&P 500 trades at a forward 12-month price/earnings ratio of 22.4, above the five-year average of 20 and the 10-year average of 18.7, according to <span class="companylink">FactSet</span>.</p>
<p class="articleParagraph enarticleParagraph" >The market's weighting in giant tech stocks like <span class="companylink">Nvidia</span>, <span class="companylink">Microsoft</span>, and <span class="companylink">Alphabet</span> is another concern. The top 10 stocks in the S&P 500, which is weighted by market cap, constitute nearly 40% of the index. Most are AI-related names.</p>
<p class="articleParagraph enarticleParagraph" >Few pros would recommend exiting big tech stocks altogether. But if you have an outsize allocation, now is the time to trim. Within equities, <span class="companylink">Vanguard</span>'s Khang recommends that retirees keep less than 50% in the S&P 500.</p>
<p class="articleParagraph enarticleParagraph" >For the remainder, he suggests diversifying within the U.S. and international equity universes. That may include small- and mid-cap stocks, value stocks, and non-U.S. markets.</p>
<p class="articleParagraph enarticleParagraph" >One simple way to diversify your U.S. stockholdings is with the Invesco S&P 500 Equal Weight exchange-traded fund, which holds every stock in the index in equal amounts. It doesn't have the tailwind of megacap tech, but it's faring well this year, up 10% versus 15% for the S&P 500.</p>
<p class="articleParagraph enarticleParagraph" >Value stocks are another good diversifier, since the S&P 500 is tilted toward growth. The Dow Jones Industrial Average has a value tilt, Khang says, and many of its components pay a dividend.</p>
<p class="articleParagraph enarticleParagraph" >The <span class="companylink">SPDR Dow Jones Industrial Average ETF Trust</span> is one way to play it. While the fund still has tech names like <span class="companylink">Nvidia</span>, it also owns value-oriented stocks like <span class="companylink">Verizon Communications</span> and pays a 1.5% dividend.</p>
<p class="articleParagraph enarticleParagraph" >Some fund managers are looking to diversify beyond AI. One way to do it: Look for companies that will benefit from AI spending regardless of whether the Magnificent Seven companies recoup their own massive investments. Think electricity and other infrastructure providers that are needed to build AI data centers, says Neil Hennessy, chief market strategist at Hennessy Funds.</p>
<p class="articleParagraph enarticleParagraph" >"If you think pork will go up, you don't buy the pig; you buy the feed manufacturer," Hennessy said at an investment outlook event in November.</p>
<p class="articleParagraph enarticleParagraph" >One stock Hennessy owns for his firm's investors is <span class="companylink">Granite Construction</span>. The company builds roads and sells construction materials such as asphalt. And it's benefiting indirectly from AI, as companies need access roads to data centers. "There's strong demand associated with data center infrastructure improvements and expansion and development," CEO Kyle Larkin said on a recent earnings call.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Granite</span> has a market cap of $4.7 billion and earnings growth projected at 30% in 2026 to $5.54 a share, according to consensus forecasts. Shares trade at 19 times earnings, a slight discount to the market.</p>
<p class="articleParagraph enarticleParagraph" >Small- and mid-cap stocks have other benefits, says Paul Stanley, chief investment officer of Granite Bay Wealth Management in Portsmouth, N.H. Mid-caps have lagged behind the S&P 500 for years but offer more attractive valuations, he says. They also tend to be domestically oriented industrial and financial companies that don't have as much exposure to tariffs and dollar risk. Small-caps should benefit from lower interest rates since they tend to borrow more. ETFs offering exposure include Dimensional U.S. Small Cap and iShares Russell Mid-Cap Growth.</p>
<p class="articleParagraph enarticleParagraph" >Don't neglect international stocks. After years of lagging behind U.S. stocks, international stocks are up nearly 30% this year. U.S. stocks make up about 65% of the world's equity market cap, so some pros recommend that investors hold around 35% of their stock allocation outside this country. An ETF like iShares MSCI ACWI ex U.S. is an easy way to get access.</p>
<p class="articleParagraph enarticleParagraph" >Bonds Are Back</p>
<p class="articleParagraph enarticleParagraph" >Once your equity exposure is diversified, consider your overall mix of stocks and bonds. If your stock portion has ballooned beyond its target, sell some winners and buy bonds to get your allocation back on track.</p>
<p class="articleParagraph enarticleParagraph" >The outlook for fixed income is solid, Khang says: It isn't like 2020, when paltry bond yields meant there was no alternative to stocks—a phenomenon known by the acronym TINA. Nor is it like 2022, when rapidly rising interest rates caused bond prices to plunge (since the two move inversely).</p>
<p class="articleParagraph enarticleParagraph" >Instead, bonds are back to normal valuations. Anders Persson, chief investment officer and head of global fixed income at <span class="companylink">Nuveen</span>, expects 10-year Treasuries to stay in the 4% vicinity through the end of 2026. That's a respectable coupon for retirees to clip. If the <span class="companylink">Federal Reserve</span> continues to cut rates, there's more possibility for total return, including price gains. </p>
<p class="articleParagraph enarticleParagraph" >
                     <img src="../pro/default.aspx?napc=S&_XFORMSTATE=H4sIAAAAAAAEAD2LwQrCMBAF%2f2XPIWzSNkn3KBXEi%2bAfxDTUlLaGVKzQ5t9VFN9hYGDeKmgVWL9ByBRBmyayyV3Dw%2fPJL7NN9%2bAGz3f4nURZCYmVH5z56AJME8R0u%2fT%2fX3Pcn86HX6kKLHXJjeGiNlptYbSd53303daGOYYnsJIkkwQNsIIw5%2fwCzmhgNpQAAAA%3d"/>

                  </p>
<p class="articleParagraph enarticleParagraph" >PHOTO: Illustration by Jori Bolton</p>
<p class="articleParagraph enarticleParagraph" >The <span class="companylink">iShares Core U.S. Aggregate Bond ETF</span> is a simple way to get exposure to the intermediate part of the yield curve that many pros find attractive right now. Alternatively,<span class="colorLinks">actively managed funds [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/active-bond-funds-worth-apremium-1b40315a]</span> look for opportunities throughout the whole fixed-income landscape. Examples include <span class="companylink">Nuveen</span> Strategic Income and Dodge & Cox Global Bond, the latter including international debt securities.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">Municipal bonds [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/fund-sweet-spot-muni-bonds-b929d126]</span> remain attractive for investors in higher tax brackets, since the income they generate is generally exempt from federal and state taxes for residents of the state that issued the bond. The low-cost <span class="companylink">Vanguard Tax-Exempt Bond ETF</span> is a solid choice; it yields 3.5%—or a taxable equivalent north of 5% for investors in the top three tax brackets.</p>
<p class="articleParagraph enarticleParagraph" >Manage Your Taxes Wisely</p>
<p class="articleParagraph enarticleParagraph" >If you are tweaking your portfolio, it's usually most tax-efficient to make changes within a tax-deferred retirement account like a traditional 401(k) or individual retirement account, or IRA, since buying and selling within these accounts won't generate taxable capital gains. Of course, if your goal is to realize losses to offset gains in a taxable account, confining your activity to tax-deferred accounts would be a minus.</p>
<p class="articleParagraph enarticleParagraph" >Investors age 73 and over who are subject to required minimum distributions can use their RMDs as a rebalancing tool. Consider taking your RMD from an overweighted part of your portfolio. If you don't need the money to live on, invest the proceeds in a taxable account in an asset class where you are underweight. If your <span class="colorLinks">cash cushion isn't big enough [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/stock-market-selloff-retirement-planning-38c4b07e]</span>, that's another good place to plow your proceeds. Nearly one-third of RMDs at <span class="companylink">Fidelity</span> are taken in November and December, according to a spokeswoman.</p>
<p class="articleParagraph enarticleParagraph" >This year offers slim pickings for tax-loss harvesting. That's the practice of selling losing positions to offset taxes on capital gains, which may be plentiful this year.</p>
<p class="articleParagraph enarticleParagraph" >One way for retirees to avoid capital gains—and get a tax break—is by transferring appreciated stock to charity. A "donor advised fund" offers maximum flexibility: You can transfer appreciated shares to the fund and get an immediate tax deduction, then take your time deciding how and when to disperse the proceeds to charities.</p>
<p class="articleParagraph enarticleParagraph" >Through 2025, taxpayers can typically get a tax deduction from charitable giving only if they itemize their deductions. Starting next year, <span class="colorLinks">taxpayers who take the standard deduction will receive an above-the-line charitable deduction [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/tax-rules-planning-7a14eb79]</span>of up to $1,000 for a single person and $2,000 for a married couple filing jointly. That gives taxpayers who take the standard deduction a reason to push their charitable giving into January of next year.</p>
<p class="articleParagraph enarticleParagraph" >While you can time your charitable giving, you can't time the market. That's why it's helpful to review your portfolio and rebalance on a fixed schedule, rather than in response to market gyrations. Because some of your moves may have tax implications, December is a good time to get it done.</p>
<p class="articleParagraph enarticleParagraph" >Write to Elizabeth O'Brien at <span class="colorLinks">elizabeth.obrien@barrons.com [mailto:elizabeth.obrien@barrons.com]</span>
                  </p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>ccat : Corporate/Industrial News | nadc : Advice | ncat : Content Types | nimage : Images | npag : Page One Stories</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>AGG | DIA | FIN.XX | G/FDC | G/FED | GOOGL | GVA | LLM | MORN | MSFT | N/DJN | N/GEN | N/PFN | N/WER | NVDA | R/NME | R/US | TIA.XX | VGI.XX | VTEB | VZ</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>Barrons.com | Features - Main</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Dow Jones & Company, Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document B000000020251205elc80002w</td></tr></table><br/></div></div><br/><span></span><div id="article-B000000020251205elc800008" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/bLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>CLM</b>&nbsp;</td><td>Technology Trader</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>
                           Apple Has Stayed Out Of the AI Race. It's Winning Anyway.</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>By Adam Levine </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>986 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Barron's</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>B</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>29</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025 Dow Jones & Company, Inc. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >
                        <span class="companylink">Apple</span> has emerged from the AI doghouse. The stock hit a new all-time high this past week after surging 39% since Aug. 1. The rally follows the botched rollout of Apple Intelligence, <span class="companylink">Apple</span>'s effort to integrate artificial intelligence into its devices.</p>
<p class="articleParagraph enarticleParagraph" >The big piece of Apple Intelligence was supposed to be a new version of <span class="companylink">Apple</span>'s digital personal assistant, Siri, that works like the top AI chatbots from <span class="companylink">OpenAI</span> and <span class="companylink">Alphabet</span>. A smarter assistant is something <span class="companylink">Apple</span> users have craved since Siri first arrived in 2011. But the project has been indefinitely delayed.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >The new Siri is proving to be difficult because <span class="companylink">Apple</span> came into the AI race by handicapping itself. It is the only Big Tech company that sees privacy and security as marketable features, not cost centers. Any implementation of the new Siri will need to meet <span class="companylink">Apple</span> standards in this regard, and that is proving to be a big hurdle.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Apple</span>'s strong preference is for all machine learning to happen on encrypted <span class="companylink">Apple</span> devices, leveraging special units in <span class="companylink">Apple</span>'s chips. Nothing is more private and secure. But the "frontier" language models that underlie ChatGPT and Gemini run in giant data centers, and are far too demanding for a phone. Much smaller models that can run on a phone don't yet provide a consistently good enough user experience for <span class="companylink">Apple</span>.</p>
<p class="articleParagraph enarticleParagraph" >So we wait. Meanwhile, Wall Street seems to have moved on to a new narrative: It doesn't matter if <span class="companylink">Apple</span> is late to AI.</p>
<p class="articleParagraph enarticleParagraph" >While most of Big Tech is sprinting to an AI future, <span class="companylink">Apple</span> is running a marathon. Only time will tell who is right, but I share <span class="companylink">Apple</span>'s long view of the AI boom. The company can take its time fitting AI into its products.</p>
<p class="articleParagraph enarticleParagraph" >So far, hundreds of billions in capital expenditures are bringing Big Tech to the same place: AI models that struggle to stand out from each other.</p>
<p class="articleParagraph enarticleParagraph" >It turns out that having the best AI models isn't a moat, just a fleeting advantage. Many enterprise customers have said that AI language models are becoming commoditized, most recently <span class="companylink">Salesforce</span> CEO Marc Benioff.</p>
<p class="articleParagraph enarticleParagraph" >"We use all of the large language models," he said on the company's Wednesday third-quarter earnings call. "They're all very good at this point, so we can swap them in and out. The lowest-cost one is the best one for us."</p>
<p class="articleParagraph enarticleParagraph" >There have been reports that <span class="companylink">Apple</span> is in talks with <span class="companylink">Alphabet</span> and start-up <span class="companylink">Anthropic</span> to use their AI models, fine-tuned for <span class="companylink">Apple</span> hardware, as a stopgap until the company can create its own high-performing models.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Apple</span> is pacing itself, putting user experience and privacy above expediency. Amid everyone else's AI battle, <span class="companylink">Apple</span> created its Private Cloud Compute: open-source server software written in <span class="companylink">Apple</span>'s programming language, running on <span class="companylink">Apple</span> servers that sport <span class="companylink">Apple</span> chips. As always, the company wants to own and control the whole stack, especially when it comes to privacy and security. AI chats can include very personal information, and Private Cloud Compute hides them from peering eyes, including <span class="companylink">Apple</span>'s. At some point, an upgraded Siri will arrive and it will be more secure than any other chatbot.</p>
<p class="articleParagraph enarticleParagraph" >Meanwhile, <span class="companylink">Apple</span> is keeping its powder dry, increasing capital expenditures modestly to support Private Cloud Compute. By contrast, <span class="companylink">Meta Platforms</span>, <span class="companylink">Oracle</span>, <span class="companylink">Microsoft</span>, and <span class="companylink">Google</span> are polluting their once-pristine cash flow statements and balance sheets with hundreds of billions in combined capital expenditures for AI data centers. <span class="companylink">Meta</span> stands out, spending around $70 billion on AI data centers this year, and promising more next year. It's all for its own use, not to rent out in the cloud like the others. Debt levels are rising, and depreciation expenses from capex are beginning to mount. They will keep rising.</p>
<p class="articleParagraph enarticleParagraph" >As <span class="companylink">Alphabet</span>'s depreciation was up 41%, <span class="companylink">Microsoft</span>'s 93%, and <span class="companylink">Meta</span>'s 20%, <span class="companylink">Apple</span>'s rose just 7% in the latest quarter. If a time comes where big capital outlays make sense, <span class="companylink">Apple</span> has plenty of room to do that.</p>
<p class="articleParagraph enarticleParagraph" >While <span class="companylink">Apple</span> sorts out how AI fits into its software, the company's strengths remain evident.</p>
<p class="articleParagraph enarticleParagraph" >Wall Street analysts now agree that iPhone 17 will boost device sales growth to the highest level since fiscal year 2021. Services revenue continues to grow briskly, leveraging the more than 2.3 billion <span class="companylink">Apple</span> devices being used by customers. Because it isn't raiding its cash flow statement like other big tech companies, the cash-return program will continue unabated. When <span class="companylink">Apple</span> reports its first-quarter earnings, it will likely push all-time dividend payments and share buybacks past $1 trillion. Since 2012, the company has retired nearly half of its outstanding stock, raising per share metrics by 79%.</p>
<p class="articleParagraph enarticleParagraph" >And this whole discussion brings up a bigger question: How badly does <span class="companylink">Apple</span> need AI features to sell devices? Since it became a mature category, people buy a new phone when they think they need a new phone. For better or worse, new features no longer drive smartphone sales.</p>
<p class="articleParagraph enarticleParagraph" >During the Covid-19 lockdowns of fiscal 2021, <span class="companylink">Apple</span> iPhone sales were up 39% from the year before, as customers needed new devices to work from home. Phone 16 was heavily marketed as the <span class="companylink">Apple</span> Intelligence phone, and sales were decent but no one's idea of a blockbuster. Now the iPhone 17 lineup is being sold in a more traditional <span class="companylink">Apple</span> manner, with a focus on hardware, design, and camera—and it looks to be doing much better. Those fiscal-year 2021 phones are five years old now in fiscal 2026, and people need a new one. It's as simple as that.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Apple</span> has plenty of time. Its investors should hold on for the ride.</p>
<p class="articleParagraph enarticleParagraph" >Write to Adam Levine at <span class="colorLinks">adam.levine@barrons.com [mailto:adam.levine@barrons.com]</span>
                  </p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>applc : Apple Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302 : Computers/Consumer Electronics | i3302022 : Artificial Intelligence Technologies | icomp : Computing | icph : Computer Hardware | iint : Online Service Providers | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c11 : Corporate Strategy/Planning | ccapex : Capital Expenditure | ccat : Corporate/Industrial News | gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcsci : Computer Science | gsci : Sciences/Humanities | ncat : Content Types | ncolu : Columns</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>AAPL | APBC.XX | CRM | GOOGL | I/CPR | I/ETK | I/XFFX | LLM | M/TEC | META | MSFT | N/CNW | N/DJN | N/GEN | N/SCN | N/WER | OPEN.XX | ORCL</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>Barrons.com | Technology Trader</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Dow Jones & Company, Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document B000000020251205elc800008</td></tr></table><br/></div></div><br/><span></span><div id="article-B000000020251206elc8000dx" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/bLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>CLM</b>&nbsp;</td><td>The Trader</td></tr>
<tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>Features</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>
                           Oracle Earnings Are Coming. It Can't Go Any Worse Than Last Time for the Stock.</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>By Jacob Sonenshine </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>539 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Barron's</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>B</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>33</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025 Dow Jones & Company, Inc. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >
                        <span class="companylink">Oracle</span>'s last earnings release started with a celebration and ended in disaster. Its Wednesday earnings report should work out a whole lot better.</p>
<p class="articleParagraph enarticleParagraph" >Most investors would probably prefer to forget what happened just about three months ago. <span class="companylink">Oracle</span> stock <span class="colorLinks">initially surged 36% [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/oracle-ellison-ai-stocks-nvidia-9ff8e005]</span> when the company said that its bookings had grown by 359%. Then everyone realized that most of that number came from one customer—<span class="companylink">OpenAI</span>—which may or may not be able to pay for everything it has ordered. Shares have dropped 35% since then to $214.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >The drop has coincided with analysts reducing their earnings estimates, in large part because of the higher interest and depreciation expense that comes with the tens of billions of dollars of borrowings and capital investments <span class="companylink">Oracle</span> has to make to build data centers to provide the artificial intelligence-driven data storage software its customers, including <span class="companylink">OpenAI</span>, need. If that money never materializes, the company has a big problem.</p>
<p class="articleParagraph enarticleParagraph" >That <span class="companylink">OpenAI</span> uncertainty makes it hard to predict where <span class="companylink">Oracle</span> stock will go over the long term. For now, though, its fiscal second-quarter earnings, which will hit the wires after the market closes on Wednesday, provide an opportunity to prove that the company is on a high-growth path, without having to show <span class="colorLinks">any demand from OpenAI [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/oracle-stock-price-google-open-ai-02617085]</span>, as the contract doesn't start until 2027.</p>
<p class="articleParagraph enarticleParagraph" >Analysts forecast <span class="companylink">Oracle</span>'s total sales to grow by 15% to $16.2 billion, according to <span class="companylink">FactSet</span>, powered by <span class="colorLinks">its cloud-based and AI-driven offerings [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/ai-stocks-nvidia-oracle-bitcoin-703566dd]</span>. Its older, more outdated software isn't really growing anymore, but the company is signing up new corporate customers for its highly efficient AI product. The cloud infrastructure segment grew 55% in the first quarter, and is expected to grow at a similar rate in the second quarter.</p>
<p class="articleParagraph enarticleParagraph" >The sales should translate to earnings of $1.64 a share, for a growth rate of about 12%. That's lower than the revenue growth because the newer AI business carries lower profit margins and is becoming a larger portion of the business. But the market has well understood that for a while. It's still the key ingredient in the overall growth story, and could very well become more profitable over time. Earnings could demonstrate that there is strong demand for <span class="companylink">Oracle</span>'s AI offerings, even without <span class="companylink">OpenAI</span>.</p>
<p class="articleParagraph enarticleParagraph" >That's why, when it comes to the stock, "we view current weakness as a buying opportunity ahead of its second quarter print in December," writes <span class="companylink">Mizuho Securities</span> analyst Siti Panigrahi.</p>
<p class="articleParagraph enarticleParagraph" >Traders can cue off recent technical signals. The stock has risen from a low of $198, somewhat of a key level, given that it's roughly the $200 area where buyers stepped in at the end of June <span class="colorLinks">to send the stock higher [https://www.forbes.com/sites/greatspeculations/2025/06/12/is-oracle-stock-a-buy-at-190/]</span>. Now the stock is a touch over its $211 200-day moving average, which has trended upward over the past three years. As long as the earnings picture doesn't darken, shares should rise.</p>
<p class="articleParagraph enarticleParagraph" >It doesn't take an oracle to see there just might be an opportunity in <span class="companylink">Oracle</span>.</p>
<p class="articleParagraph enarticleParagraph" >Write to Jacob Sonenshine at <span class="colorLinks">jacob.sonenshine@barrons.com [mailto:jacob.sonenshine@barrons.com]</span>
                  </p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>orcle : Oracle Corporation</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302 : Computers/Consumer Electronics | i330202 : Software | i3302021 : Applications Software | i3302022 : Artificial Intelligence Technologies | icomp : Computing | icph : Computer Hardware | iint : Online Service Providers | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c15 : Financial Performance | c151 : Earnings | c1521 : Analysts' Comments/Recommendations | ccat : Corporate/Industrial News | ncat : Content Types | ncolu : Columns | nfact : Factiva Filters | nfce : C&E Exclusion Filter | nfcpin : C&E Industry News Filter</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>I/CPR | I/ETK | I/SOF | I/XFFX | LLM | M/TEC | N/CNW | N/ERN | N/PFM | OPEN.XX | ORCL</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>Barrons.com | The Trader</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Dow Jones & Company, Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document B000000020251206elc8000dx</td></tr></table><br/></div></div><br/><span></span><div id="article-B000000020251206elc800002" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/bLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Can AI Make The U.S. More Efficient? Not Fast Enough. --- Gains in productivity, key to U.S. economic growth, may soon start to ebb. Hopes that AI will come to the rescue look misplaced.</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>By Megan Leonhardt and Adam Levine </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1351 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Barron's</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>B</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>11</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025 Dow Jones & Company, Inc. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >U.S. labor productivity growth has been on the rise in recent years, gaining an average of 2.2% a quarter since 2023 due to public and private investments, new business formation, and surging immigration. These forces are now waning, however, adding to the challenges facing the U.S. economy.</p>
<p class="articleParagraph enarticleParagraph" >Many economists and investors expect AI to come to the rescue, ushering in a productivity boom in the next few years that will lift gross domestic product and bolster U.S. competitiveness. But it may not come soon enough.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Although companies are spending hundreds of billions of dollars in a race to fully capture the benefits of artificial intelligence, the history of technological advancements argues for caution in estimating how quickly and effectively this investment will pay off. If the anticipated AI-driven productivity gains fail to materialize in the next couple of years, the U.S. could face more inflation, labor challenges, and reduced economic activity.</p>
<p class="articleParagraph enarticleParagraph" >Why Productivity Matters</p>
<p class="articleParagraph enarticleParagraph" >Labor productivity is the measure of how efficiently workers generate goods and services. As productivity rises, businesses typically need fewer employees to produce the same amount of goods or services. Thus, productivity gains can drive economic growth and help to alleviate inflationary pressures.</p>
<p class="articleParagraph enarticleParagraph" >They can also lift living standards. If the U.S. achieves an annual productivity growth rate of 2%, living standards can double every 35 years, according to John Ryding, chief economic advisor at <span class="companylink">Brean Capital</span>. But if growth accelerates to around 3%, as happened from 1995 to 2005, that allows for a doubling every 23 years. Conversely, if productivity growth slows to just 1%, living standards would double every 69 years.</p>
<p class="articleParagraph enarticleParagraph" >U.S. productivity has grown by an average of 2% annually since 1960, although growth slowed to just 1.2% a year in the 2010s. Since 2023, however, labor productivity gains have been pacing at nearly twice that rate on a quarterly basis, and they grew at a 3.3% rate in this year's second quarter.</p>
<p class="articleParagraph enarticleParagraph" >The recent gains in productivity have been driven by investment, changing labor-market dynamics, and business dynamism, rather than the direct effects of AI. Industries such as hospitality and mining have been among the biggest gainers. Productivity at restaurants, for example, <span class="colorLinks">surged more than 15% during the Covid pandemic [https://www-nber-org.ezproxy.cul.columbia.edu/papers/w33555]</span>. Restaurants saw a wave of new business from delivery, but had trouble hiring, so output went up quickly without the complementary rise in labor.</p>
<p class="articleParagraph enarticleParagraph" >"The bulk of the postpandemic productivity outperformance has been driven by higher services productivity," says <span class="companylink">Goldman Sachs Research</span> economist Manuel Abecasis.</p>
<p class="articleParagraph enarticleParagraph" >Yet the U.S. has now wrung out the majority of the productivity gains achieved through Covid-era business upgrades and workforce dynamics, while the most recent drivers of productivity growth, including full employment, fixed investment, and supply-side stability, are ebbing. As a result, the U.S. economy is approaching a potential inflection point: The factors that returned the economy to a roughly 2% annual productivity growth trend may not persist for much longer.</p>
<p class="articleParagraph enarticleParagraph" >"With labor-force growth slowing due to demographics, the U.S. economy is increasingly reliant on productivity gains to drive growth and improve living standards," says Adam Schickling, senior economist at <span class="companylink">Vanguard</span>.</p>
<p class="articleParagraph enarticleParagraph" >AI adoption has had a minimal impact on labor and productivity chiefly because it is still in the early stages. Only about 10% of businesses used any form of AI—including machine learning, natural language processing, virtual agents, and voice recognition—to produce goods or services in September, according to the Census Bureau's Business Trends and Outlook Survey. Still, that is up from 3.7% in September 2023.</p>
<p class="articleParagraph enarticleParagraph" >Other surveys, including <span class="colorLinks">one conducted by the Federal Reserve Bank of New York [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/ai-jobs-layoffs-new-york-fed-report-617a9bd4]</span> in September and <span class="colorLinks">another by ADP [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/ai-jobs-wage-gains-layoffs-economy-e3adeba2]</span> in October, put regular usage of AI by businesses and workers at much higher levels. But the technology isn't in "regular use" by the majority of American companies, and may take years to generate substantial economic dividends.</p>
<p class="articleParagraph enarticleParagraph" >Economists have written extensively about productivity gains from technological innovation. Indeed, Joel Mokyr, Philippe Aghion, and Peter Howitt won this year's Nobel Prize in Economic Sciences for establishing the theoretical underpinnings of this dynamic.</p>
<p class="articleParagraph enarticleParagraph" >Stanford economist Erik Brynjolfsson has also been a leading thinker on the subject. As personal computer use mushroomed in the 1980s and '90s, a mystery unfolded: Where was the productivity growth that so many anticipated? From 1977, when the Apple II computer was released, to 1989, when Brynjolfsson began writing about the "productivity paradox," U.S. productivity grew by a meager 1.3% annually, even as PC prices plummeted and sales rose.</p>
<p class="articleParagraph enarticleParagraph" >"You can see the computer age everywhere but in the productivity statistics," Robert Solow, a Nobel laureate in economics, said in 1987.</p>
<p class="articleParagraph enarticleParagraph" >But that changed in the mid-1990s, and U.S. productivity grew by 3% a year over the following decade. To address the lag between deployment and productivity gains, Brynjolfsson developed what he called the productivity J-curve, which charts the path of productivity growth following the introduction of a new technology.</p>
<p class="articleParagraph enarticleParagraph" >Technological innovations like AI can reduce productivity growth at first, before the benefits accrue years or even decades later. Electric motors were first used in factories in the 1880s, but the productivity benefits didn't accrue until 30 years later.</p>
<p class="articleParagraph enarticleParagraph" >"Technology by itself rarely delivers productivity just when you plug it in," Brynjolfsson told Barron's. "What almost always has to happen is that you have to rethink your business processes. You need to reskill your workforce. You may need to develop new products and services. All this reinvention and co-invention adds a ton of value, but it also takes time."</p>
<p class="articleParagraph enarticleParagraph" >Economic Implications</p>
<p class="articleParagraph enarticleParagraph" >The U.S. is still a few years away from substantial commercial adoption of AI, and perhaps even further away from the promised benefits. Meanwhile, the <span class="companylink">Congressional Budget Office</span>
                     <span class="colorLinks">projects [https://www.cbo.gov/system/files/2025-07/61546-NABE.pdf]</span> that annual productivity gains will need to average at least 1.4% over the next decade for the U.S. economy to grow by 2% a year. But productivity growth may not achieve that pace, given labor trends and the delayed boost from AI adoption.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Goldman Sachs Research</span>
                     <span class="colorLinks">estimates [https://www.goldmansachs.com/insights/articles/what-is-the-us-economys-potential-growth-rate]</span> that labor-force growth will contribute just 0.3 percentage point to potential gross domestic product growth over the next few years, given an aging U.S. population and immigration curbs on worker supply. That is down from 0.8 percentage point since 2019.</p>
<p class="articleParagraph enarticleParagraph" >Consumer spending, which accounts for a majority of GDP growth, is also expected to slow from recent highs. Personal consumption expenditures rose by 2.9% in 2024, but the National Association of Business Economics' November <span class="colorLinks">consensus forecast [https://nabe.com/NABE/Surveys/Outlook_Surveys/November_2025_Outlook_Survey_Summary.aspx?_gl=1*1dwe9ki*_ga*NjcxNTIyOTYxLjE3NTE5MDEzMjA.*_ga_J48PFZK49C*czE3NjQxMDE1MTEkbzU0JGcxJHQxNzY0MTAxNTE2JGo1NSRsMCRoMjk2MDM4MTA0]</span> was for just 2.5% growth in 2025 and 1.8% in 2026.</p>
<p class="articleParagraph enarticleParagraph" >That leaves productivity growth to do the heavy lifting. Yet the <span class="companylink">Congressional Budget Office</span>
                     <span class="colorLinks">projects [https://www.cbo.gov/system/files/2025-07/61546-NABE.pdf]</span> that annual productivity gains will average just 1.3% through the end of the decade. As a result, most economists expect U.S. GDP growth to fall in coming years below the 2% rate considered healthy for advanced economies. Economists surveyed by <span class="companylink">FactSet</span> expect the economy to grow by 1.8% in 2026 and 1.9% in 2027.</p>
<p class="articleParagraph enarticleParagraph" >Slowing productivity growth, together with growing government obligations, could force policymakers to make difficult decisions around taxation, public spending, and entitlement outlays, <span class="companylink">Vanguard</span>'s Schickling says.</p>
<p class="articleParagraph enarticleParagraph" >"If we're not getting labor growth and productivity gains, the risk of a recession rises," says Gerald Cohen, chief economist at the Frank H. Kenan Institute of Private Enterprise and a professor at the <span class="companylink">University of North Carolina at Chapel Hill</span>.</p>
<p class="articleParagraph enarticleParagraph" >There is little doubt that the U.S. is in the early stages of the AI boom. But it is unlikely that this emerging technology will be able to reverse sagging productivity trends soon.</p>
<p class="articleParagraph enarticleParagraph" >Write to Megan Leonhardt at <span class="colorLinks">megan.leonhardt@barrons.com [mailto:megan.leonhardt@barrons.com]</span> and Adam Levine at <span class="colorLinks">adam.levine@barrons.com [mailto:adam.levine@barrons.com]</span>
                  </p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>e11 : Economic Performance/Indicators | e1101 : Economic Growth/Recession | e1115 : Employment Cost/Productivity Figures | ecat : Economic News | ncat : Content Types | npag : Page One Stories</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>BMI.XX | LLM | N/DJN | N/GENI | N/IEN | N/WER | R/NME | R/US | VGI.XX</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>Barrons.com | Features - Main</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Dow Jones & Company, Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document B000000020251206elc800002</td></tr></table><br/></div></div><br/><span></span><div id="article-B000000020251205elc80002t" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/bLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>CLM</b>&nbsp;</td><td>Other Voices</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Most Important Industry Isn't AI. It's Healthcare.</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>By Alí R. Bustamante </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>886 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Barron's</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>B</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>62</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025 Dow Jones & Company, Inc. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >
                        <img src="../pro/default.aspx?napc=S&_XFORMSTATE=H4sIAAAAAAAEAD2LywrCMBBF%2f2XWIUymD5NZSgVxI%2fgHMQ01pdaQFhXa%2fLuK4l0cOHDuonhRaN5gFDVDm0a2yV3C3cvRPyab5uAGL7f4HSFVirDyg9MfnUFsGGK6nfv%2frznsjqf9r6wLLEsjtZbKaDJruNrOyz76bm3DFMMTRMkkiKEBUTDmnF%2b8c3NOlAAAAA%3d%3d"/>

                     </p>
<p class="articleParagraph enarticleParagraph" >PHOTO: Illustration by Juanjo Gasull</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >About the author: Ali R. Bustamante is a professor of practice at the University of New Orleans Department of Economics and Finance.</p>
<p class="articleParagraph enarticleParagraph" >Scan the <span class="colorLinks">latest market commentary [http://www.barrons.com.ezproxy.cul.columbia.edu/articles/ai-is-driving-growth-but-it-isnt-the-only-game-in-town-1d13e435?]</span> and you will hear a familiar refrain: Artificial intelligence is propping up the U.S. economy. Analysts see soaring share prices for <span class="companylink">Nvidia</span>, <span class="companylink">Meta Platforms</span>, and <span class="companylink">Microsoft</span>, along with a wave of data center construction, and conclude that America's economic resilience rests on AI's shoulders.</p>
<p class="articleParagraph enarticleParagraph" >Confusing the stock market with the real economy is the oldest analytical mistake in finance. AI is propping up the stock market. But healthcare is propping up the economy.</p>
<p class="articleParagraph enarticleParagraph" >The difference is unmistakable when you look at how these two industries behave in the labor market. Healthcare is the only major sector whose share of total U.S. employment rose in every recession and under nearly every macroeconomic condition of the past 25 years. It has never posted a sustained decline—not during the 2001 economic downturn, not during the 2008-09 global financial crisis, and not during the Covid-19 pandemic-related recession of 2020.</p>
<p class="articleParagraph enarticleParagraph" >Meanwhile, the much-celebrated AI boom is barely visible in labor-market data. Information sector employment—the broadest proxy for tech—shrank from 2.7% to 1.8% as a share of total jobs during the past 25 years. At the same time, tech firms' equity valuations have skyrocketed.</p>
<p class="articleParagraph enarticleParagraph" >Employment data should force investors to rethink what is actually sustaining the economy. Healthcare and social assistance employ more than 20 million Americans. The <span class="companylink">Bureau of Labor Statistics</span> projects it to account for about 38% of all new U.S. jobs over the next decade, far outpacing tech, manufacturing, and construction combined.</p>
<p class="articleParagraph enarticleParagraph" >The data tell a clear story of steady growth. In 2000, healthcare accounted for just over 8.2% of all nonfarm jobs. By 2025, it had climbed to 11.4%. Crucially, this rise has rarely reversed. No other major sector is close to its level of stability.</p>
<p class="articleParagraph enarticleParagraph" >Professional and business services, manufacturing, construction, information, retail, accommodation and food services all contracted during downturns. Healthcare didn't just avoid decline. It expanded, month after month, at the exact moment the rest of the economy was breaking. Some of this reflects disproportionate job losses in lower-wage sectors during economic downturns, but it also reflects the essential nature of healthcare. Demand for medical care doesn't disappear in a recession; it intensifies. <span class="colorLinks">Economic stress worsens [https://pmc-ncbi-nlm-nih-gov.ezproxy.cul.columbia.edu/articles/PMC7307016/]</span> short and long-term health outcomes. It drives up chronic illness, emergency care, and behavioral health needs, forcing hospitals and clinics to staff up precisely when other employers are cutting back.</p>
<p class="articleParagraph enarticleParagraph" >And healthcare jobs are strong multipliers at the local level. Hospitals and ambulatory care providers support employment in transportation companies, food vendors, educational programs, janitorial services, construction firms, and biomedical suppliers. When a hospital expands, the surrounding economy expands with it. When a hospital closes, the economy immediately contracts. No AI firm has that kind of footprint in Baton Rouge, Des Moines, or Fresno.</p>
<p class="articleParagraph enarticleParagraph" >This brings us to the real danger on the horizon. The U.S. economy is leaning on a sector whose future stability isn't guaranteed. It is policy-dependent. Unlike AI, whose trajectory is tied to capital markets, investor sentiment, and technological progress, healthcare's stability depends directly on federal and state policy.</p>
<p class="articleParagraph enarticleParagraph" >Three major healthcare supports are at risk. The enhanced Affordable Care Act subsidies, expanded under the American Rescue Plan, are scheduled to expire at the end of the year. <span class="colorLinks">If they lapse [https://www.barrons/com/livecoverage/sundayshows1130/cards/trump-s-idea-to-extend-aca-plan-subsidies-not-fully-baked-hassett-dPgPuKzLmkfYP3lmy5ln]</span>, premiums for millions could jump by hundreds of dollars a month. Millions could lose coverage entirely. Rising uninsured rates increase uncompensated-care burdens for hospitals, strain budgets, slow hiring, and weaken regional growth.</p>
<p class="articleParagraph enarticleParagraph" >In addition, Medicaid and Children's Health Insurance Program enrollment has already fallen sharply as states unwind Covid-19-era continuous-coverage rules. From March 2023 to July 2025, more than 17 million people<span class="colorLinks">lost coverage [https://www.kff.org/medicaid/medicaid-enrollment-and-unwinding-tracker/]</span>. Hospitals can't absorb that level of uncompensated care without cutting services or closing outright. More than 700 rural hospitals are currently <span class="colorLinks">at risk of closing [https://ruralhospitals.chqpr.org/Overview.html]</span>. And when a hospital closes, a regional economy loses one of its few recession-proof anchors.</p>
<p class="articleParagraph enarticleParagraph" >Finally, the healthcare workforce, already strained by shortages of nurses, medical assistants, mental health counselors, and clinical lab workers, faces an uncertain federal funding landscape. President Donald Trump's proposed fiscal year 2026 budget includes over $400 million in cuts to workforce programs that expand the supply of nurses, physicians, behavioral health specialists, and other healthcare workers. Without stable investments in training and education, health systems can't staff new units, expand services, or meet rising demand. A recession-proof sector becomes recession sensitive.</p>
<p class="articleParagraph enarticleParagraph" >If policymakers allow Medicaid funding to erode, <span class="companylink">ACA</span> subsidies to expire, and the health workforce pipeline to thin, they will weaken the very sector that has kept the economy stable through every crisis of the 21st century. No amount of AI-driven equity exuberance will be enough to keep the real economy from feeling the shock.</p>
<p class="articleParagraph enarticleParagraph" >Guest commentaries like this one are written by authors outside the Barron's newsroom. They reflect the perspective and opinions of the authors. Submit feedback and commentary pitches to ideas@barrons.com.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i257 : Pharmaceuticals | i951 : Healthcare/Life Sciences | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>ncat : Content Types | ncolu : Columns | nedc : Commentaries/Opinions | nfact : Factiva Filters | nfcpex : C&E Executive News Filter | nimage : Images</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>I/ETK | M/HCR | M/TEC | META | MSFT | N/DJN | N/WER | NVDA | R/NME | R/US</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>Barrons.com | Other Voices</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Dow Jones & Company, Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document B000000020251205elc80002t</td></tr></table><br/></div></div><br/><span></span><div id="article-B000000020251206elc80008d" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/bLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>CLM</b>&nbsp;</td><td>Barron's Mailbag</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Play a Waiting Game With Roblox Stock</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>901 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Barron's</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>B</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>63</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025 Dow Jones & Company, Inc. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >To the Editor: <span class="companylink">Roblox</span> continues to show impressive growth in users and revenue, yet the company still faces significant financial challenges ("<span class="colorLinks">Roblox Isn't Playing Games. Why the Stock Could Jump 50% [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/wp-bar-0001516638]</span>," Cover Story, Nov. 25). Despite strong gains in bookings, <span class="companylink">Roblox</span> posted a large net loss for 2024 and expects a loss in 2025. I would be cautious. I'd let the kids under 17 play the game, but I'll just stay on the sidelines and watch for now.</p>
<p class="articleParagraph enarticleParagraph" >Martin Blumberg Melville, N.Y.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Google</span>'s Other Antitrust Suit</p>
<p class="articleParagraph enarticleParagraph" >To the Editor: Due to its recent artificial-intelligence successes, <span class="companylink">Alphabet</span>'s <span class="companylink">Google</span> was mentioned favorably a couple of times in the latest issue, including in "<span class="colorLinks">Google Stock Has Been the Clear AI Winner—and the Gains Could Keep Coming [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/buy-alphabet-stock-price-google-googl-3e3f606d]</span>" (Nov. 24). The stock has doubled since the spring on the news in September that the search antitrust case was resolved, and on unusually lenient terms. The judge specifically cited the potential for AI to disrupt <span class="companylink">Google</span>'s search monopoly as an excuse to avoid structural penalties. But there's another case out there, the "ad tech" antitrust case. <span class="companylink">Google</span> has been found guilty of monopolistic behavior here, too, and this judge may be inclined to "correct" the unusually lenient penalties meted out in the first case, especially seeing how things are truly turning out with AI. Cards and dice may not have a memory, but judges and the public do.</p>
<p class="articleParagraph enarticleParagraph" >Jim Hemenway Niwot, Colo.</p>
<p class="articleParagraph enarticleParagraph" >To the Editor: <span class="companylink">Google</span> has been an AI winner compared with <span class="companylink">Meta Platforms</span>, right? That's not true, according to the figure depicting their respective, quarterly pretax GAAP income margins in percentages over the past seven quarters. Overall, <span class="companylink">Meta</span> outperformed <span class="companylink">Google</span> in five of the past seven quarters. Nevertheless, I put my money in <span class="companylink">Google</span>.</p>
<p class="articleParagraph enarticleParagraph" >Erik H. Schot Lauderdale-by-the-Sea, Fla.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Blue Owl</span>'s Warning</p>
<p class="articleParagraph enarticleParagraph" >To the Editor: This is all seriously scary stuff ("<span class="colorLinks">Blue Owl Is Having a Rough Year. Its Co-CEO Says Investors' Fears Are 'Ungrounded [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/blue-owl-co-ceo-fears-ungrounded-7bc0a3e5]</span>,' " Interview, Nov. 20). Marc Lipschultz posits that either <span class="companylink">Blue Owl</span> is mispriced, or the market is due for a serious reversal. If the latter is true, then <span class="companylink">Blue Owl Capital</span>'s troubles so far this year were, in retrospect, the canary in the coal mine.</p>
<p class="articleParagraph enarticleParagraph" >Ralph Fisher On Barrons.com</p>
<p class="articleParagraph enarticleParagraph" >Doing Nothing Is Hard</p>
<p class="articleParagraph enarticleParagraph" >To the Editor: Trading less is rare but very sound advice ("<span class="colorLinks">Don't Just Trade Something, Sit There [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/nvidia-rate-cuts-options-stocks-139d894a]</span>," The Striking Price, Nov. 26). Most of us are simply not equipped to be active traders. We don't have access to the information, and most aren't able to understand the trends, to be able to trade actively and profitably. Even the pros that have those advantages can't consistently beat the indexes.</p>
<p class="articleParagraph enarticleParagraph" >Thus, traders are mostly rolling the dice with at best 50/50 odds of executing profitable trades. Those trading decisions are being mainly driven by emotions, the talking heads on TV, or news articles. Being a long-term buy-and-hold investor, and doing nothing, is harder. But over the long term, the odds of success are much higher.</p>
<p class="articleParagraph enarticleParagraph" >David Feliciano On Barrons.com</p>
<p class="articleParagraph enarticleParagraph" >Tax Code Changes</p>
<p class="articleParagraph enarticleParagraph" >To the Editor: I wish Congress would stop changing the tax code every year or so ("<span class="colorLinks">Tax Rules Are Changing. What to Do Before Year End [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/tax-rules-planning-7a14eb79]</span>," Guide to Wealth, Nov. 25). In addition to being annoying, it makes planning for business and retirement difficult. Of course, the tax-prep industry loves it. The never-ending complexity forces more people to get help. Most people have real lives and don't want to spend a lot of time keeping up with the latest changes. It really is crazy what we put people through every year.</p>
<p class="articleParagraph enarticleParagraph" >Walt Busalacchi On Barrons.com</p>
<p class="articleParagraph enarticleParagraph" >Calling Out the Fed</p>
<p class="articleParagraph enarticleParagraph" >To the Editor: <span class="companylink">Allianz</span> Chief Economic Advisor Mohamed El-Erian is spot on in his assessment that the <span class="companylink">Federal Reserve</span> was too late in responding to the inflation surge of 2021, that it was a "miracle" we avoided a recession, that the Fed continues to be late, and that its 2% inflation target is unrealistic and unnecessary ("<span class="colorLinks">The Bull's Wild Ride: What We Have Here Is a Worrywart Market [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/bull-sputtering-worrywart-stock-market-2d561565]</span>," Up & Down Wall Street, Nov. 21).</p>
<p class="articleParagraph enarticleParagraph" >Many economists are terrible at forecasting the future because they can't get the present correct. El-Erian is different. Why? He isn't solely an academic. El-Erian has decades of experience on Wall Street, understands Main Street, and has skin in the game. Experience is the greatest teacher. Academics hate that expression. Hopefully, it will all change come May.</p>
<p class="articleParagraph enarticleParagraph" >Tom Verdi Providence, R.I.</p>
<p class="articleParagraph enarticleParagraph" >Trump's Wins</p>
<p class="articleParagraph enarticleParagraph" >To the Editor: I appreciated Christopher Smart's honest perspective on President Donald Trump's economic wins this year ("<span class="colorLinks">Trump's Record on the Economy Actually Has Wins. Ten Things to Be Grateful for This Thanksgiving [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/trump-economy-good-for-investment-thanksgiving-28cfb6ef]</span>," Nov. 21).</p>
<p class="articleParagraph enarticleParagraph" >In today's climate of partisan politics, it was refreshing to read an objective analysis. Offering recognition to someone with whom you disagree is a lost art. Whether in investing or politics, focusing on facts while ignoring the noise remains the best approach.</p>
<p class="articleParagraph enarticleParagraph" >Jonathan I. Shenkman West Hempstead, N.Y.</p>
<p class="articleParagraph enarticleParagraph" >Send letters to: <span class="colorLinks">mail@barrons.com [mailto:mail@barrons.com]</span>. To be considered for publication, correspondence must bear the writer's name, address, and phone number. Letters are subject to editing.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>gognew : Google LLC | goog : Alphabet Inc. | rblxu : Roblox Corporation</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302 : Computers/Consumer Electronics | i330202 : Software | i3302021 : Applications Software | i3302022 : Artificial Intelligence Technologies | i4941 : Toys/Games | i8395464 : Internet Search Engines | icnp : Consumer Goods | icomp : Computing | icph : Computer Hardware | igamsof : Games Software | iint : Online Service Providers | ilgood : Leisure/Travel Goods | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c15 : Financial Performance | c151 : Earnings | ccat : Corporate/Industrial News | ncat : Content Types | ncolu : Columns | nfact : Factiva Filters | nfce : C&E Exclusion Filter | nfcpin : C&E Industry News Filter | niwe : IWE Filter | nlet : Letters | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>G/FED | GOOGL | I/CPR | I/ETK | I/ISV | I/LTG | I/SOF | I/TMF | M/NCY | M/TEC | META | N/CNW | N/DJN | N/ERN | N/PFM | N/WER | OWL | RBLX</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>Barrons.com | Mailbag</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Dow Jones & Company, Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document B000000020251206elc80008d</td></tr></table><br/></div></div><br/><span></span><div id="article-B000000020251206elc80005l" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/bLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>CLM</b>&nbsp;</td><td>Research Reports</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Research Reports --- How Analysts Size Up Companies</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>894 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Barron's</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>B</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>41</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025 Dow Jones & Company, Inc. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >These reports, excerpted and edited by Barron's, were issued recently by investment and research firms. The reports are a sampling of analysts' thinking; they should not be considered the views or recommendations of Barron's. Some of the reports' issuers have provided, or hope to provide, investment-banking or other services to the companies being analyzed.</p>
<p class="articleParagraph enarticleParagraph" >
                        <span class="companylink">Uber Technologies</span> • <span class="companylink">UBER</span>-NYSE Buy • $87.57 on Dec. 3 by Gordon Haskett Recently, <span class="companylink">Uber</span>'s chief financial officer, Prashanth Mahendra-Rajah, presented at an investor conference. We continue to be impressed by management's ability to lay the groundwork for sustainable long-term growth drivers and the company's ability to manage insurance costs in a variety of ways. On the autonomous-vehicle front, we remain bullish on the company's strategy and continue to see <span class="companylink">Uber</span> as a beneficiary of growing robo-taxi adoption.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Over the long term, we believe that <span class="companylink">Uber</span> has multiple levers in place to continue driving double-digit percentage growth (membership, new mobility offerings, focus on less dense/penetrated markets where a large percentage of the population lives).</p>
<p class="articleParagraph enarticleParagraph" >Furthermore, with 1) plenty of runway ahead to further grow advertising revenue, which should largely flow through to the bottom line, and 2) a sizable repurchase authorization, we see the shares offering a compelling value. Target price: $122.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">American Eagle Outfitters</span> • AEO-NYSE Hold • $20.83 on Dec. 2 by TD Cowen Third-quarter 2025 was better than expected, with overall comps up 4% vs. Street's 2.7%, and fourth-quarter 2025 guide exceeded expectations, with comps in the range of 8%-9% vs. Street's 2%. The holiday season is off to a strong start, but we monitor consumer sentiment and spending as well as inventory management. We remain at Hold as we look for the durability and longevity of recent momentum. Target price: $23.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">GitLab</span> • GTLB-Nasdaq Buy • $39.60 on Dec. 3 by BTIG GitLab posted mixed fiscal third-quarter result, with decent upside on revenue and a downtick in current remaining performance obligations/bookings. In addition, the company guided fiscal fourth-quarter 2026 in line with prior expectations. Most notably, <span class="companylink">GitLab</span> posted revenue of $244.4 million/24.6% year over year, about 2% ahead of our estimate of $239 million/ 21.9% (Street $239.3 million). While positive, CRPO growth decelerated to 28% in fiscal third quarter versus 31% in fiscal second quarter and 34% in fiscal first quarter.</p>
<p class="articleParagraph enarticleParagraph" >Similarly, CRPO bookings growth slowed to 19% from 23% last quarter. Lastly, the fiscal fourth-quarter revenue guide $251.5 million/19% Y/Y was in line with Street and prior expectations. On the call, management sounded upbeat about early interest in the Duo Agent platform and the potential for artificial intelligence to expand <span class="companylink">GitLab</span>'s addressable market long term.</p>
<p class="articleParagraph enarticleParagraph" >However...there is still some uncertainty around the timing of new go-to-market initiatives to drive improved new logo wins. And some questions remain on the potential for self-managed customers to upgrade in order to take advantage of the Duo Agent platform. Target price: $52.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Apple</span> • AAPL-Nasdaq Outperform • $283.10 on Dec. 2 by Evercore ISI App Store revenue growth deaccelerated in November, with revs growing 6% Y/Y versus 8% last quarter....We estimate that the App Store represents about 20% of Services revenue and believe that faster-growing Services segments (<span class="companylink">Apple</span> Pay, iCloud, Licensing, etc.) could help offset slower App Store growth in the December quarter. Target price: $300.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">CrowdStrike Holdings</span> • CRWD-Nasdaq Neutral • $516.55 on Dec. 2 by Guggenheim CrowdStrike Holdings reported a solid fiscal third quarter, exceeding consensus estimates of both revenue and annual recurring revenue, or ARR, and raised fiscal 2026 revenue guidance by $24 million at the midpoint. We estimate that adjusted new ARR grew 46%, although this was on easier outage-affected year-ago comps, and business momentum as measured by a two-year stack of new ARR growth also improved in the quarter. Adjusted new ARR growth excludes contribution from acquisitions of low-single digit millions, which we assume is about $4 million....</p>
<p class="articleParagraph enarticleParagraph" >We were impressed by the numbers put up this quarter by the company. Management remained optimistic on its future opportunities with its Next-Gen SIEM, Cloud, Next-Gen Identity, and Endpoint solutions as well as the growing popularity of its Falcon Flex model....</p>
<p class="articleParagraph enarticleParagraph" >The smoke is slowly starting to clear, and the path for growth is becoming more apparent, but with shares trading at 23 times enterprise value/next 12 months recurring revenue and 72 times our EV/NTM free cash flow, we remain Neutral.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">GoDaddy</span> • GDDY-NYSE Perform • $128.31 on Dec. 3 by <span class="companylink">Oppenheimer</span> We attended <span class="companylink">GoDaddy</span>'s annual investor dinner, and came away incrementally positive on management's strategy to improve attach and monetization with <span class="companylink">Airo</span> AI. <span class="companylink">GoDaddy</span> demoed the AI app builder, agentic workflows for core small and midsize customers, and <span class="companylink">Airo</span> for WordPress professionals. Management indicated that agentic functionality will sit behind a paywall with monetization via subscription tiers. Equally important, management emphasized that managing AI costs will be a priority....</p>
<p class="articleParagraph enarticleParagraph" >Management continues to drive productivity within the Care organization utilizing AI to oversee everyday support tickets, while limiting human involvement on revenue-generating/retention interactions.</p>
<p class="articleParagraph enarticleParagraph" >To be considered for this section, material should be sent to <span class="colorLinks">Research@barrons.com [mailto:Research@barrons.com]</span>.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>ubrib : Uber International B.V. | ubrti : Uber Technologies Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302 : Computers/Consumer Electronics | i722 : Taxi/Limousine Services | icomp : Computing | icph : Computer Hardware | iecom : E-commerce | iint : Online Service Providers | irailtr : Land Transport | iridhps : Ride-Hailing Platforms/Services | itech : Technology | itnsv : Sharing/On-demand Economy Services | itsp : Transportation/Logistics</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c15 : Financial Performance | c151 : Earnings | c1513 : Sales Figures | c1521 : Analysts' Comments/Recommendations | ccat : Corporate/Industrial News | ncat : Content Types | ncolu : Columns | nfact : Factiva Filters | nfce : C&E Exclusion Filter | nfcpin : C&E Industry News Filter</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>AEO | CRWD | GDDY | GTLB | I/ETK | I/ISV | I/PSG | I/RTR | M/TEC | M/TRSH | N/ANL | N/ARG | N/CNW | N/DJN | N/ERN | N/PFM | N/SLS | N/WER | R/NME | R/US | UBER</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>Barrons.com | Research Reports</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Dow Jones & Company, Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document B000000020251206elc80005l</td></tr></table><br/></div></div><br/><span></span><div id="article-B000000020251205elc800002" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/bLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>CLM</b>&nbsp;</td><td>Funds</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>A Celebrity Manager Doesn't Guarantee Big Returns</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>By Debbie Carlson </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1150 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Barron's</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>B</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>26</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025 Dow Jones & Company, Inc. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >
                        <img src="../pro/default.aspx?napc=S&_XFORMSTATE=H4sIAAAAAAAEAD2LwQrCMBAF%2fyXnEDabNKZ7lAriRfAPYhpqSltDKiq0%2bXdFxXcYGJi3SFok1G8QcEOszRO57C%2fxHsQUHrPLt%2biHILbwHQJWEqEKg7cfZ3xDLOXruf%2f%2fmsPueNr%2fSqNAy1pYK9BoqdY4ui6IPoVubeOc4pNxTciRWMO4IiilvADwZ5V1lAAAAA%3d%3d"/>

                     </p>
<p class="articleParagraph enarticleParagraph" >The Dan Ives Wedbush AI Revolution exchange-traded fund has $960 million in assets. Above, Ives in September. PHOTO: Tasos Katopodis/<span class="companylink">Getty Images</span> for <span class="companylink">Eightco Holdings</span> and BitMine</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >What's in a name? For two exchange-traded fund issuers, a way to stand out in a crowded field and gather significant assets.</p>
<p class="articleParagraph enarticleParagraph" >Nearly six months <span class="colorLinks">after its debut [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/dan-ives-wedbush-etf-ai-stocks-9c59b87a]</span>, the Dan Ives Wedbush AI Revolution ETF—named after the widely followed <span class="companylink">Wedbush</span> technology analyst known for his bullish calls and frequent media appearances—boasts $960 million in assets.</p>
<p class="articleParagraph enarticleParagraph" >Meanwhile, Fundstrat Granny Shots U.S. Large Cap has $3.8 billion in its coffers, 13 months after popular Wall Street strategist <span class="colorLinks">Tom Lee [https://www.youtube.com/watch?v=16u324ffjZI]</span>, Fundstrat's chief investment officer and head of research, launched the ETF.</p>
<p class="articleParagraph enarticleParagraph" >With more than 4,000 ETFs available and dozens launching daily, most ETFs are considered successful if they attract $100 million in assets. Hoovering up nearly a billion or more suggests star power—and a hot market—is behind the ETFs' explosive growth, says Rick Wedell, chief investment officer at RFG Advisory.</p>
<p class="articleParagraph enarticleParagraph" >Buyers seem to be drawn by the names and are staying for the performance. Albeit having a short comparable record of three months, the Ives fund is up 13%, versus 9.2% for the Technology Select Sector SPDR ETF. The Granny Shots fund is up 20.3% on a one-year basis, beating the S&P 500's 14.8% return.</p>
<p class="articleParagraph enarticleParagraph" >The Ives fund is a thematic index fund based off the Dan Ives AI 30, a list of his 30 best ideas and names across the artificial-intelligence ecosystem. Although structured as an index fund with quarterly rebalancing, <span class="companylink">Wedbush</span> will update the holdings if Ives changes his list, giving it an active tilt, says Cullen Rogers, the firm's chief investment officer.</p>
<p class="articleParagraph enarticleParagraph" >Making it an index fund allows Ives to focus on research, rather than portfolio management. That leverages Ives' skill, Rogers says, "as opposed to putting him in a different seat and asking different questions."</p>
<p class="articleParagraph enarticleParagraph" >The Ives fund overlaps other tech indexes, with holdings such as <span class="companylink">Nvidia</span> and <span class="companylink">Microsoft</span>, but Rogers says the broader tech indexes don't own lesser-known names such as nuclear energy company <span class="companylink">Oklo</span> and IT company <span class="companylink">Zscaler</span> that could produce future outsize returns. "Those could be the next <span class="companylink">Nvidia</span>," he says.</p>
<p class="articleParagraph enarticleParagraph" >The Granny Shots fund, whose name refers to someone making an underhanded basketball free-throw shot, is a large-cap blend fund actively managed by <span class="companylink">Lee</span> and three other portfolio managers. It's based on a core list of names culled from Fundstrat's thematic research on long-term "supercycle" trends, which they believe can improve stock earnings.</p>
<p class="articleParagraph enarticleParagraph" >The fund targets three short-term themes, such as seasonality, with four longer-term themes, including energy/cybersecurity and millennial preferences. Stocks need to align with at least two themes to be included. Holdings include <span class="companylink">Microsoft</span>, <span class="companylink">CrowdStrike Holdings</span>, and <span class="companylink">Monster Beverage</span>.</p>
<p class="articleParagraph enarticleParagraph" >Thematic funds often are narrowly focused on a single theme or target a trend. Lee says Granny Shots differs from typical thematic funds with its multiple themes and longer-term focus. "It's the opposite of [trendy]. That's why it's called a granny shot," he says. "We're not trying to make cool shots. We're trying to make shots that go in."</p>
<p class="articleParagraph enarticleParagraph" >
                     Deborah Fuhr, founder of independent research firm ETFGI, says both Ives' and <span class="companylink">Lee</span>'s funds benefit from current thematic tailwinds and large-cap growth outperformance. <span class="companylink">Lee</span>'s fund, with its multithemed holdings in AI, energy security, and cybersecurity, "is kind of in the sweet spot right now in terms of performance," she says.</p>
<p class="articleParagraph enarticleParagraph" >Star-powered ETFs aren't new. One of the earliest was <span class="companylink">Pimco</span>'s ETF version of its <span class="companylink">Pimco</span> Total Return fund, which launched in 2012 and leaned on then-manager Bill Gross' name to draw attention, Fuhr says.</p>
<p class="articleParagraph enarticleParagraph" >More recent ETF launches by well-known investors include O'Shares ETFs, a suite of funds that follow factor-based indexes from Kevin O'Leary of Shark Tank fame, and ETFs issued by Strive Asset Management founder and biotech entrepreneur Vivek Ramaswamy, which were originally marketed as being "anti-woke." He has since left the firm to run for various political offices.</p>
<p class="articleParagraph enarticleParagraph" >Big names aren't always a draw. The Atlas America ETF <span class="colorLinks">launched by Nouriel Roubini [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/dr-doom-dr-realistnouriel-roubini-08c34e02]</span> has only $18 million in assets. Nor do star-powered funds always succeed: U.S.-based ETFs from commodities trader Jim Rogers were delisted, as was an ETF from <span class="companylink">YouTube</span> financial personality Kevin Paffrath.</p>
<p class="articleParagraph enarticleParagraph" >Actively managed thematic funds with star managers can be subject to volatile performance. A prime example is Cathie Wood's $7.8 billion <span class="companylink">ARK Innovation ETF</span>. It is up 37% in the past year but has lost 6.2% on an annualized five-year basis. The fund gained 152.8% in 2020 but lost 23.4% in 2021 and 67% in 2022, according to <span class="companylink">Morningstar</span>.</p>
<p class="articleParagraph enarticleParagraph" >Compared with plain-vanilla index funds, thematic and actively managed ETFs tend to have higher fees, which Fuhr says are a drag on performance. That said, Ives' and <span class="companylink">Lee</span>'s funds each cost 0.75% annually to own, but they are outperforming broader indexes net of fees.</p>
<p class="articleParagraph enarticleParagraph" >Whether the outperformance lasts remains to be seen. S&P Global <span class="colorLinks">research [https://www-spglobal-com.ezproxy.cul.columbia.edu/spdji/en/spiva/article/institutional-spiva-scorecard]</span> shows that at least 80% of actively managed equity funds underperformed their benchmarks over the past 10 years after deducting fees.</p>
<p class="articleParagraph enarticleParagraph" >The market cycle may turn, which could make some strategies underperform. However, Lee says since Fundstrat started its research list of its best investment ideas in 2019, the names they've selected outperformed the S&P 500 annually, including in 2022, by losing less in that down year.</p>
<p class="articleParagraph enarticleParagraph" >Fuhr and Wedell say the popularity of these two funds also says something about the ETF industry as much as the strategies themselves. Asset managers are embracing ETFs for their tradability and tax efficiency, and investors are buying active ETFs.</p>
<p class="articleParagraph enarticleParagraph" >It's relatively cheap to launch an ETF, but it's hard for issuers to get them distributed to major brokerages, so they need other methods to grab attention, Fuhr says. As such, investors should expect to see more thematic and star-power funds come to market.</p>
<p class="articleParagraph enarticleParagraph" >Among the ETFs launched by financial celebrities, there are those that have serious strategies and others that are just a way to grab attention, Wedell says.</p>
<p class="articleParagraph enarticleParagraph" >As more buzzy ETFs arrive, investors need to tread carefully. A <span class="colorLinks">2022 study [https://papers-ssrn-com.ezproxy.cul.columbia.edu/sol3/papers.cfm?abstract_id=3765063]</span> about attention-grabbing ETFs, focusing on thematic funds, showed that in the first five years after launch, specialized ETFs lose about 30% of their risk-adjusted value.</p>
<p class="articleParagraph enarticleParagraph" >Francesco Franzoni, a professor at USI Lugano and the Swiss Finance Institute who contributed to the study, says three main features hurt performance: overvalued holdings as the fund managers chased trends, a lack of diversification, and higher fees.</p>
<p class="articleParagraph enarticleParagraph" >Write to <span class="colorLinks">editors@barrons.com [mailto:editors@barrons.com]</span>
                  </p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>rvtjaj : Eightco Holdings Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i81502 : Trusts/Funds/Financial Vehicles | iblock : Blockchain Technology | ibnk : Banking/Credit | iextrfu : Exchange Traded Funds | ifinal : Financial Services | iinv : Investing/Securities | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>ccat : Corporate/Industrial News | ncat : Content Types | ncolu : Columns | nimage : Images | redit : Selection of Top Stories/Trends/Analysis | reqr : Suggested Reading – Industry News | reqrbc : Suggested Reading – Banking/Credit | reqris : Suggested Reading – Investing/Securities</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>ALIZY | ALV.XE | ARKK | I/ETK | I/EXT | I/FDS | I/IAV | M/FCL | M/TEC | MNST | N/CNW | N/DJN | N/WER | NVDA | OKLO | WBM.XX | ZS</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>Barrons.com | Funds</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Dow Jones & Company, Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document B000000020251205elc800002</td></tr></table><br/></div></div><br/><span></span><div id="article-WPCOM00020251205elc8002jp" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/wpcomLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>Style</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>From Rosalía to Addison Rae, these artists made the year's best albums </span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>By Chris Richards </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1045 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>washingtonpost.com</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>WPCOM</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025, The Washington Post Co. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >My favorite albums of 2025 weren't exactly magic tricks, but you had to listen to them closely to keep from getting head-faked. A country music traditionalist was the freshest in his field. A middle-aged rapper sounded more alive than many half his age. There were conceptual pop albums with concepts that didn't matter, and a TikTok superstar who was better at singing hooks than dancing to them. Each of these albums rewarded curiosity, attention and interrogation — rewards that felt significant as the soulless head fakery of AI continues to puree this world into a soup of meaningless superficialities.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >10. Rosalía, 'Lux' Phew. This aggressively orchestral, opera-curious, polylingual ode to saints and martyrs across the ages feels like way too much — so go ahead and get ravished, mind-blown, tongue-tied, misty-eyed, whatever you need to feel in unfeeling times. Does any of it feel better than smiling along with Rosalía when she giggles during that one pause in "La Perla"? She's at her best when she's undercutting her grandeur with her humanity, and I'm convinced she's resourceful enough to have made an even better album with two paper clips and a rubber band. Maybe that's what's next.</p>
<p class="articleParagraph enarticleParagraph" >9. Zach Top, 'Ain't in It for My Health' The Nashville industrial complex remains unsure how to wrap its arms around this handsome young singer of handsomely forlorn country songs. Is he a big-hat revivalist being readied to inherit the chair Chris Stapleton keeps warm? Or is he a change agent powerful enough to make the world forget about Morgan Wallen? Turns out, the quietly seismic shift on "Ain't in It for My Health" isn't a matter of style so much as attitude: Top's music communicates sadness without self-pity.</p>
<p class="articleParagraph enarticleParagraph" >8. Rafiq Bhatia, 'Environments' Here's a wonderful jazzlike album that uses techy tools — digital sampling software that allows Bhatia to summon all kinds of heavy weather from his electric guitar — to generate simple results. A song titled "Rain on the Canopy: Melting Sky" pantomimes the pitter of rainfall. "Aviary I: Sunrise" evokes a whole lot of birds waking up. If this music sounds like the world, be reminded that this world sounds like music.</p>
<p class="articleParagraph enarticleParagraph" >7. Intermission, 'Power Corrupts' One thing that kept Joy Division from becoming the most spirit-depleting rock band to ever exist was the drumming — namely, those dance-friendly flashes that gave the group's romantic gloom-saying a countervailing levity. Flash ahead four decades, and this San Diego punk quartet is inverting the idea, interrupting their shapeless growls with four-on-the-floor beats that make gravity feel like it's pulling harder than it should. Suddenly, your dancing shoes are <span class="companylink">Vans</span> with waffle soles made of lead.</p>
<p class="articleParagraph enarticleParagraph" >6. Nourished by Time, 'The Passionate Ones' Prince casts a long shadow across today's pop music, but it's hard to locate Baltimore's Marcus Brown of Nourished by Time in that purpled darkness. His music feels more Princely than it sounds, foremost because Brown avoids his hero's shrieky falsetto in favor of a wide baritone that feels oddly desirous, as if he were eager to drink up the entire world. "We don't have to be so average," Brown bellows, "and I say that with love."</p>
<p class="articleParagraph enarticleParagraph" >5. James Brandon Lewis, 'Apple Cores' The Brooklyn-based saxophonist kept two heroes at the front of his mind during this session — the jazz trumpeter Don Cherry and the jazz critic Amiri Baraka — but, as ever, the back of his brain refused to keep quiet, teeming with thoughts of John Coltrane, molecular biology, the paintings of Paul Klee, the poetry of Aimé Césaire and lots more. Add surplus thought waves from drummer Chad Taylor and bassist/guitarist Josh Werner and we might begin to understand why these no-nonsense improvisations sound this remarkable. Broadening minds funneled into tightening grooves.</p>
<p class="articleParagraph enarticleParagraph" >4. Bruiser Wolf, 'Potluck' What this fabulous, 40-something Detroit rapper fails to mention when he explains how his "lyrics flow from my heart to my larynx" is that, en route, the rhymes seem to get filtered through our collective memories of '80s game show hosts, '70s cinema pimps, the verses of CeeLo Green circa Goodie Mob's first two albums and various Hanna-Barbera cartoons. Snagglepuss, even!</p>
<p class="articleParagraph enarticleParagraph" >3. FKA Twigs, 'Eusexua' That album title is an unfortunate portmanteau of "sex" and "euphoria," which Twigs tried to frame as a buzzword/concept during this album's rollout, going as far as to describe "Eusexua" as a state of being akin to the "pinnacle of human experience." A meaningless idea, but then the British pop singer threw all of her melodic agility and granular attention to detail into it, and now, achieving "Eusexua" feels strangely and impossibly sweet, like devouring a Honda-size tuft of cotton candy one molecule at a time.</p>
<p class="articleParagraph enarticleParagraph" >2. Playboi Carti, 'Music' With 2020's "Whole Lotta Red" still in the lead for rap record of the decade, Carti steps down from the mountaintop, but with his feet still planted on some kind of edge. The edge of existence, maybe? Throughout these half-abstracted, entirely exhilarating rap songs, the Atlanta auteur sounds like he's erasing himself, masking his voice in deceptively anodyne rhymes, playing footsie with oblivion. He originally titled it "I Am Music," then scrubbed himself from that, too. The air buzzing around our heads is all that's left.</p>
<p class="articleParagraph enarticleParagraph" >1. Addison Rae, 'Addison' "Don't ask too many questions." Sorry, but when a TikTok star with 88 million followers sings that line on the finest song of the year's most luscious pop album, we have no choice. For starters: Is making parasocial dance videos the best preparation for 21st-century pop craft? Or is this just what happens once the Madonna songbook fully seeps into the digital groundwater? Or does Madonna only have as much to do with this music as Lana, Enya and Charli XCX? Is "Addison" the first true opus of the post-"Brat" wave? If so, where's everybody else? Why does this woman sound so singular, so scrupulous, so savvy, so alone?</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gent : Arts/Entertainment | gmusic : Music</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>entertainment | music</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Washington Post</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document WPCOM00020251205elc8002jp</td></tr></table><br/></div></div><br/><span></span><div id="article-DAYB000020251205elc80003e" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daybLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Washington Daybook - General News Events</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Federal Information & News Dispatch, Inc./Agence France-Presse </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>92 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Washington Daybook</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYB</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Federal Information & News Dispatch, Inc. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Advisory Technology - Discussion</p>
<p class="articleParagraph enarticleParagraph" >SPONSOR: The <span class="companylink">Federal Communications Bar Association</span> (FCBA) - The Tech Bar</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >TOPIC/SUBJECT: holds a virtual discussion, beginning at 1 p.m., on "Algorithms & Atoms: Mitigating the Next Generation of Tech Risk."</p>
<p class="articleParagraph enarticleParagraph" >AGENDA: Highlight:</p>
<p class="articleParagraph enarticleParagraph" >-- 1:05 p.m.: Discussion on "AI Security"</p>
<p class="articleParagraph enarticleParagraph" >DATE: December 8, 2025</p>
<p class="articleParagraph enarticleParagraph" >LOCATION: None given</p>
<p class="articleParagraph enarticleParagraph" >CONTACT: 202-293-4000, fcba@fcba.org [Note: Register at <span class="colorLinks">https://www.fcba.org/event/cle-seminar-algorithms-atoms-mitigating-the-next-generation-of-tech-risk/ [https://www.fcba.org/event/cle-seminar-algorithms-atoms-mitigating-the-next-generation-of-tech-risk/]</span> ]</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>fdcmba : Federal Communications Bar Association</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gpir : Politics/International Relations | gpol : Domestic Politics | ncal : Calendar of Events | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usdc : Washington DC | uss : Southern U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Federal Information & News Dispatch LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYB000020251205elc80003e</td></tr></table><br/></div></div><br/><span></span><div id="article-DAYB000020251205elc80003j" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daybLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Washington Daybook - General News Events</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Federal Information & News Dispatch, Inc./Agence France-Presse </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>153 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Washington Daybook</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYB</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Federal Information & News Dispatch, Inc. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >11 a.m. Technology - Discussion</p>
<p class="articleParagraph enarticleParagraph" >SPONSOR: <span class="companylink">The George Washington University (GWU)</span> Elliott School of International Affairs</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >TOPIC/SUBJECT: holds a discussion on "Navigating Taiwan's AI Future: Policy, Innovation, and Governance."</p>
<p class="articleParagraph enarticleParagraph" >PARTICIPANTS: Cheng Ming Wang, director general of digital services at <span class="companylink">Taiwan's Ministry of Digital Affairs</span>; Hsin-Chung Liao, associate professor at <span class="companylink">National Chengchi University</span> and chair of National Chengchi University's Department of Public Administration; and Susan Aaronson, <span class="companylink">GWU</span> research professor and director of <span class="companylink">GWU</span>'s Digital Trade and Data Governance Hub</p>
<p class="articleParagraph enarticleParagraph" >DATE: December 8, 2025</p>
<p class="articleParagraph enarticleParagraph" >LOCATION: GWU Elliott School, 1957 E Street NW, Room 505, Washington, D.C.</p>
<p class="articleParagraph enarticleParagraph" >CONTACT: Cate Douglass, 609-235-7859 or cdouglass@gwu.edu or gwmedia@gwu.edu; <span class="colorLinks">http://elliott.gwu.edu [http://elliott.gwu.edu]</span> [Note: Register at <span class="colorLinks">https://calendar.gwu.edu/event/taiwan-roundtable-navigating-taiwans-ai-future-policy-innovation-and-governance [https://calendar.gwu.edu/event/taiwan-roundtable-navigating-taiwans-ai-future-policy-innovation-and-governance]</span> ]</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>grgwhu : George Washington University | nchenu : National Chengchi University | twnmdf : Taiwan Ministry of Digital Affairs</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gedu : Education | gpir : Politics/International Relations | gpol : Domestic Politics | guni : University/College | ncal : Calendar of Events | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | asiaz : Asia | chinaz : Greater China | devgcoz : Emerging Market Countries | easiaz : East Asia | namz : North America | taiwan : Taiwan | usa : United States | usdc : Washington DC | uss : Southern U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Federal Information & News Dispatch LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYB000020251205elc80003j</td></tr></table><br/></div></div><br/><span></span><div id="article-DAYB000020251205elc80003n" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daybLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Washington Daybook - General News Events</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Federal Information & News Dispatch, Inc./Agence France-Presse </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>153 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Washington Daybook</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYB</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Federal Information & News Dispatch, Inc. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >2 p.m. Technology - Discussion</p>
<p class="articleParagraph enarticleParagraph" >SPONSOR: <span class="companylink">The Brookings Institution</span>
                     </p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >TOPIC/SUBJECT: holds a discussion on "The Future of the Internet in the Age of AI."</p>
<p class="articleParagraph enarticleParagraph" >AGENDA: Highlights:</p>
<p class="articleParagraph enarticleParagraph" >-- 2 p.m.: Vint Cerf, vice president and chief internet evangelist at <span class="companylink">Google</span>, delivers remarks</p>
<p class="articleParagraph enarticleParagraph" >-- 2:35 p.m.: Karen Kornbluh, visiting fellow at the Center for Democracy and Technology; and Chris Lewis, president and CEO of <span class="companylink">Public Knowledge</span>, participate in a panel discussion on "The Commercial Internet and the Explosion of Broadband"</p>
<p class="articleParagraph enarticleParagraph" >-- 3:35 p.m.: Panel discussion on "Internet Policy and AI Governance: Connections and Convergences"</p>
<p class="articleParagraph enarticleParagraph" >DATE: December 8, 2025</p>
<p class="articleParagraph enarticleParagraph" >LOCATION: <span class="companylink">Brookings Institution</span>, 1775 Massachusetts Avenue NW, Falk Auditorium, Washington, D.C.</p>
<p class="articleParagraph enarticleParagraph" >CONTACT: 202-797-6105, events@brookings.edu [Note: Registration and livestream at <span class="colorLinks">https://www.brookings.edu/events/the-future-of-the-internet-in-the-age-of-ai/ [https://www.brookings.edu/events/the-future-of-the-internet-in-the-age-of-ai/]</span> ]</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>brooit : The Brookings Institution | pubknw : Public Knowledge</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcsci : Computer Science | gpir : Politics/International Relations | gpol : Domestic Politics | gsci : Sciences/Humanities | ncal : Calendar of Events | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usdc : Washington DC | uss : Southern U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Federal Information & News Dispatch LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYB000020251205elc80003n</td></tr></table><br/></div></div><br/><span></span><div id="article-DAYB000020251205elc80003p" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daybLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Washington Daybook - General News Events</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Federal Information & News Dispatch, Inc./Agence France-Presse </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>141 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Washington Daybook</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYB</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Federal Information & News Dispatch, Inc. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >6 p.m. Technology - Discussion</p>
<p class="articleParagraph enarticleParagraph" >SPONSOR: The <span class="companylink">American Enterprise Institute for Public Policy Research</span> (AEI)</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >TOPIC/SUBJECT: holds a discussion on "Maximizing School Improvement by 2035 Means Integrating AI into Classrooms Today."</p>
<p class="articleParagraph enarticleParagraph" >PARTICIPANTS: Shanika Hope, director of Americas and knowledge, skill and learning at <span class="companylink">Google</span>; Alex Kotran, CEO of the AI Education Project; Dan Meyer, vice president of user growth at <span class="companylink">Amplify</span>; Jake Tawney, vice president of curriculum at Great Hearts Academies; and Nat Malkus, AEI deputy director of education policy studies</p>
<p class="articleParagraph enarticleParagraph" >DATE: December 8, 2025</p>
<p class="articleParagraph enarticleParagraph" >LOCATION: AEI, 1789 Massachusetts Avenue NW, Auditorium, Washington, D.C.</p>
<p class="articleParagraph enarticleParagraph" >CONTACT: 202-862-5829, mediaservices@aei.org [Note: Register at <span class="colorLinks">https://www.aei.org/events/education-policy-debate-series-maximizing-school-improvement-by-2035-means-integrating-ai-into-classrooms-today/ [https://www.aei.org/events/education-policy-debate-series-maximizing-school-improvement-by-2035-means-integrating-ai-into-classrooms-today/]</span> ]</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>aepppr : American Enterprise Institute for Public Policy Research</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcsci : Computer Science | gpir : Politics/International Relations | gpol : Domestic Politics | gsci : Sciences/Humanities | ncal : Calendar of Events | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usdc : Washington DC | uss : Southern U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Federal Information & News Dispatch LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYB000020251205elc80003p</td></tr></table><br/></div></div><br/><span></span><div id="article-DAYB000020251205elc800039" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daybLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>The Washington Daybook - General News Events</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Federal Information & News Dispatch, Inc./Agence France-Presse </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>93 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Washington Daybook</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYB</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © 2025 Federal Information & News Dispatch, Inc. All rights reserved </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Advisory Foreign Affairs - Discussion</p>
<p class="articleParagraph enarticleParagraph" >SPONSOR: The Center for Strategic and International Studies (<span class="companylink">CSIS</span>)</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >TOPIC/SUBJECT: holds a virtual discussion, beginning at 8:30 a.m., on "Previewing India's AI Impact Summit."</p>
<p class="articleParagraph enarticleParagraph" >PARTICIPANTS: Shri Krishnan, secretary of the <span class="companylink">Indian Ministry of Electronics and Information Technology</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >DATE: December 8, 2025</p>
<p class="articleParagraph enarticleParagraph" >LOCATION: None given</p>
<p class="articleParagraph enarticleParagraph" >CONTACT: Sofia Chavez, 202-775-7317, SChavez@csis.org [Note: Register at <span class="colorLinks">https://www.csis.org/events/previewing-indias-ai-impact-summit-meity-secretary-s-krishnan [https://www.csis.org/events/previewing-indias-ai-impact-summit-meity-secretary-s-krishnan]</span> ]</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>inmcit : India Ministry of Electronics and Information Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gpir : Politics/International Relations | gpol : Domestic Politics | ncal : Calendar of Events | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usdc : Washington DC | uss : Southern U.S.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Federal Information & News Dispatch LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYB000020251205elc800039</td></tr></table><br/></div></div><br/><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WC45695020251205elc800002"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  NewsJudge orders DTE to keep power on at troubled Detroit apartment complexJolie Sherman</b><div class="leadFields"><a href="javascript:void(0)">WXYZ</a>, 11:00 PM, 7 December 2025, 597 words,  Jolie Sherman, (English)</div><div class="snippet ensnippet"> DETROIT (WXYZ) — Dozens of residents at the Leland House Apartments in downtown Detroit received temporary relief Thursday when a federal judge approved the financing needed to keep the building operating.</div>
<div>(Document WC45695020251205elc800002)</div><br/></td></tr>
						</table>
					</div>
				</div><span></span><div id="article-IEUEV00020251205elc800001" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/ieuevLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>Better Finance</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Better Finance conference: From Fraud to Accountability – Tackling Financial Crime Across Europe</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>169 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>8 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Insight EU Events (IEU-E)</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>IEUEV</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025. Comecon Media GmbH </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Conference</p>
<p class="articleParagraph enarticleParagraph" >Date: 9 December 2025</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Time: 15:00 – 17:30</p>
<p class="articleParagraph enarticleParagraph" >Location: State of Hessen Rep to the EU (Rue Montoyer 21, Brussels)</p>
<p class="articleParagraph enarticleParagraph" >Europe’s capital markets participation is challenged by the flood of scams, misconduct, and systemic risks that retail investors and savers face online. In turn, trust in Europe’s capital markets continues to erode. And, with new digital innovations sparked by AI, financial crimes have accelerated, with fraudsters becoming increasingly sophisticated. To tackle such crimes, regulatory and enforcement responses must keep pace.</p>
<p class="articleParagraph enarticleParagraph" >To examine the problem and understand how Europe can strengthen its defences against financial crime, BETTER FINANCE is hosting a high-level conference bringing together policymakers, regulators, industry leaders, and consumer advocates. The programme features the launch of a BETTER FINANCE Paper on Scam, expert insights from leading organisations, and a panel debate on how shared responsibility can ensure accountability and better protection for investors across the EU.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">Programme and registration [https://betterfinance.eu/event/fraud-financial-crime-europe-conference/]</span>
                  </p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gcrim : Crime/Legal Action | gfinc : Financial Crime</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>eurz : Europe</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>Better Finance | Tackling Financial Crime</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Comecon</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document IEUEV00020251205elc800001</td></tr></table><br/></div></div><br/><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WC48892020251204elc800002"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  A New Era Begins: From Systems of Record to Systems of Action</b><div class="leadFields"><a href="javascript:void(0)">Quality Digest</a>, 12:02 PM, 8 December 2025, 1637 words, (English)</div><div class="snippet ensnippet"> (QAD Inc: Santa Barbara, CA) -- QAD, a company transforming manufacturing and supply chains with intelligent, adaptive solutions, has outlined a bold shift to a reimagined manufacturing platform designed specifically for midmarket ...</div>
<div>(Document WC48892020251204elc800002)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WC60927020251203elc8001p6"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  artificial intelligence LSEG Brings Market Data Into ChatGPT</b><div class="leadFields"><a href="javascript:void(0)">PYMNTS.com</a>, 07:00 PM, 7 December 2025, 590 words, (English)</div><div class="snippet ensnippet"> The integration will begin the week of Dec. 8 and will allow users with LSEG credentials to access Financial Analytics content, real-time market data, research and news from within ChatGPT through a connector built using the Model Context ...</div>
<div>(Document WC60927020251203elc8001p6)</div><br/></td></tr>
						</table>
					</div>
				</div><span></span><div id="article-TAIP000020251206elc700003" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/taipLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Lai reiterates commitment to defense SHARED ISSUE:Chinas authoritarian expansion impacts the international community, the foreign minister said, calling for a security mechanism with like-minded countries</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>By Su Yong-yao and Lee I-chia </td></tr>
<tr><td align="right" valign="top" class="index"><b>CR</b>&nbsp;</td><td>Staff reporters, with Reuters </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>630 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Taipei Times</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>TAIP</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© Copyright 2025 The Taipei Times. All rights reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Taiwan reiterated its commitment to bolstering its self-defense to uphold regional peace, as ranking officials expressed appreciation to the US for prioritizing deterring a conflict over Taiwan and highlighting the importance of the security of the first island chain in the US latest National Security Strategy.</p>
<p class="articleParagraph enarticleParagraph" >Greatly appreciate that the #US National Security Strategy prioritizes deterring a conflict over Taiwan & highlights the security of the First Island Chain. #Taiwan will continue to be a reliable partner deeply committed to strengthening our self-defense to uphold regional peace, President William Lai wrote on X yesterday.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >The 33-page report comes as Beijing increases pressure on Taiwan and Japan, deploying vessels across East Asian waters last week in its largest maritime show of force to date.</p>
<p class="articleParagraph enarticleParagraph" >The documents language on Taiwan is stronger than the National Security Strategy produced during US President Donald Trumps first term in office. The document in 2017 mentioned Taiwan three times in a single sentence, echoing longstanding diplomatic language.</p>
<p class="articleParagraph enarticleParagraph" >Taiwan was mentioned seven times in the report published under former US president Joe Bidens administration.</p>
<p class="articleParagraph enarticleParagraph" >The new report mentions Taiwan eight times across three paragraphs, and concludes that there is, rightly, much focus on Taiwan, because of its strategic location in trade-rich waters and dominance in semiconductor manufacturing.</p>
<p class="articleParagraph enarticleParagraph" >We will build a military capable of denying aggression anywhere, in the chain of islands stretching from Japan to Southeast Asia, it said.</p>
<p class="articleParagraph enarticleParagraph" >But the American military cannot, and should not have to, do this alone. Our allies must step up and spend and more importantly do much more for collective defense, it added.</p>
<p class="articleParagraph enarticleParagraph" >That would reinforce US and allies capacity to deny any attempt to seize Taiwan or any other steps that would make defending that island impossible, the report said.</p>
<p class="articleParagraph enarticleParagraph" >The Ministry of Foreign Affairs yesterday thanked the US for pointing out Taiwans importance in key supply chains and geopolitical strategy, as well as stressing that the US and its allies would collaborate to ensure the nations safety.</p>
<p class="articleParagraph enarticleParagraph" >The Trump administration has demonstrated support for Taiwan, including Trump signing the amended Taiwan Assurance Implementation Act and the US announcing planned arms sale to the nation, the ministry added.</p>
<p class="articleParagraph enarticleParagraph" >Taiwan will continue to work with the US on security to ensure safety and stability in the Taiwan Strait and the Indo-Pacific region, the ministry said.</p>
<p class="articleParagraph enarticleParagraph" >The government would also continue to take actions that would enhance the nations defense capabilities, such as the eight-year NT$1.25 trillion (US$39.8 billion) special defense budget proposed by Lai, to demonstrate Taiwans determination and willpower to firmly defend itself and the status quo, it added.</p>
<p class="articleParagraph enarticleParagraph" >Minister of Foreign Affairs Lin Chia-lung said Chinas authoritarian expansion does not only affect Taiwan, but also impacts the entire Indo-Pacific region and the international community.</p>
<p class="articleParagraph enarticleParagraph" >National security is the basic condition for survival; without security as a guarantee, prosperity cannot be discussed, Lin wrote on <span class="companylink">Facebook</span> yesterday, expressing hope that Taiwan could build a security communication mechanism with countries that share the same ideals.</p>
<p class="articleParagraph enarticleParagraph" >Commenting on Taiwan-US economic and trade cooperation, Lin said that given the US-China competition and the reshuffling of the global supply chain, investments made by Taiwanese companies in the US represent not risks, but opportunities.</p>
<p class="articleParagraph enarticleParagraph" >Taiwan is willing to establish mutually beneficial cooperation with the US in industries such as semiconductors, servers, robotics and artificial intelligence (AI), he said, adding that by building a comprehensive AI industry and integrating into the US innovation ecosystem, Taiwan and the US could achieve shared prosperity.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gcns : National/Public Security | gdip : International Relations | gpir : Politics/International Relations | gpol : Domestic Politics | gsec : State Security Measures/Policies | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nnam : News Agency Materials</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | asiaz : Asia | chinaz : Greater China | devgcoz : Emerging Market Countries | easiaz : East Asia | namz : North America | taiwan : Taiwan | usa : United States</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Liberty Times Ltd.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document TAIP000020251206elc700003</td></tr></table><br/></div></div><br/><span></span><div id="article-PHSTAR0020251206elc70000t" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/phstarLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Southeast Asian leaders confident in APEC growth – survey</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>Jasper Emmanuel Arcalas </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>451 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>The Philippine Star</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>PHSTAR</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>(c) 2025 Philstar Global Corporation </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >MANILA, Philippines — Majority of Southeast Asian leaders are optimistic about the economic growth within the <span class="companylink">Asia Pacific Economic Cooperation</span> (<span class="companylink">APEC</span>) region, but less than half expressed the same sentiment for the global economy, according to a survey conducted by <span class="companylink">Deloitte</span>.</p>
<p class="articleParagraph enarticleParagraph" >The British multinational firm surveyed 1,252 senior <span class="companylink">APEC</span> business leaders across 18 economies, including the Philippines, in over a dozen industries to gather their insights on the growth of their companies and the global economy.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >The respondents included more than 270 leaders in Southeast Asia, according to <span class="companylink">Deloitte</span>.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Deloitte</span> noted that three-fourths of the surveyed Southeast Asian leaders expressed confidence in the opportunities within the APEC region, while 66 percent were optimistic about the region’s overall economy.</p>
<p class="articleParagraph enarticleParagraph" >However, only 46 percent of Southeast Asian leaders expressed positive sentiment toward the global economy, the survey showed.</p>
<p class="articleParagraph enarticleParagraph" >“Leaders across Southeast Asia are confident in their own companies’ performance, see tangible opportunities across the APEC region, yet remain cautious about the broader global outlook,” said Eugene Ho, chief executive officer, Deloitte SEA.</p>
<p class="articleParagraph enarticleParagraph" >“We see this as a ‘certainty gap’ that leaders must bridge with strategic vision that turns disruption into opportunity,” Ho added.</p>
<p class="articleParagraph enarticleParagraph" >Ho explained that Southeast Asian business leaders are diversifying their firms’ respective supply chains and delaying major investments to manage business risks amid geopolitical uncertainties. Businesses in the region are using technology as a “key” growth driver, Ho said, since they are now focusing on innovation and sustainability over the long term.</p>
<p class="articleParagraph enarticleParagraph" >“Beyond immediate concerns, leaders are embedding AI into operational resilience and preparing for mandatory sustainability reporting and sustainable financing,” he said.</p>
<p class="articleParagraph enarticleParagraph" >“This purposeful agility positions businesses in the region for sustained growth, supported by cooperation within the APEC bloc,” he added.</p>
<p class="articleParagraph enarticleParagraph" >Businesses in Southeast Asia continue to prioritize growth, but their strategies for achieving this growth are shifting, with a greater emphasis on operational efficiency in the context of innovation-led expansion and new cross-border value opportunities, according to <span class="companylink">Deloitte</span>.</p>
<p class="articleParagraph enarticleParagraph" >At present, at least 45 percent of Southeast Asian business leaders surveyed identified technology application as their priority for growth leverage. However, in three years, 47 percent said they will focus on new products and innovation, compared to the 28 percent response today, <span class="companylink">Deloitte</span> said.</p>
<p class="articleParagraph enarticleParagraph" >“Geographic expansion is also gaining momentum, with executives expecting more than half of their revenue to come from <span class="companylink">APEC</span> economies, rising from 17 percent today to 35 percent in three years,” <span class="companylink">Deloitte</span> said.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">The British multinational firm surveyed 1,252 senior APEC business leaders across 18 economies, including the Philippines, in over a dozen industries to gather their insights on the growth of their companies and the global economy. [https://media.philstar.com/photos/2025/12/06/1_2025-12-06_18-03-05425_thumbnail.jpg]</span>
                  </p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>apecoo : Asia-Pacific Economic Cooperation</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c41 : Management | ccat : Corporate/Industrial News | e11 : Economic Performance/Indicators | ecat : Economic News | ncat : Content Types | nfact : Factiva Filters | nfcpin : C&E Industry News Filter | nsur : Surveys/Polls</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | asiaz : Asia | devgcoz : Emerging Market Countries | dvpcoz : Developing Economies | phlns : Philippines | seasiaz : Southeast Asia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Philstar Global Corporation</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document PHSTAR0020251206elc70000t</td></tr></table><br/></div></div><br/><span></span><div id="article-PHSTAR0020251206elc70000l" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/phstarLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Machines and manipulation</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>Francis J. Kong </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>800 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>The Philippine Star</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>PHSTAR</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>(c) 2025 Philstar Global Corporation </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Every generation has its inventions.</p>
<p class="articleParagraph enarticleParagraph" >But ours as well, ours invented something that now invents us.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >At this year’s WOBI New York, several speakers painted a picture both astonishing and alarming: a world where artificial intelligence doesn’t just serve us – it shapes us.</p>
<p class="articleParagraph enarticleParagraph" >We’ve entered an era where algorithms not only predict what we want but quietly persuade us to want it. One speaker compared TikTok to a “behavioral drug disguised as entertainment.” It’s hard to argue with that.</p>
<p class="articleParagraph enarticleParagraph" >The app’s genius lies not in what it shows you, but in what it withholds. Endless scroll. Quick dopamine. No finish line.</p>
<p class="articleParagraph enarticleParagraph" >That’s not technology; that’s psychology – with code.</p>
<p class="articleParagraph enarticleParagraph" >There is now what is called “the illusion of choice.”</p>
<p class="articleParagraph enarticleParagraph" >We once believed the internet would democratize opportunity and give every voice a chance. But somewhere along the way, it turned into an attention casino where the house always wins.</p>
<p class="articleParagraph enarticleParagraph" >AI now decides who gets noticed, who gets muted, what trends, and what disappears. And while we think we’re scrolling freely, our choices have been pre-filtered, optimized, and monetized long before our thumbs moved.</p>
<p class="articleParagraph enarticleParagraph" >As one analyst quipped at WOBI, “The algorithm doesn’t care what you watch. It only cares that you keep watching.”</p>
<p class="articleParagraph enarticleParagraph" >And we keep watching – until time blurs, focus fades, and real conversations feel inconvenient.</p>
<p class="articleParagraph enarticleParagraph" >Many young people believe they are entrepreneurs because they sell products or services online. However, what they do not understand is that they are simply changing their employment and are now working for social media platforms as their new employers. Slaving under the spell and command of the algorithms instead of a boss, they have learned to loathe and criticize.</p>
<p class="articleParagraph enarticleParagraph" >Then came the economic side of AI – a far more sobering discussion.</p>
<p class="articleParagraph enarticleParagraph" >While AI creates convenience, it also creates concentration. Ten tech giants now account for more than half of the global market capitalization. They build the platforms, own the data, and lease the intelligence.</p>
<p class="articleParagraph enarticleParagraph" >The rest of us? We’re the users, the data sources, or the entertained.</p>
<p class="articleParagraph enarticleParagraph" >Some experts warn that this is not innovation; it’s feudalism with faster Wi-Fi.</p>
<p class="articleParagraph enarticleParagraph" >We once celebrated capitalism for rewarding creativity and risk-taking. But today’s digital economy often rewards scale and surveillance instead.</p>
<p class="articleParagraph enarticleParagraph" >The small business owner can’t compete with an algorithm that knows their customer better than they do.</p>
<p class="articleParagraph enarticleParagraph" >So while the world celebrates AI breakthroughs, the truth is that economic power narrows – and social trust thins.</p>
<p class="articleParagraph enarticleParagraph" >And in the midst of it all comes what is now known as “The Epidemic of Loneliness.”</p>
<p class="articleParagraph enarticleParagraph" >In all this noise, people feel more alone.</p>
<p class="articleParagraph enarticleParagraph" >Data shows that young men, especially, are retreating from real relationships and meaningful work. One WOBI speaker called it “a generation of young people raised online but starving for connection.”</p>
<p class="articleParagraph enarticleParagraph" >We live surrounded by “friends” but devoid of friendship.</p>
<p class="articleParagraph enarticleParagraph" >We share everything but reveal nothing.</p>
<p class="articleParagraph enarticleParagraph" >We are informed but rarely transformed.</p>
<p class="articleParagraph enarticleParagraph" >AI can simulate empathy, even generate “compassionate” responses.</p>
<p class="articleParagraph enarticleParagraph" >But it cannot care. It cannot listen between the lines.</p>
<p class="articleParagraph enarticleParagraph" >It cannot sit in silence with you.</p>
<p class="articleParagraph enarticleParagraph" >Technology may make us efficient – but it does not make us whole.</p>
<p class="articleParagraph enarticleParagraph" >Progress with a pulse</p>
<p class="articleParagraph enarticleParagraph" >Now, don’t get me wrong – I’m not anti-technology.</p>
<p class="articleParagraph enarticleParagraph" >I write columns on my workstation, prepare lessons on my iPad, communicate through apps, post thoughts and reflections every day on <span class="companylink">Facebook</span>, <span class="companylink">Instagram</span>, <span class="companylink">LinkedIn</span>, Threads, Bluesky, and my blog page – and yes, sometimes even let AI help outline my ideas.</p>
<p class="articleParagraph enarticleParagraph" >But the keyword is help.</p>
<p class="articleParagraph enarticleParagraph" >The moment the tool starts thinking for you, it ceases to be a tool.</p>
<p class="articleParagraph enarticleParagraph" >AI is meant to assist human wisdom – not replace it.</p>
<p class="articleParagraph enarticleParagraph" >But wisdom requires reflection, and reflection demands quiet – something our digital lives rarely offer anymore.</p>
<p class="articleParagraph enarticleParagraph" >We cannot automate meaning.</p>
<p class="articleParagraph enarticleParagraph" >We cannot outsource our humanity.</p>
<p class="articleParagraph enarticleParagraph" >We cannot download purpose.</p>
<p class="articleParagraph enarticleParagraph" >So where do we go from here?</p>
<p class="articleParagraph enarticleParagraph" >Maybe it starts small – just like James Clear’s “one percent better” philosophy from another WOBI session I attended.</p>
<p class="articleParagraph enarticleParagraph" >Perhaps we reclaim our attention one percent at a time.</p>
<p class="articleParagraph enarticleParagraph" >We put down the phone at dinner.</p>
<p class="articleParagraph enarticleParagraph" >We talk to a friend without checking notifications.</p>
<p class="articleParagraph enarticleParagraph" >We read something that doesn’t glow.</p>
<p class="articleParagraph enarticleParagraph" >Because the danger isn’t that machines will become more human.</p>
<p class="articleParagraph enarticleParagraph" >It’s that humans will become more machine-like – efficient, reactive, and emotionally flat.</p>
<p class="articleParagraph enarticleParagraph" >The age of AI doesn’t demand that we outsmart technology.</p>
<p class="articleParagraph enarticleParagraph" >It calls us to out-human it.</p>
<p class="articleParagraph enarticleParagraph" >In the end, the real intelligence that matters isn’t artificial – it’s authentic.</p>
<p class="articleParagraph enarticleParagraph" >And that, my friends, is still something no algorithm can replicate.</p>
<p class="articleParagraph enarticleParagraph" >* * *</p>
<p class="articleParagraph enarticleParagraph" >Catch Kongversations with Francis on <span class="companylink">YouTube</span> and all major podcast platforms – <span class="companylink">Spotify</span>, <span class="companylink">Apple</span> Podcasts, <span class="companylink">Google</span> Podcasts, and more. Plus, listen to Inspiring Excellence wherever you stream.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | asiaz : Asia | devgcoz : Emerging Market Countries | dvpcoz : Developing Economies | phlns : Philippines | seasiaz : Southeast Asia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Philstar Global Corporation</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document PHSTAR0020251206elc70000l</td></tr></table><br/></div></div><br/><span></span><div id="article-PHSTAR0020251206elc70000j" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/phstarLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>De Asis wins AWEN award for championing Filipino brands</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>626 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>The Philippine Star</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>PHSTAR</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>(c) 2025 Philstar Global Corporation </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >MANILA, Philippines — Her story is about a lifelong belief that strong Filipino brands can change the trajectory of entire families.</p>
<p class="articleParagraph enarticleParagraph" >Catherine “Karen” De Asis has spent most of her career in rooms where ideas about brands, identity and purpose are shaped.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >But this year, the conversation shifts to her: de Asis has been named the Philippines’ 2025 ASEAN Women Entrepreneurs Awardee, joining honorees from across Southeast Asia at ceremonies in Phnom Penh on November 21.</p>
<p class="articleParagraph enarticleParagraph" >For someone who prefers to keep the spotlight on her clients, the recognition feels both unexpected and deeply aligned with the mission she’s carried for over a decade.</p>
<p class="articleParagraph enarticleParagraph" >“I’ve always believed that when Filipino family businesses succeed, the social impact is exponential,” de Asis says.</p>
<p class="articleParagraph enarticleParagraph" >“Success creates expansion, and expansion creates jobs.”</p>
<p class="articleParagraph enarticleParagraph" >Why family businesses matter</p>
<p class="articleParagraph enarticleParagraph" >De Asis is a founding board member and chief brand strategist of MKS Marketing Consulting, a firm she helped build with a sharp focus: empower Filipino-owned companies to compete – and win – at home and abroad.</p>
<p class="articleParagraph enarticleParagraph" >Her philosophy is straightforward.</p>
<p class="articleParagraph enarticleParagraph" >“We’ve restricted our services to Filipino family businesses because visionary owners are quick, flexible and deeply invested,” De Asis says.</p>
<p class="articleParagraph enarticleParagraph" >“When they succeed, the entire ecosystem benefits.”</p>
<p class="articleParagraph enarticleParagraph" >This perspective emerged from years of observing how charitable efforts, while valuable, often struggle with sustainability.</p>
<p class="articleParagraph enarticleParagraph" >Entrepreneurship, she argues, is generational. It can transform communities without relying on intermittent support.</p>
<p class="articleParagraph enarticleParagraph" >Brand stories that changed businesses</p>
<p class="articleParagraph enarticleParagraph" >At MKS, De Asis has led transformative brand work – some behind the scenes, others now category leaders.</p>
<p class="articleParagraph enarticleParagraph" >Several of these case studies were part of the ASEAN Women Entrepreneurs Network (AWEN) deliberations.</p>
<p class="articleParagraph enarticleParagraph" >One retail brand had plateaued at 47 branches after 30 years.</p>
<p class="articleParagraph enarticleParagraph" >Following a full brand restructuring, it grew to 350 company-owned stores in 15 years.</p>
<p class="articleParagraph enarticleParagraph" >“That kind of expansion creates thousands of jobs,” De Asis says.</p>
<p class="articleParagraph enarticleParagraph" >“That’s real impact.”</p>
<p class="articleParagraph enarticleParagraph" >Another client started as a newcomer selling only air conditioners. A decade later, it offers a full line of major appliances sold through more than a thousand dealer outlets nationwide.</p>
<p class="articleParagraph enarticleParagraph" >A third case was a drugstore chain with minimal awareness that now stands as a strong challenger in its category.</p>
<p class="articleParagraph enarticleParagraph" >For De Asis, these aren’t just success stories – they’re proof points.</p>
<p class="articleParagraph enarticleParagraph" >“Creative intelligence isn’t optional in branding,” she notes.</p>
<p class="articleParagraph enarticleParagraph" >“You’re shaping not just market leaders or challengers, but the livelihoods of people who depend on these businesses.”</p>
<p class="articleParagraph enarticleParagraph" >From strategy rooms to lecture halls</p>
<p class="articleParagraph enarticleParagraph" >De Asis’ work spans retail, food, health care, consumer durables, telecommunications, travel, fashion and property development.</p>
<p class="articleParagraph enarticleParagraph" >She has led numerous brand campaigns, including one that became the first in the Philippines to use AI-generated imagery across TV, digital and print.</p>
<p class="articleParagraph enarticleParagraph" >De Asis’ academic path reflects the same discipline: a Doctor of Education in leadership and corporate social responsibility from De La Salle University (with highest distinction), an MBA with distinction from the Ateneo Graduate School of Business, and executive programs at <span class="companylink">Stanford University</span> and <span class="companylink">Oxford University</span>’s Saïd Business School.</p>
<p class="articleParagraph enarticleParagraph" >Career built on advocacy</p>
<p class="articleParagraph enarticleParagraph" >Her recognitions – <span class="companylink">Agora</span> Awardee for Excellence in Marketing Education, Outstanding Faculty Award from De La Salle, and the Philippines Women Leadership Excellence distinction – map out a career grounded in rigor and conviction.</p>
<p class="articleParagraph enarticleParagraph" >De Asis has taught in top Philippine business programs and once shared the stage with branding pioneer Al Ries.</p>
<p class="articleParagraph enarticleParagraph" >She later authored Color Folders in the Mind, considered the first Philippine rule book on brand management.</p>
<p class="articleParagraph enarticleParagraph" >Yet despite the honors, De Asis circles back to the same belief that has anchored her work all these years.</p>
<p class="articleParagraph enarticleParagraph" >“Ultimately,” she says, “our advocacy is simple: help visionary Filipino businesses succeed so more Filipino families can thrive for decades and generations to come.”</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">Catherine “Karen” De Asis [https://media.philstar.com/photos/2025/12/06/11_2025-12-06_17-41-36550_thumbnail.jpg]</span>
                  </p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>ccat : Corporate/Industrial News | gaward : Awards | gcat : Political/General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | asiaz : Asia | devgcoz : Emerging Market Countries | dvpcoz : Developing Economies | phlns : Philippines | seasiaz : Southeast Asia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Philstar Global Corporation</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document PHSTAR0020251206elc70000j</td></tr></table><br/></div></div><br/><span></span><div id="article-YOMSHI0020251206elc70000c" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/yomshiLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>Society</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Ex-teacher indicted over deepfake child pornography</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>The Yomiuri Shimbun </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>405 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>The Japan News</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>YOMSHI</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>2</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2025 The Japan News All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >NAGOYA - A male former teacher in Nagoya was indicted Friday for allegedly possessing sexually explicit images of young girls created using generative artificial intelligence based on real images of children.</p>
<p class="articleParagraph enarticleParagraph" >The Nagoya District Public Prosecutors Office indicted Shota Suito, a 34-year-old former teacher at a Nagoya municipal elementary school, on suspicion of violating the Law on Punishment of Activities Relating to Child Prostitution and Child Pornography, and the Protection of Children.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >This marks the first time in Japan that sexual deepfakes - obscene images created using generative AI - have been judged to constitute child pornography and the first time a person has been prosecuted under this law.</p>
<p class="articleParagraph enarticleParagraph" >The Aichi prefectural police sent papers on Suito to prosecutors in November.</p>
<p class="articleParagraph enarticleParagraph" >Suito is already on trial for charges including violating a different law against photographing sexual postures and for allegedly sharing material, including images of minors that were secretly taken, in a group chat.</p>
<p class="articleParagraph enarticleParagraph" >According to the indictment in the latest case, Suito allegedly possessed two sexually explicit images of children at his home in March. The images had been edited and processed using a site that created AI-generated images. He had sent images of two children stored at his school to another person and had them create nude images using generative AI, according to the prefectural police.</p>
<p class="articleParagraph enarticleParagraph" >Possession and production of sexual images of children are regulated under the child pornography law, but past judicial precedents assumed the victims were real children, which made it difficult to apply the law to deepfakes. The prefectural police determined the images constituted child pornography, citing factors such as identifying real children from the faces in the sexual AI-generated images.</p>
<p class="articleParagraph enarticleParagraph" >"Determining that sexually explicit images of children created using generative AI constitute child pornography challenges the premise of the child pornography law, which requires depictions of real life images," said Associate Prof. Masaki Ueda of Kanagawa University, an expert on obscenity regulations. "This judgment is conducive to preventing cases in which parts of images of real children are used, with them being sexually depicted using generative AI."</p>
<p class="articleParagraph enarticleParagraph" >However, Ueda also said there is a risk the lines could be blurred between creative works and actual child pornography. "It is necessary to limit the scope of the law's application, for example, by requiring that the faces in the original image and the generated image be identical."</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gchlab : Child Abuse | gcom : Society/Community | gcrim : Crime/Legal Action | gcsci : Computer Science | ggenai : Generative AI | gporn : Pornography | grape : Sex Crimes | gsci : Sciences/Humanities | gsoc : Social Issues</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>aichi : Chubu | apacz : Asia Pacific | asiaz : Asia | easiaz : East Asia | jap : Japan</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>The Yomiuri Shimbun</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document YOMSHI0020251206elc70000c</td></tr></table><br/></div></div><br/><span></span><div id="article-MANI000020251206elc700015" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/maniLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>New AI tools for PH firms unveiled</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>The Manila Times </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>434 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>The Manila Times</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>MANI</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025. The Manila Times </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >A GLOBAL technology company providing cloud, software and AI services, introduced new artificial intelligence features at its Ignite 2025 conference on Wednesday, presenting tools designed to support the Philippines' accelerating shift toward digital transformation.</p>
<p class="articleParagraph enarticleParagraph" >
                        <span class="companylink">The Department of Information and Communications Technology</span> has been urging wider adoption of digital systems and AI across the country. With sectors such as business process outsourcing and small and medium enterprises facing rising security and productivity demands, <span class="companylink">Microsoft</span> said its latest AI tools releases are intended to help organizations modernize operations while keeping data protected.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >One of the highlighted tools, Agent 365, serves as a centralized platform for managing AI-driven workflows. The system is aimed at large corporations and outsourcing firms that handle multiple AI tools and strict compliance requirements. The BPO industry employs about 1.82 million Filipinos and generates $38 billion in annual revenue.</p>
<p class="articleParagraph enarticleParagraph" >The company also emphasized security as a foundation of its AI ecosystem, citing the need for Philippine organizations to comply with the Data Privacy Act of 2012 and other sector-specific regulations. <span class="companylink">Microsoft</span> said its zero-trust approach is designed to mitigate risks from emerging AI threats as local businesses expand their use of automated systems.</p>
<p class="articleParagraph enarticleParagraph" >For SMEs, which make up 99.5 percent of registered businesses in the country, <span class="companylink">Microsoft</span> introduced agent-driven business applications that move beyond traditional record-keeping. These tools are built to deliver real-time insights and automated processes for faster decision-making. Industry forecasts project the Philippine AI market to reach $1.025 billion in 2025.</p>
<p class="articleParagraph enarticleParagraph" >Productivity tools combining Copilot and AI agents were also presented, with <span class="companylink">Microsoft</span> saying they can help automate routine tasks and support hybrid work setups. The company cited a survey showing that 74 percent of Filipino business leaders believe AI could improve workplace efficiency.</p>
<p class="articleParagraph enarticleParagraph" >On the security front, <span class="companylink">Microsoft</span> announced Edge for Business, which it described as a secure enterprise AI browser. The platform integrates security functions and productivity features directly into the browser environment, a move the company says can help Philippine firms reduce costs and strengthen cyber defenses.</p>
<p class="articleParagraph enarticleParagraph" >"Philippine businesses are ready to embrace the next wave of intelligent technology. These innovations from <span class="companylink">Microsoft</span> Ignite 2025 will help organizations harness AI responsibly while ensuring security and compliance," a <span class="companylink">Microsoft</span> spokesman said. "Our mission is to empower every enterprise, from SMEs to large conglomerates, to thrive in this new era of digital transformation."</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Microsoft</span> said the developments align with the <span class="companylink">DICT</span>'s goal of a digitally empowered Philippines and position local firms to take advantage of new AI-driven opportunities.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>incotk : Department of Information and Communications Technology | mcrost : Microsoft Corporation</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302 : Computers/Consumer Electronics | i330202 : Software | i3302021 : Applications Software | i3302022 : Artificial Intelligence Technologies | icomp : Computing | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c22 : New Products/Services | ccat : Corporate/Industrial News | cexpro : Products/Services | csmlbs : Small/Medium Businesses | gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcsci : Computer Science | gdatap : Privacy Issues/Information Security | gsci : Sciences/Humanities | ncat : Content Types | nfact : Factiva Filters | nfcpex : C&E Executive News Filter | nfcpin : C&E Industry News Filter</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | asiaz : Asia | devgcoz : Emerging Market Countries | dvpcoz : Developing Economies | phlns : Philippines | seasiaz : Southeast Asia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>The Manila Times Publishing Corp.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document MANI000020251206elc700015</td></tr></table><br/></div></div><br/><span></span><div id="article-MANI000020251206elc70000z" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/maniLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Preparing Pinoys for the future of work</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>By Derrick Latreille </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1096 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>The Manila Times</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>MANI</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025. The Manila Times </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >ARTIFICIAL intelligence (AI) is not just another buzzword. It is the force that is rapidly reshaping the future of work. For Filipinos, the challenge is no longer about finding that one "dream job" and holding onto it for life. Instead, the question we should be asking ourselves is: Which fast-moving wave am I going to ride?</p>
<p class="articleParagraph enarticleParagraph" >We should view AI as the big tsunami. It is redefining how companies approach productivity and decision-making — two things that are critical to every organization. Yet many still hesitate to dive in. In this era, lifelong learning is no longer a slow, leisurely pursuit. It means being aggressive in picking up skills as fast as the world changes, as those who delay risk being left behind.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >This urgency is not optional because the demand for skilled professionals across industries, especially in tech and business fields, continues to accelerate. Roles in software development — particularly DevOps (the integration of software development and information technology operations to deliver quick and high-quality performance) — and systems networking remain strong, alongside cloud computing, hardware engineering and cybersecurity. Anything with the word "data" in front of it — data management, data analytics, data engineering, data science — is in demand. Blockchain expertise is another field that continues to expand.</p>
<p class="articleParagraph enarticleParagraph" >Preparing for the future of work</p>
<p class="articleParagraph enarticleParagraph" >While there is a spike in these careers, the demand curves will not look the same. Cybersecurity, for instance, will remain a growth area for decades, as seen by United States projections that indicate nearly 30 percent growth in cybersecurity jobs by 2033. By contrast, the current surge in software development may taper off as automation tools advance. It will be subtle. You won't notice layoffs in five years, but you'll notice it's hard to get started as a software developer. Understanding these trends helps us see the future of work better and allows us to prepare for it by learning early and strategically.</p>
<p class="articleParagraph enarticleParagraph" >Alongside this, there is also a rising demand in areas that require a blend of digital expertise and business savvy, like e-commerce, logistics and marketing technology. At Mapúa Malayan Digital College (MMDC), industry-relevant programs in Data Analytics, Software Development, Network and Cybersecurity, Marketing Technology and Entrepreneurship Technology are the direct response to what the market is seeking today. Our goal is simple: to give learners the tools and experiences they need to move where opportunity is going.</p>
<p class="articleParagraph enarticleParagraph" >Given the pace of tech innovations today, students and professionals alike must build core technical fluency to thrive. This means learning to code — and not so much because you'll be building a lot of code. A lot of people working with AI need to use code blocks, just needing to be able to read them to understand what they are doing. Many marketing technologists use code snippets.</p>
<p class="articleParagraph enarticleParagraph" >Again, reading, understanding and choosing is the key. Python and Java are two of the most versatile languages today and are great starting points if you want to get into the tech industry. Just as important are solid math skills, statistics and probability, and basic algebra, which underlie data analytics — the biggest and fastest-moving field. More than keeping up with the future of work, those who grasp these fundamentals will help shape it. In other words, you don't need to predict the future; you need to build the skills that make you adaptable to whatever comes next.</p>
<p class="articleParagraph enarticleParagraph" >At the same time, robotics is advancing at a pace that will likewise transform industries from manufacturing to healthcare. Edge computing will decongest the cloud by pushing computation to the device level. Spatial computing, including augmented reality (AR), virtual reality (VR) and quantum technologies, is also on the rise due to cost breakthroughs.</p>
<p class="articleParagraph enarticleParagraph" >For many, hearing this list might sound overwhelming. But I want to emphasize this: far from being saturated, technology fields will continue to need talent. That is why, even for early and mid-career professionals, it is never too late to pivot and enter a new industry. Coding, after all, is not primarily a math skill; it is a language skill. If you can learn a foreign language, you can learn to code. Remember that the sooner you begin, the more waves you'll be able to ride.</p>
<p class="articleParagraph enarticleParagraph" >Beyond technical skills, soft skills will determine who will rise fastest. Organizations like <span class="companylink">LinkedIn</span>, the <span class="companylink">World Economic Forum</span> and Harvard Business Review consistently rank communication, adaptability, creative thinking and emotional intelligence as the most important skills to have when entering the workplace. Alongside these are resilience, strategic thinking and collaboration.</p>
<p class="articleParagraph enarticleParagraph" >These skills are vital to learn early in one's career. Most classrooms don't teach this. This is why it is important for young professionals to keep volunteering for projects, whether at school, work or even in the community. Doing things that are hard and unfamiliar is when real growth happens.</p>
<p class="articleParagraph enarticleParagraph" >At MMDC, we have built our academic approach around this philosophy. Our Projects, Problems, Cases (PPC) learning model puts students in industry-inspired scenarios from day one. Instead of listening to lectures, they engage in solving real-world problems that companies actually face. This approach strengthens students' technical mastery while developing critical thinking, leadership and collaboration — traits employers are looking for.</p>
<p class="articleParagraph enarticleParagraph" >This model reflects our belief that education should not simply prepare students to pass exams; it should prepare them to ride tsunamis. The workplace is moving too quickly, so students must be empowered to think for themselves, adapt to new tools and work in diverse teams. At MMDC, we aim to build this mindset of helping Filipinos not just find jobs, but future-proof their careers. By embedding PPC into our programs, graduates leave with the skills, confidence and agility to apply their learning where it is needed most.</p>
<p class="articleParagraph enarticleParagraph" >The world is changing faster than ever. Ultimately, the future of work will be about catching waves as they form, adjusting your skills as they go and knowing when to leap onto the next wave. A lot of people are thinking, "What's my dream job?" Instead, it should be "What tsunami am I going to ride?" The tsunamis of AI, data and emerging technologies are already here. The only question left is: Are you going to ride them?</p>
<p class="articleParagraph enarticleParagraph" >Derrick Latreille, is the chief learning officer at Mapúa Malayan Digital College, a digital-first, technology-focused college in the Philippines designed to offer flexible, industry-aligned degree programs. It is part of the Mapúa University and Malayan Colleges network under the Yuchengco Group of Companies.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcsci : Computer Science | gsci : Sciences/Humanities</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | asiaz : Asia | devgcoz : Emerging Market Countries | dvpcoz : Developing Economies | phlns : Philippines | seasiaz : Southeast Asia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>The Manila Times Publishing Corp.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document MANI000020251206elc70000z</td></tr></table><br/></div></div><br/><span></span><div id="article-MANI000020251206elc700010" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/maniLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>
                           Salesforce bets on PH's AI future</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>Noemi Lardizabal-Dado Let's Talk Social </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>834 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>The Manila Times</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>MANI</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025. The Manila Times </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >AI has become a constant headline, but <span class="companylink">Salesforce</span>'s move to open a new office in Manila lands at a moment when Filipino businesses are figuring out how the technology will shape their growth and the skills their teams will need next. Access Partnership's report, in collaboration with <span class="companylink">Google</span>, projects that AI adoption could generate P2.8 trillion in economic benefits for Philippine businesses by 2030 through productivity gains and cost savings.</p>
<p class="articleParagraph enarticleParagraph" >It was in this setting that Gavin Barfield, vice president and CTO for Solutions for Salesforce Asean, said we would look back at this period as "one of, if not the most significant technology innovations of our lifetime." He wasn't talking about a new app. He was pointing to a shift in how work happens, how people relate to systems, and how companies operate at scale.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >That shift is what <span class="companylink">Salesforce</span> calls the agentic enterprise. And Manila is now one of its new homes.</p>
<p class="articleParagraph enarticleParagraph" >From the cloud era to the agentic era</p>
<p class="articleParagraph enarticleParagraph" >He has seen several waves of technology reshape organizations. <span class="companylink">Salesforce</span> helped pioneer cloud computing about 25 years ago. Businesses then absorbed mobile, social and predictive AI. What comes next, he said, is a deeper shift.</p>
<p class="articleParagraph enarticleParagraph" >AI systems are no longer just answering questions. They are beginning to take on tasks, coordinate actions and draw from knowledge scattered across an organization. The shift, he said, will change "the way that we work in the office, the way that we interact at home, [and] the way that we contact companies."</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Salesforce</span> is positioning Agentforce 360 as the foundation for that change. It brings together apps, data, metadata and agents in one unified environment so employees can hand off routine work and focus on higher-value decisions.</p>
<p class="articleParagraph enarticleParagraph" >To illustrate this, he pointed to <span class="companylink">Salesforce</span>'s own use of these tools. The company now runs more than 14 internal agents, including one on help.<span class="companylink">salesforce.com</span> that searches thousands of documents such as manuals, release notes and past resolutions to answer customer questions around the clock. It has already helped save roughly $1 million in analyzed support costs.</p>
<p class="articleParagraph enarticleParagraph" >A unified view of data</p>
<p class="articleParagraph enarticleParagraph" >AI agents only work as well as the information they learn from, and for most companies, that information lives in scattered systems and formats.</p>
<p class="articleParagraph enarticleParagraph" >Data 360 is <span class="companylink">Salesforce</span>'s response. It brings together structured data and unstructured content such as emails, <span class="companylink">Slack</span> threads, PowerPoints, Word files and internal manuals to form a single, usable view. He noted how much intelligence sits in these materials because that's where companies store the details of how work actually gets done. When agents can interpret this, they act with context instead of guesswork.</p>
<p class="articleParagraph enarticleParagraph" >He also didn't gloss over the hurdles. Many companies struggle with bad or untrusted data, bolt-on solutions, unclear priorities and do-it-yourself efforts that never progress beyond a proof of concept. As he puts it, a lot of projects get stuck in "POC-land" because data and governance aren't addressed early enough.</p>
<p class="articleParagraph enarticleParagraph" >This is part of what <span class="companylink">Salesforce</span> calls the "agentic divide": the growing gap between companies eager to adopt AI and those unable to deploy it at scale.</p>
<p class="articleParagraph enarticleParagraph" >Building capability, testing real-world impact</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Salesforce</span>'s expansion isn't just about platforms. The company has committed to training 12,000 Filipino workers in customer relationship management and AI skills over the next five years, on top of the 53,000 who have already completed courses on Trailhead. Government leaders welcomed the focus on talent development. <span class="companylink">DICT</span> Secretary Henry Aguda said AI "will unlock new innovations, sharpen efficiencies and create opportunities for every sector," noting that access to trusted platforms helps Filipino companies compete more effectively.</p>
<p class="articleParagraph enarticleParagraph" >Local organizations are already putting these tools to work. <span class="companylink">Ayala Land</span>, BPI, Meralco, <span class="companylink">Philippine Airlines</span>, <span class="companylink">PLDT</span> and Maxicare use <span class="companylink">Salesforce</span> CRM and AI to improve customer experience and create new revenue opportunities. <span class="companylink">Converge ICT Solutions</span> is going even further. It is building one of the country's first generative AI contact centers, aiming for a 30 percent efficiency lift as part of its broader push to evolve from a traditional telco into a full technology company.</p>
<p class="articleParagraph enarticleParagraph" >Together, these efforts show how capability building and early adoption reinforce each other. As more Filipino workers gain AI skills, more companies are ready to test real-world applications that deliver measurable results.</p>
<p class="articleParagraph enarticleParagraph" >Beyond the technology</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Salesforce</span>'s involvement in the DOST's Starbooks initiative, supporting 475 students in Mindoro and Bataan, adds a human layer to its expansion. It hints at how access, skills and opportunity remain central to any digital shift.</p>
<p class="articleParagraph enarticleParagraph" >And maybe that's the real story here. Agentic AI will reshape workflows and business models, but progress ultimately depends on people: the workers learning new tools, the teams redesigning processes and the institutions making room for innovation. The Philippines has the momentum. The question now is how widely and how well this transformation will be shared.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>salesf : Salesforce Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302 : Computers/Consumer Electronics | i330202 : Software | i3302021 : Applications Software | i3302022 : Artificial Intelligence Technologies | icomp : Computing | icrmsw : Customer Relationship Management Software | ientrps : Enterprise Management Software | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>ccat : Corporate/Industrial News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | asiaz : Asia | devgcoz : Emerging Market Countries | dvpcoz : Developing Economies | manil : Manila | phlns : Philippines | seasiaz : Southeast Asia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>The Manila Times Publishing Corp.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document MANI000020251206elc700010</td></tr></table><br/></div></div><br/><span></span><div id="article-MANI000020251206elc700014" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/maniLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>AI tie-up aims to curb digital fraud risk</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>The Manila Times </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>367 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>The Manila Times</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>MANI</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025. The Manila Times </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >TRUSTING Social Philippines, an AI-focused credit scoring and identity verification firm, and Ecofinance, a digital lending company behind consumer brand Honey Loan, have partnered to strengthen security measures amid rising fraud cases affecting online financial transactions in the country.</p>
<p class="articleParagraph enarticleParagraph" >The agreement, announced this week, enables Ecofinance to use Trusting Social's artificial intelligence tools for customer verification and fraud detection. The integration includes facial recognition, cross-ID validation and real-time monitoring to help block threats such as phishing, SIM swapping and deepfake-enabled identity fraud.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Ecofinance said the move supports its goal of widening access to digital lending while ensuring consumer protection. Citing a 2025 <span class="companylink">World Bank</span> report, the company noted that more than half of Filipino adults still do not own a formal financial account, underscoring the need for secure digital channels. Fraud incidents in the Philippines were reported to be nearly 150 percent higher than the global average, with identity-related cases up 121 percent in 2024.</p>
<p class="articleParagraph enarticleParagraph" >"Inclusive lending needs to go hand in hand with secure lending. We view our partnership with Trusting Social not as compliance, but as a critical investment in protecting our customers," Ecofinance Philippines General Manager Kirill Kalashnikov said. "We're building a digital experience that ensures every customer receives not just financial freedom, but peace of mind."</p>
<p class="articleParagraph enarticleParagraph" >Trusting Social said its experience in regional markets, including Vietnam and Indonesia, would help adapt AI-driven security measures to local conditions. The company expects the deployment to streamline onboarding processes, improve document checks and reduce operational costs for Ecofinance.</p>
<p class="articleParagraph enarticleParagraph" >"In a market where digital trust is under attack, Trusting Social strongly echoes Ecofinance's priority of security — it's the only way to build sustainable inclusion," Trusting Social Philippines CEO Johnny Escaler said. "Our job is to provide the AI intelligence needed to protect that journey — to confidently verify who they are and shield them from the rising wave of digital crime."</p>
<p class="articleParagraph enarticleParagraph" >The companies said the initiative supports the <span class="companylink">Bangko Sentral ng Pilipinas</span>' call for improved fraud management under the Anti-Financial Account Scamming Act and related directives requiring financial institutions to implement real-time detection systems.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302022 : Artificial Intelligence Technologies | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c12 : Corporate Crime/Legal Action | c17 : Corporate Funding | c173 : Financing Agreements | ccat : Corporate/Industrial News | gcat : Political/General News | gcrim : Crime/Legal Action | gfraud : Fraud | ncat : Content Types | nfact : Factiva Filters | nfcpex : C&E Executive News Filter | nfcpin : C&E Industry News Filter</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | asiaz : Asia | devgcoz : Emerging Market Countries | dvpcoz : Developing Economies | phlns : Philippines | seasiaz : Southeast Asia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>The Manila Times Publishing Corp.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document MANI000020251206elc700014</td></tr></table><br/></div></div><br/><span></span><div id="article-MANI000020251206elc700011" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/maniLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>2026 prediction: Identity crisis — The rise of 'It wasn't me — my AI did it!'</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>Tony Maghirang Tech Space </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>934 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>The Manila Times</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>MANI</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025. The Manila Times </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >AS we head into the new year 2026, <span class="companylink">Jumio</span>, the leader in AI-powered identity intelligence, sought to define the next phase of APAC's digital economy in terms of the shift from AI assistance to AI autonomy.</p>
<p class="articleParagraph enarticleParagraph" >In this paradigm shift, Jumio APAC Managing Director Ee Khoon Oon predicts that as AI agents begin moving money across borders and signing contracts across Asean, APAC will face a crisis where "my AI did it" becomes a common defense. This will force regulators in Singapore and beyond to evolve from traditional KYC (Know Your Customer) to KYA — Know Your Agent — requiring a verifiable chain of custody that binds every autonomous action back to a human biometric, especially for high-risk sectors.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >This need for accountability is the thread connecting all predictions for 2026: from the "AI versus AI" fraud arms race and the rise of continuous liveness detection to the global push toward reusable digital identity under frameworks like eIDAS 2.0. The consensus is clear: Identity is no longer a one-time check. It is becoming a continuous, portable anchor in a world where we can no longer distinguish human from machine by sight alone.</p>
<p class="articleParagraph enarticleParagraph" >Following are more 2026 predictions:</p>
<p class="articleParagraph enarticleParagraph" >
                     Robert Prigge, chief executive: Identity intelligence will separate market leaders</p>
<p class="articleParagraph enarticleParagraph" >In 2026, identity will either be a company's strongest differentiator or its weakest link. We are entering an era where AI is transforming business and transforming fraud. The cost is not just revenue loss, but long-term reputational damage, regulatory exposure and erosion of customer trust.</p>
<p class="articleParagraph enarticleParagraph" >Identity verification must become continuous, adaptive and anticipatory, predicting and preventing risk before it occurs while remaining nearly invisible to the end user. Identity intelligence brings together data across identity, historical, behavioral and risk checks to build a dynamic view of a user over time. Instead of verifying once and hoping for the best, organizations can continuously assess trust in the background, adapting to new signals as they emerge. When fraud happens, customers do not blame the criminal, they blame the brand.</p>
<p class="articleParagraph enarticleParagraph" >Bala Kumar, chief product and technology officer: 2026 — The breakout year for reusable identity</p>
<p class="articleParagraph enarticleParagraph" >In 2026, reusable identity will move from industry buzzword to operational reality, and it will fundamentally change how authentication works. Once an individual is verified with high assurance, their identity becomes portable, allowing them to authenticate seamlessly anywhere across a trusted network without repeating onboarding. This is the moment authentication collapses into onboarding. Identity stops being a one-off check and becomes a persistent asset.</p>
<p class="articleParagraph enarticleParagraph" >As reusable identity becomes the standard, the market will split into two groups: those with a reusable identity network at global critical mass and everyone else. The first group will deliver frictionless experiences, detect fraud patterns that emerge only across ecosystems and materially reduce operational costs. Everyone else will be stuck re-verifying users repeatedly — paying more, converting less and missing the signals that matter.</p>
<p class="articleParagraph enarticleParagraph" >Reusable identity is not a feature. It is the new architecture of trust, and 2026 is the year it becomes unavoidable.</p>
<p class="articleParagraph enarticleParagraph" >Reinhard Hochrieser, senior vice president of product and technology: A potential change in identity verification for social media platforms</p>
<p class="articleParagraph enarticleParagraph" >Social media platforms have begun using behavioral analytics for age verification, but the approach is fraught with consumer privacy tensions and challenges. As new biometric detection tools emerge, companies will begin to deploy a multifold approach, implementing AI age estimation combined with advanced liveness detection to prove a user is real, the correct age and to prevent deepfakes. Digital identity wallets will also begin to emerge due to their data minimization benefits, allowing users to prove their age without revealing other personal data.</p>
<p class="articleParagraph enarticleParagraph" >Additionally, by recognizing the security risks posed by centralized data storage — a honeypot for fraudsters — companies will increasingly move to decentralize customer data. As privacy becomes more important across the globe, those who shift away from the current approach to identity verification will see increased compliance, reduced security risks from sophisticated fraud attacks and greater privacy awareness in 2026.</p>
<p class="articleParagraph enarticleParagraph" >Ashwin Sugavanam, vice president of AI and identity analytics: The advent of advanced liveness and layered defense strategies</p>
<p class="articleParagraph enarticleParagraph" >AI has made it alarmingly easy to create fraudulent digital identities. The once-reliable barrier of biometric authentication is being compromised through camera injection attacks, where AI-generated or manipulated images are inserted into live video streams to deceive even highly sophisticated security systems.</p>
<p class="articleParagraph enarticleParagraph" >We are entering an era defined not just by AI-driven fraud but by a perpetual arms race between adversarial AI and defensive AI. Multimodal liveness detection — combining visual, auditory and motion-based signals — will remain critical, but its real power lies in how it integrates into a broader identity intelligence framework. This includes leveraging cross-customer fraud intelligence, behavioral biometrics and transaction risk analytics to identify patterns of fraud before they manifest.</p>
<p class="articleParagraph enarticleParagraph" >Alix Melchy, vice president of AI: AI and privacy in future fraud prevention</p>
<p class="articleParagraph enarticleParagraph" >AI agents are making fraud more accessible and personalized than ever before. As we look ahead to 2026, fraud prevention in the digital space will be about balancing security with user experience. Intelligent friction will become a critical strategy for businesses to address this challenge. By using AI and machine learning, companies will be able to tailor verification steps based on a user's risk profile and behavior. Low-risk users will experience a smoother, less intrusive process, while high-risk users will be flagged for additional scrutiny. This approach will help organizations meet compliance and security requirements without alienating legitimate users.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>jumioi : Jumio, Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302 : Computers/Consumer Electronics | i330202 : Software | i3302021 : Applications Software | i3302022 : Artificial Intelligence Technologies | icomp : Computing | iphmetrix : Biometrics Technology | isecpri : Security/Privacy Software | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcns : National/Public Security | gcsci : Computer Science | gsci : Sciences/Humanities | gsec : State Security Measures/Policies</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>The Manila Times Publishing Corp.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document MANI000020251206elc700011</td></tr></table><br/></div></div><br/><span></span><div id="article-NORTHT0020251206elc700004" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/northtLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>NewsLiftout</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>‘YOU’LL CHANGE THE WORLD’</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>Vanessa Marsh US Correspondent </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1324 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>NT News</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>NORTHT</td></tr><tr><td align="right" valign="top" class="index"><b>ED</b>&nbsp;</td><td>NTNews</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>29</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© News Pty Limited. No redistribution is permitted. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Australia’s bold experiment must succeed for humanity’s sake, says child safety expert Jonathan Haidt</p>
<p class="articleParagraph enarticleParagraph" >LET THEM BE KIDS Australia’s looming social media ban for under-16s is the most significant child‑protection measure ever taken anywhere in the world and dwarfs all other efforts, a leading ­expert has declared.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Renowned child safety advocate Jonathan Haidt predicted tech behemoths would “play dirty” in the coming weeks to try to discredit the efficacy of Australia’s new laws in a bid to protect their bottom lines, but warned failure was not an option with the world watching, poised to follow suit.</p>
<p class="articleParagraph enarticleParagraph" >Dr Haidt, who authored best-selling book The Anxious Generation, which exposed how the rise in smartphones and social media caused an explosion in child mental illness rates around the world, said he expected the benefits of the ban to become evident within months.</p>
<p class="articleParagraph enarticleParagraph" >“Whatever the difficulties and of course, there will be difficulties in implementing this, this is a very bold law, but whatever the difficulties, imagine not doing it,” he said.</p>
<p class="articleParagraph enarticleParagraph" >“Imagine that we condemn the rest of humanity, we ­condemn the kids who are infants today, imagine we condemn them to growing up scrolling and watching short videos and falling in love with AI companions and not living life in the world.</p>
<p class="articleParagraph enarticleParagraph" >“If Australia didn’t do this, or if Australia doesn’t succeed and we stick with business as usual, I think the effect on ­humanity is incalculable, so it has to be done.” In a heartening prediction for parents fearing intense withdrawal symptoms from their children, Dr Haidt noted surveys in which more than half of Gen Z participants said they wished social media was never invented.</p>
<p class="articleParagraph enarticleParagraph" >“They see that it’s a trap, but they just can’t get out of it on their own,” he said. “My … prediction is that Australian kids will surprise adults by being less upset about this than the adults expect.” Dr Haidt referenced his own teenage daughter’s experience when phones were banned at her high school in September. Within two weeks she commented how much happier kids were playing games and cards and talking instead of silently scrolling.</p>
<p class="articleParagraph enarticleParagraph" >Dr Haidt has written a new handbook for tweens to be released in the coming weeks, named The Amazing Generation, which empowers kids to choose a life not dominated by screens.</p>
<p class="articleParagraph enarticleParagraph" >Asked about speculation the companies could intentionally switch off adult accounts to cause backlash, Dr Haidt said history showed the tech ­behemoths would do anything to win.</p>
<p class="articleParagraph enarticleParagraph" >“We can expect that they will work hard to make it look like the Australia law is not working,” he said. “We can expect they will play dirty.</p>
<p class="articleParagraph enarticleParagraph" >“They play to win, and there’s a lot of money at stake. So that is my expectation.” But Dr Haidt said it was critical Australia forged ahead despite the David and Goliath-like battle, and “absolutely crucial” that other countries eyeing a similar move followed suit.</p>
<p class="articleParagraph enarticleParagraph" >“Given that many other countries are likely to follow, that is going to create a rock slide, a tidal wave, a global movement for change,” he said. “Parents everywhere are upset about this, they just thought there was nothing they could do and Australia is showing us, wait, we do get to say how companies treat our children. And so I think Australia is going first, but since so many other countries are already announcing that they’re going to do it, I think this is going to change the world.” Greece, the UK, France and Fiji are among the countries to announce aspirations to implement similar bans pending the outcome of Australia’s world-leading move.</p>
<p class="articleParagraph enarticleParagraph" >“What Australia did is by far the biggest thing that has ever been done to protect children,” Dr Haidt said.</p>
<p class="articleParagraph enarticleParagraph" >“It dwarfs everything else.</p>
<p class="articleParagraph enarticleParagraph" >“We can mess around with making algorithms safer for eight-year-olds, we can mess around with content moderation – none of that stuff is going to move the needle.” Speaking from his office in New York, Dr Haidt said it was imperative Australia didn’t rest after implementing the ­social media ban, warning a far more dangerous and sinister threat was looming – AI.</p>
<p class="articleParagraph enarticleParagraph" >He described the chatbots and companion technology as the “next uncontrolled mass experiment that Silicon Valley wants to perform on the world’s children”.</p>
<p class="articleParagraph enarticleParagraph" >“We missed the window to act with social media because we were all in awe of these products,” he said, quoting the phrase, “fool me once, shame on you, fool me twice, shame on me”.</p>
<p class="articleParagraph enarticleParagraph" >“We’re now entering a new phase of digital childhood as an even more transformative technology rolls in like a tidal wave.</p>
<p class="articleParagraph enarticleParagraph" >“This time, we will not be able to say we didn’t know, ­because we know there are ­already so many dead kids.” Over the past year, there has been a surge in disturbing cases from around the world in which AI chatbots allegedly sexually groomed children and coached others to suicide.</p>
<p class="articleParagraph enarticleParagraph" >“There’s never been a technology that came on this fast and this deadly to childhood so if we don’t do something, if we don’t do anything to stop this, then we are idiots,” Dr Haidt said.</p>
<p class="articleParagraph enarticleParagraph" >Australia’s social media laws will come into effect from Wednesday, increasing the legal age of access to social media from 13 to 16 for ­Australian kids.</p>
<p class="articleParagraph enarticleParagraph" >The new legislation was brought about by advocacy from <span class="companylink">News Corp Australia</span>’s Let Them Be Kids campaign, which highlighted the devastating harms being caused to kids through social media.</p>
<p class="articleParagraph enarticleParagraph" >Dr Haidt, who advocated for an under-16 social media ban in his book, said it was ­unrealistic to expect a 100 per cent compliance rate, with kids likely to find workarounds and social media companies unlikely to block every single underage user.</p>
<p class="articleParagraph enarticleParagraph" >“It doesn’t matter if a few per cent are still on it, what matters is the norm, what matters is the social pressure,” he said.</p>
<p class="articleParagraph enarticleParagraph" >“And if half of all kids are on it, then there’s a lot of pressure on the other half to be on but if it’s only a few per cent, then … we’re free.</p>
<p class="articleParagraph enarticleParagraph" >“So success is changing the norm, and the norm right now is that kids open their first ­social media accounts around the age of eight or nine, usually with TikTok, and that all has to stop.” Dr Haidt predicted a teething period in the first few months to work through technical glitches.</p>
<p class="articleParagraph enarticleParagraph" >“I’m not expecting the Australia bill to work properly in December, but I am expecting that it will work well by February,” he said.</p>
<p class="articleParagraph enarticleParagraph" >“I would guess that within three months after that, we’ll start seeing visible changes in behaviour.</p>
<p class="articleParagraph enarticleParagraph" >“So it’ll take the kids a little while to remember how to play, to remember how to interact with each other directly, and a lot will depend on whether Australian parents give them back a real childhood.” Hitting back at critics who argued the access choice should remain with parents or that the laws were censorship, Dr Haidt said the decision was already taken out of parents’ hands because any child who could access the internet could create as many accounts as they wanted.</p>
<p class="articleParagraph enarticleParagraph" >“Parents are desperate for help because the tech industry has taken childhood and put us into a collective action trap,” he said.</p>
<p class="articleParagraph enarticleParagraph" >“On the idea that it’s censorship, there’s no issues about content here. “In fact, kids can still see content. “The Australia bill is well written, so it’s not about blocking kids from seeing things, it’s about contract law and I would ask any sceptic, what is the right age at which your child can sign a contract with a gigantic company that is known to prey on children?”</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RF</b>&nbsp;</td><td><br/>Northern Territory News-20251207-NTNews-29-000496793510 </td></tr>
<tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gbook : Books | gcat : Political/General News | gent : Arts/Entertainment</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | ausnz : Australia/Oceania | austr : Australia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Nationwide News Pty Ltd.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document NORTHT0020251206elc700004</td></tr></table><br/></div></div><br/><span></span><div id="article-NORTHT0020251206elc70000t" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/northtLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>WBusiness</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>SHARE tips</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>195 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>NT News</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>NORTHT</td></tr><tr><td align="right" valign="top" class="index"><b>ED</b>&nbsp;</td><td>NTNews</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>45</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© News Pty Limited. No redistribution is permitted. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Abigail Cowley Bell Potter Securities</p>
<p class="articleParagraph enarticleParagraph" >BUY Stockland Group (SGP) Strong dividend yield and will benefit from first home buyers’ government scheme.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Wisetech Global</span> (WTC) Recently took a share price hit following a sector downgrade and ongoing governance issues. This presents an attractive buying opportunity.</p>
<p class="articleParagraph enarticleParagraph" >HOLD <span class="companylink">Telstra Group</span> (TLS) <span class="companylink">Telstra</span> expect the demand for mobile and digital connectivity to increase in the near term as driven by AI and continued technological advancements.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Goodman Group</span> (GMG) Focusing on property investment in logistics and infrastructure, a booming industry. GMG is integrating AI and robotics into operations.</p>
<p class="articleParagraph enarticleParagraph" >SELL Transurban Group (TCL) Currently trading at a desirable price to take profits. The company faces regulatory risks with the NSW toll reform.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Bendigo & Adelaide Bank</span> (BEN) Financials have significantly underperformed, and its dividend sustainability appears under pressure. Deficiencies in AML/CTF controls have been self-reported to <span class="companylink">AUSTRAC</span>.</p>
<p class="articleParagraph enarticleParagraph" >The company and analyst may have long or short positions on these stocks. These tips don’t take into account individual financial situations. They are summaries only. Readers should obtain copies of the full research reports and disclosures and seek financialadvice before investing</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RF</b>&nbsp;</td><td><br/>Northern Territory News-20251207-NTNews-45-000496788057 </td></tr>
<tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>atraly : Australian Transaction Reports and Analysis Centre | bgobs : Bendigo & Adelaide Bank Ltd | tcoma : Telstra Group Ltd</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i7902 : Telecommunication Services | i79022 : Wireless Telecommunications Services | i7902202 : Mobile Telecommunications | i814 : Banking | i81402 : Commercial Banking | ibnk : Banking/Credit | ifinal : Financial Services</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | ausnz : Australia/Oceania | austr : Australia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Nationwide News Pty Ltd.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document NORTHT0020251206elc70000t</td></tr></table><br/></div></div><br/><span></span><div id="article-COUMAI0020251206elc70003r" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/coumaiLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>WNews</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Building a $1bn dream from scratch</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>Chris Herde </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>766 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Courier Mail</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>COUMAI</td></tr><tr><td align="right" valign="top" class="index"><b>ED</b>&nbsp;</td><td>CourierMail</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>63</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© News Pty Limited. No redistribution is permitted. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >
                        Michael McNab believes he’s an “accidental” businessman and his main ambition when he started his eponymous company almost three decades ago was to feed his family and stay afloat. And from these small beginnings, the Toowoomba-based company now has a staff of more than 700 and is on track for $1bn plus in turnover in the 2026 financial year.</p>
<p class="articleParagraph enarticleParagraph" >But Mr McNab would be the first to admit it hasn’t all been clear sailing, especially during the pandemic, when rumours swept the construction sector that the group would soon be joining the swelling ranks of failed builders.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >“We were never in trouble. We were more honest than other builders. Our costs escalated 30 per cent in 18 months and if builders in Queensland say they didn’t have a tough time over covid they are bullshitting," he said.</p>
<p class="articleParagraph enarticleParagraph" >“But the rumours about us were savage and they came from a couple of people. We know who they are. We were never going anywhere. We were hurting like everyone else. It was terrible and we were down in the trenches working hard. The noise was upsetting our staff, but it wasn’t upsetting me because I wasn’t worried.” In the darkest days of the pandemic, revenue in the 2021 financial year dropped to $399m with a wafer-thin $3.5m net profit before tax, which dropped to $2.4m the year after. However, the company recovered and in FY25 McNab reported $834m in revenue and $29.1m profit which included significant investments in technology and innovation projects to support in the company’s five-year growth and productivity goals.</p>
<p class="articleParagraph enarticleParagraph" >McNab’s vertically integrated model, with its construction, development and building services arms, as well as its construction supplies and hire business, has made the company into one of Queensland’s largest private builders and it will celebrate a 30th anniversary in March 2026.</p>
<p class="articleParagraph enarticleParagraph" >The son of teachers, Mr McNab arrived in Toowoomba as a 14-year-old and never really left, after attending St Mary’s Christian Brothers College. He then studied civil engineering at the then-Darling Downs Institute of Advanced Education.</p>
<p class="articleParagraph enarticleParagraph" >He said he started out working as an engineer but his career pivoted when he began working on steel fabrication.</p>
<p class="articleParagraph enarticleParagraph" >“That introduced me to the building industry via industrial steel fabrication tilt panels and that sort of work,” he said.</p>
<p class="articleParagraph enarticleParagraph" >“My first building of substance was doing a job for Craig Black’s Black Toyota in Dalby and we finished a job for him just the other day.</p>
<p class="articleParagraph enarticleParagraph" >“I had a bank guarantee on my house when I started which thankfully I didn’t have to fully mortgage to the hilt. I was a good technician. I knew how to build. I was good with people. But I was probably more of an accidental businessperson.” Once focused on the agri-industrial and cold storage end of the industry, the group has moved more into residential projects and from 10-level buildings has graduated to 20 and 30-storey apartment towers.</p>
<p class="articleParagraph enarticleParagraph" >McNab also has well established Gold Coast, Sunshine Coast and Southeast Asia offices, and opened a Sydney office this year. But they probably won’t be moving too far away from South East Queensland.</p>
<p class="articleParagraph enarticleParagraph" >“We’re in no rush to go elsewhere. I talk about my Ts – Toowoomba, Tewantin and Tweed. It’s a nice little triangle of four-and-a-half million people and growing at 30,000 to 50,000 people a year and we also have the Olympics coming,” Mr McNab said.</p>
<p class="articleParagraph enarticleParagraph" >Mr McNab, 61, said the pandemic also convinced him that to was time to relinquish the day-to-day running of the company. In July 1, 2024 he handed the chief executive job to Kunjan Ganatra, who joined in 2020 as chief financial officer after 10 years with Flight Centre. Now the executive chairman, Mr McNab said the growth of the business necessitated the change.</p>
<p class="articleParagraph enarticleParagraph" >“Covid wasn’t fun and the last few years of it was tough. I knew that I needed to make a change. My own operating style wasn’t going to match what the business needed for the next chapter,” he said.</p>
<p class="articleParagraph enarticleParagraph" >Part of the “new chapter” is the presence of AI. “We have to be careful with it ... We’re using it on low- hanging fruit but we have a team working really hard on it to help drive efficiency,” Mt McNab said. “The building industry has never been good at embracing the new.”</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RF</b>&nbsp;</td><td><br/>The Courier-Mail-20251207-CourierMail-63-000496777933 </td></tr>
<tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i501 : Building Construction | iconst : Construction | icre : Real Estate/Construction</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>ccat : Corporate/Industrial News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | ausnz : Australia/Oceania | austr : Australia | queensl : Queensland</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Nationwide News Pty Ltd.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document COUMAI0020251206elc70003r</td></tr></table><br/></div></div><br/><span></span><div id="article-ADVTSR0020251206elc70002h" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/advtsrLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>WNews</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Chatbots’ influence on voters</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>136 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>The Advertiser</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>ADVTSR</td></tr><tr><td align="right" valign="top" class="index"><b>ED</b>&nbsp;</td><td>Advertiser</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>35</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© News Pty Limited. No redistribution is permitted. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Talking to AI chatbots could influence the attitudes and intentions of voters in elections, according to a new study.</p>
<p class="articleParagraph enarticleParagraph" >Researchers carried out experiments involving conversations with an AI model programmed to advocate for one of the candidates in the 2024 US presidential election or the 2025 national elections in Canada and Poland.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >The model was instructed to be positive, respectful, and facts-based, and acknowledge each individual’s views while making its own compelling arguments. The study, published in Nature, found views towards a person’s preferred US presidential candidate were strengthened slightly when speaking to a chatbot with aligned views. It noted a stronger effect in chatbots persuading people who were at first opposed to the candidate being advocated for by the AI model.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RF</b>&nbsp;</td><td><br/>The Advertiser-20251207-Advertiser-35-000496792001 </td></tr>
<tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcsci : Computer Science | gpir : Politics/International Relations | gpol : Domestic Politics | gsci : Sciences/Humanities | gvote : Elections</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | ausnz : Australia/Oceania | austr : Australia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Nationwide News Pty Ltd.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document ADVTSR0020251206elc70002h</td></tr></table><br/></div></div><br/><span></span><div id="article-SUNSTT0020251206elc700012" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/sunsttLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>AI IN THE WORKPLACE: OPTIMISM FOR THE ‘NEANDERTHALS’ AMONG US</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1435 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Sunday Star-Times</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>SUNSTT</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>20</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2025 Fairfax New Zealand Limited. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >So terrorised are tax professionals by the threat of AI, hundreds stayed on at the end of a two-day tax conference in Auckland late last month to hear about the “threats and opportunities” artificial intelligence posed to their lives.</p>
<p class="articleParagraph enarticleParagraph" >The spectre of AI taking jobs is haunting the nightmares of many knowledge workers.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >A Reuters/<span class="companylink">Ipsos</span> poll shows 71% of respondents were concerned AI will put too many people out of work permanently, leading potentially to societal instability and conflict, and a recent global report claimed 40% of 18- to 24-year-olds were suffering AI-induced anxiety.</p>
<p class="articleParagraph enarticleParagraph" >Accountants at the <span class="companylink">Chartered Accountants Australia New Zealand</span> conference at the Cordis hotel were first treated to the threat, embodied by the language choices of PwC’s Kayur Patel, who presented a world split between clever technology-adopters and “Neanderthals”.</p>
<p class="articleParagraph enarticleParagraph" >“I live in a world where the idea that I would walk up to the office, open up my laptop, open up Word, and write some text into Word like a Neanderthal, kind of gives me an allergic reaction,” Patel said.</p>
<p class="articleParagraph enarticleParagraph" >In popular culture, “Neanderthal” is often used to describe people about to be replaced by a more highly evolved form of life, though science has upended that idea. Neanderthals are now seen as having been an advanced and adaptive people who didn’t die out, but interbred with “modern humans” to contribute to the development of the people we are now.</p>
<p class="articleParagraph enarticleParagraph" >Patel even had a timeline for technology Neanderthals.</p>
<p class="articleParagraph enarticleParagraph" >“I’m not saying we’re all in that world now, but you’re all going to be in that world sometime within the next 12 to 18 months. Of that I’m very sure,” he said. “In the next 12 to 18 months, all knowledge work will be done in conjunction with these tools.”</p>
<p class="articleParagraph enarticleParagraph" >Non-Neanderthals need not worry, at least in the short term.</p>
<p class="articleParagraph enarticleParagraph" >“For those of us that get onto this journey now, we’re going have some amazing margin gains,” he said. “But for those of us that don’t, we’ll eventually turn up to a tank war with muskets and we’ll be in a bit of strife.”</p>
<p class="articleParagraph enarticleParagraph" >Seasoned tax adviser Geof Nightingale has seen a few more waves of technology hyping than the much younger Patel, and he’s more optimistic about people’s ability to adapt.</p>
<p class="articleParagraph enarticleParagraph" >“Is this technology wave different? Well, in some ways it’s not, in my view. It’s just another wave of technology for our profession, and since I’ve been in it, we’ve been through a range of technology adoptions, and we’ve proven remarkably adaptive and remarkably resilient as a profession,” he said.</p>
<p class="articleParagraph enarticleParagraph" >When he started work, accountants didn’t have computers.</p>
<p class="articleParagraph enarticleParagraph" >“We had adding machines and eight-column ledgers. Then we got a computer on our desks, and that was pretty exciting,” he said.</p>
<p class="articleParagraph enarticleParagraph" >“Then sometime around the mid-90s we got access to internet and email. None of that changed us too much really. It speeds things up. It made us more insightful. We could bring better insights to our clients. It made us easier to deal with.”</p>
<p class="articleParagraph enarticleParagraph" >It did mean that some of the “secret knowledge” accountants and tax advisers had was suddenly available for free on the internet. “So as advisers we had to move up the value chain to stay relevant,” he said, and they did it. They adapted.</p>
<p class="articleParagraph enarticleParagraph" >“We brought in each of those waves of change. There was always speculation, if you go back to the old media, that accounting was a sunset industry, and would be fully automated. There will be no jobs. Don’t send your kids off to do commerce. But that didn’t happen. There are more accounting jobs now than ever.”</p>
<p class="articleParagraph enarticleParagraph" >Then in late 2022, generative AI was released into the market, Nightingale said.</p>
<p class="articleParagraph enarticleParagraph" >“Again, warnings of massive disruption in professional services, unemployment for graduates, etc, etc. And here we are in late 2025, three years on, and in my view, the profession has yet to profoundly change,” he said. “Based on that experience, I’m a great optimist, and I believe that just like previous waves of technology, we will adapt as a profession to generative AI.”</p>
<p class="articleParagraph enarticleParagraph" >Nightingale’s advice to accountants was to experiment with AI tools, and: “Don’t be afraid of them.”</p>
<p class="articleParagraph enarticleParagraph" >“I’ve been using AI extensively in my own little tax advisory business. It’s an incredible tool,” he said.</p>
<p class="articleParagraph enarticleParagraph" >It was more powerful than those previous technology waves.</p>
<p class="articleParagraph enarticleParagraph" >“As professionals, it’s reaching into our experiences, reaching into our judgment, and our wisdom, and those are the things that we ultimately sell. We are going to need to adapt to that and adapt to that quickly.”</p>
<p class="articleParagraph enarticleParagraph" >But, he said: “In the foreseeable future, they [AI tools] are not going to replace human to human contact, human to human delivery of services, human judgment, and relationships, and ultimately the thing that we all trade in is trust.”</p>
<p class="articleParagraph enarticleParagraph" >Accountants should adopt the tools, and learn how to use them, while taking great care with their governance.</p>
<p class="articleParagraph enarticleParagraph" >Nightingale cited an embarrassing failure by consulting firm <span class="companylink">Deloitte</span> in Australia which provided a report to government which contained AI-generated errors.</p>
<p class="articleParagraph enarticleParagraph" >“You need some governance over AI, but you actually need to allow, permit, and encourage the adoption of retail AI tools within your business to discover what the value and where the use cases are.”</p>
<p class="articleParagraph enarticleParagraph" >Nightingale said AI should feature in all planning and strategy sessions.</p>
<p class="articleParagraph enarticleParagraph" >He recommended businesses use an off-the-shelf subscription AI, and not try to build their own systems as AI was moving so fast.</p>
<p class="articleParagraph enarticleParagraph" >“It should be cancellable because you don’t know what’s going to happen in six months or 12 months, and almost certainly, if you invest deeply in bespoke tools, you’re going to end up with stranded capital.”</p>
<p class="articleParagraph enarticleParagraph" >Data protection was a must for accountants, and Patel advised only to use AI that guaranteed data would remain secret, and not be used by the AI’s controllers to train their AI.</p>
<p class="articleParagraph enarticleParagraph" >Patel advised accountants to choose a “business” AI, something like ChatGPT for business, which guaranteed that.</p>
<p class="articleParagraph enarticleParagraph" >“They sign in blood, ‘We will not take your data and use it’. Even if you pay for the consumer version of those tools, that is not the case,” Patel said.</p>
<p class="articleParagraph enarticleParagraph" >Patrick O’Doherty, chief data officer at Inland Revenue - Te Tari Taake, viewed AI as “just the next wave of digital innovation”.</p>
<p class="articleParagraph enarticleParagraph" >“At the fundamental core of our digital ambition is around how do we get our people ready for what’s coming next,” he said.</p>
<p class="articleParagraph enarticleParagraph" >Workers needed to have “digital dexterity”.</p>
<p class="articleParagraph enarticleParagraph" >Inland Revenue was rolling out <span class="companylink">Microsoft</span> 365 Copilot to all staff by the middle of next year, and training them to use it.</p>
<p class="articleParagraph enarticleParagraph" >It was also trialling Snowflake Cortex AI to automate processes. So far, that had been focused primarily on audits, but its use would expand.</p>
<p class="articleParagraph enarticleParagraph" >There was a glimmer of hope for accountants and tax advisers, who might get more work, if the taxman was able to do more audits by using AI.</p>
<p class="articleParagraph enarticleParagraph" >One accountant in the audience was worried AI might reduce the critical thinking capacities of juniors, if they didn’t have to struggle with learning the trade of accounting and tax.</p>
<p class="articleParagraph enarticleParagraph" >There are increasing concerns that AI may make users lazier and less critical, partly based on a relatively limited study from MIT in the United States.</p>
<p class="articleParagraph enarticleParagraph" >Patel’s answer was to use AI to train juniors to recreate the tough early learning years he and, many years before, Nightingale had experienced.</p>
<p class="articleParagraph enarticleParagraph" >The question has also been posed: if AI does the lower-level grunt work, why do older practitioners need to hire young people at all?</p>
<p class="articleParagraph enarticleParagraph" >Nightingale wasn’t convinced.</p>
<p class="articleParagraph enarticleParagraph" >“You can read all the doomsday articles about no graduate roles and no junior roles and everything, but you know, in the last 18 months that there’s mixed views emerging on that. The <span class="companylink">Wharton School of Business</span> did a survey just recently which said 49% of chief people officers were looking at hiring more graduates to deal with AI, not less,” he said.</p>
<p class="articleParagraph enarticleParagraph" >One firm prediction he had, however, was that the “old hourly rate model” would finally die. “Firms are going to have to think really hard about how they change their economic model. How much time a human spends on your work is not going to be relevant.”</p>
<p class="articleParagraph enarticleParagraph" >What do you think? Email sundayletters@stuff.co.nz. Don’t forget to include your address.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>nzioca : Chartered Accountants Australia and New Zealand</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302022 : Artificial Intelligence Technologies | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcsci : Computer Science | gjob : Labor Issues | gsci : Sciences/Humanities</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Fairfax New Zealand Limited</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document SUNSTT0020251206elc700012</td></tr></table><br/></div></div><br/><span></span><div id="article-SUNSTT0020251206elc70000m" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/sunsttLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>FINTECHS PLOT TO CHANGE NZ MORTGAGE LENDING FOREVER</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>868 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Sunday Star-Times</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>SUNSTT</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>19</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2025 Fairfax New Zealand Limited. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Founders of start-up companies preparing to vie for a share of the country’s $75 billion or so mortgage business say open banking means their task is about to get a lot easier – and big banks, which claim a 93% home loan market share right now, will have to compete much harder for borrowers.</p>
<p class="articleParagraph enarticleParagraph" >Anteup, Zoro Mortgages and Homely are all readying to launch in the new year, hoping to break through the vested interests that have made the task of shopping around for a home loan gruelling.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Matthew Williams, co-founder of Anteup, said he and co-founder Adam Joyce, chief executive of Marlborough Wine, decided to launch their business over a beer.</p>
<p class="articleParagraph enarticleParagraph" >“We were talking about how difficult it was to shop around for a home loan,” Williams said.</p>
<p class="articleParagraph enarticleParagraph" >Someone in search of the best deal, and wanting to get a quote for a loan from multiple banks and other lenders, had to approach every business separately, or give up and ask a mortgage adviser/broker to seek a loan on their behalf.</p>
<p class="articleParagraph enarticleParagraph" >Williams’ and Joyce’s idea was to launch an online service where people could make one application, which would be submitted to every lender registered with Anteup.</p>
<p class="articleParagraph enarticleParagraph" >The lenders would then submit their best loan offers in a process similar to that experienced by buyers tendering bids on a property.</p>
<p class="articleParagraph enarticleParagraph" >Anteup would launch with the tag line: “You’re not applying for a loan – they’re applying for you.”</p>
<p class="articleParagraph enarticleParagraph" >Open banking changes the game</p>
<p class="articleParagraph enarticleParagraph" >Open banking, which became fully regulated this month, should make applying for loans simpler, and quicker.</p>
<p class="articleParagraph enarticleParagraph" >Open banking allows ordinary people to give permission to trusted organisations like Anteup and Homely to retrieve their information, like bank account statements, directly from banks and use it for limited, specific purposes.</p>
<p class="articleParagraph enarticleParagraph" >In Anteup and Homely’s case, this would be to automatically build a bid to take to lenders detailing the finances and money habits of people seeking loans.</p>
<p class="articleParagraph enarticleParagraph" >Roy Chowdhury, founder of Homely, said research indicated that preparing an application for a bank took around 24 hours of labour, but that would change as a result of the efficient systems Homely had built using AI and open banking.</p>
<p class="articleParagraph enarticleParagraph" >Chowdhury said the current system of a new application for each lender, with the alternative of sitting down with a broker, was “outdated” for younger people, who expected easy and seamless digital applications.</p>
<p class="articleParagraph enarticleParagraph" >And he said early signs indicated the market was ready for change.</p>
<p class="articleParagraph enarticleParagraph" >Like Anteup, Homely is in late-stage beta-testing mode, with word-of-mouth having secured it customers to test its systems and be amongst its first to apply for loans.</p>
<p class="articleParagraph enarticleParagraph" >“That demonstrates there is a market out there,” Chowdhury said.</p>
<p class="articleParagraph enarticleParagraph" >Borrowers have been promised this before</p>
<p class="articleParagraph enarticleParagraph" >The idea of digital loan marketplaces where lenders compete for loans is not new.</p>
<p class="articleParagraph enarticleParagraph" >Back in 2008, a company called Fundit was launched to do just that. Fundit said it would allow people wanting a home loan to make one application, and banks and other lenders would make their best offers to the borrower. But the business didn’t get any cut-through.</p>
<p class="articleParagraph enarticleParagraph" >Josh Daniell from Dosh, which is a fintech offering home loans online, recalled Fundit. He said a family member was an equity investor and had told him it failed because banks were not keen to play ball, in fear of angering mortgage brokers. Daniell said his family member told him banks were afraid of upsetting the growing mortgage broker market.</p>
<p class="articleParagraph enarticleParagraph" >But around the world, fintechs have been winning market share, and the challengers believe open banking, AI, and reduced public patience for the big banks’ stranglehold on markets have made change inevitable.</p>
<p class="articleParagraph enarticleParagraph" >Customers have also had 20 years of training to do much of their commerce on their phones.</p>
<p class="articleParagraph enarticleParagraph" >Will the banks play ball?</p>
<p class="articleParagraph enarticleParagraph" >Neither Chowdhury or Matthews would say which lenders had signed up to bid on applications made through their site.</p>
<p class="articleParagraph enarticleParagraph" >However, Chowdhury said Homely already had several big banks, several small banks, and several non-bank lenders that specialised in lending to people who did not fit bank lending criteria, including people with spotty credit records or unusual income and employment situations.</p>
<p class="articleParagraph enarticleParagraph" >Matthews said about half of mortgage lenders had agreed to participate, and the other half had “not said no”.</p>
<p class="articleParagraph enarticleParagraph" >Zoro Mortgages, operated by Zoro AI Limited, is also readying for launch with the similar idea: “Lenders bid for you: You don’t chase banks.”</p>
<p class="articleParagraph enarticleParagraph" >Co-founder Sam Jeffs said busy young professionals currently had to take several days off work to in effect shop around.</p>
<p class="articleParagraph enarticleParagraph" >Zoro Mortgages’ plan was to focus on simple refinancing, a term used for people upping sticks and taking their loan from one lender to another.</p>
<p class="articleParagraph enarticleParagraph" >It’s been a big driver of activity in recent months as banks have engaged in a “cash-back” war to gain market share.</p>
<p class="articleParagraph enarticleParagraph" >ANZ, for example, is offering up to 1.5% cash-back up to $30,000 to new borrowers shifting from a rival lender, provided they borrow enough and they are not seeking a loan for more than 80% of the value of the property against which it is secured.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i814 : Banking | i81402 : Commercial Banking | i8150103 : Mortgage Banks/Real Estate Credit | ibnk : Banking/Credit | ifinal : Financial Services</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c22 : New Products/Services | ccat : Corporate/Industrial News | cexpro : Products/Services | ncat : Content Types | nfact : Factiva Filters | nfcpin : C&E Industry News Filter</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | ausnz : Australia/Oceania | nz : New Zealand</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Fairfax New Zealand Limited</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document SUNSTT0020251206elc70000m</td></tr></table><br/></div></div><br/><span></span><div id="article-SUNSTT0020251206elc70000h" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/sunsttLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>A CHEFS’ GUIDE TO CHRISTMAS GIFTS FOR THE FOODIE IN YOUR LIFE</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1477 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Sunday Star-Times</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>SUNSTT</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>35</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2025 Fairfax New Zealand Limited. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >If you really love your foodie, I would get them a proper top-quality chef’s knife. Cooking is a real joy if you have a tool that slices and chops through your prep list, fast and neatly, and in the big moments - like carving up the baked ham - the family and friends will be in awe!</p>
<p class="articleParagraph enarticleParagraph" >If they already have a go-to all-around knife, then get them a specialty knife such as a smaller utility one for boning chickens or filleting fish. Even a decent bread knife will be appreciated.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >I recommend a hand-forged Japanese Utility Knife, Seisuke brand, 135mm. $265.</p>
<p class="articleParagraph enarticleParagraph" >Mark Wallbank, co-owner, The Blue Breeze Inn, Ponsonby</p>
<p class="articleParagraph enarticleParagraph" >This year, I’m so busy in the restaurant that everyone will be getting my Quincessential Quince Paste. I first tasted quince paste in 2000 and was instantly hooked. When I opened Rocco, in Ponsonby, I put it on the menu paired with Manchego cheese - but the imported versions were full of preservatives, and I knew we could do better. So I planted a quince orchard on my Pukekohe property and began perfecting our own. Today, Quincessential is grown, harvested and handmade by us using only natural fruit pectin, with no artificial setting agents. It melts in your mouth, pairs beautifully with cheese or duck liver parfait, is wonderful even on toast for breakfast, lunch or dinner with a slice of Manchego cheese!</p>
<p class="articleParagraph enarticleParagraph" >Find it at Farro, Maison Vauron, Moore Wilson’s and The Blue Breeze Inn.</p>
<p class="articleParagraph enarticleParagraph" >Zennon Wijlens, head chef/co-owner Paris Butter, Herne Bay</p>
<p class="articleParagraph enarticleParagraph" >I discovered The Southerly a few months ago and immediately became obsessed with their whisky-infused Mānuka Honey. It’s a high-quality Mānuka Honey with a gentle infusion of 8-year-old single malt from New Zealand. The flavour is rich and silky with a warm whisky note that lingers across the palate. It’s ultra-premium and something you’d open slowly to savour.</p>
<p class="articleParagraph enarticleParagraph" >For an incredible gift, the new Cardrona Distillery Rose Rabbit Barrel Aged Cherry Liqueur is unbeatable. Central Otago cherries, aged in ex-bourbon casks, produce a complete, rounded cherry profile with beautiful depth. It’s elegant, festive and the perfect treat for Christmas.</p>
<p class="articleParagraph enarticleParagraph" >Sid Sahrawat, owner and executive chef, Cassia and The French Café, central Auckland</p>
<p class="articleParagraph enarticleParagraph" >Long Kiwi summer days call for gifts that make entertaining effortless. My stand-out this year is the Ninja slushy maker - frozen margaritas on repeat with zero fuss, and the kids love their own juice-only versions. It’s become a lifesaver when friends gather in the sun. Another favourite is my Huski cooler, which I actually received last Christmas from my in-laws. I’ve used it constantly, perfect for keeping bubbles or a crisp white chilled on the deck without dragging out an ice bucket, and brilliant for beach picnics too. Now if only we could take the slushy maker to the beach!</p>
<p class="articleParagraph enarticleParagraph" >Al Brown, Depot, Fed Deli, Auckland CBD</p>
<p class="articleParagraph enarticleParagraph" >I always think the ultimate foodie gift is a knife. There are many out there to choose from, however if you really want a very special hand-made knife, check out Champion Knives.</p>
<p class="articleParagraph enarticleParagraph" >I have a personal connection to this amazing brand. Hayden Scott was my sous chef for nearly 15 years, we travelled the world together cooking, filming, putting on events. Hayden moved on a couple of years ago, to realise his ultimate dream of making unique hand-made knives.</p>
<p class="articleParagraph enarticleParagraph" >He is an extraordinary craftsman and his work is not only beautiful to the eye, but because of his background the knives are perfectly balanced and, of course, incredibly functional.</p>
<p class="articleParagraph enarticleParagraph" >You, or your special person who loves to cook, will treasure a “Champion” knife forever, and it will be passed on for generations to come. Hayden also teaches knife- making, and puts on courses from time to time... another possible and unique gift idea.</p>
<p class="articleParagraph enarticleParagraph" >Rebecca Smidt, co-owner Cazador and San Ray, central Auckland</p>
<p class="articleParagraph enarticleParagraph" >If I were gifting to a food lover I’d wrap up a giant jar of La Explanada Anchovy Stuffed Olives. I’m crazy about them, they’re ideal on a grazing board, for garnishing a martini, or snacking alongside a fino sherry - so all the summer bases are covered.</p>
<p class="articleParagraph enarticleParagraph" >They’re available in large format at Sabato in Mt Eden, and I’d suggest the large size because once you pop...</p>
<p class="articleParagraph enarticleParagraph" >Nic Watt, executive chef, Canting, Masu and Inca, Auckland</p>
<p class="articleParagraph enarticleParagraph" >I always look for gifts that can be used within the same season, as we all want a gift that we can open and use straight away.</p>
<p class="articleParagraph enarticleParagraph" >With the rise of smash burgers and recently learning how a breakfast burger can be someone’s start of the day, runny eggs and all, this year my go-to gift is a Burger Meat Press.</p>
<p class="articleParagraph enarticleParagraph" >A weighted round press that is used to flatten burger patties while cooking and can double as a fish weight to ensure even pressure and super crispy skin. I would opt for the latter, while the breakfast burger types can have that perfect smash pattie effect.</p>
<p class="articleParagraph enarticleParagraph" >The Mako Press is a solid choice.</p>
<p class="articleParagraph enarticleParagraph" >Krishna Botica, co-owner Café Hanoi, Ghost Street, Perch, Auckland</p>
<p class="articleParagraph enarticleParagraph" >Go for a combo that’ll change their week nights: an Instant Pot plus a quick one-on-one Christmas card voucher for a session on “how to boss your AI”. I never thought I would use The Instant Pot as much as I have... it’s a true all-in-one workhorse, turning out healthy one-pot dinners in a fraction of the usual time, perfect for time-poor food lovers who still care about flavour and nutrition. Pair it with an oven and stove top, a short AI lesson: snap a few photos of the fridge and pantry, ask for five family dinners, batch-cook on Sunday, and you’ve got roughly a week of tasty, varied meals in around four hours – Christmas panic officially cancelled.</p>
<p class="articleParagraph enarticleParagraph" >John Lawrence, chef, Boulcott Street Bistro, Wellington</p>
<p class="articleParagraph enarticleParagraph" >‘Tis the season – at least for me – to be opening a lot of oysters, so my top pick has to be the Toadfish oyster knife from <span class="companylink">Victorinox</span>.</p>
<p class="articleParagraph enarticleParagraph" >After more than 30 years in commercial kitchens and having tried dozens of knives, this one stands out as the most well-designed and functional for Pacific-style oysters. The ergonomic handle and robust stainless-steel blade, with its slightly curved, rounded tip, make shucking far less of a task.</p>
<p class="articleParagraph enarticleParagraph" >And if you’re after a great budget option, their paring knife is also excellent – reliable, versatile and cheap as chips. A perfect stocking stuffer for any seafood lover.</p>
<p class="articleParagraph enarticleParagraph" >Josh Emett, Michelin star chef</p>
<p class="articleParagraph enarticleParagraph" >I’ve loved Tony Sly’s handmade ceramics for years. Each piece is hand-shaped in Raglan, with soft, coastal-inspired glazes that give them a really beautiful, understated look. I love giving them as gifts because they feel timeless, functional, and completely Kiwi. Supporting local artisans is really important to me, especially at Christmas, and Tony Sly’s work ticks all the boxes.</p>
<p class="articleParagraph enarticleParagraph" >The range is amazing, from serving platters to bowls and mugs, and they’ve just released a new eggshell blue glaze that I’m absolutely loving.</p>
<p class="articleParagraph enarticleParagraph" >They’re the kind of gifts people keep for years.</p>
<p class="articleParagraph enarticleParagraph" >Sophie Egan, restaurant manager, Gilt Brasserie, central Auckland</p>
<p class="articleParagraph enarticleParagraph" >I’m always on the hunt for thoughtful gifts, and I can’t go past By The Bottle. They’re a small independent wine store in Auckland that also ships nationwide. The team are always amazing at helping me find the perfect bottles when I have no idea what I’m looking for. Their gift packs and wine selection are so thoughtfully put together.</p>
<p class="articleParagraph enarticleParagraph" >I especially love the Kiwi Negroni pack, a Negroni made entirely with New Zealand ingredients, which is perfect for any thirsty friend, family member, or colleague. Their vermouth selection is fantastic too, especially the full Saison range, which I love on a hot afternoon in the sun. With some of my favourite New Zealand and international wine producers and an epic range of spirits, you’re sure to find something delicious and gift-worthy.</p>
<p class="articleParagraph enarticleParagraph" >
                     Marisa Bidois, CEO, Restaurant Association</p>
<p class="articleParagraph enarticleParagraph" >As someone who eats out a lot – occupational hazard of being the Restaurant Association CEO – my go-to foodie gift is a Restaurant Association Gift Voucher. It takes all the guesswork out because the recipient gets to choose exactly where they want to eat from more than 1500 restaurants, cafés and bars around the country.</p>
<p class="articleParagraph enarticleParagraph" >Whether they’re into neighbourhood gems, special-occasion spots or just want to try somewhere new, the choice is entirely theirs. I love that it gives people an experience rather than another thing to unwrap, and it supports our incredible hospitality community at the same time.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gfod : Food/Drink | glife : Living/Lifestyle</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | auckl : Auckland | ausnz : Australia/Oceania | nz : New Zealand</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Fairfax New Zealand Limited</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document SUNSTT0020251206elc70000h</td></tr></table><br/></div></div><br/><span></span><div id="article-BUSWNZ0020251206elc700002" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/buswnzLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>the-life</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>IM6 performance review: luxury, fast</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1833 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>BusinessDesk</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>BUSWNZ</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright © NZME Publishing Ltd </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Pros</p>
<p class="articleParagraph enarticleParagraph" >* Astonishingly smooth and refined</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >* 4WS is a brilliant feature</p>
<p class="articleParagraph enarticleParagraph" >* A tech-lover's delight in so many ways</p>
<p class="articleParagraph enarticleParagraph" >Cons</p>
<p class="articleParagraph enarticleParagraph" >* Some driver assists/automation need work</p>
<p class="articleParagraph enarticleParagraph" >* Performance out of sync with IM6 character</p>
<p class="articleParagraph enarticleParagraph" >* Far from the most stylish SUV around</p>
<p class="articleParagraph enarticleParagraph" >IM ("Intelligence in Motion") is MG’s luxury brand, as Lexus is to <span class="companylink">Toyota</span>. Hence, this new car is officially the “IM6 Presented by MG Motor”. No octagonal brand-badges; instead, you get the IM-signature dots and lines.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">Click to view image [https://media.businessdesk.co.nz/file/c_fill,w_700,q_100/cs4yeR4xEL5xtYh368GzsOPZyfoxZ5zU8anKX1bv.jpg]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >IM6 PERFORMANCE: POWERTRAIN 100kWh battery with dual electric motors, single-speed transmission, AWD OUTPUT 572kW/802Nm (200kW/302Nm front, 372kW/500Nm rear) EFFICIENCY Range 505km (WLTP) SIZE 4904mm long, 2470kg PRICE $89,900.</p>
<p class="articleParagraph enarticleParagraph" >The IM6 does indeed offer an impressively luxurious experience. It’s pure-electric, but even the relative silence of that is enhanced by active noise cancellation, quelling road noise to the extent that sometimes all you can hear is a whoosh of wind around the A-pillar.</p>
<p class="articleParagraph enarticleParagraph" >Don’t be distracted by this flagship Performance model’s outrageous turn of speed (0-100km/h in 3.4 seconds), Advanced Air Suspension with Continuous Control Damping (CCD) and sophisticated 4-wheel steering (4WS) system. Or indeed the name. It's not really a sports/performance SUV.</p>
<p class="articleParagraph enarticleParagraph" >All that tech is still in pursuit of luxury, refinement, and ease of use. The steering is devoid of feel, and the suspension is incredibly soft in Comfort mode, though still pretty well controlled.</p>
<p class="articleParagraph enarticleParagraph" >For faster driving, it’s pretty precise, and Sport mode stiffens things up, but there's still a lot of body movement. There’s impressive traction from the AWD system and Pirelli Scorpion tyres, and the IM6 really can hustle if you want it to. But it’s more enjoyable when you don’t.</p>
<p class="articleParagraph enarticleParagraph" >The 4WS is fantastic. At low speed (60km/h and below), the rear wheels turn up to 6 degrees in the opposite direction to the front wheels, giving the 4.9-long IM6 the same turning circle as a supermini. At motorway speeds, they turn up to 12 degrees the same way, making lane changes so, so smooth. The same function can also be forced at low speed to allow the car to “crab” away from a close kerb.</p>
<p class="articleParagraph enarticleParagraph" >If you equate minimalism with luxury, the IM6 cabin will please you. Lots of rounded surfaces and no buttons on the dashboard, although there are two scroll controls on the steering wheel, <span class="companylink">Tesla</span>-style.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">Click to view image [https://media.businessdesk.co.nz/file/c_fill,w_700,q_100/ZXjZLYtqvtN6oie9RMgfsrCEgHedcKSZrPgo9Odw.jpg]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Dover Beige interior is pretty in-your-face. But there is a darker option.</p>
<p class="articleParagraph enarticleParagraph" >The “synthetic leather” is slightly questionable; it has a sticky feeling and is not ideal in hot summer weather, although the IM6 is not alone in that choice. But the very light Dover Beige colour does give the interior an other-worldly quality. Don’t worry, if that’s not you, there’s also Highland Grey.</p>
<p class="articleParagraph enarticleParagraph" >The dual-screen layout takes a little getting used to, but it’s ergonomically quite clever. There’s 26.3 inches of “immersive” screen along the dashboard, and a separate 10.5-inch display embedded in the centre console. The left-hand section of the main display and the smaller screen are interchangeable in some ways, but it helps to think of the top section as mainly for display and the individual 10.5-inch screen as a work surface. For example, when you have <span class="companylink">Apple</span> CarPlay or Android Auto running, that can take over the top screen while the bottom one remains free for other functions, including shortcuts to the likes of the One Touch iAD parking functions (more about those in a minute).</p>
<p class="articleParagraph enarticleParagraph" >There are some nifty shortcuts over the shortcuts, too. A two-finger swipe up or down on the central screen activates climate temperature, or left and right for fan speed.</p>
<p class="articleParagraph enarticleParagraph" >The IM6 is really all about driver-assistance tech. It’s absolutely loaded with advanced AI-enhanced features.</p>
<p class="articleParagraph enarticleParagraph" >Speaking or surprise-and-delight stuff, you might notice little embroidered spots around the cabin (dashboard, seatbacks) labelled “IM Mag”. They’re built-in magnetic mounts for a phone or tablet.</p>
<p class="articleParagraph enarticleParagraph" >The IM6 is really all about the driver-assistance tech. It’s absolutely loaded with advanced AI-enhanced features, although not all of them seem completely sorted for Kiwi conditions.</p>
<p class="articleParagraph enarticleParagraph" >Kudos to the camera system … in concept.</p>
<p class="articleParagraph enarticleParagraph" >There are 9 HD cameras and 12 sensors around the car, enabling some pretty cool features.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">Click to view image [https://media.businessdesk.co.nz/file/c_fill,w_700,q_100/jbYPqzE97II1QneHPwodouTmOK9PdE10ZLve6AKJ.jpg]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Cameras and sensors akimbo, all around the car.</p>
<p class="articleParagraph enarticleParagraph" >The physical rearview mirror in the cabin can be folded away completely; that’s because you can summon a virtual one with one click of a steering wheel button, bringing up a live feed on the dashboard for a few seconds whenever you need it.</p>
<p class="articleParagraph enarticleParagraph" >And to be honest, you may as well fold the real thing away, because you can’t see a thing out of the tiny rear window.</p>
<p class="articleParagraph enarticleParagraph" >For turning or lane changes, you also get a live feed on the appropriate side of the instrument panel. But it gets more clever than that, because the cameras are used to create an invisible A-pillar, giving you a clear view as you’re turning.</p>
<p class="articleParagraph enarticleParagraph" >So far, so good. As well as giving a complete 360-degree view for parking, the cameras also help run the automated parking features, of which there are many. Ask the car to find a space (there’s a one-touch shortcut on the lower screen) and the cameras scan all possible options; then you choose, and once you hit go, the car can do the rest.</p>
<p class="articleParagraph enarticleParagraph" >It’s simple to activate, which is half the battle with this kind of stuff. And in a lightly trafficked space, it works brilliantly, with impressive speed.</p>
<p class="articleParagraph enarticleParagraph" >Introduce other cars, even slow-moving ones, and you’re in trouble; we tried several times to use both the parking and “pull out” features in busy parking lots, and we had to give up, lest we sparked a series of road-rage incidents.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">Click to view image [https://media.businessdesk.co.nz/file/c_fill,w_700,q_100/BFoNhNEvdGePF9vlsFbM6acqvvQC0Uz7lhLu0KAl.jpg]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >There's a Nap-mode for the front seats; nice, not totally flat though.</p>
<p class="articleParagraph enarticleParagraph" >The IM6 can theoretically retrace the last 100m it drove in reverse – a similar feature to that offered on every current <span class="companylink">BMW</span>, although the German cars can only do 50m. However, speaking as somebody who has the ideal test environment –a narrow 60m-long hedge-lined driveway, I have to report the IM6 simply couldn’t do it, repeatedly giving up and telling me “manual reversing is recommended”.</p>
<p class="articleParagraph enarticleParagraph" >Every <span class="companylink">BMW</span> I’d had for the past few years has been able to automatically back up the same driveway for the allocated 50m at quite an alarming speed. Just saying.</p>
<p class="articleParagraph enarticleParagraph" >And at the risk of this sounding like a list of IM6 grievances, the automated lane-change function was a bit tricky, too. In full assist mode (that’s one click on the transmission stalk for adaptive cruise, then another for steering assistance), the IM6 can theoretically change lanes itself when you activate the indicator.</p>
<p class="articleParagraph enarticleParagraph" >The window is fairly small for Kiwi conditions (over 75km/h) and it's a lot more picky about ideal conditions than other similar systems we've used in recent years. Just be prepared to see "impossible" and "take control" pop up on the dashboard quite a bit.</p>
<p class="articleParagraph enarticleParagraph" >You get the picture: the IM6 is equipped with a staggering array of automated features. Some of them are brilliant, some of them are not quite there. The good thing is that over-the-air (OTA) updates and the speed at which Chinese brands seem able to respond to country-specific feedback of this nature mean that even if you buy an IM6 right now, it’ll get better with new software down the track.</p>
<p class="articleParagraph enarticleParagraph" >But should you buy one? There’s a lot to like about the IM6: refinement, space and all that tech, even if some of it hasn’t reached its full potential. Despite the frustrations, it’s a likeable and often-impressive luxury EV.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">Click to view image [https://media.businessdesk.co.nz/file/c_fill,w_700,q_100/VGlRnO1Y4BFQyBVntd0ouUMbTGQVbTIbAdQ6dtLT.jpg]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Deeply impressive loadspace. And deep, with more storage underneath.</p>
<p class="articleParagraph enarticleParagraph" >But we reckon the $12k-cheaper Platinum looks like the smarter buy, partly because it has a more appropriate model name and partly because 0-100km/h in 5.4sec seems perfectly quick for this car.</p>
<p class="articleParagraph enarticleParagraph" >The Platinum has the same 100kWh battery but a bigger range (555km versus 505km), the same 21-inch wheels and fancy rubber, and while it’s RWD, it has the same 4WS system as this Performance.</p>
<p class="articleParagraph enarticleParagraph" >You do miss out on the Advanced Air Suspension; if that’s a must-have, you can add it for $5,500, although you’re sneaking up towards Performance price again then.</p>
<p class="articleParagraph enarticleParagraph" >How much is the IM6 Performance?</p>
<p class="articleParagraph enarticleParagraph" >The Performance is the flagship model and is priced at $89,990. The less powerful Platinum and Premium versions are $77,900 and $66,900, respectively.</p>
<p class="articleParagraph enarticleParagraph" >What are the key statistics for the IM6 Performance?</p>
<p class="articleParagraph enarticleParagraph" >The 100kWh battery feeds dual electric motors. IM does not quote combined outputs, preferring to separate front and rear at 200kW/372Nm and 302kW/500Nm. It's fast: 0-100km/h in just 3.4 seconds.</p>
<p class="articleParagraph enarticleParagraph" >Is the IM6 Performance efficient?</p>
<p class="articleParagraph enarticleParagraph" >That extreme performance takes its toll on power consumption. The Performance has a WLTP range of 505km, compared to the Platinum with the same battery at 555km.</p>
<p class="articleParagraph enarticleParagraph" >Is the IM6 Performance good to drive?</p>
<p class="articleParagraph enarticleParagraph" >Despite the outrageous acceleration, the IM6 Performance is primarily a luxury car. It's extremely quiet with active noise cancellation, and the AWD/4WS technology makes it a super-smooth machine on the road.</p>
<p class="articleParagraph enarticleParagraph" >Is the IM6 Performance practical?</p>
<p class="articleParagraph enarticleParagraph" >Extremely so. The long wheelbase and flat floor provide generous passenger accommodation, and the boot is big for the class at 646 litres, even if it's slightly smaller than the RWD models. You also get a 32l frunk up front and another small storage space underneath the boot floor.</p>
<p class="articleParagraph enarticleParagraph" >What do we like about the IM6 Performance?</p>
<p class="articleParagraph enarticleParagraph" >It's a true luxury car, with extreme refinement and lots of technology that makes for a super-smooth drive (including 4WS and a "comfort stop" braking function).</p>
<p class="articleParagraph enarticleParagraph" >If you're wowed by the acceleration, the price is pretty good: we can't think of a car that can go faster for less. And it's a truly spacious and practical family SUV.</p>
<p class="articleParagraph enarticleParagraph" >What don’t we like about the IM6 Performance?</p>
<p class="articleParagraph enarticleParagraph" >The styling is an acquired taste (and apologies to both <span class="companylink">Tesla</span> and <span class="companylink">Aston Martin</span>).</p>
<p class="articleParagraph enarticleParagraph" >Some of the automated-drive technology needs work: we really struggled with the likes of the 100m reversing assistant and automatic lane change.</p>
<p class="articleParagraph enarticleParagraph" >What kind of person would the IM6 Performance suit?</p>
<p class="articleParagraph enarticleParagraph" >An EV enthusiast who wants a real luxury experience and is fascinated by the high level of technology in the IM6.</p>
<p class="articleParagraph enarticleParagraph" >This story was originally published on <span class="colorLinks">Driven Car Guide [https://www.drivencarguide.co.nz/]</span>.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i353 : Motor Vehicle Parts | iaut : Automotive</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>ccat : Corporate/Industrial News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | ausnz : Australia/Oceania | nz : New Zealand</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>the-life</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>NZME Publishing Ltd.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document BUSWNZ0020251206elc700002</td></tr></table><br/></div></div><br/><span></span><div id="article-NZHLD00020251206elc700012" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/nzhldLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>General News</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Why a Swedish central banker matters more than Ikea’s meatballs</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1151 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>The New Zealand Herald</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>NZHLD</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>A025</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025 NZME Publishing Ltd. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Two Swedes were in the news this week, with entrances at both ends of the hype scale</p>
<p class="articleParagraph enarticleParagraph" >Valkommen till vart konstiga lilla land ...</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >It’s been a huge week for Swedish arrivals in New Zealand — possibly the biggest in our history.</p>
<p class="articleParagraph enarticleParagraph" >Abba never made it to New Zealand and we can hardly count the visit of 1980s Swedish pop icons Roxette in 2015, as they were some 30 years past their prime.</p>
<p class="articleParagraph enarticleParagraph" >A Swedish botanist, Daniel Solander, accompanied Captain Cook on his first voyage to New Zealand in 1768-1771 — there’s an island in the Foveaux Strait named after him.</p>
<p class="articleParagraph enarticleParagraph" >So maybe we have to go back more than 300 years to find a week of such cultural connection between our two nations.</p>
<p class="articleParagraph enarticleParagraph" >We’ve never had a Swedish royal visit or even a prime ministerial one, as far as I can see.</p>
<p class="articleParagraph enarticleParagraph" >You have to head a long way down the rabbit hole to find conspiracy theorists wacky enough to make the case that the Vikings made it here.</p>
<p class="articleParagraph enarticleParagraph" >Thanks to Scandinavia’s colonial success in Britain and France’s Normandy, we probably share a lot of genetic history, though.</p>
<p class="articleParagraph enarticleParagraph" >Anyway, in a weird coincidence, furniture and homeware <span class="companylink">Ikea</span> opened its doors and shared its meatball-flavoured vision for cheap designer homeware the same week that we valkommened our new Swedish Reserve Bank Governor.</p>
<p class="articleParagraph enarticleParagraph" >The Swedish phrase at the top of this column, according to my best AI-assisted research efforts, says: “Welcome to our strange little country.”</p>
<p class="articleParagraph enarticleParagraph" >I know enough about the Swedish (I’ve visited once and have Swedish friends) to recognise that they might themselves share some affinity with being a slightly weird small country.</p>
<p class="articleParagraph enarticleParagraph" >We are both actually quite big countries, but with small populations.</p>
<p class="articleParagraph enarticleParagraph" >If you put New Zealand on a map of Europe, we’d stretch from London to Italy.</p>
<p class="articleParagraph enarticleParagraph" >Sweden is about 70% larger by landmass but has double the population, at 10 million.</p>
<p class="articleParagraph enarticleParagraph" >It’s the largest of the Scandinavian countries. We are the largest of the Polynesian island groups.</p>
<p class="articleParagraph enarticleParagraph" >I’m sure we both like to think we punch above our weight on the world stage.</p>
<p class="articleParagraph enarticleParagraph" >We also both copped weird peripheral characters in The Muppets.</p>
<p class="articleParagraph enarticleParagraph" >If the meatball-loving Swedish Chef represents a problematic stereotype to be overcome, I don’t know where to begin with the maniacal fish-throwing Lew Zealand.</p>
<p class="articleParagraph enarticleParagraph" >The Swedes also take their dairy industry very seriously, even though their economy is no longer based on agricultural production.</p>
<p class="articleParagraph enarticleParagraph" >One of the strongest business connections between our two countries has been the relationship with Swedish dairy agricultural machinery manufacturer DeLaval. A specialist in milking technology, it has had a foothold in New Zealand since 1924.</p>
<p class="articleParagraph enarticleParagraph" >I suspect our new Governor, Dr Anna Breman’s first clue that we’re an odd place would be the asymmetric media coverage around the two debuts.</p>
<p class="articleParagraph enarticleParagraph" >If you based their relative significance on the local media, you’d assume the arrival of <span class="companylink">Ikea</span> was vastly more auspicious.</p>
<p class="articleParagraph enarticleParagraph" >We have a deep-seated cultural insecurity in this country around the need to attract global retail chains.</p>
<p class="articleParagraph enarticleParagraph" >Speculation about the prospect of <span class="companylink">Ikea</span> opening in New Zealand dates back to 2004, according to New Zealand Herald records.</p>
<p class="articleParagraph enarticleParagraph" >No wonder we were so excited this week.</p>
<p class="articleParagraph enarticleParagraph" >We’ve been triumphantly celebrating the arrival of international chains since <span class="companylink">Kentucky Fried Chicken</span> opened here in 1974.</p>
<p class="articleParagraph enarticleParagraph" >I’m old enough to remember the collective joy at the opening of each new <span class="companylink">McDonald</span>’s location through the 1980s.</p>
<p class="articleParagraph enarticleParagraph" >We’ve got a <span class="companylink">Costco</span> now — and a <span class="companylink">Taco Bell</span> and a <span class="companylink">Krispy Kreme</span>.</p>
<p class="articleParagraph enarticleParagraph" >If only we could convince <span class="companylink">Aldi</span> to open a cut-price supermarket chain, we could finally relax and take our place in the realm of fully developed economies.</p>
<p class="articleParagraph enarticleParagraph" >The Prime Minister turned out to cut the ribbon at <span class="companylink">Ikea</span> and bask in the warm populist glow of the news coverage.</p>
<p class="articleParagraph enarticleParagraph" >I’m sure <span class="companylink">Ikea</span> will create some jobs and make some consumers happy, but its arrival does strike me as bad news for many local retailers.</p>
<p class="articleParagraph enarticleParagraph" >And I’m pretty sure its owners will be planning to add to the drain on our current account by taking profits back home to Sweden.</p>
<p class="articleParagraph enarticleParagraph" >I’m sure the PM was equally enthusiastic about welcoming Bremen privately in Wellington.</p>
<p class="articleParagraph enarticleParagraph" >Finance Minister Nicola Willis definitely was when she introduced her at a press conference in September.</p>
<p class="articleParagraph enarticleParagraph" >In the grand scheme, Breman’s arrival is considerably more important in my clearly biased view.</p>
<p class="articleParagraph enarticleParagraph" >It is significant for many reasons. She is our first female Reserve Bank Governor.</p>
<p class="articleParagraph enarticleParagraph" >That doesn’t tell us anything one way or another about how she will shape monetary policy or banking regulation. But, as Willis has already noted, it is great to see just from a “breaking the glass ceiling” point of view.</p>
<p class="articleParagraph enarticleParagraph" >Much more important for the Reserve Bank is that she is from anywhere but here.</p>
<p class="articleParagraph enarticleParagraph" >Given the horror year (some would argue several) that the <span class="companylink">RBNZ</span> has had, it needs a clean break.</p>
<p class="articleParagraph enarticleParagraph" >Breman is unencumbered by political baggage. In what little we’ve seen of her publicly so far, she seems to be considered, articulate and a consummate professional.</p>
<p class="articleParagraph enarticleParagraph" >It will be interesting to see to what extent that buffers her from the intense public scrutiny the Reserve Bank can face.</p>
<p class="articleParagraph enarticleParagraph" >Monetary policy is followed like a sport in New Zealand, and that’s not quite the case in many other countries, including Sweden.</p>
<p class="articleParagraph enarticleParagraph" >It’s big news, of course, but often confined to financial and economic news.</p>
<p class="articleParagraph enarticleParagraph" >Here, the major news organisations (including the Herald) live blog the decisions.</p>
<p class="articleParagraph enarticleParagraph" >I think that has something to do with the way Kiwis fix their mortgages for a variety of relatively short terms.</p>
<p class="articleParagraph enarticleParagraph" >In Sweden and Australia, most people float along at variable market rates. In the US, people fix for decades.</p>
<p class="articleParagraph enarticleParagraph" >But here, we try to pick the best term, between six months and five years, to lock in and beat the bank.</p>
<p class="articleParagraph enarticleParagraph" >That makes monetary policy a national obsession rivalling our passion for rugby. That is to say, we have a lot of armchair experts.</p>
<p class="articleParagraph enarticleParagraph" >I hope Breman has been warned that she is stepping into a role that might be the equivalent of being the first international All Blacks coach.</p>
<p class="articleParagraph enarticleParagraph" >Next year, as the media hype around <span class="companylink">Ikea</span> fades, the focus on interest rates and inflation will grow.</p>
<p class="articleParagraph enarticleParagraph" >New Zealand might be in recovery, but it is not yet recovered.</p>
<p class="articleParagraph enarticleParagraph" >The Reserve Bank’s role next year will likely be more nuanced than it has been in the past few years. I hope it is.</p>
<p class="articleParagraph enarticleParagraph" >The world remains a volatile place, and the next monetary policy challenge will not be far away.</p>
<p class="articleParagraph enarticleParagraph" >I wish Breman well in her efforts to guide policy and in the more difficult task of communicating that policy to this strange little country.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>ikea : IKEA International AB | ingkah : Ingka Holding B.V.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i64 : Retail/Wholesale | i648 : Household Goods Retailing | i6481 : Furniture/Home Furnishings Retailing | i654 : Specialty Retailing | iretail : Retail</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>grugu : Rugby Union | gspo : Sports | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | ausnz : Australia/Oceania | eecz : European Union Countries | eurz : Europe | nordz : Northern Europe | nz : New Zealand | swed : Sweden</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>NZME Publishing Ltd.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document NZHLD00020251206elc700012</td></tr></table><br/></div></div><br/><span></span><div id="article-PARALL0020251206elc70025x" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/parallLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>MIL-OSI USA: Joint Statement of the 21st Meeting of the India-USA Joint Working Group on Counter Terrorism (JWG-CT) and 7th Designations Dialogue</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>538 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>ForeignAffairs.co.nz</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>PARALL</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025.  Multimedia Investments Ltd.  All rights reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Source: <span class="companylink">United States Department of State</span>
                     </p>
<p class="articleParagraph enarticleParagraph" >Office of the Spokesperson</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Joint Statement of the 21st Meeting of the India-USA Joint Working Group on Counter Terrorism (JWG-CT) and 7th Designations Dialogue</p>
<p class="articleParagraph enarticleParagraph" >Media Note</p>
<p class="articleParagraph enarticleParagraph" >December 6, 2025</p>
<p class="articleParagraph enarticleParagraph" >The text of the following statement was released by the Governments of the United States of America and the Republic of India on the occasion of the 21st Meeting of the India-USA Joint Working Group on Counter Terrorism (JWG-CT) and 7th Designations Dialogue.</p>
<p class="articleParagraph enarticleParagraph" >Begin Text</p>
<p class="articleParagraph enarticleParagraph" >India and the United States of America held the 21st Meeting of the India-USA Joint Working Group (JWG) on Counter Terrorism (CT) and the 7th Designations Dialogue on 3 December 2025 in New Delhi. Dr. Vinod Bahade, Joint Secretary (Counter Terrorism) in the <span class="companylink">Ministry of External Affairs of India</span>, and Ms. Monica Jacobsen, Senior Bureau Official in the Bureau of Counterterrorism in the <span class="companylink">United States Department of State</span>, led their respective delegations.</p>
<p class="articleParagraph enarticleParagraph" >The meetings underscored the importance of bilateral cooperation in countering terrorism, reflecting the spirit and breadth of the India-USA Comprehensive Global Strategic Partnership. Both sides unequivocally condemned terrorism in all its forms and manifestations, including cross border terrorism. They expressed concern over the increasing use of unmanned aerial vehicles (UAVs), drones, and AI for terrorist purposes. They strongly condemned the terrorist attack in Pahalgam, Jammu and Kashmir on 22 April 2025, and the recent heinous terror incident near the Red Fort, New Delhi on 10 November 2025, and stressed that those responsible for terrorism should be held accountable.</p>
<p class="articleParagraph enarticleParagraph" >The two sides reviewed a wide range of traditional and emerging threats and challenges such as terrorist recruitment, abuse of technology for terrorist purposes, and financing of terrorism. Both sides discussed ways to strengthen cooperation against challenges, including through training, cybersecurity, exchange of best practices, and information sharing through continued bilateral and multilateral efforts.</p>
<p class="articleParagraph enarticleParagraph" >Participants from India and the United States discussed strengthening law enforcement and judicial cooperation, including through information sharing and cooperation on mutual legal assistance requests.</p>
<p class="articleParagraph enarticleParagraph" >Both sides emphasized that confronting terrorism requires concerted action in a sustained and comprehensive manner. Against this backdrop, the two sides renewed their commitment to strengthening multilateral cooperation in the field of countering terrorism including in the <span class="companylink">UN</span>, Quad and the <span class="companylink">Financial Action Task Force (FATF)</span>. The two sides called for additional designations of <span class="companylink">ISIS</span> and al-Qa’ida affiliates, and <span class="companylink">Lashkar-e-Tayyiba</span> (LeT) and <span class="companylink">Jaish-e-Mohammad</span> (<span class="companylink">JeM</span>) and their proxy groups, supporters, sponsors, financiers and backers, under the <span class="companylink">UN</span> 1267 sanctions regime, ensuring their members face a global asset freeze, travel ban, and arms embargo. Underscoring the growing convergence between India and the United States on counterterrorism, the Indian side thanked the <span class="companylink">U.S. Department of State</span> for designating The Resistance Front (TRF), a proxy of LeT, as both a Foreign Terrorist Organization (FTO) and a Specially Designated Global Terrorist (SDGT).</p>
<p class="articleParagraph enarticleParagraph" >Both sides decided to hold the next meeting of the Joint Working Group on Counter Terrorism and Designations Dialogue in the United States on a mutually convenient date.</p>
<p class="articleParagraph enarticleParagraph" >End Text</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">MIL OSI USA News [http://milnz.co.nz/mil-osi-aggregation/]</span> -</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>inmea : India Ministry of External Affairs | usstat : United States Department of State</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gcns : National/Public Security | gcrim : Crime/Legal Action | gdip : International Relations | gfinc : Financial Crime | gpir : Politics/International Relations | grisk : Risk News | gterr : Terrorism | gtfin : Terrorism Financing | ncat : Content Types | nfact : Factiva Filters | nfcpex : C&E Executive News Filter</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>asiaz : Asia | delhi : Delhi | devgcoz : Emerging Market Countries | dvpcoz : Developing Economies | india : India | namz : North America | sasiaz : South Asia | usa : United States</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>AM-NC,Americas,Artificial Intelligence,Asia,Asia Pacific,Crime,CTF,DJF,India,KB,Machine Learning,MIL-OSI,Technology,terrorism,Transport,United States of America,Vehicles</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Multimedia Investments Ltd</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document PARALL0020251206elc70025x</td></tr></table><br/></div></div><br/><span></span><div id="article-PARALL0020251206elc70025y" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/parallLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>MIL-OSI Africa: Egypt: President El-Sisi Meets Prime Minister and Minister of Education and Technical Education</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>619 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>ForeignAffairs.co.nz</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>PARALL</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025.  Multimedia Investments Ltd.  All rights reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Source: APO</p>
<p class="articleParagraph enarticleParagraph" >
                        <span class="colorLinks">. [https://www.africa-newsroom.com/files/download/04cd40fd065257d]</span>
                     </p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Today, President Abdel Fattah El-Sisi met with Prime Minister Dr. Mostafa Madbouly, and Minister of Education and Technical Education Mr. Mohamed Abdel Latif.</p>
<p class="articleParagraph enarticleParagraph" >Spokesman for the Presidency Ambassador Mohamed El-Shennawy stated that the meeting reviewed several ministry work files. The Minister of Education and Technical Education offered a presentation on the execution status of teaching Programming and Artificial Intelligence (AI) as part of the curricula for the first year of secondary school, starting from the current academic year 2025/2026. In this regard, the minister highlighted that the inclusion of this subject is part of the State’s vision for digital transformation and educational development, and meets the requirements of the technological revolution and its associated changes in the labor market.</p>
<p class="articleParagraph enarticleParagraph" >The minister pointed out that the demand for the Japanese "QUREO" Programming and AI platform has exceeded all expectations, with over 236,000 students completing the full training content. He explained that secondary stage graduates who study the subject receive an accredited international certificate in programming from <span class="companylink">Hiroshima University</span> in Japan. In the same context, the Minister added that the Programming and AI subject will also be introduced in Technical Education starting from the academic year 2026/2027.</p>
<p class="articleParagraph enarticleParagraph" >The meeting also reviewed the Ministry of Education’s efforts to develop the technical education system through the expansion of Applied Technology Schools, which reached 115 schools during the 2025/2026 academic year, and linking study with practical training through partnerships with the private sector. This is in addition to signing international partnerships to grant graduates accredited international certificates, providing them with job opportunities in both the local and international markets. President El-Sisi stressed the necessity of exerting maximum effort to elevate the scientific and professional level of technical education graduates, in light of the growing needs of the labor market.</p>
<p class="articleParagraph enarticleParagraph" >The meeting also touched on the developments of the Japanese schools in Egypt, where President El-Sisi gave directives to increase their number to 500 schools over the next five years. The Minister also reviewed the results of his field tours to follow up on the educational process in various governorates. He noted the Ministry’s success in addressing accumulated challenges, including eliminating the shortage of teachers in core subjects, reducing student density in classes to less than 50 students, and ensuring the timely delivery of textbooks.</p>
<p class="articleParagraph enarticleParagraph" >The Minister of Education and Technical Education also reviewed the developments in applying the Egyptian Baccalaureate Certificate system. He outlined the multiple opportunities and diverse tracks it offers for taking exams, which suit students’ aptitudes and abilities. He also pointed out that this system ends the single-chance exam of the general secondary system, and highlighted the increasing student interest in the Baccalaureate system, with the enrollment rate exceeding 90% of the total number of students in the first phase of secondary school this current academic year.</p>
<p class="articleParagraph enarticleParagraph" >President El-Sisi emphasized the necessity of firm action against cheating cases and directed that the penalty be tightened for anyone proven to be involved in cheating in the General Secondary examinations.</p>
<p class="articleParagraph enarticleParagraph" >Furthermore, the President directed the continuous exertion of all necessary effort and the taking of appropriate measures to care for teachers and provide them with continuous incentives, including improving their economic status. President El-Sisi reiterated the importance of continuing to enforce discipline and consolidate positive moral values within the educational system, non-tolerance of any lapse in this matter, and the taking of urgent and decisive accountability measures towards any transgression or misconduct.</p>
<p class="articleParagraph enarticleParagraph" >Distributed by APO Group on behalf of Presidency of the Arab Republic of Egypt.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">MIL OSI Africa [http://milnz.co.nz/mil-osi-aggregation/]</span> -</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcntsu : Continuing Education | gcsci : Computer Science | gedu : Education | gjob : Labor Issues | gpir : Politics/International Relations | gpol : Domestic Politics | gscho : School | gsci : Sciences/Humanities</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>africaz : Africa | asiaz : Asia | devgcoz : Emerging Market Countries | dvpcoz : Developing Economies | egypt : Egypt | meastz : Middle East | medz : Mediterranean Countries | nafrz : North Africa</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>Africa,AM-NC,Artificial Intelligence,Asia,Asia Pacific,CTF,DJF,Education,KB,Machine Learning,Middle East,MIL-OSI,Technology,Transport,Universities</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Multimedia Investments Ltd</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document PARALL0020251206elc70025y</td></tr></table><br/></div></div><br/><span></span><div id="article-SLMO000020251206elc7002jp" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/slmoLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>C</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Off to a solid start; A look at the Thanksgiving shopping weekend and what's next (copy);</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>ANNE D'INNOCENZIO, Associated Press </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1175 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>St. Louis Post-Dispatch</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>SLMO</td></tr><tr><td align="right" valign="top" class="index"><b>ED</b>&nbsp;</td><td>01</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>1</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025, St. Louis Post-Dispatch.  All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Holiday retail sales</p>
<p class="articleParagraph enarticleParagraph" >NEW YORK - The nation's shoppers may feel gloomy about the economy, but they certainly were in the mood to shop over the five-day Thanksgiving weekend that wrapped up on Cyber Monday.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >As Wall Street analysts and retailers sift through the data from the weekend - the unofficial start to the season and a good barometer of shoppers' financial health and the strength of the economy - the figures show that shoppers went online and in stores to scour for deals on everything from TVs to clothing. But all that economic uncertainty did affect spending. Shoppers were very focused and selective, some malls reported.</p>
<p class="articleParagraph enarticleParagraph" >Of course, the weekend looks a lot different from 15 years ago, when shoppers camped out in the wee hours of the morning and fought in store aisles for doorbusters like TVs. Shoppers are still heading to stores, but the biggest growth is online, which now accounts for 30% of total holiday sales. That's up from 15% in 2012, according to the <span class="companylink">National Retail Federation</span>, the nation's largest retail trade group.</p>
<p class="articleParagraph enarticleParagraph" >Adobe Analytics reported Tuesday that so-called Cyber Week - the five-day period from Thanksgiving to Cyber Monday - brought in $44.2 billion online overall, up 7.7% year-over-year, bolstered by record spending online during Black Friday.</p>
<p class="articleParagraph enarticleParagraph" >On Cyber Monday, consumers spent $14.25 billion, up 7.1% and making it again the year's biggest online shopping day.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">National Retail Federation</span>'s CEO Matt Shay said Tuesday that shoppers wall off the winter holidays from all the economic noise, building a moat around the season.</p>
<p class="articleParagraph enarticleParagraph" >"The holidays is really very much an emotional purchase," Shay said. "Families plan for it. They invest in it. And as a component of the holidays, the five-day Thanksgiving weekend is really the psychological kickoff of the holidays."</p>
<p class="articleParagraph enarticleParagraph" >Based on the group's survey of shoppers from the weekend, Shay called the period a "very solid beginning" to the holiday season.</p>
<p class="articleParagraph enarticleParagraph" >The group still expects sales over November and December of between $1.01 trillion and $1.02 trillion. That would be up 3.7% to 4.2% more than last year.</p>
<p class="articleParagraph enarticleParagraph" >Here's a look at the data, the discounts, and what's next for retailers among other issues:</p>
<p class="articleParagraph enarticleParagraph" >The latest data shows record traffic</p>
<p class="articleParagraph enarticleParagraph" >Software company <span class="companylink">Salesforce</span> reported that for Cyber Week - it measures from Nov. 25 through Monday - global online sales increased to $336.6 billion, up 7% compared with the year-ago period. U.S. online sales increased to $79.6 billion, up 5% year for that week, compared with the year-ago period.</p>
<p class="articleParagraph enarticleParagraph" >The Mall of America in Bloomington, Minnesota, reported on Tuesday that more than 235,000 people visited the iconic center on Black Friday, making it the busiest Black Friday on record in the mall's history. The traffic number was up 8.5% compared with the same day on 2024 and nearly 2% above pre-pandemic 2019, the mall said.</p>
<p class="articleParagraph enarticleParagraph" >Mastercard SpendingPulse, which tracks in-person and online spending, reported Saturday that overall Black Friday sales excluding automotive rose 4.1% from a year ago. The retail sales indicator, not adjusted for inflation, showed online sales jumped by double digits - 10.4% - while in-store purchases inched up 1.7%.</p>
<p class="articleParagraph enarticleParagraph" >Still, shoppers were laser-focused.</p>
<p class="articleParagraph enarticleParagraph" >William Lewis, marketing director of Westfield Garden State Plaza in Paramus, New Jersey, noted on Black Friday that, "People are definitely buying." But Lewis noted that shoppers are more targeted and have done their homework ahead of time on social media or store sites.</p>
<p class="articleParagraph enarticleParagraph" >"They know exactly where they are going," he added.</p>
<p class="articleParagraph enarticleParagraph" >Discounts were generous, but don't expect them to get better</p>
<p class="articleParagraph enarticleParagraph" >Ahead of the Thanksgiving weekend, promotions didn't come as early as last year or were more muted, according to some malls and analysts. But for the big weekend, retailers ramped up discounting to be in line with last year's sales event, according to <span class="companylink">Adobe</span> and big malls like Mall of America.</p>
<p class="articleParagraph enarticleParagraph" >But if shoppers were waiting for prices to go down after this weekend, that may not be the best strategy. Discounts won't improve on many items, and stores came into the season with leaner inventory amid an uncertain economy, analysts said.</p>
<p class="articleParagraph enarticleParagraph" >Vivek Pandya, <span class="companylink">Adobe</span>'s director of Adobe Digital Insights, noted that prior to the Thanksgiving weekend kickoff, discounts on average ranged from 10% to 17% and then accelerated to an average range of 18% to 30% for the holiday kickoff.</p>
<p class="articleParagraph enarticleParagraph" >But he expects that retailers will likely pull back from those discounts and will hover a little above what shoppers saw to the run-up of Black Friday. The exception would be poor-selling seasonal items, which need to be sold before Dec. 25, Pandya said.</p>
<p class="articleParagraph enarticleParagraph" >As for inventory, there were fears of empty shelves when tariff rates ballooned in April, but analysts said stores were able to navigate the vacillating tariff policy, bringing in goods at lower rates.</p>
<p class="articleParagraph enarticleParagraph" >Nikki Baird, vice president of strategy at <span class="companylink">Aptos</span>, a retail technology firm which works with fashion clients, noted that, "I think consumers will continue to find the things that they're looking for, but there will be fewer choices."</p>
<p class="articleParagraph enarticleParagraph" >Some shoppers relied on artificial intelligence tools</p>
<p class="articleParagraph enarticleParagraph" >Shoppers also used AI tools to track prices or get gift recommendations, though the usage is still modest. On Cyber Monday, AI traffic to U.S. retail sites - measured by shoppers clicking on a link - increased by nearly eightfold, according to <span class="companylink">Adobe</span>. From Nov. 1 through Dec. 1, AI traffic is up nearly ninefold, it said.</p>
<p class="articleParagraph enarticleParagraph" >The services were used most in categories including video games, appliances, electronics, toys, and personal care products, according to <span class="companylink">Adobe</span>.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Salesforce</span> reported that across Cyber Week, AI and agents influenced 20% of all orders, accounting for $67 billion in global sales. In the U.S., AI and agents drove 17% of orders, or $13.5 billion in sales. The figure encompasses everything from a ChatGPT query to AI-supplied gift suggestions on a retailer's website.</p>
<p class="articleParagraph enarticleParagraph" >What's next</p>
<p class="articleParagraph enarticleParagraph" >The Thanksgiving weekend is a key barometer of spending for the season. But with worries of rising prices, will shoppers taper their spending as the season progresses?</p>
<p class="articleParagraph enarticleParagraph" >Baird said she will be looking at the period between the post-Thanksgiving weekend and the last week before Christmas to see whether spending keeps up.</p>
<p class="articleParagraph enarticleParagraph" >"I think that will help us answer the question of whether this was a concentration of spending or a trend of spending," she said.</p>
<p class="articleParagraph enarticleParagraph" >Tiffany Yeh, a managing director and partner at <span class="companylink">Boston Consulting Group</span>, believes there will be strong spending throughout the rest of the holiday season. Her concern is what's in store for 2026. Yeh cited the consultancy's shopper surveys pointing to consumers delaying purchases in order to spend during the holidays. She wonders if shoppers will mute their spending or instead steady their buying.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>natrfe : National Retail Federation</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i64 : Retail/Wholesale | iretail : Retail</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nnam : News Agency Materials</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usc : Midwest U.S. | usmo : Missouri</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>St. Louis Post-Dispatch</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document SLMO000020251206elc7002jp</td></tr></table><br/></div></div><br/><span></span><div id="article-SLMO000020251206elc7002bd" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/slmoLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>C</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>AI takes bigger role in holiday shopping; How new AI tools are changing holiday shopping in 2025;</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>ANNE D'INNOCENZIO, Associated Press </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1271 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>St. Louis Post-Dispatch</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>SLMO</td></tr><tr><td align="right" valign="top" class="index"><b>ED</b>&nbsp;</td><td>01</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>2</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025, St. Louis Post-Dispatch.  All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >NEW YORK - Major retail chains and tech companies are offering new or updated artificial intelligence tools in time for the holiday shopping season, hoping to give consumers an easier gift-buying experience and themselves an augmented share of online spending.</p>
<p class="articleParagraph enarticleParagraph" >Although AI-powered purchases are in early stages, the shopping assistants and agents rolled out by the likes of <span class="companylink">Walmart</span>, <span class="companylink">Amazon</span> and <span class="companylink">Google</span> can do more than the chatbots of holidays past. The latest versions were designed to provide personalized product recommendations, track prices and place some orders through unscripted "conversations" with customers.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Those features are in addition to shopping updates from AI platforms like <span class="companylink">OpenAI</span>'s ChatGPT and Google Gemini. In one of the season's most talked-about launches, <span class="companylink">Google</span> this month introduced an AI agent that can be instructed to call local stores to ask if a desired product is in stock.</p>
<p class="articleParagraph enarticleParagraph" >San Francisco software company <span class="companylink">Salesforce</span> estimated that AI would influence $73 billion, or 22%, of all global sales in one way or another from the Tuesday before Thanksgiving through Monday after the holiday, according to Caila Schwartz, <span class="companylink">Salesforce</span>'s director of consumer insights.</p>
<p class="articleParagraph enarticleParagraph" >The figure, which stood at $60 billion a year ago, encompasses everything from a ChatGPT query to AI-supplied gift suggestions on a retailer's website, Schwartz said.</p>
<p class="articleParagraph enarticleParagraph" >Despite the advancements, AI's impact on holiday shopping will be "relatively limited" this year since not every shopping site has useful tools and not every shopper is willing to try them, said Brad Jashinsky, a senior retail industry analyst at information technology research and consulting firm <span class="companylink">Gartner</span>.</p>
<p class="articleParagraph enarticleParagraph" >"The more retailers that launch these tools, the better they get, and the more that consumers get comfortable and start to seek them out," Jashinsky said. "But customer behavior takes a long time to change."</p>
<p class="articleParagraph enarticleParagraph" >Here are three ways the technology is poised to influence holiday shopping habits in 2025:</p>
<p class="articleParagraph enarticleParagraph" >Bypassing the search bar</p>
<p class="articleParagraph enarticleParagraph" >AI's potential to simplify the search for the perfect present is most apparent so far in tools that promise to give shoppers faster and more detailed results than a web browser with a lot fewer clicks.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">OpenAI</span> upgraded ChatGPT with a shopping research feature that provides personalized buyers' guides. The information comes from product pages, reviews, prices and a user's previous interactions with the chatbot. The tool works best for complicated products like electronics and appliances or for "detail-heavy" items like beauty or sporting goods, <span class="companylink">OpenAI</span> said.</p>
<p class="articleParagraph enarticleParagraph" >Then there's Rufus, the shopping assistant that <span class="companylink">Amazon</span> rolled out last year. It now remembers information customers previously fed it, like having four children who all like board games, for example. A user's browsing and purchase history, as well as reviews, are used to personalize recommendations.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Google</span> upgraded its AI Mode search tool to provide answers to detailed questions composed in natural language. For example, users can tell the agent they want to buy a casual sweater to wear with a skirt or jeans in New York in January.</p>
<p class="articleParagraph enarticleParagraph" >Responses are pulled from <span class="companylink">Google</span>'s 50 billion product listings. The tool can also produce charts with side-by-side comparisons of prices, features, reviews and other factors. Previously, shoppers had to use keywords, filters and product links to find the information they needed.</p>
<p class="articleParagraph enarticleParagraph" >"This is an expansionary moment, I think, for all of technology and for commerce," Lilian Rincon, vice president of product, consumer shopping at <span class="companylink">Google</span>, recently told <span class="companylink">The Associated Press</span>.</p>
<p class="articleParagraph enarticleParagraph" >Meanwhile, <span class="companylink">Walmart</span>'s AI shopping assistant, Sparky, offers occasion-based recommendations and synthesizes reviews. An AI-powered gift finder on Target's app, exclusive to the holidays, responds to prompts such as the recipient's age and special hobbies.</p>
<p class="articleParagraph enarticleParagraph" >New pricing tools and alerts</p>
<p class="articleParagraph enarticleParagraph" >Tools for tracking online prices have been around for years, including CamelCamelCamel, a third-party service for <span class="companylink">Amazon</span> prices, as well as <span class="companylink">Paypal</span>'s Honey browser extension for monitoring thousands of online shops.</p>
<p class="articleParagraph enarticleParagraph" >This holiday season, shoppers have new options.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Amazon</span> launched a 90-day pricing history tracker this month for virtually everything it sells. Shoppers also now can set up alerts to receive notifications when prices on specific items fall within their budgets.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Google</span>, which for years had a basic price tracker, launched a more advanced version that lets users refine their requests with details like a garment's size and color. <span class="companylink">Microsoft</span>'s Copilot also launched a price tracker this year.</p>
<p class="articleParagraph enarticleParagraph" >Jason Goldberg, chief commerce strategy officer at <span class="companylink">Publicis Groupe</span>, said he thinks the new pricing tools will add more pressure on retailers to make sure their prices are competitive.</p>
<p class="articleParagraph enarticleParagraph" >"A lot of consumers that weren't even looking for price alerts are going to discover price alerts for the first time," Goldberg predicted.</p>
<p class="articleParagraph enarticleParagraph" >New ways to buy</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Amazon</span>, <span class="companylink">OpenAI</span> and <span class="companylink">Google</span> are racing to create tools that would allow for seamless AI-powered shopping by taking consumers from browsing to buying within the same program instead of having to go to a retailer's website to complete a purchase.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">OpenAI</span> launched a new instant checkout feature that lets users buy products suggested by ChatGPT without leaving the app. Users can order merchandise from <span class="companylink">Etsy</span> sellers and from some brands that use <span class="companylink">Shopify</span>, including <span class="companylink">Glossier</span>, Skims and <span class="companylink">Spanx</span>.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">OpenAI</span> and <span class="companylink">Walmart</span> announced a similar deal in October, saying the partnership would allow ChatGPT members to use the instant checkout feature to shop for nearly everything available on <span class="companylink">Walmart</span>'s website except for fresh food. For now, however, the feature only supports buying one item at a time.</p>
<p class="articleParagraph enarticleParagraph" >A different deal Target struck with <span class="companylink">OpenAI</span> lets shoppers put multiple items in a cart on ChatGPT, including fresh food products. But when customers are ready to pay for their orders, they are directed away from the chatbot to the Target app.</p>
<p class="articleParagraph enarticleParagraph" >New tools from <span class="companylink">Amazon</span> and <span class="companylink">Google</span> will give shoppers a taste of having autonomous AI assistants do the buying for them. While the services still are limited, "agentic AI" is intended to be more independent and advanced than the generative AI chatbots that excel at research and writing, experts say.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Amazon</span> is now letting Rufus automatically purchase items for customers who click an "auto buy" button while setting up price alerts. Once a product's price drops to the desired level, customers receive notice of their completed orders and have a limited window to cancel, the company said.</p>
<p class="articleParagraph enarticleParagraph" >The e-commerce giant also started allowing shoppers to use Rufus searches for brand-name products on the <span class="companylink">Amazon</span> app as a gateway to other retailers. If <span class="companylink">Amazon</span> doesn't carry a desired item in its store, a "Shop Direct" button will take them to the website of a place that does.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Google</span>'s AI Mode price tracker also includes a "buy for me" option that automatically makes a customer's purchase through <span class="companylink">Google</span> Pay when the price is right. The feature is available for products sold by <span class="companylink">Wayfair</span>, <span class="companylink">Chewy</span>, Quince and some <span class="companylink">Shopify</span> merchants, and <span class="companylink">Google</span> expects to keep adding more stores, the company said. sellers.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Google</span> also expanded its web browser with an automated AI call feature that phones local businesses on behalf of customers looking for information or specific products. <span class="companylink">Google</span>'s program discloses to the store that it's an AI caller, and stores can choose not to participate, the company said.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Google</span> said it's applying the feature initially to specific product categories: toys, health and beauty, and electronics. Target and <span class="companylink">Walmart</span> declined to comment on whether this type of service would be part of their future plans.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>amzcom : Amazon.com, Inc. | ezxqlr : OpenAI LLC | gognew : Google LLC | goog : Alphabet Inc. | jpixti : Shopify Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302 : Computers/Consumer Electronics | i330202 : Software | i3302022 : Artificial Intelligence Technologies | i64 : Retail/Wholesale | i656000301 : Etailing | i8395464 : Internet Search Engines | icomp : Computing | iecom : E-commerce | iint : Online Service Providers | iretail : Retail | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcsci : Computer Science | gsci : Sciences/Humanities | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nnam : News Agency Materials | redit : Selection of Top Stories/Trends/Analysis | reqr : Suggested Reading – Industry News | reqrcm : Suggested Reading – Computers</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usc : Midwest U.S. | usmo : Missouri</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>St. Louis Post-Dispatch</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document SLMO000020251206elc7002bd</td></tr></table><br/></div></div><br/><span></span><div id="article-SLMO000020251206elc70028l" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/slmoLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>C</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Letters to the editor, Sunday Dec. 7; Letters to the editor, Sunday Dec. 7;</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>498 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>St. Louis Post-Dispatch</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>SLMO</td></tr><tr><td align="right" valign="top" class="index"><b>ED</b>&nbsp;</td><td>01</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>6</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025, St. Louis Post-Dispatch.  All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >St. Louis' infrastructure isn't ready for self-driving automobiles</p>
<p class="articleParagraph enarticleParagraph" >Regarding "<span class="colorLinks">Waymo planning St. Louis rollout, will manually test cars starting this week [https://www.stltoday.com/news/local/business/article_b9e29ab0-0cb4-49ad-98ff-744ec06f3028.html]</span>" (Dec. 3): There has been and will be a lot of critics skeptical of autonomous vehicles (AVs) using artificial intelligence (AI) to navigate our 4,230 lane miles of city streets. Count me as one of those critics.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >The <span class="companylink">Washington Post</span> recently released an investigation across the United States headlined "<span class="colorLinks">The deadliest roads in America [https://www.washingtonpost.com/business/interactive/2025/pedestrian-deaths-surge-road-safety/]</span>," which cites general neglect and lack of investment by transit authorities for the reason pedestrian deaths have surged. Accidents involving <span class="companylink">Waymo</span> vehicles have also been fatal to both pedestrians and other drivers.</p>
<p class="articleParagraph enarticleParagraph" >Adding self-driving or autonomous vehicles to an already aging, congested and underfunded infrastructure like St. Louis streets and Missouri highways is an exceptionally poor idea, especially when cyclist and pedestrian fatalities are already rising.</p>
<p class="articleParagraph enarticleParagraph" >I would look closely at the bills being introduced by the Missouri Legislature to see if any type of funding for road improvements is attached which would allow <span class="companylink">Waymo</span> and competitors to operate in Missouri. If <span class="companylink">Waymo</span> or another AV company wants to operate in Missouri, then it should also foot the bill to make the infrastructure safer for pedestrians in areas they operate. We should not turn our public roadways into the private proving grounds for AI and AV entrepreneurship to exploit at the cost of human life.</p>
<p class="articleParagraph enarticleParagraph" >Joe Rich</p>
<p class="articleParagraph enarticleParagraph" >St. Louis</p>
<p class="articleParagraph enarticleParagraph" >Second-tap boat bombing was a war crime. Where is Congress?</p>
<p class="articleParagraph enarticleParagraph" >Regarding the U.S. military bombings of boats off of Venezuela: This is clearly an act of war in international waters, so the rules of war should apply. Dropping a second bomb to kill two survivors is simply a war crime. This is the same as executing a wounded enemy on the battlefield.</p>
<p class="articleParagraph enarticleParagraph" >Doing this is exactly what some brave senators warned of when they made a video reminding our armed forces that their oath requires them not to follow illegal orders. The order to drop the second bomb was clearly illegal. The Nuremberg trials established that following illegal orders is a war crime punishable by death.</p>
<p class="articleParagraph enarticleParagraph" >Making a video does not qualify. So I am asking Missouri Sens. Josh Hawley and Eric Schmitt: Where do you stand? Will you be brave and support an inquiry or will you simply follow the president no matter what?</p>
<p class="articleParagraph enarticleParagraph" >Our legislative branch is an embarrassment and should be called out for it. Congress is in charge of oversight. So start doing it!</p>
<p class="articleParagraph enarticleParagraph" >Paul Schroeder</p>
<p class="articleParagraph enarticleParagraph" >Florissant</p>
<p class="articleParagraph enarticleParagraph" >On immigration debate, remember we're all from somewhere else</p>
<p class="articleParagraph enarticleParagraph" >Unless you identify as Native American (less than 3% of the U.S. population), you are the descendent of refugees, slaves or immigrants.</p>
<p class="articleParagraph enarticleParagraph" >Would you like to be forcibly returned to your country of origin? ("<span class="colorLinks">Editorial: Trump's anti-immigration net entangles an Afghan who aided America [https://www.stltoday.com/opinion/editorial/article_5a61fa31-903d-4844-8753-0af2fb1f9c48.html]</span>," Dec. 3.)</p>
<p class="articleParagraph enarticleParagraph" >Don Owen</p>
<p class="articleParagraph enarticleParagraph" >Ballwin</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>goog : Alphabet Inc. | waymmo : Waymo LLC</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>iadrive : Autonomous Driving Technologies | iaut : Automotive | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcns : National/Public Security | gcrim : Crime/Legal Action | gcsci : Computer Science | gdef : Armed Forces | grisk : Risk News | gsci : Sciences/Humanities | gvio : Military Operations | gwar : War Crimes | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | niwe : IWE Filter | nlet : Letters | nrgn : Routine General News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | usa : United States | usc : Midwest U.S. | usmo : Missouri</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>St. Louis Post-Dispatch</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document SLMO000020251206elc70028l</td></tr></table><br/></div></div><br/><span></span><div id="article-SHD0000020251206elc70001m" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/shdLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>News</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Ukraine: AI drones to transform Ukraine war</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>David Crowe | Europe correspondent </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1227 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Sun Herald</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>SHD</td></tr><tr><td align="right" valign="top" class="index"><b>ED</b>&nbsp;</td><td>First</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>6</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>© 2025 Copyright John Fairfax Holdings Limited. <span class="colorLinks">www.smh.com.au [http://www.smh.com.au]</span>
               </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Lviv, Ukraine: Old ideas about war are being tossed aside in an arms race to win the battle for Ukraine and a new breed of drone may decide which side claims victory.</p>
<p class="articleParagraph enarticleParagraph" >Young companies are forming across Ukraine in a bid to defeat Russian forces with faster and more powerful drones that can operate on land, sea and air.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >And they understand that if they do not build autonomous drones that use artificial intelligence to find their targets, their enemies will get there first.</p>
<p class="articleParagraph enarticleParagraph" >"We are very, very close to the war of drones," says Sergii Gunko, the chief operating officer of Tank Bureau, a tech company in Lviv.</p>
<p class="articleParagraph enarticleParagraph" >"There will be less and less direct contact for people who see each other, take their guns and shoot. You have a kill zone that is approximately 20 to 25 kilometres wide and is constantly increasing."</p>
<p class="articleParagraph enarticleParagraph" >Gunko outlines this future at a gathering of Ukrainian defence developers in Lviv, in western Ukraine, where they meet to share knowledge and attract investors.</p>
<p class="articleParagraph enarticleParagraph" >Tank Bureau is a member of a group called Iron Cluster, which began in Lviv and now has 90 members across Ukraine. The young companies mirror the attitude of Silicon Valley start-ups, but their goal is to win a war.</p>
<p class="articleParagraph enarticleParagraph" >The "kill zone" means the concept of a "front line" is becoming obsolete as aerial drones roam across the line of contact and limit the movements of the opposing army. The size of this zone is defined by the range of the drones. Gunko believes the width of the zone is likely to extend to about 100 kilometres over time.</p>
<p class="articleParagraph enarticleParagraph" >The war in Ukraine has already been transformed by first-person view, or FPV, drones, which operators fly via very long spools of fibre-optic cable that prevent them from being jammed. While they have disadvantages, such as an operating distance limited to the length of a wire, they are highly resilient and their range is increasing all the time.</p>
<p class="articleParagraph enarticleParagraph" >Meanwhile, another company, DoD Solution, is installing software in military drones to create autonomous weapons that can complete missions using artificial intelligence.</p>
<p class="articleParagraph enarticleParagraph" >DoD Solution co-founder Ivan Oleksii says the software will not replace the best drone pilots, given the human skill required to control the devices in battle, but will help address the scarcity of drone pilots. Oleksii says some pilots may be very good and others may be average, which means it will be better to use AI drones in large numbers.</p>
<p class="articleParagraph enarticleParagraph" >The DoD software is loaded onto a module, he says, which has been fitted to military drones used by the Ukrainian Armed Forces to prove it works in the field. A human operator can direct a drone on some parts of its journey, then switch to AI so it completes the task on its own. This can be useful when a drone goes on a long-distance mission outside radio range.</p>
<p class="articleParagraph enarticleParagraph" >"It totally makes sense to use it," says Oleksii. "Once those AI tools can perform better than the average soldier, defence forces will start to use this human factor on something else. If they can automate it, they will automate it."</p>
<p class="articleParagraph enarticleParagraph" >None of these concepts is hidden from the Russians, although the technology itself is guarded carefully. Ukraine bans the export of this software and hardware, a contentious decision because it limits revenue for its own start-ups. The companies spoke about their work but would not divulge confidential details.</p>
<p class="articleParagraph enarticleParagraph" >The race on the Russian side is led by groups such as <span class="companylink">Rubicon</span>, a military unit set up in the middle of last year and credited with using drones to push Ukrainian soldiers back in the Kursk region.</p>
<p class="articleParagraph enarticleParagraph" >The head of the Ukrainian Aerial Reconnaissance Support Centre, Maria Berlinska, said in August that <span class="companylink">Rubicon</span> had "brilliant management" and was Russia's best technology unit. This means there is an open question about whether the start-up culture on the Ukrainian side will be able to match the centralised military power of the Kremlin.</p>
<p class="articleParagraph enarticleParagraph" >While Australian companies have joined this arms race, and the Department of Defence is buying and developing drones, few countries can match the intensity of the work in Ukraine. The war is a tragedy, but also a laboratory. Lessons from the front are acted on urgently because so many in Ukraine see this as an existential challenge for their nation.</p>
<p class="articleParagraph enarticleParagraph" >"The innovation cycle of technology that we're used to, like in <span class="companylink">NATO</span> or other countries, can take years to create something new," says Iron Cluster co-founder and global project lead Andrii Makhnyk.</p>
<p class="articleParagraph enarticleParagraph" >"We don't have this time, and that's why our innovation cycle started to be more like six months to three months.</p>
<p class="articleParagraph enarticleParagraph" >"It's a constant fight between sword and shield."</p>
<p class="articleParagraph enarticleParagraph" >Access to technology is a factor. Russia has built devastating drones using designs from Iran and hardware from China. So far, Ukrainian companies are still buying Chinese components, but they worry about restrictions that will hurt their ability to win the arms race.</p>
<p class="articleParagraph enarticleParagraph" >Not all the work leads to the production of offensive weapons. Farsight Vision, another Iron Cluster member, is taking data gathered from aerial drones and feeding it into mapping software to create 3D maps of Russian positions. "Our system is actively used by the Ukrainian military now," says Oksana Vakshynska, the engagement manager at</p>
<p class="articleParagraph enarticleParagraph" >the company.</p>
<p class="articleParagraph enarticleParagraph" >"The tool is used by reconnaissance units, whose task is to investigate an area and it is also used in the planning of operations because, before you plan the operation, you need to understand the area."</p>
<p class="articleParagraph enarticleParagraph" >It is also used in drone schools, so the pilots-in-training are working with real examples of the enemy terrain.</p>
<p class="articleParagraph enarticleParagraph" >Naval drones are proving their capacity in Ukrainian strikes on Russian oil tankers. Ukraine claims success in damaging two tankers in the Black Sea last week and it has released video of its "Sea Baby" drones heading toward their targets like small automated speedboats.</p>
<p class="articleParagraph enarticleParagraph" >Gunko, of Tank Bureau, says the future will be about using large numbers of small drones rather than a small number of large offensive weapons.</p>
<p class="articleParagraph enarticleParagraph" >"If you look at what's going on in the war, you see that it's better not to invest in a big ship but in smaller drones like the Sea Baby," he says.</p>
<p class="articleParagraph enarticleParagraph" >In the same way, Russia has pierced Ukrainian air defences by launching hundreds of drones in a single night.</p>
<p class="articleParagraph enarticleParagraph" >This is the logic that led Tank Bureau to develop ground drones - known as UGVs, for unmanned ground vehicles - as a way to help Ukraine on the battlefield.</p>
<p class="articleParagraph enarticleParagraph" >While China has released video of drones that appear to walk like dogs, the Ukrainian UGVs use tracks like tanks. Tank Bureau puts its vehicles on display at defence gatherings in Kyiv and Lviv this week.</p>
<p class="articleParagraph enarticleParagraph" >Gunko said brigade commanders would choose hundreds of UGVs as an alternative to a tank because of the greater impact of a large number of uncrewed vehicles. Western countries, he added, needed to adapt to what is happening in Ukraine and think again about the idea that large ships and aircraft can withstand attack from large numbers of smaller drones.</p>
<p class="articleParagraph enarticleParagraph" >"This is the main trend, I think, for modern war, in the world we see now," Gunko says.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcns : National/Public Security | gcsci : Computer Science | gdef : Armed Forces | grisk : Risk News | gsci : Sciences/Humanities | gvio : Military Operations</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>asiaz : Asia | dvpcoz : Developing Economies | eeurz : Central/Eastern Europe | eurz : Europe | russ : Russia | ukrn : Ukraine</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Fairfax Media Management Pty Limited</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document SHD0000020251206elc70001m</td></tr></table><br/></div></div><br/><span></span><div id="article-SAGE000020251206elc700009" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/sageLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>News</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Exclusive: Gas giant seeks to dodge scheme to slash prices</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>Mike Foley | Climate and energy correspondent </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>940 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Sunday Age</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>SAGE</td></tr><tr><td align="right" valign="top" class="index"><b>ED</b>&nbsp;</td><td>First</td></tr><tr><td align="right" valign="top" class="index"><b>PG</b>&nbsp;</td><td>1</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>(c) 2025 Copyright John Fairfax Holdings Limited. <span class="colorLinks">www.theage.com.au [http://www.theage.com.au]</span>
               </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Commonwealth plans to slash gas prices for households and business by shoring up domestic supply face a last-minute attempt by one of the biggest gas exporters to sidestep the scheme.</p>
<p class="articleParagraph enarticleParagraph" >Before a federal cabinet meeting tomorrow to consider a national gas reserve, manufacturers are demanding the Albanese government ignore pressure from an east coast exporter for Labor to ditch its preferred policy.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Authorities warn that millions of gas-connected homes in Victoria and NSW face shortfalls unless more local gas is made available, while businesses warn they could be forced to close because of soaring energy costs.</p>
<p class="articleParagraph enarticleParagraph" >In response, Canberra has been reviewing gas exports, and will consider the details of how to reserve gas for Australian consumers. The government is expected to announce its gas reservation policy before breaking for the holiday season.</p>
<p class="articleParagraph enarticleParagraph" >"Australian gas should be available to Australian users at reasonable prices," Energy Minister Chris Bowen said yesterday.</p>
<p class="articleParagraph enarticleParagraph" >Manufacturers, unions and the Victorian government are pressing the federal government to impose an east coast reservation scheme, forcing exporters to hold back a certain amount for the local market to help limit the impact of global markets and put downward pressure on local energy costs.</p>
<p class="articleParagraph enarticleParagraph" >Independent analysis has found that a 6 per cent increase in domestic gas supply could cut wholesale gas prices by up to 20 per cent, which would help lower prices for millions of Victorian and NSW households, as well as manufacturers that rely on the fuel.</p>
<p class="articleParagraph enarticleParagraph" >The price of wholesale gas on the east coast has tripled from $4 a gigajoule to more than $12 since 2015. In that time, three massive liquefied natural gas (LNG) hubs in Gladstone, Queensland, began their lucrative export trade - and tied local prices to the global market.</p>
<p class="articleParagraph enarticleParagraph" >Global prices surged in 2022 when Russia invaded Ukraine, resulting in sanctions on Russian gas exports and a global energy crunch. Meanwhile, the cheapest domestic gas reserves, including vast Bass Strait oil and gas fields that have supplied the south-eastern states for decades, have begun rapidly drying up, causing uncertainty of supply, which has also pushed prices up.</p>
<p class="articleParagraph enarticleParagraph" >"Given our increasingly crippling energy issues with price and supply, we have reached the point that conditions on gas producers to ensure viable domestic supply are becoming necessary," said <span class="companylink">Australian Industry Group</span> chief executive Innes Willox, who represents manufacturers and other large gas users.</p>
<p class="articleParagraph enarticleParagraph" >Gas, like minerals, is owned by the Commonwealth, and companies pay royalties for the right to sell it. The federal government sets the conditions for exports of the fuel, and it would use this power to force exporters to send more to the local market.</p>
<p class="articleParagraph enarticleParagraph" >The government's preferred model for its reservation scheme is "export permitting", in which LNG producers would have to guarantee a certain volume of supply to the local market to gain the right to send shipments overseas.</p>
<p class="articleParagraph enarticleParagraph" >One of the three exporters, the Santos-led GLNG joint venture, has been in Canberra over the past week lobbying against an export-permitting scheme because it would hit its operations the hardest. Along with Australian company <span class="companylink">Santos</span>, GLNG members include Malaysia's <span class="companylink">Petronas</span>, French TotalEnergies and South Korea's <span class="companylink">Kogas</span>.</p>
<p class="articleParagraph enarticleParagraph" >GLNG favours a model that would force gas producers to draw their supply for a domestic reservation only from what is known as uncontracted gas, which means supplies that are excess to what is required to meet long-term contracts with Asian trading partners.</p>
<p class="articleParagraph enarticleParagraph" >This model would hit GLNG's two rivals. That is because GLNG has not developed enough of its own reserves to fulfil long-term contracts, meaning it has to buy domestic gas to convert into LNG for export, and therefore does not have any spare uncontracted gas to sell.</p>
<p class="articleParagraph enarticleParagraph" >Brisbane-headquartered GLNG withdraws more gas from the domestic market than it puts in, while <span class="companylink">Shell</span>'s QCLNG and <span class="companylink">Origin Energy</span>'s <span class="companylink">Australia Pacific LNG</span> venture (APLNG) produce gas that supplies the local market.</p>
<p class="articleParagraph enarticleParagraph" >A scheme that targeted only uncontracted exports would leave APLNG and QCLNG providing all the gas for the reservation scheme.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Shell</span>'s QCLNG rejected moves to nobble the export-permitting scheme and said the burden should fall evenly across all three gas ventures.</p>
<p class="articleParagraph enarticleParagraph" >"All three exporters should contribute equally to the reservation, right from the start," a <span class="companylink">Shell</span> spokeswoman said.</p>
<p class="articleParagraph enarticleParagraph" >Management at APLNG also hit back at Santos' lobbying. "All exporters should contribute to Australia's domestic gas supply with no exceptions and no loopholes," an APLNG spokeswoman said.</p>
<p class="articleParagraph enarticleParagraph" >AI Group has backed an export-permitting scheme, which does not force gas producers to break existing supply contracts.</p>
<p class="articleParagraph enarticleParagraph" >However, Willox urged government to resist any moves by gas companies to tweak the scheme to shift the cost burden, arguing gas prices had risen due to the creation of an export industry that tied Australian gas to international prices.</p>
<p class="articleParagraph enarticleParagraph" >"The obligations should sit very clearly with gas exporters. They have transformed the market and control the vast bulk of gas in the ground. They should bear the onus for ensuring that Australian gas users and the broader economy aren't left high and dry," he said.</p>
<p class="articleParagraph enarticleParagraph" >Manufacturing Australia, a coalition of chief executives of companies including <span class="companylink">BlueScope Steel</span>, <span class="companylink">Brickworks</span>, Tomago Aluminium and Dulux Group, said prices must fall urgently to keep manufacturers in business, and urged the government to target uncontracted gas and new supplies for a local reservation.</p>
<p class="articleParagraph enarticleParagraph" >"Gas reservation will only work if we put enough gas into the local market to see prices fall," Manufacturing Australia chief executive officer Ben Eade said.</p>
<p class="articleParagraph enarticleParagraph" >Santos was contacted for comment.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c312 : Corporate/Industry Exports | c314 : Prices | ccat : Corporate/Industrial News | cdom : Markets/Marketing | gcat : Political/General News | gpir : Politics/International Relations | gpol : Domestic Politics | ncat : Content Types | npag : Page One Stories</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>apacz : Asia Pacific | ausnz : Australia/Oceania | austr : Australia | victor : Victoria (Australia)</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Fairfax Media Management Pty Limited</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document SAGE000020251206elc700009</td></tr></table><br/></div></div><br/><span></span><div id="article-PARALL0020251206elc7001xi" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/parallLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>MIL-OSI Video: How AI is preserving a Māori language & | WEF | Top Stories of the Week</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>352 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>ForeignAffairs.co.nz</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>PARALL</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025.  Multimedia Investments Ltd.  All rights reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Source: <span class="companylink">World Economic Forum</span> (video statements)</p>
<p class="articleParagraph enarticleParagraph" >0:14 - This Māori leader trained AI to speak his language and preserve its wisdom: Peter-Lucas Jones, CEO of Te Hiku Media, a Māori media company, explains how they built an AI to help transcribe 30 years of Māori-language archival recordings, in which Indigenous elders passed down their priceless knowledge and customs.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >4:11 - Think financial education isn’t for everyone? 3 money myths busted by the experts: In an age of rising living costs, understanding how to manage money is a critical life skill. Three financial experts - Ana Mahony of AdditionWealth, Saira Malik of Nuveeninv and Oluwatosin Olaseinde of MoneyAfrica - bust common money myths.</p>
<p class="articleParagraph enarticleParagraph" >8:06 - Quantum technology could help make aircraft stronger and safer: Quantum holds promise in a number of areas of product design, from developing corrosion-resistant alloys in aerospace engineering to accelerating the discovery of new medication in pharmaceutical manufacturing.</p>
<p class="articleParagraph enarticleParagraph" >11:31 - This solution could be an alternative to antibiotics. AI is making it possible: Phages are viruses that attack bacteria. French start-up Phagos is using AI to help find the right phages, with promising results for the fight against AMR. Here, co-founders Adèle James and Alexandros Pantalis explain how it works. Phagos is a <span class="companylink">World Economic Forum</span> 2025 Technology Pioneer.</p>
<p class="articleParagraph enarticleParagraph" >**************************************************************************</p>
<p class="articleParagraph enarticleParagraph" >The <span class="companylink">World Economic Forum</span> is the International Organization for Public-Private Cooperation. It provides a global, impartial and not-for-profit platform for meaningful connection between stakeholders to establish trust, and build initiatives for cooperation and progress.</p>
<p class="articleParagraph enarticleParagraph" >Find out more below:</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">World Economic Forum</span> Website ► <span class="colorLinks">http://www.weforum.org/ [http://www.weforum.org/]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">YouTube</span> ► <span class="colorLinks">https://www.youtube.com/wef [https://www.youtube.com/wef]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">LinkedIn</span> ► <span class="colorLinks">https://www.linkedin.com/company/world-economic-forum [https://www.linkedin.com/company/world-economic-forum]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Facebook</span> ► <span class="colorLinks">https://www.facebook.com/worldeconomicforum/ [https://www.facebook.com/worldeconomicforum/]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Instagram</span> ► <span class="colorLinks">https://www.instagram.com/worldeconomicforum/ [https://www.instagram.com/worldeconomicforum/]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >X ► <span class="colorLinks">https://twitter.com/wef [https://twitter.com/wef]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >TikTok ► <span class="colorLinks">https://www.tiktok.com/@worldeconomicforum [https://www.tiktok.com/@worldeconomicforum]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">WhatsApp</span> ► <span class="colorLinks">https://www.whatsapp.com/channel/0029VaDcHBKGZNCihKxwiD0L [https://www.whatsapp.com/channel/0029VaDcHBKGZNCihKxwiD0L]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >Threads ► <span class="colorLinks">https://www.threads.com/@worldeconomicforum [https://www.threads.com/@worldeconomicforum]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Flipboard</span> ► <span class="colorLinks">https://flipboard.com/@WEF [https://flipboard.com/@WEF]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >#WorldEconomicForum #wef</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">https://www.youtube.com/watch?v=m0Qvx0W1yhA [https://www.youtube.com/watch?v=m0Qvx0W1yhA]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">MIL OSI Video [https://milnz.co.nz/mil-osi-aggregation/]</span> -</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>wecof : World Economic Forum</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302022 : Artificial Intelligence Technologies | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gpir : Politics/International Relations | gpol : Domestic Politics</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>Africa,AM-NC,Artificial Intelligence,Aviation,Business,CTF,DJF,Economy,France,KB,Machine Learning,MIL-OSI,MIL-OSI-Video,Technology,Video</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Multimedia Investments Ltd</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document PARALL0020251206elc7001xi</td></tr></table><br/></div></div><br/><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WC57785020251202elc70000u"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  GTA 6 animation leak surfaced from Ex-Rockstar animator’s demo reel</b><div class="leadFields"><a href="javascript:void(0)">India TV News</a>, 02:00 PM, 7 December 2025, 453 words, (English)</div><div class="snippet ensnippet"> New GTA 6 animation footage has leaked online through the demo reel of a former Rockstar Games animator, Ben Chue. The clip shows early animation tests, including bicycle interactions and character movement. Although the footage does not ...</div>
<div>(Document WC57785020251202elc70000u)</div><br/></td></tr>
						</table>
					</div>
				</div><span></span><div id="article-BTDY000020251202elc700003" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/btdyLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>Deep Dive</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>With a new campus in Dubai and a course on AI, IIMA is preparing the leaders of the future</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>George Skaria </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>768 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Business Today</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>BTDY</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025. Living Media India Ltd </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >With a campus in Dubai and initiatives such as an AI programme, IIMA is looking to equip its students with the tools to navigate an uncertain world.</p>
<p class="articleParagraph enarticleParagraph" >The indian Institute of Management, Ahmedabad (IIMA), has topped the BT-MDRA India's Best B-Schools rankings. But it is an unenviable position: even though it became a proud winner, it did so with a slender margin relative to the next on the list, had to face many challenges through the year like economic uncertainty due to global conflicts and growth of artificial intelligence (AI), and is staring at a landscape where management education and the nature of companies will be sharply rewritten.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Bharat Bhasker, Director of IIMA, says, "We are in a good shape and trying to keep up with the changing technology, but the pace [of adaptation] has to be enhanced. We have to work doubly hard to catch up and remain relevant to the business environment which is unfolding in front of us."</p>
<p class="articleParagraph enarticleParagraph" >New initiatives</p>
<p class="articleParagraph enarticleParagraph" >To be sure, IIMA took a host of new initiatives: a one-year MBA programme from Dubai, a blended MBA in business analytics and an AI programme, a restructured syllabus introducing new courses with specific focus on technology and finance, and innovative industry partnerships like the one with <span class="companylink">Novo Nordisk</span> to strengthen research on a critical subject like managing the obesity ecosystem.</p>
<p class="articleParagraph enarticleParagraph" >Professor Bhasker says the Dubai campus has a larger objective of amplifying India's soft power. IIMA believes that Dubai is a global future city with particular focus on AI, finance and tourism. Further, it is easier for the school to reach out to other geographical regions like Africa, Northern Europe and countries that are part of the <span class="companylink">Commonwealth of Independent States</span>, like Belarus and Azerbaijan.</p>
<p class="articleParagraph enarticleParagraph" >Despite the current uncertainties in the global economic environment, IIMA achieved 100% placements, with an average salary of Rs 34.45 lakh per annum and the highest compensation at Rs 1.46 crore. Around 168 firms participated in the placement process. The sectors from where the companies recruited were the traditional ones of consulting, investment banking and fintech, while conglomerates like <span class="companylink">Adani Enterprises</span> and the <span class="companylink">Essar Group</span> hired students from the general management MBA category.</p>
<p class="articleParagraph enarticleParagraph" >Interestingly, five students opted for the IIMAvericks fellowship, under which the school will mentor and support them financially for two years to set up entrepreneurial ventures.</p>
<p class="articleParagraph enarticleParagraph" >The innovative "Dream Application" policy, which empowers students to pursue their preferred sectors and roles, continued to be a highlight of the process, with 132 such applications recorded during the placement season.</p>
<p class="articleParagraph enarticleParagraph" >One of the most important ways through which IIMA is looking to create leaders who can thrive in an uncertain environment is to enhance the quality of its engagements with industry. Traditionally, this was done through summer internship and the placement process.</p>
<p class="articleParagraph enarticleParagraph" >For example, its intensive faculty development programmes seek to update them with knowledge of the new technologies. Further, the institution is focusing more on executive education programmes and consulting by the faculty to understand better the challenges that companies are facing and transfer that knowledge to the students.</p>
<p class="articleParagraph enarticleParagraph" >Changes afoot</p>
<p class="articleParagraph enarticleParagraph" >The impact of the changing environment in India and abroad is evident on IIMA. A pioneer of the two-year MBA, it has started its Dubai campus with a one-year MBA. The rationale was that there is an urgent need for good managerial talent there and, therefore, the sooner students get a degree the faster they will be able to get jobs. IIMA is also taking a relook at its well-regarded case studies, which typically used to run into about 30-40 pages. Now, with immersive technologies, students will be able to absorb cases in depth in a shorter print version of, say, 10 pages.</p>
<p class="articleParagraph enarticleParagraph" >The Common Admission Test, which all the IIMs and many other private business schools use to select the students, could be less preferred in the future because of the intense competition that it generates. Many B-schools and students now prefer GRE or GMAT. Competition in the market will also grow because foreign higher education institutions are slowly coming to India. Others have their presence in India online and physically especially in the shorter executive education programmes, the growth of private sector-backed business schools and the setting up of multiple campuses.</p>
<p class="articleParagraph enarticleParagraph" >So, even as IIMA is the highest ranked B-school in this year's BT-MDRA survey, it has its work cut out for the coming years.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302022 : Artificial Intelligence Technologies | itech : Technology</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcsci : Computer Science | gedu : Education | gsci : Sciences/Humanities</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>asiaz : Asia | devgcoz : Emerging Market Countries | dubai : Dubai | dvpcoz : Developing Economies | india : India | meastz : Middle East | sasiaz : South Asia | uae : United Arab Emirates</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>T.V. Today Network Ltd.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document BTDY000020251202elc700003</td></tr></table><br/></div></div><br/><span></span><div id="article-BTDY000020251202elc700001" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/btdyLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>Interview</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Management education must move faster to stay relevant: IIMA's Bharat Bhasker</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>Siddharth Zarabi </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1322 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Business Today</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>BTDY</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025. Living Media India Ltd </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Bharat Bhasker, Director, IIM Ahmedabad, on why Indian management education must begin to echo what is happening at the workplace.</p>
<p class="articleParagraph enarticleParagraph" >Indian Institute of Management Ahmedabad (IIMA) leads BT-MDRA's 26th annual ranking of India's Best B-Schools, reaffirming its status as the country's premier management institution. For its Director, Bharat Bhasker, the real story is the race to keep management education relevant amid tectonic shifts in technology and global business. Edited excerpts from an interview with BT:</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Q: How would you describe the state of management education in India?</p>
<p class="articleParagraph enarticleParagraph" >A: Our management education ecosystem is in good shape, but the changes that are happening are quite drastic in nature. Technologies like artificial intelligence (AI), blockchain, robotics and autonomous systems will significantly impact workplaces. Management education must immediately begin to echo what is happening in the actual workplace, because ultimately, you are creating leaders for the future.</p>
<p class="articleParagraph enarticleParagraph" >A key challenge is how quickly we can transform the current curriculum into a new one, which incorporates and reflects the changing reality. We must also prepare graduates for uncertainty in global trade practices and shifting supply chains.</p>
<p class="articleParagraph enarticleParagraph" >Our management schools must immediately accelerate adaptation. We are in good shape and are trying to keep pace with the changing technology over time, but the pace must be accelerated to remain relevant.</p>
<p class="articleParagraph enarticleParagraph" >Q:The syllabus of IIMA has undergone a major transformation. What has changed?</p>
<p class="articleParagraph enarticleParagraph" >A: Our curriculum is designed to transform graduates into business leaders, reflecting industry realities. A major mechanism is the case study method, which mirrors real scenarios; adopting the latest cases brings industry reflection into the classroom.</p>
<p class="articleParagraph enarticleParagraph" >More importantly, technology often moves faster than industry adoption. We prepare our students to become business leaders who lead the industry in technology adoption and drive change.</p>
<p class="articleParagraph enarticleParagraph" >Over the past year, we have introduced technology-oriented courses, including AI in human resources, AI-driven fintech, and technology-driven global supply chain management.</p>
<p class="articleParagraph enarticleParagraph" >We integrate these shifts, and our students are being prepared to absorb all that information and be ready for the future business environment. Sometimes industry leads us; sometimes we lead industry by preparing students who will take new technologies into organisations.</p>
<p class="articleParagraph enarticleParagraph" >Q: Overall, is Indian management education well-positioned for the transition that is underway?</p>
<p class="articleParagraph enarticleParagraph" >A: There are layers in the system. The top institutes are preparing well and transforming quickly. Others are lagging and would take longer to adapt. We are well-positioned, but the transition must include the entire ecosystem. Top institutes must help bring others along so the broader economy benefits, not only high-end industry.</p>
<p class="articleParagraph enarticleParagraph" >Q: What is your sense of job placements this year, and how can industry and academia respond to any dips?</p>
<p class="articleParagraph enarticleParagraph" >A: Industry engagement should not be limited to placements, as they are only an outcome. Engagement must begin during the transformation stage, ensuring students understand current industry practices. That is why our core curriculum is taught by faculty and electives by numerous industry practitioners.</p>
<p class="articleParagraph enarticleParagraph" >Faculty must remain updated on industry practices. Research advances knowledge, but faculty must also understand how new technologies affect organisations. We engage deeply with industry through executive education and consulting. In consulting, faculty work closely with companies, understand their challenges and develop solutions—gaining practical insight on applying theory.</p>
<p class="articleParagraph enarticleParagraph" >In executive education, I don't think industry people come to learn from us. We do impart education to them, but at the same time, we learn a lot from industry people because in interactive discussions in classrooms, they bring out the nuances of what is happening in the industry.</p>
<p class="articleParagraph enarticleParagraph" >Industry engagement must, therefore, be holistic—from teaching to consulting to executive education—with knowledge flowing back into the curriculum. Placements as an outcome will automatically happen if the institute is involved in an integrated fashion with industry. Technology often moves faster than industry adoption. We prepare our students to become business leaders who lead the industry in technology adoption and drive change. We have introduced courses like AI in human resources. -Bharat Bhasker, Director, IIM Ahmedabad</p>
<p class="articleParagraph enarticleParagraph" >Q: What will the management classroom of the future look like?</p>
<p class="articleParagraph enarticleParagraph" >A: Even before Covid, technology made blended and online classrooms feasible. The pandemic only accelerated the adoption. Blended learning will grow for two reasons.</p>
<p class="articleParagraph enarticleParagraph" >First, a growing economy cannot rely only on training fresh graduates. People already in the industry must be prepared for new technologies, management practices, and transitions from technical to managerial roles. Working with mid- and senior-management professionals has always been important, and technology now removes many physical-meeting constraints. Executive education increasingly uses hybrid formats where leaders spend some time on campus and learn the rest while working.</p>
<p class="articleParagraph enarticleParagraph" >Second, blended learning is a force multiplier. A move from a $5-trillion to a $30-trillion economy would require a multiple-fold increase in managerial capacity. Residential programmes alone cannot meet this scale.</p>
<p class="articleParagraph enarticleParagraph" >That is why we launched the Blended Post Graduate Programme in Management (BPGP), a blended MBA-equivalent programme for working professionals, which is now in its second batch.</p>
<p class="articleParagraph enarticleParagraph" >We are also launching an MBA in Business Analytics and AI, because the modern manager must be technology-savvy.</p>
<p class="articleParagraph enarticleParagraph" >Blended learning is essential to meet India's scale and leadership needs.</p>
<p class="articleParagraph enarticleParagraph" >Q: Who is an ideal student for IIM Ahmedabad? What profile, background, skills, and work experience matter most?</p>
<p class="articleParagraph enarticleParagraph" >A: Ideal work experience is easier to define: a couple of years in industry, so students understand organisational dynamics. Fresh graduates often struggle with this.</p>
<p class="articleParagraph enarticleParagraph" >Indian Institute of Technology (IIT) graduates are welcome; they have proven ability, but the ideal student is not limited to IIT. Today, you don't need to go into the depths of a technological development. Technology is accessible today; what matters is one's ability to apply it.</p>
<p class="articleParagraph enarticleParagraph" >Our motto Vidya Viniyoga Vikasa means development through the application of knowledge. The ideal student has an open mindset, willing to engage with technology and apply knowledge for development, regardless of whether they come from commerce, science or an arts background.</p>
<p class="articleParagraph enarticleParagraph" >Q: Does the Common Admission Test (CAT) exam help you select such students?</p>
<p class="articleParagraph enarticleParagraph" >A: Only to an extent. The exam acts as a filter. After shortlisting, we assess the mindset through interviews, group discussions and case study writing. CAT tests analytical and verbal abilities, as the key requirement is an analytical mindset for solving business problems. Blended learning is a force multiplier. A move to a $30-trillion economy would require a multiple-fold increase in managerial capacity. Residential programmes alone cannot meet this scale. -Bharat Bhasker, Director, IIM Ahmedabad</p>
<p class="articleParagraph enarticleParagraph" >Q: How do you view the multiple-campus model now that IIMA has a Dubai campus?</p>
<p class="articleParagraph enarticleParagraph" >A: India must show its capabilities and lead the Global South. Our philosophy emphasises collective development.</p>
<p class="articleParagraph enarticleParagraph" >When the Global South grows, India grows. Dubai fits into a deliberate strategy: enabling the Global South to benefit from our capabilities while strengthening India through shared education and future trade. Multi-country campuses allow us to understand regional business contexts, write case studies from those markets and bring that learning back to India. We aim to prepare leaders for global business, not only in India.</p>
<p class="articleParagraph enarticleParagraph" >Q: How do you view the entry of foreign universities in India under the new education policy?</p>
<p class="articleParagraph enarticleParagraph" >A: I welcome them. India's educational capacity cannot meet the scale of growth we foresee. We need far more engineering and management graduates than Indian institutions alone can produce. Foreign universities expand the pool and help prepare talent for the emerging economy.</p>
<p class="articleParagraph enarticleParagraph" >Given the limited quality seats, many students go abroad. If foreign universities operate here, students receive comparable education at a lower cost, the currency stays in India, and parents benefit.</p>
<p class="articleParagraph enarticleParagraph" >But quality must match that of the parent campus. Regulators must ensure only strong institutions and faculty enter. If quality is maintained, foreign universities are a win-win. ￼</p>
<p class="articleParagraph enarticleParagraph" >@szarabi</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c41 : Management | ccat : Corporate/Industrial News | gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcsci : Computer Science | gedu : Education | gsci : Sciences/Humanities | ncat : Content Types | nfact : Factiva Filters | nfcpex : C&E Executive News Filter | nfcpin : C&E Industry News Filter | niex : Interviews with Corporate Executives | nitv : Interviews</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>ahemda : Ahmedabad | asiaz : Asia | devgcoz : Emerging Market Countries | dvpcoz : Developing Economies | gujar : Gujarat | india : India | sasiaz : South Asia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>T.V. Today Network Ltd.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document BTDY000020251202elc700001</td></tr></table><br/></div></div><br/><span></span><div id="article-BTDY000020251202elc700002" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/btdyLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>Deep Dive</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>
                           IIMC emerges strong despite a challenging economic landscape</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>George Skaria </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>660 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Business Today</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>BTDY</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025. Living Media India Ltd </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >
                        <span class="companylink">IIMC</span> stays its ground despite job market pressures, launches cutting-edge courses in AI, corporate sustainability, and private equity.</p>
<p class="articleParagraph enarticleParagraph" >The more things change, the more they remain the same. This adage by the 19th-century French critic, author, and journalist Jean-Baptiste Alphonse Karr seems apt in the case of Indian Institute of Management-Calcutta (<span class="companylink">IIMC</span>) when comparing its progress this year in the BT-MDRA Best B-Schools Survey to the previous year. With U.S. President Donald Trump's tariffs on several countries, including India, and the growing use of artificial intelligence (AI) leading to job cuts, business schools were expected to feel the impact on placements, admissions, and recruitment trends. <span class="companylink">IIMC</span> has been able to hold its ground.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >"As the global business environment grows increasingly dynamic, IIM Calcutta leads with purpose—fostering innovation, nurturing responsible leadership, and preparing professionals to thrive in an ever-evolving world," says Professor Alok Kumar Rai, Director, <span class="companylink">IIMC</span>.</p>
<p class="articleParagraph enarticleParagraph" >The consulting sector led the hiring process in academic year (AY) 2024-25. Further, there was an increase in the number of Indian start-ups participating in the hiring process. The median salary increased from Rs 30 lakh per annum in the previous academic year to Rs 34 lakh.</p>
<p class="articleParagraph enarticleParagraph" >Frequent recruiters, from <span class="companylink">Google</span>, <span class="companylink">Amazon</span> and ITC to the lesser-known ones like TrueTech and Saifee Hospital, continued hiring. The approved intake of the MBA programme remained the same at 480, though. In AY 2024-25, it introduced 14 new courses; the Executive Education (ExEd) division delivered 167 programmes that engaged over 8,000 professionals across open, customised, and consultancy formats.</p>
<p class="articleParagraph enarticleParagraph" >AY 2024-25 was also the year that saw the launch of cutting-edge courses in AI, corporate sustainability, and private equity. It was also a year of new initiatives, sponsorships, and expanded industry networks, marked by a significant increase in industry engagement, doubling the number of interactions to over 105 industry experts compared to the previous year. There was also a rise in inclusivity-based courses such as Climate Change, Managing Diversity & Inclusivity, and Responsible AI. Further, two new modules were introduced in international immersion programmes, including a live project on Hydrogen in SDA Bocconi Milan, Italy (main campus), and entrepreneurship in <span class="companylink">ESADE</span>, Barcelona, Spain. Despite these value additions, the fees remained unchanged at Rs 31 lakh. These developments align with <span class="companylink">IIMC</span>'s traditional strengths, like a loyal network of 45,000 alumni built over six decades, a vibrant campus culture, and a consistent rise in global rankings.</p>
<p class="articleParagraph enarticleParagraph" >"IIMs, like IITs, have a robust entrance exam, which is followed by a rigorous group discussion. That ensures good control over the quality of students who come into an institution like <span class="companylink">IIMC</span>. Additionally, the quality of faculty, including those who have come from abroad, plays an important role," says Mathew Eipe, an alumnus of the 1977 batch and former Executive Director (retired) of <span class="companylink">Godrej Industries</span>.</p>
<p class="articleParagraph enarticleParagraph" >Eipe (who also holds an engineering degree from IIT Bombay) said that the combination of an engineering degree and an MBA is perhaps more common at <span class="companylink">IIMC</span> than at some of the other IIMs. Successful business leaders with an MBA-engineering combination from IIM-C include Sumant Sinha (Renew), K. Ganesh (GrowthStory, Big Basket, Portea) and T.V. Narendran (<span class="companylink">Tata Steel</span>).</p>
<p class="articleParagraph enarticleParagraph" >Despite all the cheers, the institution needs to consider a few indicators going forward. This year, in the learning experience parameter in the BT-MDRA study, <span class="companylink">IIMC</span> was edged out of the top position, slipping to second place from last year's number one ranking. Similarly, in the placements parameter, IIM-C, which held the top position last year, has moved down to second place this year. One reason for this shift is that IIM Ahmedabad did not participate in the survey last year. Professor Alok Kumar Rai took over as the Director just four months back. The coming year will show who wins the catch-up game.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i983 : Educational Services | i9831 : Business Schools | ibcs : Business/Consumer Services</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>ccat : Corporate/Industrial News | gcat : Political/General News | gjob : Labor Issues</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>asiaz : Asia | devgcoz : Emerging Market Countries | dvpcoz : Developing Economies | india : India | sasiaz : South Asia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>T.V. Today Network Ltd.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document BTDY000020251202elc700002</td></tr></table><br/></div></div><br/><span></span><div id="article-BTDY000020251202elc700004" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/btdyLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>Cover Story</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>BT-MDRA India's Best B-Schools Ranking: The Best Stay the Course</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>George Skaria </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1550 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>7 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Business Today</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>BTDY</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025. Living Media India Ltd </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >B-Schools rise up to meet the challenge of a hiring slowdown and ai by focusing on research, real-world case studies and simulations, and more structured industry collaboration.</p>
<p class="articleParagraph enarticleParagraph" >It is perhaps fitting that the two oldest IIMs, IIM Ahmedabad and IIM Calcutta, have come up tops by claiming first and second ranks, respectively, in the BT-MDRA India's Best B-Schools Survey in which 270 business schools participated. It was a close call: the difference between the two schools was just 0.6 points. The others in the Top Five are IIM Lucknow, SP Jain Institute of Management and Research and IIM Indore. In fact, there has been just a small reshuffle in the top ten pack. Further, in the top ten, the majority—six—are government-owned IIMs, while the other four are privately-owned and managed.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >The ostensible lack of action at the top hides the varied undercurrents in India's B-school landscape captured by the BT-MDRA study. For one, B-school education continues to be in heavy demand as students look to encash its value as a ticket to a good corporate career. The average batch strength of top 100 B-schools rose from 1,076 in 2024 to 1,173 in 2025. "The demand for business education continues to grow steadily, and our numbers reflect this trend. For academic year 2025–26, 61,595 students applied to NMIMS, an increase from the previous year," says Papiya De, Program Chairperson, SVKM's Narsee Monjee Institute of Management Studies, Mumbai.</p>
<p class="articleParagraph enarticleParagraph" >That said, the challenges remain, with global uncertainty, including trade wars, and fast adoption of technology such as AI, making companies wary of hiring. "On the placement front, we are aligned with global and national trends. The job market has remained muted over the past year, and like many B-schools, we have observed companies reducing the number of internships offers. This is not a reflection on student quality but a broader hiring slowdown," says Papiya De.</p>
<p class="articleParagraph enarticleParagraph" >The dip in placements is showing in salaries as well. The average salary at the Top 25 B-schools dipped from Rs 23.12 lakh in 2023 to Rs 22.7 lakh in 2025. Average salaries for graduates from the Top 25 B-schools have recorded the slowest growth (16%) in five years. This, along with a much larger fee increase, has lowered the return on investment of a B-school degree. The average course fee of Top 25 colleges rose from Rs 18.78 lakh to Rs 20.17 lakh. It has risen nearly 23% in the last five years.</p>
<p class="articleParagraph enarticleParagraph" >Is this a blip? For some, yes. Despite the above challenges, McKinsey seems to be gung-ho about hiring India's B-school graduates; in the past three years, 77% of its hiring in India has been from leading B-schools. It has expanded the MBA summer hiring by 41% over the last two years. MBA graduates have long been an integral part of its talent pipeline. The firm says they bring in a wide variety of competencies: problem-solving acumen, leadership potential, and diverse professional experience along with analytical rigour, collaborative mindset, and global perspective.</p>
<p class="articleParagraph enarticleParagraph" >Given the challenges of a world in flux, what practices are B-schools adopting to thrive and make a mark?</p>
<p class="articleParagraph enarticleParagraph" >One MBA</p>
<p class="articleParagraph enarticleParagraph" >The demand for a good management education encouraged many B-schools such as NMIMS, IIMA, Great Lakes Institute and SP Jain to open multiple campuses. But they often struggle to tap the full synergies between them. The School of Management, NMIMS, is out to change that. It has made its flagship MBA programme into an integrated offering designed to give students across campuses a seamless experience. Its 'One MBA' initiative is more than a shared syllabus. It is a structured system in which faculty members from every campus come together to jointly design, deliver, and evaluate the course. The initiative is anchored by the Mumbai campus, which hosts weekly faculty meetings and collaborative workshops where course plans, teaching approaches, evaluation methods, and even classroom experiences are aligned. As a result, not only is the curriculum common, but so are the assessments, right down to the question papers.</p>
<p class="articleParagraph enarticleParagraph" >The school believes this model serves many purposes. Academically, the quality of learning is not determined by the campus where a student is studying. Strategically, it strengthens collaboration among faculty by enabling cross-learning and shared ownership of outcomes. "In essence, One MBA is not just a process, it is a philosophy to democratise excellence, to build a cohesive academic community, and to uphold a single promise across campuses. Every MBA student of NMIMS receives the same experience. This multi-campus coordination programme idea, championed by its Vice Chancellor, operates on a simple but powerful principle: one programme, one standard, multiple locations," says Papiya De.</p>
<p class="articleParagraph enarticleParagraph" >Human Resources</p>
<p class="articleParagraph enarticleParagraph" >India Inc is increasingly adopting analytics in decision-making and embedding AI in all aspects of the employee life cycle including hiring. That organisations today hire for skills rather than for roles has put a sharper focus on continuous skill development. Retaining talent becomes a lot easier when employees see paths to grow within an organisation and can expand their skill sets. Companies are, therefore, providing employees opportunities to unlearn and upskill themselves, either through funding these programmes or running them in the organisation. B-schools are making efforts to tap the trend. The number of management development programmes (MDPs) by the Top 25 B-schools rose from 57 in 2024 to 68 in 2025.</p>
<p class="articleParagraph enarticleParagraph" >B-schools are realising that traditional classroom learning must be supplemented with practical experiences such as real-world case studies and simulations, a more structured industry collaboration and ongoing mentorship with industry leaders. This will make the transition from campus to corporate life easier.</p>
<p class="articleParagraph enarticleParagraph" >"B-schools need to adapt their curriculum to keep up with the requirements of corporates. In the age of AI, there has to be a growing focus on developing human-centric leadership qualities like empathy, conflict management, emotional intelligence and resilience. These need to be in built in the curriculum," says Lopamudra Banerjee, CHRO, <span class="companylink">Carrier Midea India</span> (CMI).</p>
<p class="articleParagraph enarticleParagraph" >CMI believes B-schools also need to design the curricula to address the complexities of global HR management, international labour laws, sustainability, and ethical decision-making to prepare leaders for a responsible role in a complex business environment. The company hires a large part of entry-level managers from engineering colleges and B-schools.</p>
<p class="articleParagraph enarticleParagraph" >Practice Plus Research</p>
<p class="articleParagraph enarticleParagraph" >Indian B-schools have come a long way in research—as many as 95% permanent faculty among Top 25 B-schools has a PhD, according to the BT-MDRA study. Investing in cutting-edge research and faculty is the key for a bump in global rankings, say experts.</p>
<p class="articleParagraph enarticleParagraph" >Also important is academia-industry linkages. Take the case of Anilesh Seth. After a BTech from <span class="companylink">IIT Madras</span> and an MBA from IIMB, Seth, with about 40 years of corporate experience spanning mostly IT services companies and global capability centres (GCCs), decided to do a PhD. Research had always fascinated him. Over a breakfast conversation, a professor at <span class="companylink">Indian School of Business</span> asked if he was interested in a PhD. He found himself applying for the Executive Fellow Program in Management. The fee was relatively steep: around Rs 56 lakh. He had left his corporate career in 2000 and had been consulting and teaching ever since. The doctorate from <span class="companylink">ISB</span> added significant credibility to his profile. "I believe it has aided in opening several opportunities for me," he says. "I chose to do my research in the very contemporary topic of GCCs. I have been working in this field for the past 25 years. The research has added to both my credibility as well as my ability to view the field with a 'pracademic' lens. I have used my research on several occasions in my consulting assignments as well as in the GCC leadership programme that I co-designed at IIM Bangalore," says Seth.</p>
<p class="articleParagraph enarticleParagraph" >Rethinking Leadership</p>
<p class="articleParagraph enarticleParagraph" >Amid the focus on challenges being faced by companies, business school leadership often forgets the need to rethink its role. While Peter Drucker conceptualised many foundational theories of management thinking, in recent years, his vision has largely been forgotten by companies with managements focusing on shareholder capitalism. But in the new geopolitical and geoeconomic environment, there have been some calls to bring businesses and business education back to Drucker's vision that companies serve as a vital force towards global peace and prosperity.</p>
<p class="articleParagraph enarticleParagraph" >CMI's Lopamudra sums up. "Ethical leadership isn't just about doing what's right—it's about doing what's right when it's hard. In today's competitive world, leaders are often faced with decisions that test not only their business acumen but also their moral compass. Ethical leadership means making choices that prioritise integrity, fairness, and respect—even when no one is watching. Can organisations be both ethical and agile? Can we build cultures where doing the right thing isn't just encouraged but its expected?"</p>
<p class="articleParagraph enarticleParagraph" >If that is the goal of businesses and business leaders today, B-schools also need to rethink their goals. This is the moment to fix B-schools.</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i983 : Educational Services | i9831 : Business Schools | ibcs : Business/Consumer Services</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>gcat : Political/General News | gedu : Education | gjob : Labor Issues | gscho : School | ncat : Content Types | npag : Page One Stories | nran : Rankings</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>asiaz : Asia | devgcoz : Emerging Market Countries | dvpcoz : Developing Economies | india : India | sasiaz : South Asia</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>T.V. Today Network Ltd.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document BTDY000020251202elc700004</td></tr></table><br/></div></div><br/><span></span><div id="article-DAYMO00020251206elc60000q" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/daymoLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>National</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>Govt announces Shs100b grants to boost climate-smart agriculture</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>Khalil Ibrahim Manzil </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>393 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>6 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>Daily Monitor</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>DAYMO</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025 Nation Media Group. All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >The Uganda government has launched 24 competitive research grants worth around Shs100 billion ($350 million), aimed at strengthening the country’s scientific capacity and improving the resilience of its food systems amid escalating climate challenges.</p>
<p class="articleParagraph enarticleParagraph" >The grants, part of the Uganda Climate-Smart Agricultural Transformation Project (UCSATP), focus on drought-tolerant crops, AI-powered pest prediction systems, renewable energy technologies, soil fertility improvement, and livestock productivity.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >They are implemented by the National Agricultural Research Organisation (NARO) in partnership with the <span class="companylink">Ministry of Agriculture, Animal Industry and Fisheries</span> (MAAIF) and supported by the <span class="companylink">World Bank</span>.</p>
<p class="articleParagraph enarticleParagraph" >Speaking at the launch at the National Agriculture Crop Resources Institute (NaCRRI) in Namulonge (Wakiso District) on December 5, the Minister of State for Agriculture, Fred Kyakulaga Bwino, warned that climate change is already reshaping Uganda’s agricultural landscape.</p>
<p class="articleParagraph enarticleParagraph" >“The seasons are unpredictable these days. Sometimes the rainy season comes late, and the dry season arrives early. Meanwhile, pests and diseases are emerging with new aggression—these are real challenges,” he noted.</p>
<p class="articleParagraph enarticleParagraph" >Kyakulaga described the research grants as “hope in practical form,” noting that the innovations funded under the program will help farmers adapt to drought, floods, and rising temperatures.</p>
<p class="articleParagraph enarticleParagraph" >Addressing students pursuing advanced agricultural studies, Dr Bosco Obua, Acting Director of Research and Graduate Training at Kyambogo University, stressed the importance of completing programs on time.</p>
<p class="articleParagraph enarticleParagraph" >“Most students struggle with tuition. They may pay for the first semester, then fail to raise money for the second. Unfortunately, the Auditor General criticizes public universities for retaining students longer than expected, yet the real issue is the students’ inability to afford continuous enrollment,” he said.</p>
<p class="articleParagraph enarticleParagraph" >Dr Julia Kigozi, Dean of the School of Food Technology at <span class="companylink">Makerere University</span>, assured stakeholders that universities are improving systems to ensure timely graduation.</p>
<p class="articleParagraph enarticleParagraph" >“We are committed to ensuring that students complete their programs on time so that vacancies are available for new entrants. We are improving systems to make sure students submit regular updates about their academic progress. This has been a challenge, but it is being resolved,” she explained.</p>
<p class="articleParagraph enarticleParagraph" >The newly launched grants are expected to accelerate innovation, support young researchers, and ultimately strengthen Uganda’s capacity to adapt to climate change while safeguarding national food security.</p>
<p class="articleParagraph enarticleParagraph" >&gt;&gt;&gt;Stay updated by following our WhatsApp and Telegram channels;</p>
<p class="articleParagraph enarticleParagraph" >Daily Monitor Telegram channel</p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>ugmaaf : Uganda Ministry of Agriculture, Animal Industry and Fisheries</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c13 : Regulation/Government Policy | c341 : Government Aid/Grants | ccat : Corporate/Industrial News | ncat : Content Types | nfact : Factiva Filters | nfcpin : C&E Industry News Filter</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>africaz : Africa | dvpcoz : Developing Economies | eafrz : East Africa | uganda : Uganda</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Nation Media Group Limited</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document DAYMO00020251206elc60000q</td></tr></table><br/></div></div><br/><span></span><div id="article-MRKWC00020251203elc300231" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/mrkwcLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>SE</b>&nbsp;</td><td>Investing</td></tr>
<tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>AI needs power desperately. Here's how to invest in companies profiting from the pain. The shortage is a lucrative opportunity — but the window is brief</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>BY</b>&nbsp;</td><td>By Jurica Dujmovic </td></tr>
<tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>1627 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>6 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>ET</b>&nbsp;</td><td>02:12 PM</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>MarketWatch</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>MRKWC</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025 MarketWatch, Inc.  All Rights Reserved. </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >
                        <img src="../pro/default.aspx?napc=S&_XFORMSTATE=H4sIAAAAAAAEAD2LsQrCMBQA%2fyVzCC95tknfagVRROjiHNNQU2oNqajQ5t%2ftIC4HB3ezpFlCtYKAl8TaNJJN7hZeXoz%2bPdn0DG7w4tQcL1sAUKAKqQD94HA1lIxrYjE9rv3%2fqw%2b7c7P%2flSXKwkhhjEDUulrC3XZe9NF3SxumGD6Mb0hxRaxmHAlyzl9lXl5QlAAAAA%3d%3d"/>

                     </p>
<p class="articleParagraph enarticleParagraph" >AI computing workloads could consume around 500 terawatt-hours annually by 2027 — about twice the U.K.'s total electricity consumption in 2023. PHOTO: <span class="companylink">Getty Images</span>
                     </p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Rising infrastructure costs and mounting capital constraints are <span class="colorLinks">deflating the AI boom. [https://www-marketwatch-com.ezproxy.cul.columbia.edu/story/everyones-asking-the-wrong-question-about-an-ai-bubble-here-are-the-stocks-to-buy-and-when-b3fddce5]</span> The hyperscalers can't solve their computing problems fast enough, and that's creating a rare arbitrage opportunity.</p>
<p class="articleParagraph enarticleParagraph" >The solution right now isn't building data centers. The current investment opportunity lies in the temporary gap between exploding AI demand and the physical constraints of centralized infrastructure expansion. A handful of companies are exploiting this window — which likely will be a 24-to-36- month opportunity. For investors who understand the timing, it's a compelling hedge against the AI infrastructure bottleneck.</p>
<p class="articleParagraph enarticleParagraph" >Physical barriers</p>
<p class="articleParagraph enarticleParagraph" >AI's limiting factor is no longer algorithms or data — it's the brute-force physics of data-center expansion. Training large models demands tens of thousands of GPUs, dedicated networking and enormous power consumption. <span class="companylink">Gartner</span> forecasts that <span class="colorLinks">40% of AI data centers will face power constraints by 2027. [https://datacentremagazine.com/critical-environments/gartner-power-shortages-could-limit-40-of-ai-data-centres]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >The math is brutally simple: AI computing workloads could consume around 500 terawatt-hours annually by 2027 — about twice the U.K.'s total electricity consumption in 2023. This demand spike is already showing up in the grid.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Dominion Energy</span> D, the biggest utility company in Virginia, <span class="colorLinks">nearly doubled [https://www.datacenterdynamics.com/en/news/dominion-energy-nearly-doubles-data-center-capacity-under-contract-to-40gw/]</span> its data-center power capacity under contract between July and December 2024, and the trend has persisted.</p>
<p class="articleParagraph enarticleParagraph" >Even with Microsoft MSFT, Alphabet GOOG GOOGL, Amazon.com AMZN and <span class="companylink">Meta Platforms</span>
                     <span class="companylink">META</span> spending a combined $370 billion on capex in 2025, they can't build fast enough. Construction and commissioning typically take 12 to 36 months, but when you include permitting and power-grid build-outs, a full data-center project can stretch to three to six years.</p>
<p class="articleParagraph enarticleParagraph" >Time and money</p>
<p class="articleParagraph enarticleParagraph" >This time gap is the entire investment thesis.</p>
<p class="articleParagraph enarticleParagraph" >When essential resources become expensive and concentrated, parallel markets emerge. We saw this with electricity co-ops in the early 20th century, independent oil producers during <span class="companylink">OPEC</span>'s reign and broadband resellers in the early internet era.</p>
<p class="articleParagraph enarticleParagraph" >With AI, the scarce resource is GPU computing. Several companies are building marketplaces that aggregate idle capacity — consumer GPUs, academic clusters, enterprise overstock — and resell it at a fraction of centralized data-center costs.</p>
<p class="articleParagraph enarticleParagraph" >The economics for these companies are compelling during this shortage window:</p>
<p class="articleParagraph enarticleParagraph" >Cost structure advantage: Alternative networks don't finance data centers with debt. They pay participants directly for computing capacity through incentive structures, converting spare capacity into productive assets. The cost of scaling shifts from massive capex to distributed incentives.</p>
<p class="articleParagraph enarticleParagraph" >Speed to market: While hyperscalers wait 18 to 36 months for new facilities, these networks can add capacity node by node, with no billion-dollar commitments up front.</p>
<p class="articleParagraph enarticleParagraph" >Arbitrage pricing: These companies are capturing demand from the smaller labs, indie studios, emerging markets and others that are priced out of <span class="companylink">AWS</span> GPU pricing but still need computing.</p>
<p class="articleParagraph enarticleParagraph" >The catch? The explosive growth window is finite. These networks will remain viable alternatives even after constraints ease — serving cost-sensitive workloads, emerging markets and indie developers — but the opportunity for substantial investment gains compresses as growth normalizes and hyperscalers' capacity comes online.</p>
<p class="articleParagraph enarticleParagraph" >Read: <span class="colorLinks">AI data centers need juice. The next hot stocks give it. [https://www-barrons-com.ezproxy.cul.columbia.edu/articles/ai-data-center-power-grid-natural-gas-09ca55ad]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >How to play the computing shortage</p>
<p class="articleParagraph enarticleParagraph" >Again, this isn't a moonshot bet. It's an infrastructure hedge with a defined window. Here are three approaches, ranked by risk profile:</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">Render Network [https://rendernetwork.com/]</span> : Aggregates idle GPU capacity from individuals and studios, reselling to the highest bidder for rendering and AI workloads. Think of it as <span class="companylink">Airbnb</span> for GPUs — idle capacity that would otherwise sit dormant gets monetized, and users get computing at a fraction of data-center pricing. Rather than operating expensive data centers, Render pays a fraction of that cost to harvest capacity from thousands of computers.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">io.net [https://io.net/]</span> : Focuses on generic GPU computing for AI training and inference. The platform aggregates capacity from data centers, crypto miners and consumer hardware, creating a distributed alternative to centralized cloud providers. Its network is newer and more speculative than Render, but it's capturing demand from AI startups that can't afford or access hyperscaler GPU allocations.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="colorLinks">Akash Network [https://akash.network/]</span> : Takes the concept broader, offering a marketplace for general cloud computing and storage beyond just GPUs. This positions it as infrastructure for the full stack, not just AI-specific workloads. Akash is a privately held company but it does have a tradeable crypto token, AKT. This is the highest-risk play in this category, but offers the most diversified exposure if decentralized computing extends beyond AI.</p>
<p class="articleParagraph enarticleParagraph" >These are crypto token plays — not stocks</p>
<p class="articleParagraph enarticleParagraph" >Before going further, understand what you're actually buying. All three of these networks operate through native cryptocurrency tokens, not traditional equity. There is no stock ticker, no brokerage-account access and no public-equity wrapper for these businesses.</p>
<p class="articleParagraph enarticleParagraph" >Direct exposure requires navigating cryptocurrency exchanges:</p>
<p class="articleParagraph enarticleParagraph" >* Render Network (RENDER) trades on <span class="companylink">Coinbase</span>, <span class="companylink">Binance</span> and Kraken.</p>
<p class="articleParagraph enarticleParagraph" >* <span class="colorLinks">io.net [https://urldefense.com/v3/__http://io.net__;!!F0Stn7g!GQQTF59Rt6H3jzRoTPOi4xOP8yEF-jvP_gC8ImMoR_HD7yv6Nyu5Cy5-xp5HBwPWTGBlWAaG3m6jNRBOIIhOvVo5$]</span> (IO) is listed on select crypto exchanges such as Binance and <span class="colorLinks">Gate.io [https://urldefense.com/v3/__http://Gate.io__;!!F0Stn7g!GQQTF59Rt6H3jzRoTPOi4xOP8yEF-jvP_gC8ImMoR_HD7yv6Nyu5Cy5-xp5HBwPWTGBlWAaG3m6jNRBOIAlJH5SA$]</span>, with liquidity varying by venue and region.</p>
<p class="articleParagraph enarticleParagraph" >* Akash Network (AKT) trades on <span class="companylink">Coinbase</span>, Kraken and similar venues.</p>
<p class="articleParagraph enarticleParagraph" >This means dealing with crypto custody — whether through exchange accounts or self-custody wallets — and accepting the regulatory uncertainty that comes with token investments. If you're not comfortable with that infrastructure, this thesis won't work for you.</p>
<p class="articleParagraph enarticleParagraph" >For investors who prefer traditional equity exposure, the closest alternatives are second-order beneficiaries of the same capacity constraint:</p>
<p class="articleParagraph enarticleParagraph" >* Data-center operators: <span class="companylink">Equinix</span> EQIX, Digital Realty Trust DLR</p>
<p class="articleParagraph enarticleParagraph" >* Power infrastructure: <span class="companylink">Dominion Energy</span>, Duke Energy DUK, NextEra Energy NEE</p>
<p class="articleParagraph enarticleParagraph" >* GPU supply chain: Nvidia NVDA, Broadcom AVGO, Super Micro Computer SMCI</p>
<p class="articleParagraph enarticleParagraph" >But here's the critical distinction: These publicly traded companies benefit from the shortage itself — not from the temporary arbitrage window created by aggregating idle distributed capacity. They'll do well regardless of whether decentralized computing succeeds. What they won't give you is direct exposure to the specific dislocation that is going on now.</p>
<p class="articleParagraph enarticleParagraph" >Risk factors</p>
<p class="articleParagraph enarticleParagraph" >Let's be clear about what could go wrong with this arbitrage strategy:</p>
<p class="articleParagraph enarticleParagraph" >Performance and reliability: Distributed GPU networks face inherent challenges with performance variance, latency and quality control. Enterprise customers paying for AI infrastructure demand reliability. If these networks can't match centralized performance, the arbitrage doesn't matter — customers won't switch.</p>
<p class="articleParagraph enarticleParagraph" >Security and compliance: Regulated industries won't run sensitive workloads on unknown hardware scattered globally. These networks are limited to specific use cases where data sovereignty and compliance aren't blockers.</p>
<p class="articleParagraph enarticleParagraph" >Hyperscaler catch-up timeline: The base case assumes these constraints ease through 2027-'29 as new data centers and power infrastructure come online. If power constraints extend beyond 2029, the high-growth window for these companies stays open.</p>
<p class="articleParagraph enarticleParagraph" >Regulatory uncertainty: Several of these networks operate in regulatory gray areas. If governments decide to regulate decentralized computing infrastructure, costs increase and flexibility decreases.</p>
<p class="articleParagraph enarticleParagraph" >Crypto market contagion: These tokens trade on crypto exchanges and correlate with broader crypto markets. A bitcoin crash or crypto regulatory crackdown could affect these assets regardless of fundamentals.</p>
<p class="articleParagraph enarticleParagraph" >The investment timeline</p>
<p class="articleParagraph enarticleParagraph" >The window runs from early 2026 through 2027–'28, which is the core 24–to-36-month period. The broader infrastructure constraint lasts longer, but the outsized arbitrage compresses as hyperscalers come online. This aligns with the infrastructure constraint timeline I've been tracking, but extends beyond the initial shortage as power-grid limitations persist.</p>
<p class="articleParagraph enarticleParagraph" >Q1 2026: Begin building positions as the 2027 power constraint window becomes consensus view. Dollar-cost average to smooth volatility.</p>
<p class="articleParagraph enarticleParagraph" >Q2 2026-Q2 2027: Peak growth opportunity as AI demand continues accelerating while centralized capacity remains severely constrained. These networks capture maximum long-tail demand priced out of hyperscaler infrastructure.</p>
<p class="articleParagraph enarticleParagraph" >Q3 2027-Q2 2028: Growth continues, but begins normalizing as new data centers come online and power-grid upgrades progress. Monitor hyperscaler capacity announcements closely — each major facility completion incrementally compresses the arbitrage.</p>
<p class="articleParagraph enarticleParagraph" >Q3 2028-Q4 2029: Maturation phase. These networks settle into specialized roles — emerging markets, cost-sensitive workloads, indie developers. They remain viable businesses but growth normalizes.</p>
<p class="articleParagraph enarticleParagraph" >It is important to understand that this isn't a binary "it works until it doesn't" thesis. It's a maturation curve where networks transition from high-growth arbitrage plays to steady-state infrastructure alternatives.</p>
<p class="articleParagraph enarticleParagraph" >The broader implication</p>
<p class="articleParagraph enarticleParagraph" >If GPU aggregation networks prove they can deliver reliable computing at competitive prices during the 2026-'28 constraint period, they will establish legitimacy. Even if hyperscalers eventually recapture market share, these networks will have carved out niches in emerging markets, indie studios and cost-sensitive workloads.</p>
<p class="articleParagraph enarticleParagraph" >The bull case: Aggregation networks will use the arbitrage window to build defensible positions, then graduate from tactical plays to structural alternatives.</p>
<p class="articleParagraph enarticleParagraph" >The bear case? Networks are temporary stop-gaps that will get crushed the moment the next wave of data centers begin operating.</p>
<p class="articleParagraph enarticleParagraph" >The honest assessment? These networks will gorge themselves during the feast years, then adapt to leaner times, remaining profitable, just not explosive.</p>
<p class="articleParagraph enarticleParagraph" >For investors, that means treating this as what it is: a defined-window arbitrage play with asymmetric upside if the shortage persists longer than expected, and manageable downside if you size positions appropriately and respect the exit timeline.</p>
<p class="articleParagraph enarticleParagraph" >The AI infrastructure buildout is real. The computing shortage is real. The 2027-'29 constraint window is real. The question is whether you're positioned to profit from the temporary dislocation before the market normalizes.</p>
<p class="articleParagraph enarticleParagraph" >Read: <span class="colorLinks">AI has real problems. The smart money is investing in the companies solving them now. [https://www-marketwatch-com.ezproxy.cul.columbia.edu/story/investors-are-buying-into-ai-that-cant-spell-smart-money-is-buying-these-stocks-b1e8fbc0]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >More: <span class="colorLinks">The AI boom is over — here's your bubble survival guide [https://www-marketwatch-com.ezproxy.cul.columbia.edu/story/everyones-asking-the-wrong-question-about-an-ai-bubble-here-are-the-stocks-to-buy-and-when-b3fddce5]</span>
                  </p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i1 : Energy | i16 : Electricity/Gas Utilities | i3302 : Computers/Consumer Electronics | i34531 : Semiconductors | i8394 : Computer Services | ibcs : Business/Consumer Services | ibnk : Banking/Credit | icomp : Computing | icph : Computer Hardware | idcent : Data Centers/Colocation Services | idserv : Data Services | ifinal : Financial Services | iindele : Industrial Electronics | iindstrls : Industrial Goods | iint : Online Service Providers | iintcir : Integrated Circuits | iinv : Investing/Securities | itech : Technology | iutil : Utilities | ividbd : Graphics Processing Units</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>c21 : Output/Production | c25 : Information Technology | ccat : Corporate/Industrial News | cexpro : Products/Services | cpshrt : Product Shortage | gaiml : Artificial Intelligence/Machine Learning | gcat : Political/General News | gcsci : Computer Science | gsci : Sciences/Humanities | m11 : Equity Markets | m15 : Derivatives Markets | mcat : Commodity/Financial Market News | nadc : Advice | ncat : Content Types | nfact : Factiva Filters | nfce : C&E Exclusion Filter | nimage : Images | niwe : IWE Filter | nrmf : Routine Market/Financial News</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>eurz : Europe | nordz : Northern Europe | uk : United Kingdom | weurz : Western Europe</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPC</b>&nbsp;</td><td><br/>c13 | c1521 | c17 | c25 | ccat | ccsr | cesg | gaiml | gcat | gcsci | gsci | I/CPR | I/ELQ | I/SEM | I/TSX | LLM | M/CYC | M/IDU | M/TEC | m11 | m15 | N/CNW | N/GEN | N/OPC | N/SCN | nadc | ncat | nedc | nfact | nfcpex | P/ESG</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>ai | arbitrage | Artificial Intelligence | artificialintelligence | bigtech | capital | compute | computing | Debt and Debt Management | Debt and Debt Mgt | electric | electricity | equities | funding | government | gpu | hyperscaler | infrastructure | internet | MarketWatch App Screens | MarketWatch Associated Press | MarketWatch Commerce | MarketWatch Headlines | MarketWatch.com | MarketWatch.com Premium | MarketWatch.com Q&A | nvidia | politics | power | powergrid | regulation | shortage | stockmarket | stocks | SYND | tech | technology | WPMKTW00045908721 | Your Digital Self</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Dow Jones & Company, Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document MRKWC00020251203elc300231</td></tr></table><br/></div></div><br/><span></span><div id="article-ACWIRE0020251206elc60012x" class="article" ><div class="article enArticle"><p><img src="https://logos-factiva-com.ezproxy.cul.columbia.edu/acwireLogo.gif" onerror="this.style.display='none';"/></p>
<table cellpadding="1" cellspacing="1" border="0"><tr><td align="right" valign="top" class="index"><b>HD</b>&nbsp;</td><td><span class='enHeadline'>New to The Street Broadcasts Tonight on Bloomberg at 6:30 PM EST Featuring Roadzen, BioVie, and TY J Young Wealth</span>
</td></tr><tr><td align="right" valign="top" class="index"><b>WC</b>&nbsp;</td><td>405 words</td></tr><tr><td align="right" valign="top" class="index"><b>PD</b>&nbsp;</td><td>6 December 2025</td></tr><tr><td align="right" valign="top" class="index"><b>SN</b>&nbsp;</td><td>ACCESSWIRE</td></tr><tr><td align="right" valign="top" class="index"><b>SC</b>&nbsp;</td><td>ACWIRE</td></tr><tr><td align="right" valign="top" class="index"><b>LA</b>&nbsp;</td><td>English</td></tr><tr><td align="right" valign="top" class="index"><b>CY</b>&nbsp;</td><td>Copyright 2025. ACCESSWIRE </td></tr>
<tr><td align="right" valign="top" class="index"><p><b>LP</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Tonight's show is sponsored by commercials from <span class="companylink">Laser Photonics</span>, <span class="companylink">DataVault</span>, <span class="companylink">Aeries Technology</span>, Sustainable Green Team, <span class="companylink">PetVivo</span>, and <span class="companylink">Synergy CHC</span>
                     </p>
<p class="articleParagraph enarticleParagraph" >NEW YORK CITY, NY / <span class="colorLinks">ACCESS Newswire [https://www.accessnewswire.com/]</span> / December 6, 2025 / New to The Street, one of the nation's most established and fastest-growing financial news and sponsored-programming platforms, announces tonight's nationwide television broadcast on <span class="companylink">Bloomberg Television</span> at 6:30 PM EST. The episode features executive interviews with <span class="companylink">Roadzen</span> (NASDAQ:RDZN), <span class="companylink">BioVie</span> (NASDAQ:BIVI), and TY J Young Wealth, offering viewers cutting-edge insights across AI mobility, biotech innovation, and strategic wealth management.</p>
</td></tr><tr><td align="right" valign="top" class="index"><p><b>TD</b>&nbsp;</p></td><td><p class="articleParagraph enarticleParagraph" >Featured Interviews on Tonight's Broadcast</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">Roadzen</span> (NASDAQ:RDZN)</p>
<p class="articleParagraph enarticleParagraph" >A deep dive into <span class="companylink">Roadzen</span>'s AI-powered auto insurance platform and the company's global expansion initiatives redefining mobility risk intelligence.</p>
<p class="articleParagraph enarticleParagraph" >
                     <span class="companylink">BioVie</span> (NASDAQ:BIVI)</p>
<p class="articleParagraph enarticleParagraph" >An update on the company's advancing clinical programs targeting neurological and liver-related diseases, with insight into upcoming milestones.</p>
<p class="articleParagraph enarticleParagraph" >TY J Young Wealth</p>
<p class="articleParagraph enarticleParagraph" >A segment focused on wealth-building frameworks, financial strategy, and guidance for investors preparing for 2026 market conditions.</p>
<p class="articleParagraph enarticleParagraph" >Show Sponsors</p>
<p class="articleParagraph enarticleParagraph" >Tonight's broadcast is made possible through commercial sponsorships from leading innovators across multiple industries:</p>
<p class="articleParagraph enarticleParagraph" >* <span class="companylink">Laser Photonics</span> (NASDAQ:LASE)</p>
<p class="articleParagraph enarticleParagraph" >* DataVault Holdings (NASDAQ:DVLT)</p>
<p class="articleParagraph enarticleParagraph" >* <span class="companylink">Aeries Technology</span> (NASDAQ:AERT)</p>
<p class="articleParagraph enarticleParagraph" >* The Sustainable Green Team (OTCQX:SGTM)</p>
<p class="articleParagraph enarticleParagraph" >* <span class="companylink">PetVivo Holdings</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >* TY J Young Wealth</p>
<p class="articleParagraph enarticleParagraph" >* Synergy CHC (NASDAQ:SNYR)</p>
<p class="articleParagraph enarticleParagraph" >These sponsors support New to The Street's mission of providing unmatched national visibility for public companies through television, digital distribution, and outdoor media.</p>
<p class="articleParagraph enarticleParagraph" >About New to The Street</p>
<p class="articleParagraph enarticleParagraph" >New to The Street, produced by FMW Media, is one of America's longest-running and most influential financial television brands, approaching its 17th anniversary. The platform broadcasts sponsored programming on <span class="companylink">Bloomberg Television</span> and Fox Business, with additional distribution across digital networks and outdoor media in Times Square and the New York City Financial District.</p>
<p class="articleParagraph enarticleParagraph" >With 4 million subscribers across the New to The Street TV <span class="companylink">YouTube</span> channel and more than 800,000 followers across X, <span class="companylink">Facebook</span>, <span class="companylink">LinkedIn</span>, and <span class="companylink">Instagram</span>, the brand delivers one of the largest combined digital and social financial audiences in the United States. This ecosystem-supported by high-impact television, online video, earned media, and iconic billboard placements-provides companies with unmatched reach and credibility across the investor community.</p>
<p class="articleParagraph enarticleParagraph" >Media Contact; <span class="colorLinks">Monica@NewtoTheStreet.com [mailto:Monica@NewtoTheStreet.com]</span>
                  </p>
<p class="articleParagraph enarticleParagraph" >SOURCE: New to The Street</p>
<p class="articleParagraph enarticleParagraph" >View the original <span class="colorLinks">press release [https://www.accessnewswire.com/newsroom/en/publishing-and-media/new-to-the-street-broadcasts-tonight-on-bloomberg-at-6-30-pm-est-featuring-roa-1115262]</span> on <span class="companylink">ACCESS Newswire</span>
                  </p>
</td></tr><tr><td align="right" valign="top" class="index"><br/><b>CO</b>&nbsp;</td><td><br/>blfima : Bloomberg LP | dipfzy : Aeries Technology Inc. | eooicn : PetVivo Holdings Inc | kslfig : Roadzen Inc.</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IN</b>&nbsp;</td><td><br/>i3302022 : Artificial Intelligence Technologies | i372 : Medical Equipment/Supplies | i82 : Insurance | i8394 : Computer Services | i8395463 : Digital Content Services | i8395465 : Multimedia Content Services | i951 : Healthcare/Life Sciences | iacc : Accounting/Consulting | ibcs : Business/Consumer Services | icnsl : Business Consultancy | idiagn : Medical Diagnostic Equipment/Supplies | idistr : Media Content Distribution | ifinal : Financial Services | ifmsoft : Financial Technology | iinsurt : Insurance Technology | iint : Online Service Providers | iitcns : IT Consulting | imed : Media/Entertainment | iphmed : Medical Devices/Apparatus | itech : Technology | itheradv : Diagnostic/Therapeutic Devices</td></tr><tr><td align="right" valign="top" class="index"><br/><b>NS</b>&nbsp;</td><td><br/>ccat : Corporate/Industrial News | ncat : Content Types | npress : Press Releases</td></tr><tr><td align="right" valign="top" class="index"><br/><b>RE</b>&nbsp;</td><td><br/>namz : North America | nyc : New York City | usa : United States | use : Northeast U.S. | usny : New York State</td></tr><tr><td align="right" valign="top" class="index"><br/><b>IPD</b>&nbsp;</td><td><br/>NASDAQ:AERT | NASDAQ:BIVI | NASDAQ:DVLT | NASDAQ:LASE | NASDAQ:RDZN | NASDAQ:SNYR | New To The Street | OTCQX:SGTM</td></tr><tr><td align="right" valign="top" class="index"><br/><b>PUB</b>&nbsp;</td><td><br/>Accesswire</td></tr><tr><td align="right" valign="top" class="index"><br/><b>AN</b>&nbsp;</td><td><br/>Document ACWIRE0020251206elc60012x</td></tr></table><br/></div></div><br/><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WC58001020251206elc6004bl"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  How Good Has PG Stock Actually Been?</b><div class="leadFields"><a href="javascript:void(0)">Daily Finance</a>, 12:00 AM, 6 December 2025, 724 words,  Todd Shriber, (English)</div><div class="snippet ensnippet"> The safety offered by this consumer staples giant has come at a cost.P&amp;G hasn’t come close to keeping pace with the broader market or its sector over the past several years.</div>
<div>(Document WC58001020251206elc6004bl)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="BC00268020251206elc600003"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  Flat Like A Lake</b><div class="leadFields"><a href="javascript:void(0)">24/7 Wall St.</a>, 12:03 PM, 6 December 2025, 890 words,  Ben Briody, (English)</div><div class="snippet ensnippet"> Since Wednesday, Bitcoin has been on a slight downward trajectory, hitting a local bottom of around $88k on Friday. Since then, BTC has been slowly working its way back towards the $90k level. The war in the order books continues in this ...</div>
<div>(Document BC00268020251206elc600003)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="BC00268020251206elc600004"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  Is VOO + QQQ the Ultimate Retirement Formula?</b><div class="leadFields"><a href="javascript:void(0)">24/7 Wall St.</a>, 11:12 AM, 6 December 2025, 1354 words, (English)</div><div class="snippet ensnippet"> The Vanguard S&amp;P 500 ETF (VOO) tracks the 500 largest publicly traded companies and offers an extremely low expense ratio.The Invesco QQQ Trust (QQQ) tracks the Nasdaq-100 and focuses heavily on growth sectors like tech.</div>
<div>(Document BC00268020251206elc600004)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WC50127020251206elc6003uy"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  'Make memories’: the tragic reality of childhood DIPG and the new research giving families hope23h ago 03:53</b><div class="leadFields"><a href="javascript:void(0)">Special Broadcasting Service</a>, 02:01 PM, 6 December 2025, 544 words, (English)</div><div class="snippet ensnippet"> Pippa Rae was nine years old when she was diagnosed with diffuse intrinsic pontine glioma, or DIPG.It&#39;s a rare and aggressive type of deadly brain cancer that forms in the brain stem - it mainly affects children.</div>
<div>(Document WC50127020251206elc6003uy)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="BC00268020251206elc60000a"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  Are You Going To Receive Trump’s $2000 Stimulus Check By Christmas?</b><div class="leadFields"><a href="javascript:void(0)">24/7 Wall St.</a>, 08:14 AM, 6 December 2025, 1351 words, (English)</div><div class="snippet ensnippet"> Distribution may align with November 2026 midterm elections.Investors are using a behind the scenes move that sidesteps the wild swings of stocks and ETFs to lock in guaranteed income while they still can. They’re turning gains into ...</div>
<div>(Document BC00268020251206elc60000a)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="BC00268020251206elc600001"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  Why Retiring Early Is Hard Even When You Can Afford It</b><div class="leadFields"><a href="javascript:void(0)">24/7 Wall St.</a>, 12:00 PM, 6 December 2025, 1631 words, (English)</div><div class="snippet ensnippet"> Many high earners delay retirement despite having sufficient wealth due to identity attachment and loss aversion.The poster saved twice his retirement target by 55 but delayed to 57 chasing $6M more.</div>
<div>(Document BC00268020251206elc600001)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="BC00268020251206elc600008"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  CoreWeave Hits Profitability While Applied Digital Burns Cash Building Data Centers</b><div class="leadFields"><a href="javascript:void(0)">24/7 Wall St.</a>, 08:45 AM, 6 December 2025, 1366 words, (English)</div><div class="snippet ensnippet"> Applied Digital (APLD) posted $64.2M in Q1 revenue with 84% growth while CoreWeave (CRWV) reported $1.36B in Q3 revenue with 134% growth.CoreWeave generated $51.9M in operating income and doubled its backlog to $55B. Applied Digital lost ...</div>
<div>(Document BC00268020251206elc600008)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="BC00268020251206elc600009"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  Teva Crushes Earnings as Pfizer Struggles to Replace COVID Revenue</b><div class="leadFields"><a href="javascript:void(0)">24/7 Wall St.</a>, 08:52 AM, 6 December 2025, 1408 words, (English)</div><div class="snippet ensnippet"> Pfizer (PFE) posted Q3 revenue of $16.65B down 5.9% as COVID products declined sharply. Paxlovid dropped 55% and Comirnaty fell 20%.Teva (TEVA) delivered its 11th consecutive quarter of growth with revenue up 3.4% to $4.48B. AUSTEDO surged ...</div>
<div>(Document BC00268020251206elc600009)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="BC00268020251206elc600005"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  Netflix Doubled Your Money in 12 Months After Years of Lagging the Market</b><div class="leadFields"><a href="javascript:void(0)">24/7 Wall St.</a>, 10:11 AM, 6 December 2025, 1292 words, (English)</div><div class="snippet ensnippet"> Netflix (NFLX) generated $11.51B in Q3 2025 revenue with a 28% operating margin. A $619M Brazilian tax dispute pressured results.Netflix stock returned 92% over the past year but underperformed the S&amp;P 500 over the past decade.</div>
<div>(Document BC00268020251206elc600005)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="BC00268020251206elc600006"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  Up 96% in 2025, This Stock Will Be Added to the S&P 500 on Dec. 22</b><div class="leadFields"><a href="javascript:void(0)">24/7 Wall St.</a>, 09:12 AM, 6 December 2025, 1423 words,  Rich Duprey, (English)</div><div class="snippet ensnippet"> The S&amp;P 500 large-cap stock index undergoes quarterly rebalances to reflect evolving market conditions. Managed by S&amp;P Dow Jones Indices, the process involves evaluating companies based on criteria like market capitalization, liquidity, ...</div>
<div>(Document BC00268020251206elc600006)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="BC00268020251206elc600002"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  The Tools Every Marine Must Master Before Deployment</b><div class="leadFields"><a href="javascript:void(0)">24/7 Wall St.</a>, 12:00 PM, 6 December 2025, 3482 words, (English)</div><div class="snippet ensnippet"> The IFAK and Combat Application Tourniquet are essential trauma tools that can determine survival before medics arrive.Map and compass navigation remains critical when GPS fails or signals are jammed in combat zones.</div>
<div>(Document BC00268020251206elc600002)</div><br/></td></tr>
						</table>
					</div>
				</div><div id="carryOver">
				<div id="carryOverHeadlines">
				<table cellpadding="0" cellspacing="0" border="0" class="headlines"><tr class="headline" data-accno="WCBSTNG020251206elc6001gu"><td valign="top"><img title="HTML" src="../img/html.gif"/><b class="printheadline enHeadline">  Review & setlist: Even in a show without Keith Lockhart, Holiday Pops brings seasonal spirit</b><div class="leadFields"><a href="javascript:void(0)">Boston.com</a>, 01:37 PM, 6 December 2025, 1009 words,  Marc Hirsh, (English)</div><div class="snippet ensnippet"> How busy are the Boston Pops this time of year? Consider that on Friday night, at the very same time that it was playing in Symphony Hall, it was also in Worcester. That’s the kind of holiday magic that you can pull off when you’re ...</div>
<div>(Document WCBSTNG020251206elc6001gu)</div><br/></td></tr>
						</table>
					</div>
				</div></div></div><span><div id="pageFooter"><table width="100%" cellspacing="0" cellpadding="0" border="0" class="footerBG">
	<tr>
		<td nowrap="nowrap" width="100%" align="right"><span class="copyright">&copy; 2025 Factiva, Inc.  All rights reserved.</span></td>
		<td><div class="ftright">&nbsp;</div></td>
	</tr>
</table>
<span class='shadowL'></span><span class='shadowR'></span></div><noscript><img src="http://om.dowjoneson.com/b/ss/djfactivatesting/1/H.22.1--NS/0" height="1" width="1" border="0" alt="" /></noscript></span><script type="text/javascript">
//<![CDATA[
jQuery(document).ready (function(){
try { InitializeOmniture('djfactiva');}catch (ex) {}
try {DJOmniture.Property.SessionId = "mpgviMmM_G44WCMRTMU4WMNDBGA3TINJYGI4WMNJTMIZGMY3GMVQTANBTMFQQ";DJOmniture.Property.UserId_Ns = "E6OO2HVCQHVPFGQJS4YZ7OYDUQ";DJOmniture.Property.AccountId = "";DJOmniture.Property.FullURL = "https://global-factiva-com.ezproxy.cul.columbia.edu/hp/printsavews.aspx?ppstype=Article&pp=Print&hc=All";DJOmniture.Property.AccessCode = "0086";DJOmniture.Property.PageName = "PageNameNotSet";DJOmniture.Property.SearchType = "";DJOmniture.Property.FilterType = "";DJOmniture.Property.FilterValue = "";DJOmniture.Property.Type = "";DJOmniture.Property.ProfileName = "";DJOmniture.Property.ReportType = "";DJOmniture.Property.DataRange = "";DJOmniture.Property.FormatType = "";DJOmniture.Property.AccessionNumber = "";DJOmniture.Property.ContentType = "";DJOmniture.Property.ArticleType = "";DJOmniture.Property.Headline = "";DJOmniture.Property.Author = "";DJOmniture.Property.WordCount = "";DJOmniture.Property.PublicationDate = "";DJOmniture.Property.Source = "";DJOmniture.Property.BaseLanguage = "";DJOmniture.Property.ScreeningType = "";DJOmniture.Property.ScreeningValue = "";DJOmniture.Property.FactivaPageId = "";DJOmniture.Property.Channel = "";DJOmniture.Property.Area = "";DJOmniture.Property.Section = "";DJOmniture.Property.SearchQueryLength = "";}catch (ex) {}
});
//]]>
</script><table id="tmpJQueryTarget" border="0">
	<tr>

	</tr>
</table>

<script type="text/javascript">
//<![CDATA[
(function() {var fn = function() {$get('PageScriptManager_HiddenField').value = '';Sys.Application.remove_init(fn);};Sys.Application.add_init(fn);})();//]]>
</script>

<script src="https://global-factiva-com.ezproxy.cul.columbia.edu/CombineScriptsHandler.ashx?_TSM_HiddenField_=PageScriptManager_HiddenField&amp;_TSM_CombinedScripts_=%3b%3bAjaxControlToolkit%2c+Version%3d3.0.30930.28736%2c+Culture%3dneutral%2c+PublicKeyToken%3d28f01b0e84b6d53e%3aen-US%3ab0eefc76-0092-471b-ab62-f3ddc8240d71%3a865923e8%3bfactiva.com.ui%3aen-US%3ae646eb98-cdf8-4fce-ba1a-cebd47fe8530%3a84690f9d%3bEMG.Toolkit.Web%2c+Version%3d2.0.0.22%2c+Culture%3dneutral%2c+PublicKeyToken%3dnull%3aen-US%3ade23b191-cd1d-4c31-b7c2-c71e4d22ee9a%3a78e334ee%3a28617db2%3a235de37a%3a35bc484f%3a360cd5dc%3bAjaxControlToolkit%2c+Version%3d3.0.30930.28736%2c+Culture%3dneutral%2c+PublicKeyToken%3d28f01b0e84b6d53e%3aen-US%3ab0eefc76-0092-471b-ab62-f3ddc8240d71%3a91bd373d%3bEMG.Toolkit.Web%2c+Version%3d2.0.0.22%2c+Culture%3dneutral%2c+PublicKeyToken%3dnull%3aen-US%3ade23b191-cd1d-4c31-b7c2-c71e4d22ee9a%3a312433fe%3aa74d42b0" type="text/javascript"></script>
<script type="text/javascript">
//<![CDATA[
Sys.Application.add_init(function() {
    $create(EMG.Toolkit.Web.TableSorterBehavior, {"debug":true,"headers":"{0: { sorter: false}, 3: {sorter: false}}","id":"_jqueryPlgn","sortList":"[0,1], [1,1]","widgetZebra":"[\u0027zebra\u0027]"}, null, null, $get("tmpJQueryTarget"));
});
//]]>
</script>
</form><script type='text/javascript'>if(document.getElementById('carryOverHeadlines')){document.getElementById('carryOverHeadlines').style.display = 'block';}$(document).ready(function(){setTimeout(function(){window.print();}, 1000);});</script><script type='text/javascript'>framesViewNotReqd = false;modalEnabled = true;RequestFromModal=false;RequestFromIPad=false;SnapshotBaseUrl='https://snapshot-factiva-com.ezproxy.cul.columbia.edu';</script>
</body>
</html>"""

In [4]:
import glob # used to find all the file paths that match a specified pattern
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os

# ✅ Give your folder a clear name — e.g., use your initials or project topic
FOLDER_NAME = "factiva_ner_project"

# Build the full path
DRIVE_ROOT = "/content/drive/MyDrive"
path = os.path.join(DRIVE_ROOT, FOLDER_NAME)

# Create the folder if it doesn’t exist
os.makedirs(path, exist_ok=True)

print(f"✅ Folder ready at: {path}")


✅ Folder ready at: /content/drive/MyDrive/factiva_ner_project


In [6]:

# Write the HTML code to the file
with open(f"{path}/factiva.htm", 'w') as file: #replace with your path
    file.write(html_code)


# # For a list of variables with HTML content
# html_list = [html_code1, html_code2, html_code3]  # Replace with your actual variables

# # Iterate over the list and write each HTML content to a separate file
# for i, html_code in enumerate(html_list):
#     file_path = f"/content/drive/MyDrive/Factiva/factiva_{i}.htm"  # Replace with your path
#     with open(file_path, 'w') as file:
#         file.write(html_code)



In [7]:
files = glob.glob(f"{path}/*.htm", recursive = True) #replace with your path
files

['/content/drive/MyDrive/factiva_ner_project/factiva.htm']

In [8]:
empty_list = []
for file in files:
    data = pd.read_html(file, index_col = 0) #reads the HTML content of the file and tries to find any tables inside it.
    empty_list.extend(data) # The extend() method is used to add the data (which is a list of dataframes) from the current file to empty_list.

In [9]:
empty_list

[                           1
 0                           
 Dow Jones Factiva  Dow Jones,
                                                      1
 0                                                     
 HD   Increasing Literacy on the Scams Targeting Lat...
 BY                             Aguilar Gabriel Lorenzo
 WC                                           114 words
 PD                                      1 January 2026
 SN       Journal of Business & Technical Communication
 SC                                                FJBT
 PG                                                1-23
 VOL                 Volume 40; Issue 1; ISSN:1050-6519
 LA                                             English
 CY   © 2026 Journal of Business & Technical Communi...
 LP   This article builds a heuristic that raises th...
 TD                                                 NaN
 IN   i3302022 : Artificial Intelligence Technologie...
 NS   ccat : Corporate/Industrial News | gaiml : Art...
 IPD  Article

In [10]:
frames = pd.concat([l for l in empty_list if 'HD' in l.index.values], axis=1).T

In [11]:
frames

,HD,BY,WC,PD,SN,SC,PG,VOL,LA,CY,...,AN,RE,CO,CR,ED,IPC,CLM,SE,RF,ET
1,Increasing Literacy on the Scams Targeting Lat...,Aguilar Gabriel Lorenzo,114 words,1 January 2026,Journal of Business & Technical Communication,FJBT,1-23,Volume 40; Issue 1; ISSN:1050-6519,English,© 2026 Journal of Business & Technical Communi...,...,Document FJBT000020251202em1100001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fragmented Landscapes as Refuge for Forest Bir...,"Muhammad Azeem Akhter, Mark E Hostetler, Ihsan...",5728 words,31 December 2025,Pakistan Journal of Zoology,ASZOOG,2959,57,English,Copyright © 2025. Zoological Society of Pakistan,...,Document ASZOOG0020251203elcv00017,asiaz : Asia | devgcoz : Emerging Market Count...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Effect of Carbon Source Addition Strategies on...,"Malreddy Joshna, Baboonsundaram Ahilan, Cheryl...",4378 words,31 December 2025,Pakistan Journal of Zoology,ASZOOG,2877,57,English,Copyright © 2025. Zoological Society of Pakistan,...,Document ASZOOG0020251203elcv0000z,asiaz : Asia | dvpcoz : Developing Economies |...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,In-Vitro and In-Vivo Antibacterial Effects of ...,"Kuanhui Liu, Shihui Xie, Ting Li, Reem M. Aljo...",4095 words,31 December 2025,Pakistan Journal of Zoology,ASZOOG,2535,57,English,Copyright © 2025. Zoological Society of Pakistan,...,Document ASZOOG0020251203elcv00004,apacz : Asia Pacific | asiaz : Asia | china : ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ZELENSKY’S NEW SOLUTION,NaN,2216 words,18 December 2025,AirForces Monthly,FORAI,NaN,NaN,English,© 2025. Key Publishing Ltd. All rights reserved,...,Document FORAI00020251206elci00018,asiaz : Asia | dvpcoz : Developing Economies |...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,IIMC emerges strong despite a challenging econ...,George Skaria,660 words,7 December 2025,Business Today,BTDY,NaN,NaN,English,Copyright 2025. Living Media India Ltd,...,Document BTDY000020251202elc700002,asiaz : Asia | devgcoz : Emerging Market Count...,NaN,NaN,NaN,NaN,NaN,Deep Dive,NaN,NaN
1,BT-MDRA India's Best B-Schools Ranking: The Be...,George Skaria,1550 words,7 December 2025,Business Today,BTDY,NaN,NaN,English,Copyright 2025. Living Media India Ltd,...,Document BTDY000020251202elc700004,asiaz : Asia | devgcoz : Emerging Market Count...,NaN,NaN,NaN,NaN,NaN,Cover Story,NaN,NaN
1,Govt announces Shs100b grants to boost climate...,Khalil Ibrahim Manzil,393 words,6 December 2025,Daily Monitor,DAYMO,NaN,NaN,English,Copyright 2025 Nation Media Group. All Rights ...,...,Document DAYMO00020251206elc60000q,africaz : Africa | dvpcoz : Developing Economi...,"ugmaaf : Uganda Ministry of Agriculture, Anima...",NaN,NaN,NaN,NaN,National,NaN,NaN
1,AI needs power desperately. Here's how to inve...,By Jurica Dujmovic,1627 words,6 December 2025,MarketWatch,MRKWC,NaN,NaN,English,"Copyright 2025 MarketWatch, Inc. All Rights Re...",...,Document MRKWC00020251203elc300231,eurz : Europe | nordz : Northern Europe | uk :...,NaN,NaN,NaN,c13 | c1521 | c17 | c25 | ccat | ccsr | cesg |...,NaN,Investing,NaN,02:12 PM


In [12]:
frames.columns

Index(['HD', 'BY', 'WC', 'PD', 'SN', 'SC', 'PG', 'VOL', 'LA', 'CY', 'LP', 'TD',
       'IN', 'NS', 'IPD', 'PUB', 'AN', 'RE', 'CO', 'CR', 'ED', 'IPC', 'CLM',
       'SE', 'RF', 'ET'],
      dtype='object', name=0)

In [13]:
frames.rename(columns = {'HD': 'Headline',
                         'PD': 'Publication_Date','SN': 'Source_Name', 'LP': 'Lead Paragraph',
                          'TD': 'Body',
                         'BY':'Author_Name'}, inplace=True)

frames = frames[['Headline', 'Publication_Date', 'Source_Name', 'Lead Paragraph', 'Body', 'Author_Name']]


In [14]:
frames['Publication_Date'] = pd.to_datetime(frames['Publication_Date'])
frames.sort_values(by='Publication_Date', inplace=True)



/tmp/ipython-input-3260031202.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipython-input-3260031202.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [15]:
frames['CombinedText'] = frames['Lead Paragraph'] + " " + frames['Body']

/tmp/ipython-input-2185294090.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [16]:
frames

,Headline,Publication_Date,Source_Name,Lead Paragraph,Body,Author_Name,CombinedText
1,New to The Street Broadcasts Tonight on Bloomb...,2025-12-06,ACCESSWIRE,Tonight's show is sponsored by commercials fro...,Featured Interviews on Tonight's Broadcast Ro...,NaN,Tonight's show is sponsored by commercials fro...
1,AI needs power desperately. Here's how to inve...,2025-12-06,MarketWatch,AI computing workloads could consume around 50...,Rising infrastructure costs and mounting capit...,By Jurica Dujmovic,AI computing workloads could consume around 50...
1,Govt announces Shs100b grants to boost climate...,2025-12-06,Daily Monitor,The Uganda government has launched 24 competit...,They are implemented by the National Agricultu...,Khalil Ibrahim Manzil,The Uganda government has launched 24 competit...
1,BT-MDRA India's Best B-Schools Ranking: The Be...,2025-12-07,Business Today,B-Schools rise up to meet the challenge of a h...,The ostensible lack of action at the top hides...,George Skaria,B-Schools rise up to meet the challenge of a h...
1,Lai reiterates commitment to defense SHARED IS...,2025-12-07,Taipei Times,Taiwan reiterated its commitment to bolstering...,The 33-page report comes as Beijing increases ...,By Su Yong-yao and Lee I-chia,Taiwan reiterated its commitment to bolstering...
...,...,...,...,...,...,...,...
1,ZELENSKY’S NEW SOLUTION,2025-12-18,AirForces Monthly,Mina Adel looks at the rise of agile warfare i...,"“For me personally, the Gripen is the only fig...",NaN,Mina Adel looks at the rise of agile warfare i...
1,In-Vitro and In-Vivo Antibacterial Effects of ...,2025-12-31,Pakistan Journal of Zoology,"Key words Anti-microbial, Diarrhea, Escherichi...",INTRODUCTION Yaks are economically important a...,"Kuanhui Liu, Shihui Xie, Ting Li, Reem M. Aljo...","Key words Anti-microbial, Diarrhea, Escherichi..."
1,Effect of Carbon Source Addition Strategies on...,2025-12-31,Pakistan Journal of Zoology,"Key words Biofloc, GIF tilapia, Penaeus vannam...",INTRODUCTION Nowadays aquaculture industry is ...,"Malreddy Joshna, Baboonsundaram Ahilan, Cheryl...","Key words Biofloc, GIF tilapia, Penaeus vannam..."
1,Fragmented Landscapes as Refuge for Forest Bir...,2025-12-31,Pakistan Journal of Zoology,"Key words Biodiversity, Forest fragments, Bird...",INTRODUCTION The average human population in d...,"Muhammad Azeem Akhter, Mark E Hostetler, Ihsan...","Key words Biodiversity, Forest fragments, Bird..."


In [17]:
df = frames.reset_index()

In [18]:
save_path = os.path.join(path, "factiva.csv")
df.to_csv(save_path, index=False)

In [19]:
import os

for index, row in df.iterrows():
    file_name = f"{path}/factiva/text_file_{index + 1}.txt"  # Create a unique filename for each row
    os.makedirs(os.path.dirname(file_name), exist_ok=True) # Ensure the directory exists
    with open(file_name, 'w') as file:
        text_content = str(row['CombinedText']) if pd.notnull(row['CombinedText']) else ''  # Convert to string and handle NaN
        file.write(text_content)  # Write the text content to the file